# Office-0 full SLAM and Single-/Multi-view V3.3.1

Runs native full SLAM from RGB-D, evaluates coarse Single-/Multi-view registration on the same loop candidates, and exports paper metrics/assets.

## 1. Cấu hình Office-0

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib, importlib.util, time

REPO = Path('/kaggle/working/3DGS-SLAM-TURTLEBOT4')
VENV = Path('/kaggle/working/gs_slam_py311')
PY311 = VENV / 'bin' / 'python'
CACHE_ROOT = Path('/kaggle/working/huggingface_cache')
TOOLS_DIR = Path('/kaggle/working/office0_single_multi_setup')
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT)
os.environ['WANDB_MODE'] = 'disabled'

REUSE_LOCAL_REPO = True  # giữ checkout đang chạy của bạn nếu đã có
REUSE_USER_PACKAGES = True  # kiểm tra rồi giữ package của bạn nếu đủ
FORCE_REBUILD_EXTENSIONS = False
CUDA_TOOLKIT_HOME = ''  # tự tìm CUDA 12.x; ưu tiên 12.4 cho torch cu124

# CHẠY TỪ ĐẦU: không điền BENCH_RUNS.
FULL_SLAM_SCENES = ['office_0']
FULL_SLAM_SEED = 0
RUN_FULL_SLAM = True
FULL_SLAM_LOCAL_SCENE_ROOTS = {}  # tùy chọn: scene -> RGB-D đã attach; bỏ trống để tải
FULL_SLAM_DOWNLOAD_ROOT = REPO / 'data' / 'ReplicaMultiagent'
FULL_SLAM_STAGING_ROOT = Path('/kaggle/working/office0_rgbd_input')
FULL_SLAM_OUTPUT_BASE = Path('/kaggle/working/gslam_outputs/office0_full_slam')
FULL_SLAM_MAX_HOURS = 11.0
FULL_SLAM_MIN_FREE_DISK_GIB = 5.0
FULL_SLAM_GPU_SAMPLE_SECONDS = 10
FULL_SLAM_EXPORT_PAPER_ASSETS = True
FULL_SLAM_STRICT_EXPORTS = False  # ghi lỗi export riêng, vẫn chạy được benchmark registration
BENCH_RUNS = {}  # phần 10 tự điền sau khi full SLAM hoàn tất
BENCH_SCENES = list(FULL_SLAM_SCENES)
BENCH_EXPECTED_TOTAL_LOOPS = 8
BENCH_SEARCH_ROOTS = [FULL_SLAM_OUTPUT_BASE]
BENCH_DATA_ROOT_OVERRIDES = {}  # scene -> thư mục RGB-D mới nếu config cũ trỏ sai
BENCH_PYTHON = str(PY311)
BENCH_SEEDS = [0, 1, 2]
BENCH_ICP_SEEDS = []  # coarse-only; tránh lặp dense ICP cho mọi ablation
BENCH_RUN_ABLATIONS = True  # giữ sweep K/margin/geometry
BENCH_RUN_FPFH = True
BENCH_EXPORT_PAIR_FIGURES = False
BENCH_IMPORT_RESULTS = []  # thư mục kết quả benchmark từ phiên trước
BENCH_OUTPUT = Path('/kaggle/working/office0_single_multi_v331_benchmark')
print('Benchmark Python:', BENCH_PYTHON)
print('Full SLAM scenes:', FULL_SLAM_SCENES, '| full-run seed:', FULL_SLAM_SEED)
print('Sau full SLAM: single/multi và ICP trên cùng submap/candidate.')


## 2. Chuẩn bị repository

In [ ]:
# Clone repository; mặc định giữ checkout có sẵn.
USER_CLONE_SOURCE = 'set -euo pipefail\n\napt-get update -qq\napt-get install -y -qq git git-lfs >/dev/null\napt-get install -y -qq gcc-12 g++-12 >/dev/null 2>&1 || true\n\nREPO=/kaggle/working/3DGS-SLAM-TURTLEBOT4\nif [ -d "$REPO/.git" ]; then\n  git -C "$REPO" fetch origin main\n  git -C "$REPO" checkout main >/dev/null 2>&1\n  git -C "$REPO" reset --hard origin/main >/dev/null\nelse\n  git clone -q https://github.com/DucVu2610/3DGS-SLAM-TURTLEBOT4.git "$REPO"\nfi\n\ngit -C "$REPO" lfs pull >/dev/null 2>&1 || true\nprintf \'Repo HEAD: \'\ngit -C "$REPO" log -1 --format=\'%h | %s\'\n'
if REUSE_LOCAL_REPO and (REPO/'.git').is_dir():
    print('Reuse repo HEAD:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
else:
    subprocess.run(['bash','-c',USER_CLONE_SOURCE],check=True)


## 3. Python 3.11

In [ ]:
# Chỉ tạo môi trường Python 3.11 khi chưa có venv.
USER_BOOTSTRAP_SOURCE = "import shutil, subprocess, sys\n\nif not shutil.which('uv'):\n    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)\n\nuv = shutil.which('uv')\nif not uv:\n    raise RuntimeError('Không tìm thấy uv sau khi cài.')\n\nsubprocess.run([uv, 'python', 'install', '3.11'], check=True)\nif not PY311.exists():\n    subprocess.run([uv, 'venv', '--python', '3.11', '--seed', str(VENV)], check=True)\n\nprint(subprocess.check_output([str(PY311), '--version'], text=True).strip())\n"
if PY311.is_file():
    subprocess.run([str(PY311),'-c',"import sys; assert sys.version_info[:2]==(3,11),sys.version; print(sys.executable,sys.version)"],check=True)
else:
    exec(compile(USER_BOOTSTRAP_SOURCE,'user_bootstrap','exec'),globals())


## 4. Cài đặt dependencies

In [ ]:
# Giữ nguyên Torch/cu124, dependency và FAISS GPU của phiên bản ổn định.
USER_DEPENDENCIES_SOURCE = "import subprocess\n\nbase = [str(PY311), '-m', 'pip']\nsubprocess.run(base + ['install', '--no-cache-dir', '-U', 'pip', 'setuptools', 'wheel', 'ninja', 'packaging'], check=True)\n\n# ABI/CUDA gần môi trường gốc của repo.\nsubprocess.run(base + [\n    'install', '--no-cache-dir',\n    'torch==2.5.1', 'torchvision==0.20.1',\n    '--index-url', 'https://download.pytorch.org/whl/cu124'\n], check=True)\n\ndeps = [\n    'numpy==1.26.4', 'open3d==0.19.0', 'opencv-python-headless==4.10.0.84',\n    'plyfile==1.1', 'roma==1.5.2.1', 'graphslam==0.0.17', 'gtsam', 'addict',\n    'pyquaternion', 'pytorch-msssim==1.0.0', 'torchmetrics==1.6.3',\n    'transformers==4.49.0', 'huggingface-hub==0.29.3', 'wandb', 'trimesh',\n    'tqdm', 'imageio', 'joblib', 'scipy', 'matplotlib', 'pyyaml', 'plotly',\n    'pandas', 'scikit-learn'\n]\nsubprocess.run(base + ['install', '--no-cache-dir', '--no-warn-conflicts'] + deps, check=True)\nsubprocess.run(base + ['install', '--no-cache-dir', '--no-warn-conflicts', 'faiss-gpu-cu12==1.8.0.2'], check=True)\n\nprint('Python 3.11 dependencies: OK')\n"
probe = """import sys,torch,torchvision,numpy
assert sys.version_info[:2]==(3,11),sys.version
assert torch.__version__.split('+')[0]=='2.5.1',torch.__version__
assert torchvision.__version__.split('+')[0]=='0.20.1',torchvision.__version__
assert torch.version.cuda=='12.4',torch.version.cuda
assert numpy.__version__=='1.26.4',numpy.__version__
import open3d,cv2,plyfile,roma,graphslam,gtsam,addict,pyquaternion,pytorch_msssim,torchmetrics,transformers,huggingface_hub,wandb,trimesh,tqdm,imageio,joblib,scipy,matplotlib,yaml,plotly,pandas,sklearn,faiss,ninja,packaging
print('User dependencies and FAISS imports OK')
"""
ready=subprocess.run([str(PY311),'-c',probe],capture_output=True,text=True) if REUSE_USER_PACKAGES else None
if ready is not None and ready.returncode==0:
    print(ready.stdout)
else:
    if ready is not None: print('Need user dependency setup:',ready.stderr[-2000:])
    exec(compile(USER_DEPENDENCIES_SOURCE,'user_dependencies','exec'),globals())
    subprocess.run([str(PY311),'-c',probe],check=True)


## 5. Build CUDA extensions

In [ ]:
# Build theo phiên bản ổn định hiện tại và luôn dùng PY311 đã chọn.
USER_BUILD_MODULE_SOURCE = '"""Build using the user Office-0 extension cell; no SLAM or benchmark execution."""\nimport argparse\n\nimport hashlib\n\nimport json\n\nimport os\n\nimport re\n\nimport shutil\n\nimport subprocess\n\nimport sys\n\nfrom pathlib import Path\n\nEXTENSIONS = [(\'gaussian_rasterizer\', \'https://github.com/VladimirYugay/gaussian_rasterizer.git\', \'9c40173fcc8d9b16778a1a8040295bc2f9ebf129\'), (\'simple-knn\', \'https://github.com/VladimirYugay/simple-knn.git\', \'c7e51a06a4cd84c25e769fee29ab391fe5d5ff8d\')]\n\nBUILD_SCRIPT = \'set -euo pipefail\\n\\nPY="$1"\\nif [ ! -x "$PY" ]; then\\n  echo "Python 3.11 venv chưa tồn tại: "$PY""\\n  exit 1\\nfi\\n\\nexport CUDA_HOME="$3"\\nexport PATH="$CUDA_HOME/bin:$PATH"\\nexport MAX_JOBS=2\\n\\nif [ -x /usr/bin/gcc-12 ] && [ -x /usr/bin/g++-12 ]; then\\n  export CC=/usr/bin/gcc-12\\n  export CXX=/usr/bin/g++-12\\nelse\\n  export CC=/usr/bin/gcc\\n  export CXX=/usr/bin/g++\\nfi\\n\\nif [ -z "${TORCH_CUDA_ARCH_LIST:-}" ]; then\\nexport TORCH_CUDA_ARCH_LIST=$("$PY" - <<\\\'PY2\\\'\\nimport torch\\nif not torch.cuda.is_available():\\n    raise RuntimeError(\\\'Python 3.11 env không nhận CUDA\\\')\\nm, n = torch.cuda.get_device_capability(0)\\nprint(f\\\'{m}.{n}\\\')\\nPY2\\n)\\nfi\\n\\necho "Python: $("$PY" --version)"\\necho "CUDA arch: ${TORCH_CUDA_ARCH_LIST}"\\necho "Compiler: $CXX"\\n\\nDEPS="$2"\\nRASTER="$DEPS/gaussian_rasterizer"\\nKNN="$DEPS/simple-knn"\\nRASTER_COMMIT=9c40173fcc8d9b16778a1a8040295bc2f9ebf129\\nKNN_COMMIT=c7e51a06a4cd84c25e769fee29ab391fe5d5ff8d\\nmkdir -p "$DEPS"\\n\\n# ---------- gaussian_rasterizer ----------\\nif [ -d "$RASTER/.git" ]; then\\n  git -C "$RASTER" fetch --depth 1 origin "$RASTER_COMMIT"\\nelse\\n  rm -rf "$RASTER"\\n  git clone https://github.com/VladimirYugay/gaussian_rasterizer.git "$RASTER"\\nfi\\ngit -C "$RASTER" checkout -f "$RASTER_COMMIT"\\nrm -rf "$RASTER/build" "$RASTER/dist" "$RASTER"/*.egg-info\\nfind "$RASTER" -type f \\\\( -name \\\'*.so\\\' -o -name \\\'*.o\\\' \\\\) -delete 2>/dev/null || true\\n"$PY" -m pip uninstall -y gaussian_rasterizer gaussian-rasterizer >/dev/null 2>&1 || true\\n\\n# Cùng cơ chế notebook gốc: pip install local source, không build isolation.\\n"$PY" -m pip install --verbose --no-build-isolation --no-cache-dir "$RASTER"\\n\\n# ---------- simple-knn ----------\\nif [ -d "$KNN/.git" ]; then\\n  git -C "$KNN" fetch --depth 1 origin "$KNN_COMMIT"\\nelse\\n  rm -rf "$KNN"\\n  git clone https://github.com/VladimirYugay/simple-knn.git "$KNN"\\nfi\\ngit -C "$KNN" checkout -f "$KNN_COMMIT"\\n\\n"$PY" - "$KNN/simple_knn.cu" <<\\\'PY2\\\'\\nimport sys\\nfrom pathlib import Path\\np = Path(sys.argv[1])\\ns = p.read_text()\\nif \\\'#include <cfloat>\\\' not in s and \\\'#include <float.h>\\\' not in s:\\n    lines = s.splitlines()\\n    ids = [i for i, line in enumerate(lines) if line.strip().startswith(\\\'#include\\\')]\\n    lines.insert((ids[-1] + 1) if ids else 0, \\\'#include <cfloat>\\\')\\n    p.write_text(\\\'\\\\n\\\'.join(lines) + \\\'\\\\n\\\')\\nPY2\\n\\nrm -rf "$KNN/build" "$KNN/dist" "$KNN"/*.egg-info\\nfind "$KNN" -type f \\\\( -name \\\'*.so\\\' -o -name \\\'*.o\\\' \\\\) -delete 2>/dev/null || true\\n"$PY" -m pip uninstall -y simple_knn simple-knn >/dev/null 2>&1 || true\\n"$PY" -m pip install --verbose --no-build-isolation --no-cache-dir "$KNN"\\n\\n# ---------- import test ----------\\ncd "$4"\\n"$PY" - <<\\\'PY2\\\'\\nimport torch\\nimport gaussian_rasterizer\\nimport gaussian_rasterizer._C\\nfrom simple_knn._C import distCUDA2\\nprint(\\\'torch:\\\', torch.__version__, \\\'| CUDA:\\\', torch.version.cuda)\\nprint(\\\'GPU:\\\', torch.cuda.get_device_name(0), \\\'| CC:\\\', torch.cuda.get_device_capability(0))\\nprint(\\\'gaussian_rasterizer: OK\\\')\\nprint(\\\'simple_knn: OK\\\')\\nPY2\\n\\n\'\n\nBUILD_REFERENCE = {\'notebook\': \'office0_full_slam_pipeline_py311_paper_assets_v2.ipynb\', \'source_sha256\': \'585a005d48f05c5f83e4b4b3fbc8439704631d8e520bc33565b9dd99c4f3c452\', \'build_cell_index\': 12, \'original_build_cell_sha256\': \'be06333383b2da65c060c2c3b270bfb5732dde3e9645a998d88e891ac53a1bd9\', \'adaptations\': [\'selected PY311/path arguments\', \'selected compatible CUDA toolkit\', \'inherited all-GPU arch list\', \'quoted Python path\']}\n\ndef run(cmd, **kwargs):\n    print(\'Run:\', repr(cmd), flush=True)\n    return subprocess.run(cmd, check=True, **kwargs)\n\ndef toolkit_home(spec):\n    if spec.get(\'cuda_home\'):\n        choices = [Path(spec[\'cuda_home\'])]\n    else:\n        choices = [Path(\'/usr/local/cuda-12.4\'), Path(\'/usr/local/cuda\')]\n        nvcc = shutil.which(\'nvcc\')\n        if nvcc:\n            choices.append(Path(nvcc).resolve().parent.parent)\n    for path in choices:\n        nvcc = path / \'bin\' / \'nvcc\'\n        if not nvcc.is_file():\n            continue\n        version = subprocess.check_output([str(nvcc), \'--version\'], text=True)\n        found = re.search(r\'release\\s+(\\d+)\\.(\\d+)\', version)\n        if found and int(found[1]) == 12:\n            return path.resolve(), f\'{found[1]}.{found[2]}\'\n        if spec.get(\'cuda_home\'):\n            raise RuntimeError(f\'CUDA_TOOLKIT_HOME={path} is not a CUDA 12.x compiler for the pinned torch cu124 build. Existing compatible binaries may still be reused; rebuilding requires CUDA 12.x (prefer 12.4).\')\n    raise RuntimeError(\'Cannot rebuild pinned torch 2.5.1 cu124: no CUDA 12.x nvcc found. Keep the user versions and set CUDA_TOOLKIT_HOME to an installed CUDA 12.x toolkit (prefer 12.4). Existing verified binaries can run without nvcc.\')\n\ndef runtime_info(py, env):\n    script = """\nimport json,sys,torch,torchvision,numpy\nassert sys.version_info[:2]==(3,11),sys.version\nassert torch.__version__.split(\'+\')[0]==\'2.5.1\',torch.__version__\nassert torchvision.__version__.split(\'+\')[0]==\'0.20.1\',torchvision.__version__\nassert torch.version.cuda==\'12.4\',torch.version.cuda\nassert numpy.__version__==\'1.26.4\',numpy.__version__\nassert torch.cuda.is_available(),\'CUDA GPU is not visible\'\nprint(json.dumps({\'python\':sys.version.split()[0],\'executable\':sys.executable,\n \'torch\':torch.__version__,\'torch_cuda\':torch.version.cuda,\'numpy\':numpy.__version__,\n \'arch\':sorted(set(\'.\'.join(map(str,torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count()))),\n \'gpus\':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))\n"""\n    return json.loads(subprocess.check_output([str(py), \'-c\', script], text=True, env=env))\n\ndef installed_extensions(py, env, work):\n    # The saved wheel points to its local source via direct_url.json. Check that\n    # source\'s actual Git commit before reusing it; do not label an unknown binary\n    # as the user\'s pinned build merely because its import succeeds.\n    script = r"""\nimport hashlib,importlib.metadata as md,json,subprocess\nfrom pathlib import Path\nfrom urllib.parse import urlparse,unquote\nimport torch,gaussian_rasterizer._C as raster,simple_knn._C as knn\nexpected={\'gaussian_rasterizer\':\'9c40173fcc8d9b16778a1a8040295bc2f9ebf129\',\n          \'simple-knn\':\'c7e51a06a4cd84c25e769fee29ab391fe5d5ff8d\'}\nrows=[]\nfor name,module in [(\'gaussian_rasterizer\',raster),(\'simple-knn\',knn)]:\n dist=md.distribution(name);raw=dist.read_text(\'direct_url.json\')\n if not raw:raise RuntimeError(\'No local-source provenance for \'+name)\n source_url=json.loads(raw)[\'url\'];parsed=urlparse(source_url)\n if parsed.scheme!=\'file\':raise RuntimeError(\'Expected local-source install for \'+name)\n source=Path(unquote(parsed.path))\n head=subprocess.check_output([\'git\',\'-C\',str(source),\'rev-parse\',\'HEAD\'],text=True).strip()\n if head!=expected[name]:raise RuntimeError(\'Unexpected source commit for \'+name+\': \'+head)\n binary=Path(module.__file__)\n rows.append({\'name\':name,\'source\':str(source),\'commit\':head,\'binary\':str(binary),\n              \'binary_sha256\':hashlib.sha256(binary.read_bytes()).hexdigest()})\nvalue=knn.distCUDA2(torch.tensor([[0.,0.,1.],[.1,0.,1.],[0.,.1,1.],[.1,.1,1.]],device=\'cuda\'))\ntorch.cuda.synchronize()\nassert bool(torch.isfinite(value).all()),\'simple-knn CUDA execution failed\'\nprint(json.dumps(rows))\n"""\n    result = subprocess.run([str(py), \'-c\', script], text=True, env=env, cwd=work,\n                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    if result.returncode:\n        return None, result.stderr[-4000:]\n    try:\n        return json.loads(result.stdout.strip().splitlines()[-1]), None\n    except Exception as error:\n        return None, \'Unrecognized installed-extension probe output: \' + str(error)\n\ndef build_extensions(spec, py, env):\n    output = Path(spec[\'output_base\']); output.mkdir(parents=True, exist_ok=True)\n    work = Path(spec.get(\'work_root\', \'/kaggle/working\'))\n    info = runtime_info(py, env)\n    # Keep the same build-script text in a readable file, including when a\n    # working user build can be reused without compilation.\n    script = Path(spec[\'tools_dir\']) / \'build_extensions_user_version.sh\'\n    script.parent.mkdir(parents=True, exist_ok=True)\n    script.write_text(BUILD_SCRIPT)\n    run([\'bash\', \'-n\', str(script)])\n    installed, reason = installed_extensions(py, env, work)\n    forced = bool(spec.get(\'force_rebuild_extensions\', False))\n    if installed is not None and not forced:\n        report = {\'mode\':\'reused_verified_user_build\',\'python\':str(py),\'gpu_info\':info,\n                  \'reference\':BUILD_REFERENCE,\'installed_extensions\':installed,\n                  \'nvcc_required\':False,\'render_cuda_smoke\':\'benchmark uses production render_depth; reuse probe checks simple-knn execution\'}\n        (output/\'cuda_build_info.json\').write_text(json.dumps(report,indent=2))\n        print(\'Reuse user CUDA extensions: correct commits, imports and simple-knn GPU execution passed.\',flush=True)\n        return\n    toolkit, version = toolkit_home(spec)\n    build = Path(spec[\'build_root\'])\n    if build.resolve() in {Path(\'/\'), work.resolve(), py.parent.parent.resolve()}:\n        raise ValueError(\'build_root must be an extension-source subdirectory, not a workspace or environment root.\')\n    build.mkdir(parents=True, exist_ok=True)\n    for name,url,commit in EXTENSIONS:\n        folder=build/name\n        if (folder/\'.git\').exists():\n            actual=subprocess.check_output([\'git\',\'-C\',str(folder),\'remote\',\'get-url\',\'origin\'],text=True).strip()\n            if actual.removesuffix(\'.git\')!=url.removesuffix(\'.git\'):\n                raise RuntimeError(\'Wrong cached extension repository: \'+str(folder))\n    # These variables are inherited by bash and every PY311 subprocess. The\n    # compiler choice and uninstall/clean/install sequence are the original cell.\n    env = dict(env)\n    env[\'CUDA_HOME\'] = str(toolkit)\n    env[\'PATH\'] = str(toolkit/\'bin\') + os.pathsep + env.get(\'PATH\',\'\')\n    env[\'TORCH_CUDA_ARCH_LIST\'] = \';\'.join(info[\'arch\'])\n    env[\'MAX_JOBS\'] = \'2\'\n    log = output / \'cuda_build.log\'\n    print(\'Build with the user Office-0 cell:\', py, \'| CUDA\', version, \'| arch\', env[\'TORCH_CUDA_ARCH_LIST\'],flush=True)\n    print(\'Full compiler log:\', log, flush=True)\n    # Tee complete native compiler output. Never report only CalledProcessError.\n    with log.open(\'w\') as saved:\n        with subprocess.Popen([\'bash\', str(script), str(py), str(build), str(toolkit), str(work)],\n                env=env,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:\n            try:\n                for line in proc.stdout:\n                    saved.write(line);saved.flush();print(line,end=\'\',flush=True)\n                code = proc.wait()\n            except BaseException:\n                proc.terminate()\n                try: proc.wait(timeout=15)\n                except subprocess.TimeoutExpired: proc.kill();proc.wait()\n                raise\n    if code:\n        raise RuntimeError(f\'CUDA build exited {code}. Full compiler error is above and in {log}. Python={py}; pinned torch 2.5.1 cu124; nvcc={toolkit}.\')\n    installed, error = installed_extensions(py,env,work)\n    if installed is None:\n        raise RuntimeError(\'Post-build binary/source/GPU check failed: \'+str(error))\n    report = {\'mode\':\'rebuilt_with_user_cell\',\'python\':str(py),\'cuda_home\':str(toolkit),\'nvcc\':version,\n              \'gpu_info\':info,\'reference\':BUILD_REFERENCE,\'installed_extensions\':installed,\n              \'build_script_sha256\':hashlib.sha256(BUILD_SCRIPT.encode()).hexdigest(),\n              \'build_log\':str(log),\'previous_binary_probe\':reason}\n    (output/\'cuda_build_info.json\').write_text(json.dumps(report,indent=2))\n'
build_module_path=TOOLS_DIR/'user_build_extensions.py'
build_module_path.write_text(USER_BUILD_MODULE_SOURCE)
spec_path=TOOLS_DIR/'build_params.json'
spec_path.write_text(json.dumps({'python':str(PY311),'tools_dir':str(TOOLS_DIR),
    'output_base':str(TOOLS_DIR),'build_root':'/kaggle/working/3dgs_build_deps',
    'work_root':'/kaggle/working','cuda_home':CUDA_TOOLKIT_HOME,
    'force_rebuild_extensions':FORCE_REBUILD_EXTENSIONS},indent=2))
command="""import importlib.util,json,os,sys
from pathlib import Path
p=Path(sys.argv[1]);s=importlib.util.spec_from_file_location('user_build',p)
m=importlib.util.module_from_spec(s);s.loader.exec_module(m)
config=json.loads(Path(sys.argv[2]).read_text())
m.build_extensions(config,Path(config['python']),os.environ.copy())
"""
subprocess.run([str(PY311),'-c',command,str(build_module_path),str(spec_path)],check=True)
print('Build log (if compiled):',TOOLS_DIR/'cuda_build.log')


## 6. Kiểm tra môi trường

In [ ]:
import subprocess, textwrap

sanity = r"""
import sys, torch, importlib
from pathlib import Path
REPO = Path('/kaggle/working/3DGS-SLAM-TURTLEBOT4')
sys.path.insert(0, str(REPO))

required = {
    REPO / 'src/utils/utils.py': 'def coarse_registration_gaussian_landmark(',
    REPO / 'src/utils/magic_slam_utils.py': 'def register_submaps_depth(',
    REPO / 'src/entities/magic_slam.py': 'merged_refined.ply',
    REPO / 'run_slam.py': 'gaussian_landmark',
}
for path, marker in required.items():
    if not path.exists() or marker not in path.read_text(encoding='utf-8'):
        raise RuntimeError(f'Missing repo marker: {path.name} -> {marker}')

for module in ['open3d','faiss','roma','gtsam','gaussian_rasterizer','gaussian_rasterizer._C','simple_knn._C','src.entities.magic_slam']:
    importlib.import_module(module)

from src.utils.gpu_utils import build_agent_gpu_map
print('Python:', sys.version.split()[0])
print('GPU count:', torch.cuda.device_count())
print('GPU map:', build_agent_gpu_map([0,1], torch.cuda.device_count()))
print('Sanity: OK')
"""
subprocess.run([str(PY311), '-c', sanity], cwd=REPO, check=True)

with (TOOLS_DIR/'environment.txt').open('w') as saved:
    subprocess.run([str(PY311),'-m','pip','freeze'],stdout=saved,check=True)
print('Setup dùng cùng interpreter với benchmark:',PY311)


## 7. Chuẩn bị RGB-D Office-0

In [ ]:
# Script điều phối full SLAM được nhúng; không cần file ngoài notebook.
FULL_SLAM_RUNNER_SOURCE = '"""Run the user\'s native full SLAM, then hand complete outputs to single/multi."""\nimport argparse\nimport collections\nimport contextlib\nimport copy\nimport csv\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\n\nUSER_BASE_CONFIG = "from pathlib import Path\\nimport os, json\\n\\nREPO = Path(\'/kaggle/working/3DGS-SLAM-TURTLEBOT4\')\\nVENV = Path(\'/kaggle/working/gs_slam_py311\')\\nPY311 = VENV / \'bin\' / \'python\'\\nCACHE_ROOT = Path(\'/kaggle/working/huggingface_cache\')\\n\\nDATASET_REPO = \'voviktyl/ReplicaMultiagent\'\\nSCENE = \'office_0\'\\nSCENE_FOLDER = \'Office-0\'\\nFRAME_LIMIT = 1500\\nDOWNLOAD_ROOT = REPO / \'data\' / \'ReplicaMultiagent\'\\nDATA_ROOT = DOWNLOAD_ROOT / SCENE_FOLDER\\nBASE_CONFIG_PATH = REPO / \'configs\' / \'ReplicaMultiagent\' / \'office_0.yaml\'\\n\\nREGISTRATION_METHOD = \'gaussian_landmark\'\\nSEED = 0\\nFULL_RUN_ALLOW_FPFH_FALLBACK = False\\n\\nDINO_PATCH_PRESERVE_ASPECT_RATIO = True\\nDINO_PATCH_SHORT_EDGE = 280\\nDINO_USE_FAST = False\\nGAUSSIAN_MIN_OPACITY = 0.05\\nGAUSSIAN_DEPTH_TOLERANCE_M = 0.05\\nGAUSSIAN_MAX_LANDMARKS = 2048\\nGAUSSIAN_REFIT_INLIERS = True\\nDINO_CORR_MIN_SIMILARITY = -1.0\\nDINO_CORR_MIN_MARGIN = 0.0\\nDINO_CORR_KEEP_TOP_FRACTION = 1.0\\nDINO_CORR_DEPTH_WINDOW_RADIUS = 2\\nDINO_CORR_DEPTH_MAX_MAD_M = 0.05\\nDINO_CORR_COMPATIBILITY_THRESHOLD_M = 0.03\\nDINO_CORR_MIN_COMPATIBILITY_SUPPORT = 16\\nPATCH_LEVEL_CHECK = True\\nPATCH_NOVEL_FRACTION_THRESHOLD = 0.08\\n\\nOUTPUT_ROOT = Path(\'/kaggle/working/gslam_outputs/ReplicaMultiagent/office_0_gaussian_landmark_geo280_kf008\')\\nPIPELINE_ROOT = OUTPUT_ROOT / \'pipeline_assets\'\\nRUNTIME_CONFIG = REPO / \'configs\' / \'ReplicaMultiagent\' / \'office_0_pipeline_runtime.yaml\'\\n\\nREPRESENTATIVE_SUBMAPS_PER_AGENT = 3\\nEXPORT_REPRESENTATIVE_CKPT = True\\nEXPORT_REPRESENTATIVE_PLY = True\\nGS_VIS_MAX_POINTS_PER_SUBMAP = 30000\\nFINAL_VIS_MAX_POINTS = 90000\\n\\nos.environ[\'HF_HOME\'] = str(CACHE_ROOT)\\nprint(\'Scene:\', SCENE_FOLDER, \'| frames:\', FRAME_LIMIT)\\nprint(\'Repo:\', REPO)\\nprint(\'Python 3.11 env:\', PY311)\\nprint(\'Output:\', OUTPUT_ROOT)\\n"\nUSER_RUNTIME_CONFIG_CELL = \'import json, subprocess\\n\\nparams = {\\n    \\\'repo\\\': str(REPO), \\\'base_config\\\': str(BASE_CONFIG_PATH), \\\'runtime_config\\\': str(RUNTIME_CONFIG),\\n    \\\'input_path\\\': str(DATA_ROOT), \\\'output_path\\\': str(OUTPUT_ROOT), \\\'scene\\\': SCENE,\\n    \\\'frame_limit\\\': FRAME_LIMIT, \\\'seed\\\': SEED, \\\'registration_method\\\': REGISTRATION_METHOD,\\n    \\\'fallback_to_fpfh\\\': FULL_RUN_ALLOW_FPFH_FALLBACK,\\n    \\\'dino_patch_preserve_aspect_ratio\\\': DINO_PATCH_PRESERVE_ASPECT_RATIO,\\n    \\\'dino_patch_short_edge\\\': DINO_PATCH_SHORT_EDGE, \\\'dino_use_fast\\\': DINO_USE_FAST,\\n    \\\'gaussian_min_opacity\\\': GAUSSIAN_MIN_OPACITY,\\n    \\\'gaussian_depth_tolerance_m\\\': GAUSSIAN_DEPTH_TOLERANCE_M,\\n    \\\'gaussian_max_landmarks\\\': GAUSSIAN_MAX_LANDMARKS,\\n    \\\'gaussian_refit_inliers\\\': GAUSSIAN_REFIT_INLIERS,\\n    \\\'dino_min_similarity\\\': DINO_CORR_MIN_SIMILARITY,\\n    \\\'dino_min_margin\\\': DINO_CORR_MIN_MARGIN,\\n    \\\'dino_keep_top_fraction\\\': DINO_CORR_KEEP_TOP_FRACTION,\\n    \\\'dino_depth_window_radius\\\': DINO_CORR_DEPTH_WINDOW_RADIUS,\\n    \\\'dino_depth_max_mad_m\\\': DINO_CORR_DEPTH_MAX_MAD_M,\\n    \\\'dino_compatibility_threshold_m\\\': DINO_CORR_COMPATIBILITY_THRESHOLD_M,\\n    \\\'dino_min_compatibility_support\\\': DINO_CORR_MIN_COMPATIBILITY_SUPPORT,\\n    \\\'patch_level_check\\\': PATCH_LEVEL_CHECK,\\n    \\\'patch_novel_fraction_threshold\\\': PATCH_NOVEL_FRACTION_THRESHOLD,\\n}\\nPARAMS_JSON = Path(\\\'/kaggle/working/scene_pipeline_params.json\\\')\\nPARAMS_JSON.write_text(json.dumps(params, indent=2))\\n\\nscript = r"""\\nimport json, sys, yaml\\nfrom pathlib import Path\\np = json.loads(Path(sys.argv[1]).read_text())\\nREPO = Path(p[\\\'repo\\\']); sys.path.insert(0, str(REPO))\\nfrom src.utils.io_utils import load_config\\ncfg = load_config(p[\\\'base_config\\\'])\\ncfg.pop(\\\'inherit_from\\\', None)\\ncfg[\\\'config_path\\\'] = p[\\\'base_config\\\']\\ncfg[\\\'multi_gpu\\\'] = True\\ncfg[\\\'use_wandb\\\'] = False\\ncfg[\\\'seed\\\'] = p[\\\'seed\\\']\\ncfg[\\\'data\\\'][\\\'agent_ids\\\'] = [0,1]\\ncfg[\\\'data\\\'][\\\'scene_name\\\'] = p[\\\'scene\\\']\\ncfg[\\\'data\\\'][\\\'frame_limit\\\'] = p[\\\'frame_limit\\\']\\ncfg[\\\'data\\\'][\\\'input_path\\\'] = p[\\\'input_path\\\']\\ncfg[\\\'data\\\'][\\\'output_path\\\'] = p[\\\'output_path\\\']\\ncfg[\\\'loop_detection\\\'][\\\'feature_extractor_name\\\'] = \\\'dino\\\'\\ncfg[\\\'loop_detection\\\'][\\\'weights_path\\\'] = \\\'facebook/dinov2-small\\\'\\ncfg[\\\'loop_detection\\\'][\\\'embed_size\\\'] = 384\\ncfg[\\\'loop_detection\\\'][\\\'dino_patch_preserve_aspect_ratio\\\'] = p[\\\'dino_patch_preserve_aspect_ratio\\\']\\ncfg[\\\'loop_detection\\\'][\\\'dino_patch_short_edge\\\'] = p[\\\'dino_patch_short_edge\\\']\\ncfg[\\\'loop_detection\\\'][\\\'dino_use_fast\\\'] = p[\\\'dino_use_fast\\\']\\ncfg[\\\'submap\\\'][\\\'registration_method\\\'] = p[\\\'registration_method\\\']\\ncfg[\\\'submap\\\'][\\\'fallback_to_fpfh\\\'] = p[\\\'fallback_to_fpfh\\\']\\ncfg[\\\'submap\\\'][\\\'initial_transformation_unknown\\\'] = True\\ncfg[\\\'submap\\\'][\\\'registration_options\\\'] = {\\n    \\\'min_correspondences\\\': 8, \\\'min_ransac_inliers\\\': 3,\\n    \\\'ransac_distance_threshold\\\': 0.05, \\\'max_keypoints\\\': 4096, \\\'ratio_threshold\\\': 0.8,\\n    \\\'dino_min_similarity\\\': p[\\\'dino_min_similarity\\\'], \\\'dino_min_margin\\\': p[\\\'dino_min_margin\\\'],\\n    \\\'dino_keep_top_fraction\\\': p[\\\'dino_keep_top_fraction\\\'],\\n    \\\'dino_depth_window_radius\\\': p[\\\'dino_depth_window_radius\\\'],\\n    \\\'dino_depth_max_mad_m\\\': p[\\\'dino_depth_max_mad_m\\\'],\\n    \\\'dino_compatibility_threshold_m\\\': p[\\\'dino_compatibility_threshold_m\\\'],\\n    \\\'dino_min_compatibility_support\\\': p[\\\'dino_min_compatibility_support\\\'],\\n    \\\'gaussian_min_opacity\\\': p[\\\'gaussian_min_opacity\\\'],\\n    \\\'gaussian_depth_tolerance_m\\\': p[\\\'gaussian_depth_tolerance_m\\\'],\\n    \\\'gaussian_max_landmarks\\\': p[\\\'gaussian_max_landmarks\\\'],\\n    \\\'gaussian_refit_inliers\\\': p[\\\'gaussian_refit_inliers\\\'],\\n}\\ncfg[\\\'keyframing\\\'][\\\'patch_level_check\\\'] = p[\\\'patch_level_check\\\']\\ncfg[\\\'keyframing\\\'][\\\'patch_novel_fraction_threshold\\\'] = p[\\\'patch_novel_fraction_threshold\\\']\\nPath(p[\\\'runtime_config\\\']).write_text(yaml.safe_dump(cfg, sort_keys=False))\\nprint(\\\'Runtime config:\\\', p[\\\'runtime_config\\\'])\\nprint(\\\'Registration:\\\', cfg[\\\'submap\\\'][\\\'registration_method\\\'], \\\'| fallback:\\\', cfg[\\\'submap\\\'][\\\'fallback_to_fpfh\\\'])\\n"""\\nsubprocess.run([str(PY311), \\\'-c\\\', script, str(PARAMS_JSON)], cwd=REPO, check=True)\\nCONFIG_PATH = RUNTIME_CONFIG\\n\'\nUSER_DINO_CACHE_CELL = \'import subprocess, os\\n\\nenv = os.environ.copy(); env[\\\'HF_HOME\\\'] = str(CACHE_ROOT)\\nscript = """\\nfrom transformers import AutoImageProcessor, AutoModel\\nmodel_id=\\\'facebook/dinov2-small\\\'\\nAutoImageProcessor.from_pretrained(model_id, use_fast=False)\\nAutoModel.from_pretrained(model_id)\\nprint(\\\'DINOv2 cache: OK\\\')\\n"""\\nsubprocess.run([str(PY311), \\\'-c\\\', script], env=env, check=True)\\n\'\nUSER_LOOP_EVALUATOR = \'import pickle\\nfrom pathlib import Path\\nimport numpy as np\\nfrom src.utils.io_utils import load_config\\nfrom src.entities.datasets import get_dataset\\n\\n\\ndef geodesic_rotation_error_deg(R_a, R_b):\\n    R_err = R_a.T @ R_b\\n    cos_angle = np.clip((np.trace(R_err) - 1) / 2, -1.0, 1.0)\\n    return np.degrees(np.arccos(cos_angle))\\n\\n\\ndef build_agent_datasets(config):\\n    agents_datasets = {}\\n    input_root = Path(config["data"]["input_path"])\\n    agent_dirs = sorted(p for p in input_root.glob("*") if not p.name.startswith("."))\\n    if len(agent_dirs) < len(config["data"]["agent_ids"]):\\n        raise RuntimeError(\\n            f"input_path={input_root} chi co {len(agent_dirs)} thu muc con, "\\n            f"can {len(config[\\\'data\\\'][\\\'agent_ids\\\'])}."\\n        )\\n    for agent_id in config["data"]["agent_ids"]:\\n        per_agent_cfg = dict(config["data"])\\n        per_agent_cfg["input_path"] = str(agent_dirs[agent_id])\\n        dataset_cfg = {**per_agent_cfg, **config["cam"], **config["submap"]}\\n        agents_datasets[agent_id] = get_dataset(config["dataset_name"])(dataset_cfg)\\n    return agents_datasets\\n\\n\\ndef evaluate(loops_path, config_path, rot_thresh_deg=5.0, trans_thresh_cm=10.0,\\n             translation_unit_is_meters=True, label=""):\\n    config = load_config(config_path)\\n    agents_datasets = build_agent_datasets(config)\\n    with open(loops_path, "rb") as f:\\n        loops = pickle.load(f)\\n\\n    rows = []\\n    for l in loops:\\n        if l.source_agent_id == l.target_agent_id:\\n            continue\\n        gt_source_pose = agents_datasets[int(l.source_agent_id)][l.source_frame_id][-1]\\n        gt_target_pose = agents_datasets[int(l.target_agent_id)][l.target_frame_id][-1]\\n        gt_rel_pose = np.linalg.inv(gt_target_pose) @ gt_source_pose\\n        est_rel_pose = l.transformation\\n        t_err = np.linalg.norm(est_rel_pose[:3, 3] - gt_rel_pose[:3, 3])\\n        t_err_cm = t_err * 100.0 if translation_unit_is_meters else t_err\\n        r_err_deg = geodesic_rotation_error_deg(est_rel_pose[:3, :3], gt_rel_pose[:3, :3])\\n        success = (r_err_deg < rot_thresh_deg) and (t_err_cm < trans_thresh_cm)\\n        rows.append(dict(\\n            source=(int(l.source_agent_id), l.source_frame_id),\\n            target=(int(l.target_agent_id), l.target_frame_id),\\n            rot_err_deg=r_err_deg, trans_err_cm=t_err_cm,\\n            fitness=l.fitness, inlier_rmse=l.inlier_rmse,\\n            coarse_method_used=getattr(l, "coarse_method_used", "unknown (no patch)"),\\n            num_correspondences=getattr(l, "num_correspondences", None),\\n            success=success,\\n        ))\\n\\n    n = len(rows)\\n    if n == 0:\\n        print(f"[{label}] No inter-agent loops found.")\\n        return rows\\n    n_success = sum(r["success"] for r in rows)\\n    print(f"\\\\n=== [{label}] {loops_path} ===")\\n    print(f"N inter-agent loops: {n}")\\n    print(f"Success rate (<{rot_thresh_deg}deg AND <{trans_thresh_cm}cm): {n_success}/{n} = {100*n_success/n:.1f}%")\\n    print(f"Mean rotation error: {np.mean([r[\\\'rot_err_deg\\\'] for r in rows]):.3f} deg")\\n    print(f"Mean translation error: {np.mean([r[\\\'trans_err_cm\\\'] for r in rows]):.3f} cm")\\n    print(f"Mean ICP fitness: {np.mean([r[\\\'fitness\\\'] for r in rows]):.3f}\\\\n")\\n    for i, r in enumerate(rows):\\n        src = r["source"]; tgt = r["target"]\\n        rot = r["rot_err_deg"]; trans = r["trans_err_cm"]; fit = r["fitness"]\\n        meth = r["coarse_method_used"]; ncorr = r["num_correspondences"]; ok = r["success"]\\n        print(f"  [{i}] {src} <-> {tgt}  rot={rot:.2f}deg  trans={trans:.2f}cm  "\\n              f"fitness={fit:.3f}  method={meth}  n_corr={ncorr}  success={ok}")\\n    return rows\'\nUSER_PAPER_ASSETS_V1 = "\\nimport sys, os, json, pickle, shutil\\nfrom pathlib import Path\\nimport numpy as np\\nimport torch\\nimport matplotlib\\nmatplotlib.use(\'Agg\')\\nimport matplotlib.pyplot as plt\\nfrom PIL import Image\\nfrom plyfile import PlyData\\n\\nREPO = Path(os.environ[\'FULL_SLAM_REPO\'])\\nsys.path.insert(0, str(REPO))\\nfrom src.utils.io_utils import load_config\\nfrom src.entities.datasets import get_dataset\\nfrom src.utils.utils import load_3dgs, render_rgbd_from_3dgs\\n\\nOUTPUT_ROOT = Path(os.environ[\'FULL_SLAM_OUTPUT\'])\\nPIPELINE_ROOT = OUTPUT_ROOT / \'pipeline_assets\'\\nIMAGES_DIR = PIPELINE_ROOT / \'images\'\\nSUBMAP_EXPORT_DIR = PIPELINE_ROOT / \'submaps\'\\nIMAGES_DIR.mkdir(parents=True, exist_ok=True)\\nSUBMAP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)\\n\\nREPRESENTATIVE_SUBMAPS_PER_AGENT = 3\\nEXPORT_REPRESENTATIVE_CKPT = True\\nEXPORT_REPRESENTATIVE_PLY = True\\nGS_VIS_MAX_POINTS_PER_SUBMAP = 30000\\nFINAL_VIS_MAX_POINTS = 90000\\nGAUSSIAN_MIN_OPACITY = 0.05\\n\\ncfg = load_config(str(OUTPUT_ROOT / \'config.yaml\'))\\ninput_root = Path(cfg[\'data\'][\'input_path\'])\\nagent_dirs = sorted(p for p in input_root.iterdir() if p.is_dir() and not p.name.startswith(\'.\'))\\nagent_datasets, submap_paths = {}, {}\\nfor aid in cfg[\'data\'][\'agent_ids\']:\\n    aid = int(aid)\\n    per_agent = dict(cfg[\'data\']); per_agent[\'input_path\'] = str(agent_dirs[aid])\\n    dataset_cfg = {**per_agent, **cfg[\'cam\'], **cfg[\'submap\']}\\n    agent_datasets[aid] = get_dataset(cfg[\'dataset_name\'])(dataset_cfg)\\n    submap_paths[aid] = sorted((OUTPUT_ROOT / f\'agent_{aid}\' / \'submaps\').glob(\'*.ckpt\'))\\n    if not submap_paths[aid]: raise RuntimeError(f\'No submap checkpoints for agent {aid}\')\\n\\ndef representative_indices(n, k=3):\\n    k=min(int(k),int(n)); return sorted(set(np.linspace(0,n-1,k).round().astype(int).tolist()))\\ndef load_submap(aid, idx):\\n    return torch.load(submap_paths[int(aid)][int(idx)], map_location=\'cpu\', weights_only=False)\\ndef find_submap_for_frame(aid, frame_id):\\n    for idx,path in enumerate(submap_paths[int(aid)]):\\n        sub=torch.load(path,map_location=\'cpu\',weights_only=False)\\n        if int(sub[\'submap_start_frame_id\']) <= int(frame_id) <= int(sub[\'submap_end_frame_id\']): return idx,sub\\n        del sub\\n    raise KeyError((aid,frame_id))\\ndef image_to_uint8(image):\\n    arr=image.detach().cpu().numpy() if torch.is_tensor(image) else np.asarray(image)\\n    arr=np.squeeze(arr)\\n    if arr.ndim==3 and arr.shape[0] in (1,3,4) and arr.shape[-1] not in (1,3,4): arr=np.moveaxis(arr,0,-1)\\n    if arr.ndim==2: arr=np.repeat(arr[...,None],3,axis=-1)\\n    arr=arr[...,:3]\\n    if np.issubdtype(arr.dtype,np.floating) and np.nanmax(arr)<=1.5: arr=arr*255.0\\n    return np.clip(np.nan_to_num(arr),0,255).astype(np.uint8)\\ndef depth_to_float(depth):\\n    arr=depth.detach().cpu().numpy() if torch.is_tensor(depth) else np.asarray(depth)\\n    arr=np.squeeze(arr).astype(np.float32)\\n    if arr.ndim==3: arr=arr[0] if arr.shape[0]<=4 else arr[...,0]\\n    return arr\\ndef gaussian_points_rgb(submap,max_points=30000,min_opacity=0.05):\\n    model=load_3dgs(submap)\\n    try:\\n        xyz_t=model.get_xyz().detach(); op=model.get_opacity().detach().reshape(-1)\\n        valid=torch.isfinite(xyz_t).all(dim=1)&torch.isfinite(op)&(op>=min_opacity)\\n        idx=torch.where(valid)[0]\\n        if idx.numel()==0: raise RuntimeError(\'No valid Gaussians\')\\n        if idx.numel()>max_points: idx=idx[torch.linspace(0,idx.numel()-1,max_points,device=idx.device).long()]\\n        xyz=xyz_t[idx].cpu().float().numpy()\\n        dc=model.get_features().detach()[idx,0,:].cpu().float().numpy()\\n        rgb=np.clip(0.5+0.28209479177387814*dc,0.0,1.0)\\n        return xyz,rgb\\n    finally:\\n        del model\\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\\ndef apply_transform(points,T):\\n    points=np.asarray(points,dtype=np.float64); T=np.asarray(T,dtype=np.float64)\\n    return points@T[:3,:3].T+T[:3,3]\\ndef set_3d_equal(ax,points):\\n    p=np.asarray(points); mins,maxs=np.nanmin(p,axis=0),np.nanmax(p,axis=0)\\n    center=.5*(mins+maxs); radius=max(float(np.max(maxs-mins))*.52,1e-3)\\n    ax.set_xlim(center[0]-radius,center[0]+radius); ax.set_ylim(center[1]-radius,center[1]+radius); ax.set_zlim(center[2]-radius,center[2]+radius); ax.set_box_aspect((1,1,1))\\ndef save_scatter(path,point_sets,color_sets,elev=24,azim=-66):\\n    fig=plt.figure(figsize=(7.2,5.2),dpi=220); ax=fig.add_subplot(111,projection=\'3d\'); allp=[]\\n    for pts,cols in zip(point_sets,color_sets):\\n        allp.append(pts); ax.scatter(pts[:,0],pts[:,1],pts[:,2],c=cols,s=.25,alpha=.68,linewidths=0)\\n    set_3d_equal(ax,np.concatenate(allp,axis=0)); ax.view_init(elev=elev,azim=azim); ax.set_axis_off(); fig.tight_layout(pad=0)\\n    fig.savefig(path,bbox_inches=\'tight\',pad_inches=.02); plt.close(fig)\\n\\n# 1) RGB-D\\n_,rgb,depth,_=agent_datasets[0][min(len(agent_datasets[0])-1,max(0,len(agent_datasets[0])//2))]\\nrgb_np=image_to_uint8(rgb); depth_np=depth_to_float(depth); Image.fromarray(rgb_np).save(IMAGES_DIR/\'01_rgb_frame_agent0.png\')\\nvalid=depth_np[np.isfinite(depth_np)&(depth_np>0)]; lo,hi=(np.percentile(valid,[2,98]) if valid.size else (0.,1.))\\nfig=plt.figure(figsize=(7.2,4.1),dpi=220); ax=fig.add_subplot(111); ax.imshow(depth_np,cmap=\'magma\',vmin=lo,vmax=max(hi,lo+1e-6)); ax.set_axis_off(); fig.tight_layout(pad=0); fig.savefig(IMAGES_DIR/\'01_depth_frame_agent0.png\',bbox_inches=\'tight\',pad_inches=0); plt.close(fig)\\nfig,axes=plt.subplots(1,2,figsize=(12,4.3),dpi=220); axes[0].imshow(rgb_np); axes[0].set_axis_off(); axes[1].imshow(depth_np,cmap=\'magma\',vmin=lo,vmax=max(hi,lo+1e-6)); axes[1].set_axis_off(); fig.tight_layout(pad=.1); fig.savefig(IMAGES_DIR/\'01_rgbd_pair_agent0.png\',bbox_inches=\'tight\',pad_inches=.02); plt.close(fig)\\n\\n# 2) Pose/local trajectory\\nrep=representative_indices(len(submap_paths[0]),REPRESENTATIVE_SUBMAPS_PER_AGENT); sub_idx=rep[len(rep)//2]; sub=load_submap(0,sub_idx)\\nposes=np.asarray([p.detach().cpu().numpy() if torch.is_tensor(p) else np.asarray(p) for p in sub[\'submap_c2ws\']],dtype=np.float64); pos=poses[:,:3,3]\\nfig,ax=plt.subplots(figsize=(6.2,5.2),dpi=240); ax.plot(pos[:,0],pos[:,2],linewidth=2.0); ax.scatter(pos[:,0],pos[:,2],s=10); step=max(1,len(poses)//8)\\nfor i in range(0,len(poses),step):\\n    fwd=poses[i,:3,2]; ax.arrow(pos[i,0],pos[i,2],fwd[0]*.08,fwd[2]*.08,width=.002,head_width=.03,length_includes_head=True)\\nax.set_aspect(\'equal\',adjustable=\'box\'); ax.set_axis_off(); fig.tight_layout(pad=0); fig.savefig(IMAGES_DIR/\'02_pose_estimation_agent0.png\',bbox_inches=\'tight\',pad_inches=.02); plt.close(fig); del sub\\n\\n# 3) Representative local Gaussian submaps\\nrepresentative_manifest=[]\\nfor aid in sorted(submap_paths):\\n    for sub_idx in representative_indices(len(submap_paths[aid]),REPRESENTATIVE_SUBMAPS_PER_AGENT):\\n        sub=load_submap(aid,sub_idx); tag=f\'agent{aid}_submap{sub_idx:03d}\'\\n        color,_=render_rgbd_from_3dgs(sub); png=IMAGES_DIR/f\'03_local_gaussian_{tag}.png\'; Image.fromarray(color).save(png)\\n        ckpt=SUBMAP_EXPORT_DIR/f\'{tag}.ckpt\' if EXPORT_REPRESENTATIVE_CKPT else None\\n        if ckpt: shutil.copy2(submap_paths[aid][sub_idx],ckpt)\\n        ply=SUBMAP_EXPORT_DIR/f\'{tag}.ply\' if EXPORT_REPRESENTATIVE_PLY else None\\n        if ply:\\n            model=load_3dgs(sub)\\n            try: model.save_ply(str(ply))\\n            finally:\\n                del model\\n                if torch.cuda.is_available(): torch.cuda.empty_cache()\\n        representative_manifest.append({\'agent_id\':aid,\'submap_index\':sub_idx,\'start_frame\':int(sub[\'submap_start_frame_id\']),\'end_frame\':int(sub[\'submap_end_frame_id\']),\'render_png\':str(png),\'ckpt\':str(ckpt) if ckpt else None,\'ply\':str(ply) if ply else None})\\n        del sub\\n\\n# 4) GS-to-GS using a real inter-agent loop\\nloop_file=OUTPUT_ROOT/\'filtered_loops.pkl\'\\nwith loop_file.open(\'rb\') as f: loops=pickle.load(f)\\ninter=[l for l in loops if int(l.source_agent_id)!=int(l.target_agent_id)]\\nif not inter:\\n    with (OUTPUT_ROOT/\'loops.pkl\').open(\'rb\') as f: all_loops=pickle.load(f)\\n    inter=[l for l in all_loops if int(l.source_agent_id)!=int(l.target_agent_id)]\\ngs_loop_info=None\\nif inter:\\n    loop=inter[0]; sa,ta=int(loop.source_agent_id),int(loop.target_agent_id); sf,tf=int(loop.source_frame_id),int(loop.target_frame_id)\\n    si,ss=find_submap_for_frame(sa,sf); ti,ts=find_submap_for_frame(ta,tf)\\n    sx,srgb=gaussian_points_rgb(ss,GS_VIS_MAX_POINTS_PER_SUBMAP,GAUSSIAN_MIN_OPACITY); tx,trgb=gaussian_points_rgb(ts,GS_VIS_MAX_POINTS_PER_SUBMAP,GAUSSIAN_MIN_OPACITY)\\n    sc2w=np.asarray(ss[\'submap_c2ws\'][0],dtype=np.float64); tc2w=np.asarray(ts[\'submap_c2ws\'][0],dtype=np.float64); Tcam=np.asarray(loop.transformation,dtype=np.float64); T=tc2w@Tcam@np.linalg.inv(sc2w); salign=apply_transform(sx,T)\\n    save_scatter(IMAGES_DIR/\'05_gs_to_gs_before.png\',[sx,tx],[srgb,trgb]); save_scatter(IMAGES_DIR/\'06_gs_to_gs_after.png\',[salign,tx],[srgb,trgb])\\n    gs_loop_info={\'source\':[sa,sf],\'target\':[ta,tf],\'source_submap\':si,\'target_submap\':ti,\'fitness\':float(getattr(loop,\'fitness\',float(\'nan\'))),\'inlier_rmse\':float(getattr(loop,\'inlier_rmse\',float(\'nan\'))),\'registration_method\':getattr(loop,\'coarse_method_used\',None)}\\n    del ss,ts\\n\\n# 5) Final merged map\\ndef ply_points_rgb(path,max_points=90000):\\n    ply=PlyData.read(str(path)); v=ply[\'vertex\'].data; xyz=np.stack([v[\'x\'],v[\'y\'],v[\'z\']],axis=1).astype(np.float32); names=v.dtype.names\\n    if all(n in names for n in [\'f_dc_0\',\'f_dc_1\',\'f_dc_2\']):\\n        dc=np.stack([v[\'f_dc_0\'],v[\'f_dc_1\'],v[\'f_dc_2\']],axis=1).astype(np.float32); rgb=np.clip(.5+.28209479177387814*dc,0.,1.)\\n    else: rgb=np.full((len(xyz),3),.65,dtype=np.float32)\\n    valid=np.isfinite(xyz).all(axis=1)\\n    if \'opacity\' in names:\\n        raw=np.asarray(v[\'opacity\'],dtype=np.float32); op=1./(1.+np.exp(-np.clip(raw,-30,30))); valid &= op>=GAUSSIAN_MIN_OPACITY\\n    xyz,rgb=xyz[valid],rgb[valid]\\n    if len(xyz)>max_points:\\n        idx=np.linspace(0,len(xyz)-1,max_points).astype(np.int64); xyz,rgb=xyz[idx],rgb[idx]\\n    return xyz,rgb\\nfinal_ply=OUTPUT_ROOT/\'merged_refined.ply\'\\nif not final_ply.exists(): final_ply=OUTPUT_ROOT/\'merged_coarse.ply\'\\nfx,frgb=ply_points_rgb(final_ply,FINAL_VIS_MAX_POINTS); save_scatter(IMAGES_DIR/\'07_final_merged_map.png\',[fx],[frgb],elev=28,azim=-62)\\n\\n# 6) Contact sheet + manifest\\nagent0_local=sorted(IMAGES_DIR.glob(\'03_local_gaussian_agent0_*.png\'))\\nitems=[(\'RGB-D\',IMAGES_DIR/\'01_rgbd_pair_agent0.png\'),(\'Pose\',IMAGES_DIR/\'02_pose_estimation_agent0.png\'),(\'Local 3DGS\',agent0_local[len(agent0_local)//2] if agent0_local else None),(\'GS-to-GS\',IMAGES_DIR/\'06_gs_to_gs_after.png\'),(\'Merged map\',IMAGES_DIR/\'07_final_merged_map.png\')]\\nitems=[(n,p) for n,p in items if p is not None and p.exists()]\\nfig,axes=plt.subplots(1,len(items),figsize=(4.*len(items),4.),dpi=180); axes=[axes] if len(items)==1 else axes\\nfor ax,(name,path) in zip(axes,items): ax.imshow(Image.open(path)); ax.set_title(name,fontsize=10); ax.set_axis_off()\\nfig.tight_layout(pad=.4); fig.savefig(IMAGES_DIR/\'08_pipeline_contact_sheet.png\',bbox_inches=\'tight\',pad_inches=.04); plt.close(fig)\\n\\nimport subprocess, zipfile\\nmanifest={\'scene\':cfg[\'data\'][\'scene_name\'],\'dataset_root\':str(input_root),\'output_root\':str(OUTPUT_ROOT),\'repo_commit\':subprocess.check_output([\'git\',\'-C\',str(REPO),\'rev-parse\',\'HEAD\'],text=True).strip(),\'registration_method\':cfg[\'submap\'].get(\'registration_method\'),\'representative_submaps\':representative_manifest,\'gs_to_gs_loop\':gs_loop_info,\'images\':[str(p) for p in sorted(IMAGES_DIR.glob(\'*.png\'))],\'existing_all_submap_checkpoints\':{f\'agent_{aid}\':[str(p) for p in paths] for aid,paths in submap_paths.items()},\'final_map_ply\':str(final_ply)}\\n(PIPELINE_ROOT/\'pipeline_manifest.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))\\nzip_path=OUTPUT_ROOT/(cfg[\'data\'][\'scene_name\']+\'_pipeline_images.zip\')\\nif zip_path.exists(): zip_path.unlink()\\nwith zipfile.ZipFile(zip_path,\'w\',compression=zipfile.ZIP_DEFLATED) as zf:\\n    for p in sorted(IMAGES_DIR.glob(\'*.png\')): zf.write(p,arcname=f\'images/{p.name}\')\\n    zf.write(PIPELINE_ROOT/\'pipeline_manifest.json\',arcname=\'pipeline_manifest.json\')\\nprint(\'Pipeline assets:\',PIPELINE_ROOT)\\nprint(\'Representative submaps:\',len(representative_manifest))\\nprint(\'Image ZIP:\',zip_path)\\n"\nUSER_PAPER_ASSETS_V2 = \'import os\\nfrom pathlib import Path\\nOUTPUT_ROOT=Path(os.environ[\\\'FULL_SLAM_OUTPUT\\\'])\\nSCENE=os.environ[\\\'FULL_SLAM_SCENE\\\']\\n# ================================================================\\n# PAPER ASSETS V2\\n# Actual Gaussian ellipsoid glyphs instead of point-cloud scatter.\\n# Does NOT rerun SLAM.\\n# ================================================================\\nfrom pathlib import Path\\nimport pickle\\nimport zipfile\\nimport json\\nimport math\\n\\nimport numpy as np\\nimport torch\\nimport matplotlib\\nmatplotlib.use("Agg")\\nimport matplotlib.pyplot as plt\\n\\nfrom matplotlib.patches import Ellipse, Polygon\\nfrom matplotlib.collections import PatchCollection\\nfrom PIL import Image\\nfrom plyfile import PlyData\\n\\n# ------------------------------------------------\\n# 1. Paths\\n# ------------------------------------------------\\nif "OUTPUT_ROOT" not in globals():\\n    OUTPUT_ROOT = Path(\\n        "/kaggle/working/gslam_outputs/ReplicaMultiagent/"\\n        "office_0_gaussian_landmark_geo280_kf008"\\n    )\\nelse:\\n    OUTPUT_ROOT = Path(OUTPUT_ROOT)\\n\\nV2_ROOT = OUTPUT_ROOT / "pipeline_assets_v2"\\nV2_IMAGES = V2_ROOT / "images"\\nV2_IMAGES.mkdir(parents=True, exist_ok=True)\\n\\nGAUSSIAN_MIN_OPACITY = 0.05\\n\\n# Keep figures readable rather than drawing tens of thousands of glyphs.\\nLOCAL_MAX_GLYPHS = 1400\\nTRACKING_MAX_GLYPHS = 650\\nREG_MAX_GLYPHS = 1100\\nFINAL_MAX_GLYPHS = 2200\\n\\n# Draw ellipse at approximately +/-3 sigma.\\nGAUSSIAN_SIGMA_DIAMETER = 6.0\\n\\n# ------------------------------------------------\\n# 2. Load submap paths\\n# ------------------------------------------------\\nagent_ids = []\\nfor p in sorted(OUTPUT_ROOT.glob("agent_*")):\\n    if p.is_dir() and p.name.split("_")[-1].isdigit():\\n        agent_ids.append(int(p.name.split("_")[-1]))\\n\\nif not agent_ids:\\n    raise RuntimeError(f"No agent_* outputs under {OUTPUT_ROOT}")\\n\\nsubmap_paths = {\\n    aid: sorted((OUTPUT_ROOT / f"agent_{aid}" / "submaps").glob("*.ckpt"))\\n    for aid in agent_ids\\n}\\nfor aid, paths in submap_paths.items():\\n    if not paths:\\n        raise RuntimeError(f"No submap checkpoints for agent {aid}")\\n\\ndef load_submap(aid, idx):\\n    return torch.load(\\n        submap_paths[int(aid)][int(idx)],\\n        map_location="cpu",\\n        weights_only=False,\\n    )\\n\\ndef representative_index(n):\\n    return max(0, min(n - 1, n // 2))\\n\\ndef find_submap_for_frame(aid, frame_id):\\n    for idx, path in enumerate(submap_paths[int(aid)]):\\n        sub = torch.load(path, map_location="cpu", weights_only=False)\\n        s = int(sub["submap_start_frame_id"])\\n        e = int(sub["submap_end_frame_id"])\\n        if s <= int(frame_id) <= e:\\n            return idx, sub\\n        del sub\\n    raise KeyError((aid, frame_id))\\n\\n# ------------------------------------------------\\n# 3. Gaussian data helpers\\n# ------------------------------------------------\\nC0 = 0.28209479177387814\\n\\ndef _np(x):\\n    if torch.is_tensor(x):\\n        return x.detach().cpu().numpy()\\n    return np.asarray(x)\\n\\ndef sigmoid_np(x):\\n    x = np.clip(np.asarray(x, dtype=np.float64), -30.0, 30.0)\\n    return 1.0 / (1.0 + np.exp(-x))\\n\\ndef normalize_quat(q):\\n    q = np.asarray(q, dtype=np.float64)\\n    n = np.linalg.norm(q, axis=1, keepdims=True)\\n    return q / np.clip(n, 1e-12, None)\\n\\ndef quat_wxyz_to_rotmat(q):\\n    # GraphDECO convention: quaternion = [w, x, y, z]\\n    q = normalize_quat(q)\\n    w, x, y, z = q[:,0], q[:,1], q[:,2], q[:,3]\\n\\n    R = np.empty((len(q), 3, 3), dtype=np.float64)\\n    R[:,0,0] = 1 - 2*(y*y + z*z)\\n    R[:,0,1] = 2*(x*y - z*w)\\n    R[:,0,2] = 2*(x*z + y*w)\\n\\n    R[:,1,0] = 2*(x*y + z*w)\\n    R[:,1,1] = 1 - 2*(x*x + z*z)\\n    R[:,1,2] = 2*(y*z - x*w)\\n\\n    R[:,2,0] = 2*(x*z - y*w)\\n    R[:,2,1] = 2*(y*z + x*w)\\n    R[:,2,2] = 1 - 2*(x*x + y*y)\\n    return R\\n\\ndef gaussian_covariance(scales, quat):\\n    scales = np.asarray(scales, dtype=np.float64)\\n    R = quat_wxyz_to_rotmat(quat)\\n    D = np.zeros((len(scales), 3, 3), dtype=np.float64)\\n    D[:,0,0] = scales[:,0] ** 2\\n    D[:,1,1] = scales[:,1] ** 2\\n    D[:,2,2] = scales[:,2] ** 2\\n    return R @ D @ np.transpose(R, (0,2,1))\\n\\ndef submap_gaussians(sub, min_opacity=0.05):\\n    p = sub["gaussian_model_params"]\\n\\n    xyz = _np(p["xyz"]).astype(np.float64)\\n    raw_scaling = _np(p["scaling"]).astype(np.float64)\\n    quat = _np(p["rotation"]).astype(np.float64)\\n    raw_opacity = _np(p["opacity"]).reshape(-1).astype(np.float64)\\n    fdc = _np(p["features_dc"]).astype(np.float64)\\n\\n    if fdc.ndim == 3:\\n        dc = fdc[:,0,:]\\n    elif fdc.ndim == 2:\\n        dc = fdc[:,:3]\\n    else:\\n        raise RuntimeError(f"Unexpected features_dc shape: {fdc.shape}")\\n\\n    scales = np.exp(np.clip(raw_scaling, -20.0, 8.0))\\n    opacity = sigmoid_np(raw_opacity)\\n    rgb = np.clip(0.5 + C0 * dc, 0.0, 1.0)\\n\\n    valid = (\\n        np.isfinite(xyz).all(axis=1)\\n        & np.isfinite(scales).all(axis=1)\\n        & np.isfinite(quat).all(axis=1)\\n        & np.isfinite(opacity)\\n        & np.isfinite(rgb).all(axis=1)\\n        & (opacity >= float(min_opacity))\\n    )\\n\\n    return {\\n        "xyz": xyz[valid],\\n        "scale": scales[valid],\\n        "quat": normalize_quat(quat[valid]),\\n        "opacity": opacity[valid],\\n        "rgb": rgb[valid],\\n    }\\n\\ndef ply_gaussians(path, min_opacity=0.05):\\n    ply = PlyData.read(str(path))\\n    v = ply["vertex"].data\\n    names = set(v.dtype.names)\\n\\n    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float64)\\n\\n    dc = np.stack(\\n        [v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]],\\n        axis=1,\\n    ).astype(np.float64)\\n    rgb = np.clip(0.5 + C0 * dc, 0.0, 1.0)\\n\\n    raw_opacity = np.asarray(v["opacity"], dtype=np.float64)\\n    opacity = sigmoid_np(raw_opacity)\\n\\n    scale_names = [f"scale_{i}" for i in range(3)]\\n    rot_names = [f"rot_{i}" for i in range(4)]\\n\\n    if not all(n in names for n in scale_names + rot_names):\\n        raise RuntimeError("PLY does not contain Gaussian scale/rotation fields.")\\n\\n    raw_scaling = np.stack([v[n] for n in scale_names], axis=1).astype(np.float64)\\n    scales = np.exp(np.clip(raw_scaling, -20.0, 8.0))\\n\\n    quat = np.stack([v[n] for n in rot_names], axis=1).astype(np.float64)\\n    quat = normalize_quat(quat)\\n\\n    valid = (\\n        np.isfinite(xyz).all(axis=1)\\n        & np.isfinite(scales).all(axis=1)\\n        & np.isfinite(quat).all(axis=1)\\n        & np.isfinite(opacity)\\n        & np.isfinite(rgb).all(axis=1)\\n        & (opacity >= float(min_opacity))\\n    )\\n\\n    return {\\n        "xyz": xyz[valid],\\n        "scale": scales[valid],\\n        "quat": quat[valid],\\n        "opacity": opacity[valid],\\n        "rgb": rgb[valid],\\n    }\\n\\ndef apply_rigid_to_gaussians(g, T):\\n    T = np.asarray(T, dtype=np.float64)\\n    A = T[:3,:3]\\n    t = T[:3,3]\\n\\n    out = {\\n        "xyz": g["xyz"] @ A.T + t,\\n        "scale": g["scale"].copy(),\\n        "opacity": g["opacity"].copy(),\\n        "rgb": g["rgb"].copy(),\\n    }\\n\\n    # Rotate Gaussian orientation too.\\n    Rg = quat_wxyz_to_rotmat(g["quat"])\\n    Rnew = A[None,:,:] @ Rg\\n\\n    # Convert matrix back to quaternion is not necessary.\\n    # Keep covariance directly for transformed glyph projection.\\n    cov = gaussian_covariance(g["scale"], g["quat"])\\n    out["covariance"] = A[None,:,:] @ cov @ A.T[None,:,:]\\n    out["quat"] = g["quat"].copy()\\n    return out\\n\\n# ------------------------------------------------\\n# 4. Projection / drawing\\n# ------------------------------------------------\\n# Replica maps are most readable in X-Z top view.\\nP_XZ = np.array(\\n    [[1.0, 0.0, 0.0],\\n     [0.0, 0.0, 1.0]],\\n    dtype=np.float64,\\n)\\n\\ndef _robust_mask_uv(uv, qlo=0.4, qhi=99.6):\\n    lo = np.percentile(uv, qlo, axis=0)\\n    hi = np.percentile(uv, qhi, axis=0)\\n    return (\\n        (uv[:,0] >= lo[0]) & (uv[:,0] <= hi[0])\\n        & (uv[:,1] >= lo[1]) & (uv[:,1] <= hi[1])\\n    )\\n\\ndef _deterministic_sample(idx, max_n):\\n    idx = np.asarray(idx, dtype=np.int64)\\n    if len(idx) <= int(max_n):\\n        return idx\\n    take = np.linspace(0, len(idx)-1, int(max_n)).round().astype(np.int64)\\n    return idx[take]\\n\\ndef project_gaussian_glyphs(g, max_n=1400, subset_idx=None):\\n    xyz = np.asarray(g["xyz"], dtype=np.float64)\\n\\n    if "covariance" in g:\\n        cov = np.asarray(g["covariance"], dtype=np.float64)\\n    else:\\n        cov = gaussian_covariance(g["scale"], g["quat"])\\n\\n    uv = xyz @ P_XZ.T\\n\\n    base_idx = np.arange(len(xyz))\\n    if subset_idx is not None:\\n        subset_idx = np.asarray(subset_idx, dtype=np.int64)\\n        subset_idx = subset_idx[\\n            (subset_idx >= 0) & (subset_idx < len(xyz))\\n        ]\\n        if len(subset_idx):\\n            base_idx = subset_idx\\n\\n    if len(base_idx) == 0:\\n        raise RuntimeError("No Gaussians selected for glyph projection.")\\n\\n    uv0 = uv[base_idx]\\n    keep = _robust_mask_uv(uv0)\\n    base_idx = base_idx[keep]\\n\\n    # Prefer more opaque Gaussians but keep deterministic spatial coverage.\\n    # Sort by x/z first and take evenly across that ordering.\\n    order = np.lexsort((uv[base_idx,1], uv[base_idx,0]))\\n    base_idx = base_idx[order]\\n    idx = _deterministic_sample(base_idx, max_n)\\n\\n    uv = uv[idx]\\n    cov2 = np.einsum("ai,nij,jb->nab", P_XZ, cov[idx], P_XZ.T)\\n\\n    eigval, eigvec = np.linalg.eigh(cov2)\\n    eigval = np.clip(eigval, 1e-12, None)\\n\\n    major = np.sqrt(eigval[:,1]) * GAUSSIAN_SIGMA_DIAMETER\\n    minor = np.sqrt(eigval[:,0]) * GAUSSIAN_SIGMA_DIAMETER\\n\\n    vec = eigvec[:,:,1]\\n    angle = np.degrees(np.arctan2(vec[:,1], vec[:,0]))\\n\\n    # Clamp glyph diameter only for legibility in a paper figure.\\n    ranges = np.ptp(uv, axis=0)\\n    scene_extent = max(float(np.max(ranges)), 1e-3)\\n    min_d = 0.0025 * scene_extent\\n    max_d = 0.0450 * scene_extent\\n\\n    major = np.clip(major, min_d, max_d)\\n    minor = np.clip(minor, min_d, max_d)\\n\\n    rgb = np.clip(g["rgb"][idx], 0.0, 1.0)\\n    opacity = np.clip(g["opacity"][idx], 0.0, 1.0)\\n\\n    return {\\n        "uv": uv,\\n        "major": major,\\n        "minor": minor,\\n        "angle": angle,\\n        "rgb": rgb,\\n        "opacity": opacity,\\n        "idx": idx,\\n    }\\n\\ndef draw_glyphs(\\n    ax,\\n    g,\\n    max_n=1400,\\n    subset_idx=None,\\n    alpha_scale=0.72,\\n    extent=None,\\n):\\n    d = project_gaussian_glyphs(\\n        g,\\n        max_n=max_n,\\n        subset_idx=subset_idx,\\n    )\\n\\n    patches = [\\n        Ellipse(\\n            xy=(u, v),\\n            width=w,\\n            height=h,\\n            angle=a,\\n        )\\n        for (u, v), w, h, a in zip(\\n            d["uv"],\\n            d["major"],\\n            d["minor"],\\n            d["angle"],\\n        )\\n    ]\\n\\n    rgba = np.c_[\\n        d["rgb"],\\n        np.clip(\\n            0.14 + alpha_scale * d["opacity"],\\n            0.12,\\n            0.88,\\n        ),\\n    ]\\n\\n    pc = PatchCollection(\\n        patches,\\n        match_original=False,\\n        linewidth=0.0,\\n        edgecolor="none",\\n    )\\n    pc.set_facecolor(rgba)\\n    ax.add_collection(pc)\\n\\n    if extent is None:\\n        uv = d["uv"]\\n        lo = np.percentile(uv, [0.4], axis=0)[0]\\n        hi = np.percentile(uv, [99.6], axis=0)[0]\\n        span = np.maximum(hi - lo, 1e-3)\\n        pad = 0.07 * max(float(span[0]), float(span[1]))\\n        extent = (\\n            lo[0]-pad,\\n            hi[0]+pad,\\n            lo[1]-pad,\\n            hi[1]+pad,\\n        )\\n\\n    ax.set_xlim(extent[0], extent[1])\\n    ax.set_ylim(extent[2], extent[3])\\n    ax.set_aspect("equal", adjustable="box")\\n    ax.set_axis_off()\\n    return extent\\n\\ndef save_clean(fig, path):\\n    fig.patch.set_facecolor("white")\\n    fig.savefig(\\n        path,\\n        dpi=260,\\n        bbox_inches="tight",\\n        pad_inches=0.02,\\n        facecolor="white",\\n    )\\n    plt.close(fig)\\n\\ndef camera_triangle_2d(c2w, length=0.12, width=0.075):\\n    T = _np(c2w).astype(np.float64)\\n    c = T[:3,3]\\n    f = T[:3,2]\\n\\n    c2 = np.array([c[0], c[2]], dtype=np.float64)\\n    f2 = np.array([f[0], f[2]], dtype=np.float64)\\n    n = np.linalg.norm(f2)\\n    if n < 1e-9:\\n        f2 = np.array([1.0, 0.0])\\n    else:\\n        f2 /= n\\n\\n    r2 = np.array([-f2[1], f2[0]])\\n    tip = c2 + f2 * length\\n    p1 = c2 - f2 * 0.22 * length + r2 * width\\n    p2 = c2 - f2 * 0.22 * length - r2 * width\\n    return np.stack([tip, p1, p2], axis=0)\\n\\ndef draw_camera(ax, c2w, edge="black", lw=1.6, length=0.12):\\n    tri = camera_triangle_2d(\\n        c2w,\\n        length=length,\\n        width=0.58*length,\\n    )\\n    ax.add_patch(\\n        Polygon(\\n            tri,\\n            closed=True,\\n            fill=False,\\n            edgecolor=edge,\\n            linewidth=lw,\\n        )\\n    )\\n\\ndef poses_np(sub):\\n    return np.asarray(\\n        [_np(p) for p in sub["submap_c2ws"]],\\n        dtype=np.float64,\\n    )\\n\\n# ------------------------------------------------\\n# 5. Choose one representative local submap\\n# ------------------------------------------------\\nlocal_agent = min(agent_ids)\\nlocal_idx = representative_index(len(submap_paths[local_agent]))\\nlocal_sub = load_submap(local_agent, local_idx)\\nlocal_g = submap_gaussians(local_sub, GAUSSIAN_MIN_OPACITY)\\nlocal_poses = poses_np(local_sub)\\n\\nprint(\\n    "Representative local submap:",\\n    f"agent={local_agent}",\\n    f"submap={local_idx}",\\n    f"frames={local_sub[\\\'submap_start_frame_id\\\']}..{local_sub[\\\'submap_end_frame_id\\\']}",\\n    f"gaussians={len(local_g[\\\'xyz\\\'])}",\\n)\\n\\n# ------------------------------------------------\\n# 6. Tracking visualization\\n# Short trajectory + local Gaussian neighborhood.\\n# ------------------------------------------------\\npose_n = len(local_poses)\\ncurrent_i = max(0, min(pose_n - 1, pose_n // 2))\\n\\npose_ids = np.unique(\\n    np.linspace(\\n        max(0, current_i - max(3, pose_n//5)),\\n        min(pose_n - 1, current_i + max(3, pose_n//5)),\\n        5,\\n    ).round().astype(int)\\n)\\n\\ncur = local_poses[current_i,:3,3]\\ndist_xz = np.linalg.norm(\\n    local_g["xyz"][:,[0,2]] - cur[[0,2]],\\n    axis=1,\\n)\\n\\nradius = np.percentile(dist_xz, 38.0)\\ntracking_idx = np.where(dist_xz <= max(radius, 0.35))[0]\\n\\nfig, ax = plt.subplots(figsize=(7.4, 4.3))\\nextent = draw_glyphs(\\n    ax,\\n    local_g,\\n    max_n=TRACKING_MAX_GLYPHS,\\n    subset_idx=tracking_idx,\\n    alpha_scale=0.62,\\n)\\n\\ncenters = local_poses[pose_ids,:3,3][:,[0,2]]\\nax.plot(\\n    centers[:,0],\\n    centers[:,1],\\n    linestyle="--",\\n    linewidth=1.4,\\n    color="0.30",\\n    alpha=0.80,\\n)\\n\\nfor j, pi in enumerate(pose_ids):\\n    is_current = (pi == current_i)\\n    draw_camera(\\n        ax,\\n        local_poses[pi],\\n        edge=("black" if not is_current else "crimson"),\\n        lw=(1.5 if not is_current else 2.2),\\n        length=0.09 * max(extent[1]-extent[0], extent[3]-extent[2]),\\n    )\\n\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "02_tracking_pose_local_gaussians.png",\\n)\\n\\n# ------------------------------------------------\\n# 7. Mapping visualization\\n# Three nested spatial coverage stages from the SAME local submap.\\n# ------------------------------------------------\\ncams = local_poses[:,:3,3][:,[0,2]]\\ncam0 = cams[0]\\n\\n# Build coverage radius from growing prefix of the same submap trajectory.\\nprefix_fracs = [0.32, 0.62, 1.00]\\n\\nfig, axes = plt.subplots(1, 3, figsize=(12.6, 4.0))\\n\\nfull_uv = local_g["xyz"][:,[0,2]]\\n\\n# Use one common extent so growth is visually comparable.\\ntmp_d = project_gaussian_glyphs(\\n    local_g,\\n    max_n=LOCAL_MAX_GLYPHS,\\n)\\nuv = tmp_d["uv"]\\nlo = np.percentile(uv, 0.4, axis=0)\\nhi = np.percentile(uv, 99.6, axis=0)\\nspan = np.maximum(hi-lo, 1e-3)\\npad = 0.06 * max(float(span[0]), float(span[1]))\\ncommon_extent = (\\n    lo[0]-pad, hi[0]+pad,\\n    lo[1]-pad, hi[1]+pad,\\n)\\n\\nfor ax, frac in zip(axes, prefix_fracs):\\n    end = max(1, int(round(frac * len(cams))))\\n    used_cams = cams[:end]\\n\\n    # Approximate observed local region by distance to any camera\\n    # in the corresponding trajectory prefix.\\n    # This uses the actual final submap Gaussian set only for visualization.\\n    if len(used_cams) > 30:\\n        used_cams = used_cams[\\n            np.linspace(0, len(used_cams)-1, 30).round().astype(int)\\n        ]\\n\\n    dmin = np.min(\\n        np.linalg.norm(\\n            full_uv[:,None,:] - used_cams[None,:,:],\\n            axis=2,\\n        ),\\n        axis=1,\\n    )\\n\\n    # Same threshold across stages; increasing prefix expands coverage.\\n    threshold = np.percentile(\\n        np.linalg.norm(\\n            full_uv - np.median(full_uv, axis=0),\\n            axis=1,\\n        ),\\n        58.0,\\n    )\\n    idx = np.where(dmin <= max(float(threshold), 0.45))[0]\\n\\n    draw_glyphs(\\n        ax,\\n        local_g,\\n        max_n=max(350, int(LOCAL_MAX_GLYPHS * frac)),\\n        subset_idx=idx,\\n        alpha_scale=0.66,\\n        extent=common_extent,\\n    )\\n\\n    # Only a few key camera poses; do not clutter.\\n    if end > 1:\\n        show = np.unique(\\n            np.linspace(0, end-1, min(3, end)).round().astype(int)\\n        )\\n        for pi in show:\\n            draw_camera(\\n                ax,\\n                local_poses[pi],\\n                edge="0.25",\\n                lw=1.0,\\n                length=0.055 * max(\\n                    common_extent[1]-common_extent[0],\\n                    common_extent[3]-common_extent[2],\\n                ),\\n            )\\n\\nfig.subplots_adjust(wspace=0.02)\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "03_mapping_growth_local_gaussians.png",\\n)\\n\\n# ------------------------------------------------\\n# 8. Finalized local submap\\n# Full local submap, no trajectory.\\n# ------------------------------------------------\\nfig, ax = plt.subplots(figsize=(6.6, 4.6))\\ndraw_glyphs(\\n    ax,\\n    local_g,\\n    max_n=LOCAL_MAX_GLYPHS,\\n    alpha_scale=0.74,\\n)\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "04_finalized_local_submap.png",\\n)\\n\\n# ------------------------------------------------\\n# 9. GS-to-GS registration from a real inter-agent loop\\n# ------------------------------------------------\\nloop_file = OUTPUT_ROOT / "filtered_loops.pkl"\\nif not loop_file.exists():\\n    loop_file = OUTPUT_ROOT / "loops.pkl"\\n\\nwith loop_file.open("rb") as f:\\n    loops = pickle.load(f)\\n\\ninter = [\\n    l for l in loops\\n    if int(l.source_agent_id) != int(l.target_agent_id)\\n]\\n\\nif not inter:\\n    raise RuntimeError("No inter-agent loop available for GS-to-GS visualization.")\\n\\nloop = inter[0]\\nsa = int(loop.source_agent_id)\\nta = int(loop.target_agent_id)\\nsf = int(loop.source_frame_id)\\ntf = int(loop.target_frame_id)\\n\\nsi, ss = find_submap_for_frame(sa, sf)\\nti, ts = find_submap_for_frame(ta, tf)\\n\\nsg = submap_gaussians(ss, GAUSSIAN_MIN_OPACITY)\\ntg = submap_gaussians(ts, GAUSSIAN_MIN_OPACITY)\\n\\n# BEFORE: show source and target independently, side by side.\\nfig, axes = plt.subplots(1, 2, figsize=(10.0, 4.1))\\n\\ndraw_glyphs(\\n    axes[0],\\n    sg,\\n    max_n=REG_MAX_GLYPHS,\\n    alpha_scale=0.76,\\n)\\ndraw_glyphs(\\n    axes[1],\\n    tg,\\n    max_n=REG_MAX_GLYPHS,\\n    alpha_scale=0.76,\\n)\\n\\nfig.subplots_adjust(wspace=0.04)\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "05_gs_to_gs_source_target.png",\\n)\\n\\n# AFTER: actual estimated alignment in target submap frame.\\nsc2w = np.asarray(ss["submap_c2ws"][0], dtype=np.float64)\\ntc2w = np.asarray(ts["submap_c2ws"][0], dtype=np.float64)\\nTcam = np.asarray(loop.transformation, dtype=np.float64)\\n\\nT_align = tc2w @ Tcam @ np.linalg.inv(sc2w)\\nsg_aligned = apply_rigid_to_gaussians(sg, T_align)\\n\\n# Common tight extent based on both aligned source and target.\\nuv_all = np.concatenate(\\n    [\\n        sg_aligned["xyz"][:,[0,2]],\\n        tg["xyz"][:,[0,2]],\\n    ],\\n    axis=0,\\n)\\nmask = _robust_mask_uv(uv_all, 0.7, 99.3)\\nuvr = uv_all[mask]\\nlo = np.percentile(uvr, 0.4, axis=0)\\nhi = np.percentile(uvr, 99.6, axis=0)\\nspan = np.maximum(hi-lo, 1e-3)\\npad = 0.07 * max(float(span[0]), float(span[1]))\\nreg_extent = (\\n    lo[0]-pad, hi[0]+pad,\\n    lo[1]-pad, hi[1]+pad,\\n)\\n\\nfig, ax = plt.subplots(figsize=(6.8, 4.8))\\n\\n# Both maps keep their original Gaussian colors.\\ndraw_glyphs(\\n    ax,\\n    tg,\\n    max_n=REG_MAX_GLYPHS,\\n    alpha_scale=0.54,\\n    extent=reg_extent,\\n)\\ndraw_glyphs(\\n    ax,\\n    sg_aligned,\\n    max_n=REG_MAX_GLYPHS,\\n    alpha_scale=0.48,\\n    extent=reg_extent,\\n)\\n\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "06_gs_to_gs_aligned.png",\\n)\\n\\ngs_info = {\\n    "source_agent": sa,\\n    "source_frame": sf,\\n    "source_submap": si,\\n    "target_agent": ta,\\n    "target_frame": tf,\\n    "target_submap": ti,\\n    "fitness": float(getattr(loop, "fitness", float("nan"))),\\n    "inlier_rmse": float(getattr(loop, "inlier_rmse", float("nan"))),\\n    "registration_method": getattr(loop, "coarse_method_used", None),\\n}\\n\\n# ------------------------------------------------\\n# 10. Final merged Gaussian map\\n# ------------------------------------------------\\nfinal_ply = OUTPUT_ROOT / "merged_refined.ply"\\nif not final_ply.exists():\\n    final_ply = OUTPUT_ROOT / "merged_coarse.ply"\\n\\nif not final_ply.exists():\\n    raise FileNotFoundError("merged_refined.ply / merged_coarse.ply not found.")\\n\\nfg = ply_gaussians(\\n    final_ply,\\n    GAUSSIAN_MIN_OPACITY,\\n)\\n\\nfig, ax = plt.subplots(figsize=(7.3, 5.4))\\ndraw_glyphs(\\n    ax,\\n    fg,\\n    max_n=FINAL_MAX_GLYPHS,\\n    alpha_scale=0.66,\\n)\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "07_final_merged_gaussian_map.png",\\n)\\n\\n# ------------------------------------------------\\n# 11. Review sheet\\n# ------------------------------------------------\\nreview_items = [\\n    ("Tracking", V2_IMAGES / "02_tracking_pose_local_gaussians.png"),\\n    ("Mapping", V2_IMAGES / "03_mapping_growth_local_gaussians.png"),\\n    ("Finalized submap", V2_IMAGES / "04_finalized_local_submap.png"),\\n    ("GS pair", V2_IMAGES / "05_gs_to_gs_source_target.png"),\\n    ("GS aligned", V2_IMAGES / "06_gs_to_gs_aligned.png"),\\n    ("Merged map", V2_IMAGES / "07_final_merged_gaussian_map.png"),\\n]\\n\\nfig, axes = plt.subplots(2, 3, figsize=(14.5, 8.2))\\nfor ax, (title, p) in zip(axes.ravel(), review_items):\\n    ax.imshow(Image.open(p))\\n    ax.set_title(title, fontsize=11)\\n    ax.set_axis_off()\\n\\nfig.tight_layout(pad=0.7)\\nsave_clean(\\n    fig,\\n    V2_IMAGES / "08_pipeline_review_sheet_v2.png",\\n)\\n\\n# ------------------------------------------------\\n# 12. Manifest + ZIP\\n# ------------------------------------------------\\nmanifest = {\\n    "scene": SCENE,\\n    "visualization": "paper_assets_v2_gaussian_ellipsoid_glyphs",\\n    "local_agent": int(local_agent),\\n    "local_submap_index": int(local_idx),\\n    "local_submap_start_frame": int(local_sub["submap_start_frame_id"]),\\n    "local_submap_end_frame": int(local_sub["submap_end_frame_id"]),\\n    "gs_to_gs": gs_info,\\n    "final_map": str(final_ply),\\n    "notes": {\\n        "gaussian_colors": "SH DC natural Gaussian colors",\\n        "gaussian_shape": "scale + rotation projected to X-Z plane",\\n        "glyph_size": "3-sigma diameter, clamped only for figure legibility",\\n        "mapping_growth": "same final local submap visualized against increasing real camera-trajectory prefixes",\\n        "no_slam_rerun": True,\\n    },\\n    "images": [\\n        str(p)\\n        for p in sorted(V2_IMAGES.glob("*.png"))\\n    ],\\n}\\n\\nmanifest_path = V2_ROOT / "pipeline_manifest_v2.json"\\nmanifest_path.write_text(\\n    json.dumps(\\n        manifest,\\n        indent=2,\\n        ensure_ascii=False,\\n    ),\\n    encoding="utf-8",\\n)\\n\\nzip_path = OUTPUT_ROOT / (SCENE + "_pipeline_images_v2.zip")\\nif zip_path.exists():\\n    zip_path.unlink()\\n\\nwith zipfile.ZipFile(\\n    zip_path,\\n    "w",\\n    compression=zipfile.ZIP_DEFLATED,\\n) as zf:\\n    for p in sorted(V2_IMAGES.glob("*.png")):\\n        zf.write(p, arcname=f"images/{p.name}")\\n    zf.write(\\n        manifest_path,\\n        arcname="pipeline_manifest_v2.json",\\n    )\\n\\nprint("=" * 78)\\nprint("PAPER ASSETS V2 READY")\\nprint("=" * 78)\\nfor p in sorted(V2_IMAGES.glob("*.png")):\\n    print(" ", p.name)\\n\\nprint("\\\\nZIP:", zip_path)\\nprint(\\n    "ZIP size:",\\n    f"{zip_path.stat().st_size / 1024**2:.2f} MB"\\n)\\nprint("\\\\nGS-to-GS loop:", gs_info)\'\nSOURCE_PROVENANCE = {\'office_notebook\': \'office0_full_slam_pipeline_py311_paper_assets_v2.ipynb\', \'office_sha256\': \'585a005d48f05c5f83e4b4b3fbc8439704631d8e520bc33565b9dd99c4f3c452\', \'apart_notebook\': \'mv-slam_apart_1_heldout_v331(1).ipynb\', \'apart_sha256\': \'420a7a87b20421e9416af654b4d3becbbbf374015903dd77339d44ce7407c15c\', \'office_cell_sha256\': {\'6\': \'2d0c5e45c5825e2a4f1472687b2d59bdef591756558a4391cd8f29762cd1d5e4\', \'20\': \'068350b8800540e0e4813626dc85a088616df2d2bd92040ad54c82b4bfda07d5\', \'22\': \'765901c20ee206c6080f5f0eac05941b535d8e8c5b25e74776b8c887999fd0b3\', \'24\': \'4c5ed0d054cd8425ab9ddc3af07dfdef3b21e067ed289554bb7b48c4c0f62c5e\', \'28\': \'bde6b2984335beb342d5dcb441f2985fec8f6411bf399aa299efab9b7f4c917b\', \'33\': \'f36d06e658d79e3582ade66bc66f8e8beddb1156da2a6aa9e08929475f9366cc\'}, \'adaptations\': [\'selected scene/input/output/config paths\', \'Apart-0 existing loop035 config\', \'runtime multi_gpu follows device count\', \'monitored native CLI run and artifact validation\']}\nSCENE_FOLDERS={\'office_0\':\'Office-0\',\'apart_0\':\'Apart-0\',\'apart_1\':\'Apart-1\',\'apart_2\':\'Apart-2\'}\nFRAME_LIMITS={\'office_0\':1500,\'apart_0\':2500,\'apart_1\':2500,\'apart_2\':2500}\nREQUIRED_FILES=[\'config.yaml\',\'detected_loops.pkl\',\'loops.pkl\',\'filtered_loops.pkl\',\n    \'registration_metrics.csv\',\'registration_summary.json\',\'merged_coarse.ply\',\'merged_refined.ply\',\n    \'coarse_render_metrics.json\',\'fine_render_metrics.json\',\'walltime_stats.json\']\n\n\ndef write_json(path, value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_name(path.name+\'.tmp\')\n    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str))\n    temp.replace(path)\n\n\ndef sha_file(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()\n\n\ndef run(command, **kwargs):\n    print(\'Run:\',repr(command),flush=True)\n    return subprocess.run(command,check=True,**kwargs)\n\n\ndef validate_options(options):\n    scenes=options[\'scenes\']\n    if not scenes or len(scenes)!=len(set(scenes)) or any(s not in SCENE_FOLDERS for s in scenes):\n        raise ValueError(\'scenes must be a nonempty unique selection of Office-0/Apart-0/1/2.\')\n    if int(options[\'full_run_seed\'])<0:raise ValueError(\'full_run_seed must be nonnegative.\')\n    if Path(options[\'python\']).resolve()!=Path(sys.executable).resolve():\n        raise RuntimeError(\'All native SLAM/preparation must run in the selected PY311 interpreter.\')\n\n\ndef user_namespace(options, scene, data_root):\n    namespace={\'__builtins__\':__builtins__}\n    # Baseline settings are copied, rather than reconstructed with new thresholds.\n    with contextlib.redirect_stdout(io.StringIO()):\n        exec(compile(USER_BASE_CONFIG,\'user_base_configuration\',\'exec\'),namespace)\n    repo=Path(options[\'repo\']);tools=Path(options[\'tools_dir\'])\n    base=repo/\'configs\'/\'ReplicaMultiagent\'/(\'apart_0_gaussian_landmark_loop035.yaml\' if scene==\'apart_0\' else scene+\'.yaml\')\n    if not base.is_file():raise FileNotFoundError(\'Required user scene config: \'+str(base))\n    output=Path(options[\'output_base\'])/scene/f"seed_{int(options[\'full_run_seed\'])}"\n    namespace.update(REPO=repo,PY311=Path(options[\'python\']),VENV=Path(options[\'python\']).parent.parent,\n        CACHE_ROOT=Path(options[\'cache_root\']),SCENE=scene,SCENE_FOLDER=SCENE_FOLDERS[scene],\n        FRAME_LIMIT=FRAME_LIMITS[scene],SEED=int(options[\'full_run_seed\']),DATA_ROOT=Path(data_root),\n        OUTPUT_ROOT=output,PIPELINE_ROOT=output/\'pipeline_assets\',BASE_CONFIG_PATH=base,\n        RUNTIME_CONFIG=tools/\'runtime_configs\'/f"{scene}_seed_{int(options[\'full_run_seed\'])}.yaml",\n        FULL_PARAMS_PATH=tools/\'runtime_configs\'/f"{scene}_params.json")\n    namespace[\'RUNTIME_CONFIG\'].parent.mkdir(parents=True,exist_ok=True)\n    return namespace\n\n\ndef scene_agent_dirs(source, scene):\n    """Accept both SLAM names and the original ReplicaMultiagent part names."""\n    source=Path(source)\n    layouts=[(\'agent_0\',\'agent_1\'),(f\'{scene}_part1\',f\'{scene}_part2\')]\n    for names in layouts:\n        agents={aid:source/name for aid,name in enumerate(names)}\n        if all((agent/\'results\').is_dir() and (agent/\'traj.txt\').is_file()\n               for agent in agents.values()):\n            return agents\n    return None\n\n\ndef scene_layout_error(source, scene):\n    source=Path(source)\n    children=sorted(p.name for p in source.iterdir() if p.is_dir()) if source.is_dir() else []\n    return (f\'{source}: expected agent_0/agent_1 or {scene}_part1/{scene}_part2 \'\n            f\'(each with results/ and traj.txt). Found directories: {children[:12]}\')\n\n\ndef resolve_source(options, scene):\n    supplied=options.get(\'local_scene_roots\',{}).get(scene)\n    if supplied:\n        root=Path(supplied).expanduser()\n        for candidate in [root,root/SCENE_FOLDERS[scene],root/scene]:\n            if scene_agent_dirs(candidate,scene) is not None:return candidate.resolve()\n        raise FileNotFoundError(\'Local scene layout is invalid. \'+scene_layout_error(root,scene))\n    download=Path(options[\'download_root\']);source=download/SCENE_FOLDERS[scene]\n    if scene_agent_dirs(source,scene) is None:\n        download.mkdir(parents=True,exist_ok=True)\n        if shutil.disk_usage(download.parent).free < float(options[\'min_free_disk_gib\'])*2**30:\n            raise RuntimeError(\'Insufficient free disk before dataset download.\')\n        from huggingface_hub import snapshot_download\n        from huggingface_hub.utils import HfHubHTTPError\n        for attempt in range(6):\n            try:\n                snapshot_download(repo_id=\'voviktyl/ReplicaMultiagent\',repo_type=\'dataset\',\n                    allow_patterns=[SCENE_FOLDERS[scene]+\'/*\'],local_dir=str(download),max_workers=4)\n                break\n            except HfHubHTTPError:\n                if attempt==5:raise\n                time.sleep(min(30*(attempt+1),60))\n    if scene_agent_dirs(source,scene) is None:\n        raise FileNotFoundError(\'Downloaded scene layout is invalid. \'+scene_layout_error(source,scene))\n    return source.resolve()\n\ndef stage_scene(source, stage, frame_limit, scene):\n    source=Path(source);stage=Path(stage)\n    agents=scene_agent_dirs(source,scene)\n    if agents is None:raise FileNotFoundError(scene_layout_error(source,scene))\n    stage.mkdir(parents=True,exist_ok=True)\n    identities={}\n    for aid in [0,1]:\n        agent=agents[aid];destination=stage/f\'agent_{aid}\'\n        print(f\'RGB-D agent_{aid} <- {agent.resolve()}\',flush=True)\n        destination.mkdir(parents=True,exist_ok=True)\n        results=destination/\'results\';target=(agent/\'results\').resolve()\n        if results.is_symlink():\n            if results.resolve()!=target:raise RuntimeError(\'Staging points to another dataset; use a new staging root: \'+str(results))\n        elif results.exists():\n            raise RuntimeError(\'Expected staged results symlink, found an existing directory: \'+str(results))\n        else:results.symlink_to(target,target_is_directory=True)\n        trajectory=agent/\'traj.txt\'\n        if not trajectory.is_file():raise FileNotFoundError(trajectory)\n        rgb=sorted(target.glob(\'frame*.jpg\'));depth=sorted(target.glob(\'depth*.png\'))\n        poses=[line for line in trajectory.read_text().splitlines() if line.strip() and not line.lstrip().startswith(\'#\')]\n        if min(len(rgb),len(depth),len(poses))<frame_limit:\n            raise RuntimeError(f\'{agent}: need {frame_limit} RGB/depth/poses; got {len(rgb)}/{len(depth)}/{len(poses)}.\')\n        rgb_ids={int(re.search(r\'(\\d+)\\.jpg$\',p.name)[1]) for p in rgb}\n        depth_ids={int(re.search(r\'(\\d+)\\.png$\',p.name)[1]) for p in depth}\n        if not set(range(frame_limit)).issubset(rgb_ids&depth_ids):\n            raise RuntimeError(\'Missing/mismatched RGB-D frame IDs in the selected full sequence: \'+str(agent))\n        # Same conversion as the user notebook: preserve the original GT bytes.\n        for name in [\'traj.txt\',\'traj_converted.txt\']:\n            dest=destination/name\n            if not dest.is_file() or dest.read_bytes()!=trajectory.read_bytes():shutil.copyfile(trajectory,dest)\n        # Some datasets use another padding width; identify by parsed frame ID.\n        by_id_rgb={int(re.search(r\'(\\d+)\\.jpg$\',p.name)[1]):p for p in rgb}\n        by_id_depth={int(re.search(r\'(\\d+)\\.png$\',p.name)[1]):p for p in depth}\n        boundaries=[by_id_rgb[i] for i in [0,frame_limit-1]]+[by_id_depth[i] for i in [0,frame_limit-1]]\n        identities[str(aid)]={\'source\':str(agent.resolve()),\'rgb_count\':len(rgb),\'depth_count\':len(depth),\'pose_count\':len(poses),\n            \'traj_sha256\':sha_file(trajectory),\'boundary_image_sha256\':{p.name:sha_file(p) for p in boundaries}}\n    extras=[p.name for p in stage.iterdir() if p.is_dir() and not p.name.startswith(\'.\') and p.name not in [\'agent_0\',\'agent_1\']]\n    if extras:raise RuntimeError(\'Staging contains extra agent-like directories: \'+str(extras))\n    return identities\n\n\ndef probe_dataset(repo, config_path, limit):\n    sys.path.insert(0,str(repo))\n    import numpy as np\n    from src.utils.io_utils import load_config\n    from src.entities.datasets import get_dataset\n    config=load_config(str(config_path));root=Path(config[\'data\'][\'input_path\'])\n    rows=[]\n    for aid in [0,1]:\n        data={**config[\'data\'],\'input_path\':str(root/f\'agent_{aid}\')}\n        dataset=get_dataset(config[\'dataset_name\'])({**data,**config[\'cam\'],**config[\'submap\']})\n        if len(dataset)!=limit:raise RuntimeError(f\'Agent {aid} loader returned {len(dataset)} frames, expected {limit}.\')\n        for index in [0,limit-1]:\n            _,rgb,depth,pose=dataset[index]\n            value=lambda x:x.detach().cpu().numpy() if hasattr(x,\'detach\') else np.asarray(x)\n            if value(pose).shape!=(4,4) or not np.isfinite(value(pose)).all():raise ValueError(\'Invalid GT pose at boundary frame.\')\n            if not np.isfinite(value(rgb)).all():raise ValueError(\'Invalid RGB at boundary frame.\')\n            d=value(depth)\n            if not (np.isfinite(d)&(d>0)).any():raise ValueError(\'No valid boundary depth.\')\n        rows.append({\'agent\':aid,\'frames\':len(dataset),\'checked_frames\':[0,limit-1]})\n    return rows\n\n\ndef prepare_jobs(options):\n    validate_options(options)\n    import torch,yaml\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA is not visible in PY311.\')\n    jobs=[];repo=Path(options[\'repo\'])\n    source_hashes={str(p.relative_to(repo)):sha_file(p) for p in sorted((repo/\'src\').rglob(\'*.py\'))}\n    source_hashes[\'run_slam.py\']=sha_file(repo/\'run_slam.py\')\n    head=subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip()\n    for scene in options[\'scenes\']:\n        source=resolve_source(options,scene);stage=Path(options[\'staging_root\'])/scene\n        input_identity=stage_scene(source,stage,FRAME_LIMITS[scene],scene)\n        namespace=user_namespace(options,scene,stage)\n        runtime_cell=USER_RUNTIME_CONFIG_CELL.replace("Path(\'/kaggle/working/scene_pipeline_params.json\')",\'FULL_PARAMS_PATH\')\n        exec(compile(runtime_cell,\'user_runtime_configuration\',\'exec\'),namespace)\n        runtime=Path(namespace[\'RUNTIME_CONFIG\'])\n        config=yaml.safe_load(runtime.read_text())\n        config[\'multi_gpu\']=torch.cuda.device_count()>=2\n        runtime.write_text(yaml.safe_dump(config,sort_keys=False))\n        threshold=float(config[\'loop_detection\'][\'feature_dist_threshold\'])\n        if scene==\'apart_0\' and abs(threshold-.35)>1e-12:raise RuntimeError(\'Apart-0 must keep the user loop035 configuration.\')\n        if scene in [\'apart_1\',\'apart_2\'] and abs(threshold-.1)>1e-12:raise RuntimeError(\'Apart-1/2 frozen loop threshold must remain 0.1.\')\n        dataset_probe=probe_dataset(repo,runtime,FRAME_LIMITS[scene])\n        identity={\'config\':config,\'input_identity\':input_identity,\'repo_head\':head,\'source_hashes\':source_hashes,\n            \'python\':sys.version,\'torch\':torch.__version__,\'torch_cuda\':torch.version.cuda,\'provenance\':SOURCE_PROVENANCE}\n        job={\'scene\':scene,\'full_run_seed\':int(options[\'full_run_seed\']),\'frame_limit\':FRAME_LIMITS[scene],\n            \'input\':str(stage),\'source_input\':str(source),\'output\':str(namespace[\'OUTPUT_ROOT\']),\n            \'config\':str(runtime),\'gpu_count\':torch.cuda.device_count(),\'registration_method\':namespace[\'REGISTRATION_METHOD\'],\n            \'run_fingerprint\':digest(identity),\'identity\':identity,\'dataset_probe\':dataset_probe}\n        jobs.append(job)\n    write_json(options[\'jobs_file\'],jobs)\n    print(\'Prepared native full SLAM:\',[(j[\'scene\'],j[\'frame_limit\']) for j in jobs],flush=True)\n    if options.get(\'run_full_slam\',True):\n        # Download weights once before the long frontend run, using the user\'s cell.\n        exec(compile(USER_DINO_CACHE_CELL,\'user_dino_precache\',\'exec\'),namespace)\n    return jobs\n\n\ndef required_output_status(output):\n    output=Path(output)\n    missing=[name for name in REQUIRED_FILES if not (output/name).is_file() or (output/name).stat().st_size==0]\n    for aid in [0,1]:\n        if not list((output/f\'agent_{aid}\'/\'submaps\').glob(\'*.ckpt\')):missing.append(f\'agent_{aid}/submaps/*.ckpt\')\n    return missing\n\n\ndef file_inventory(output):\n    output=Path(output);paths=[output/name for name in REQUIRED_FILES]\n    for aid in [0,1]:paths.extend(sorted((output/f\'agent_{aid}\'/\'submaps\').glob(\'*.ckpt\')))\n    return {str(p.relative_to(output)):{\'bytes\':p.stat().st_size,\'mtime_ns\':p.stat().st_mtime_ns} for p in paths}\n\n\ndef completed_job(job):\n    output=Path(job[\'output\']);marker=output/\'RUN_COMPLETE.json\'\n    if not marker.is_file():return False\n    data=json.loads(marker.read_text())\n    if data.get(\'complete\') is not True or data.get(\'run_fingerprint\')!=job[\'run_fingerprint\']:\n        raise RuntimeError(\'Completed output belongs to a different config/input/code. Use a new FULL_SLAM_OUTPUT_BASE: \'+str(output))\n    missing=required_output_status(output)\n    if missing:raise RuntimeError(\'Completed output lost required artifacts: \'+str(missing))\n    if data[\'file_inventory\']!=file_inventory(output):raise RuntimeError(\'Completed native files changed; inspect \'+str(output))\n    return True\n\n\ndef verify_terminal_submaps(job):\n    import torch\n    rows=[]\n    for aid in [0,1]:\n        files=sorted((Path(job[\'output\'])/f\'agent_{aid}\'/\'submaps\').glob(\'*.ckpt\'))\n        first=torch.load(files[0],map_location=\'cpu\',weights_only=False)\n        last=torch.load(files[-1],map_location=\'cpu\',weights_only=False)\n        stop=int(last[\'submap_start_frame_id\'])+len(last[\'submap_c2ws\'])\n        if int(first[\'submap_start_frame_id\'])!=0 or stop!=int(job[\'frame_limit\']):\n            raise RuntimeError(f\'Agent {aid} submap coverage ends at {stop}; expected all {job["frame_limit"]} frames. Do not benchmark incomplete frontend outputs.\')\n        rows.append({\'agent\':aid,\'submaps\':len(files),\'first_frame\':0,\'last_pose_frame\':stop-1})\n    return rows\n\n\ndef verify_gaussian_ply(path):\n    count=None;properties=[];vertex=False;ended=False\n    with Path(path).open(\'rb\') as handle:\n        if handle.readline().strip()!=b\'ply\':raise ValueError(\'Not a PLY map: \'+str(path))\n        for _ in range(1000):\n            line=handle.readline().decode(\'ascii\').strip()\n            if line.startswith(\'element \'):\n                fields=line.split();vertex=fields[1]==\'vertex\'\n                if vertex:count=int(fields[2])\n            elif vertex and line.startswith(\'property \'):properties.append(line.split()[-1])\n            elif line==\'end_header\':ended=True;break\n    required={\'x\',\'y\',\'z\',\'opacity\',\'f_dc_0\',\'f_dc_1\',\'f_dc_2\',\'scale_0\',\'scale_1\',\'scale_2\',\'rot_0\',\'rot_1\',\'rot_2\',\'rot_3\'}\n    if not ended or not count or not required.issubset(properties):\n        raise ValueError(\'Expected nonempty Gaussian map with SH/opacity/scale/rotation: \'+str(path))\n    return count\n\n\ndef command_for_job(options,job):\n    command=[options[\'python\'],str(Path(options[\'repo\'])/\'run_slam.py\'),job[\'config\'],\n        \'--output_path\',job[\'output\'],\'--registration_method\',job[\'registration_method\'],\n        \'--seed\',str(job[\'full_run_seed\'])]\n    if job[\'gpu_count\']>=2:command.append(\'--multi_gpu\')\n    return command\n\n\ndef monitored_run(command,output,options):\n    output=Path(output);log=output/\'full_run.log\'\n    env=os.environ.copy();env.update(HF_HOME=options[\'cache_root\'],WANDB_MODE=\'disabled\',PYTHONUNBUFFERED=\'1\')\n    begin=time.monotonic();heartbeat=begin;sample_time=0.;samples=[]\n    print(\'FULL SLAM:\',repr(command),\'| log:\',log,flush=True)\n    with log.open(\'a\') as saved:\n        saved.write(\'\\nCOMMAND: \'+repr(command)+\'\\n\');saved.flush()\n        with subprocess.Popen(command,cwd=options[\'repo\'],env=env,stdout=saved,stderr=subprocess.STDOUT,start_new_session=True) as process:\n            try:\n                while process.poll() is None:\n                    now=time.monotonic()\n                    if now-begin>float(options[\'max_run_hours\'])*3600:raise TimeoutError(\'Native full SLAM exceeds configured hours; see \'+str(log))\n                    if now-heartbeat>=60:\n                        print(output.parent.name,\'full SLAM running:\',round((now-begin)/60,1),\'minutes |\',log,flush=True);heartbeat=now\n                    if now-sample_time>=float(options[\'gpu_sample_seconds\']):\n                        try:\n                            stats=subprocess.run([\'nvidia-smi\',\'--query-gpu=index,memory.used,utilization.gpu\',\'--format=csv,nounits,noheader\'],capture_output=True,text=True,timeout=15)\n                            if stats.returncode==0:\n                                for line in stats.stdout.splitlines():\n                                    values=[v.strip() for v in line.split(\',\')]\n                                    if len(values)!=3:continue\n                                    try:gpu,mem,util=int(values[0]),float(values[1]),float(values[2])\n                                    except ValueError:continue\n                                    samples.append({\'elapsed_s\':now-begin,\'gpu\':gpu,\'device_memory_used_mib\':mem,\'utilization_percent\':util})\n                        except (OSError,subprocess.TimeoutExpired):pass\n                        sample_time=now\n                    time.sleep(2)\n            finally:\n                if process.poll() is None:\n                    os.killpg(process.pid,signal.SIGTERM)\n                    try:process.wait(timeout=15)\n                    except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()\n                if samples:\n                    with (output/\'resource_samples.csv\').open(\'w\',newline=\'\') as handle:\n                        writer=csv.DictWriter(handle,fieldnames=list(samples[0]));writer.writeheader();writer.writerows(samples)\n                    write_json(output/\'resource_summary.json\',{\'scope\':\'whole_device_all_processes\',\'exact_process_peak\':False,\n                        \'sample_seconds\':options[\'gpu_sample_seconds\'],\'sampled_peak_device_mib\':\n                        {str(g):max(r[\'device_memory_used_mib\'] for r in samples if r[\'gpu\']==g) for g in {r[\'gpu\'] for r in samples}}})\n    if process.returncode:\n        with log.open(errors=\'replace\') as handle:print(\'\'.join(collections.deque(handle,maxlen=120)),flush=True)\n        raise subprocess.CalledProcessError(process.returncode,command)\n    return time.monotonic()-begin\n\n\ndef run_job(options,job):\n    if completed_job(job):\n        print(\'Skip completed full SLAM:\',job[\'scene\'],flush=True);return \'reused\'\n    if not options.get(\'run_full_slam\',True):raise RuntimeError(\'RUN_FULL_SLAM=False but output is incomplete: \'+job[\'output\'])\n    output=Path(job[\'output\'])\n    if output.exists() and any(output.iterdir()):\n        # Native frontend optimizer has no exact resume interface in the source notebook.\n        # Preserve the interrupted run, then restart it; never mix old/new submaps.\n        backup=output.with_name(output.name+\'_partial_\'+str(time.time_ns()))\n        output.rename(backup);print(\'Preserved incomplete native output:\',backup,flush=True)\n    output.mkdir(parents=True,exist_ok=True)\n    write_json(output/\'RUN_STARTED.json\',{\'run_fingerprint\':job[\'run_fingerprint\'],\'scene\':job[\'scene\'],\n        \'full_run_seed\':job[\'full_run_seed\'],\'started_unix\':time.time(),\'native_frontend_resume\':False})\n    begin=time.monotonic()\n    try:\n        wall=monitored_run(command_for_job(options,job),output,options)\n        missing=required_output_status(output)\n        if missing:raise RuntimeError(\'Native CLI exited successfully but full outputs are incomplete: \'+str(missing))\n        map_counts={stage:verify_gaussian_ply(output/f\'merged_{stage}.ply\') for stage in [\'coarse\',\'refined\']}\n        coverage=verify_terminal_submaps(job)\n        for name in [\'config.yaml\']: \n            if not (output/name).read_text().strip():raise RuntimeError(\'Empty native config.\')\n        for name in [\'registration_summary.json\',\'coarse_render_metrics.json\',\'fine_render_metrics.json\',\'walltime_stats.json\']:\n            json.loads((output/name).read_text())\n        write_json(output/\'RUN_COMPLETE.json\',{\'complete\':True,\'scene\':job[\'scene\'],\'full_run_seed\':job[\'full_run_seed\'],\n            \'run_fingerprint\':job[\'run_fingerprint\'],\'full_slam_wall_s\':wall,\'terminal_submaps\':coverage,\n            \'gaussian_counts\':map_counts,\n            \'file_inventory\':file_inventory(output),\'full_slam_native_method\':job[\'registration_method\'],\n            \'descriptor_ablation_is_subsequent\':True,\'identity\':job[\'identity\']})\n        print(\'Full SLAM complete:\',job[\'scene\'],\'->\',output,flush=True)\n        return \'completed\'\n    except BaseException as error:\n        write_json(output/\'RUN_FAILED.json\',{\'run_fingerprint\':job[\'run_fingerprint\'],\'error\':repr(error),\'wall_s\':time.monotonic()-begin})\n        raise\n\n\ndef run_jobs(options):\n    validate_options(options)\n    jobs=json.loads(Path(options[\'jobs_file\']).read_text());states=[]\n    for job in jobs:states.append({\'scene\':job[\'scene\'],\'status\':run_job(options,job)})\n    write_json(Path(options[\'tools_dir\'])/\'full_slam_status.json\',states)\n    return states\n\n\ndef export_native(options):\n    validate_options(options)\n    import pandas as pd\n    jobs=json.loads(Path(options[\'jobs_file\']).read_text());summary=[];evaluations=[]\n    repo=Path(options[\'repo\']);tools=Path(options[\'tools_dir\']);sys.path.insert(0,str(repo))\n    evaluator_path=tools/\'native_loop_evaluator.py\';evaluator_path.write_text(USER_LOOP_EVALUATOR)\n    spec=importlib.util.spec_from_file_location(\'native_loop_evaluator\',evaluator_path)\n    evaluator=importlib.util.module_from_spec(spec);spec.loader.exec_module(evaluator)\n    for job in jobs:\n        if not completed_job(job):raise RuntimeError(\'Export needs a completed native SLAM: \'+job[\'scene\'])\n        output=Path(job[\'output\']);context={\'scene\':job[\'scene\'],\'full_run_seed\':job[\'full_run_seed\'],\n            \'method\':\'shared_full_slam_gaussian_landmark\',\'descriptor_specific_map\':False,\n            \'native_metric_units_preserved\':True,\'native_files\':{},\'native_ate\':{}}\n        for name in [\'registration_summary.json\',\'coarse_render_metrics.json\',\'fine_render_metrics.json\',\'walltime_stats.json\']:\n            context[\'native_files\'][name]=json.loads((output/name).read_text())\n        for aid in [0,1]:\n            ate=output/f\'agent_{aid}\'/\'traj_eval\'/\'ate.json\'\n            if ate.is_file():context[\'native_ate\'][str(aid)]=json.loads(ate.read_text())\n        write_json(output/\'native_system_metrics_context.json\',context)\n        row={\'scene\':job[\'scene\'],\'full_run_seed\':job[\'full_run_seed\'],\'output\':str(output),\n            \'coarse_map_mib\':(output/\'merged_coarse.ply\').stat().st_size/2**20,\n            \'refined_map_mib\':(output/\'merged_refined.ply\').stat().st_size/2**20,\n            \'full_slam_wall_s\':json.loads((output/\'RUN_COMPLETE.json\').read_text())[\'full_slam_wall_s\'],\n            \'native_ate_agents_present\':len(context[\'native_ate\'])}\n        for aid in [0,1]:row[f\'submaps_agent{aid}\']=len(list((output/f\'agent_{aid}\'/\'submaps\').glob(\'*.ckpt\')))\n        for stage in [\'coarse\',\'fine\']:\n            metrics=context[\'native_files\'][stage+\'_render_metrics.json\']\n            for key,value in metrics.items():\n                if isinstance(value,(int,float)):row[f\'native_{stage}_{key}\']=value\n        summary.append(row)\n        status={\'scene\':job[\'scene\'],\'native_loop_eval\':{},\'paper_assets\':{},\'native_ate_agents_present\':len(context[\'native_ate\'])}\n        for filename in [\'loops.pkl\',\'filtered_loops.pkl\']:\n            try:\n                rows=evaluator.evaluate(str(output/filename),str(output/\'config.yaml\'),label=job[\'scene\']+\':\'+filename)\n                pd.DataFrame(rows).to_csv(output/(\'native_\'+filename.removesuffix(\'.pkl\')+\'_gt_cm.csv\'),index=False)\n                status[\'native_loop_eval\'][filename]=\'complete\'\n            except Exception as error:\n                status[\'native_loop_eval\'][filename]={\'error\':repr(error)}\n                print(\'Native loop diagnostic error:\',job[\'scene\'],filename,repr(error),flush=True)\n        if options.get(\'export_paper_assets\',True):\n            env=os.environ.copy();env.update(FULL_SLAM_OUTPUT=str(output),FULL_SLAM_SCENE=job[\'scene\'],\n                FULL_SLAM_REPO=str(repo),HF_HOME=options[\'cache_root\'],MPLBACKEND=\'Agg\')\n            for version,source in [(\'v1\',USER_PAPER_ASSETS_V1),(\'v2\',USER_PAPER_ASSETS_V2)]:\n                path=tools/f\'export_user_pipeline_assets_{version}.py\';path.write_text(source)\n                try:\n                    # Keep paper-export logs distinct from the native SLAM log.\n                    log=output/f\'paper_assets_{version}.log\'\n                    with log.open(\'w\') as handle:\n                        process=subprocess.run([options[\'python\'],str(path)],cwd=repo,env=env,stdout=handle,stderr=subprocess.STDOUT)\n                    if process.returncode:raise RuntimeError(f\'Paper assets {version} exited {process.returncode}; see {log}\')\n                    status[\'paper_assets\'][version]=\'complete\'\n                except Exception as error:\n                    status[\'paper_assets\'][version]={\'error\':repr(error)}\n                    print(\'Paper asset error:\',job[\'scene\'],version,repr(error),flush=True)\n                    if options.get(\'strict_exports\',False):raise\n        write_json(output/\'native_evaluation_status.json\',status);evaluations.append(status)\n    pd.DataFrame(summary).to_csv(Path(options[\'output_base\'])/\'native_full_slam_summary.csv\',index=False)\n    write_json(Path(options[\'output_base\'])/\'native_evaluation_status.json\',evaluations)\n    return evaluations\n\n\ndef handoff(options):\n    validate_options(options)\n    jobs=json.loads(Path(options[\'jobs_file\']).read_text());runs={};inputs={}\n    for job in jobs:\n        if not completed_job(job):raise RuntimeError(\'Single/multi cannot use incomplete SLAM: \'+job[\'scene\'])\n        runs[job[\'scene\']]=job[\'output\'];inputs[job[\'scene\']]=job[\'input\']\n    value={\'bench_runs\':runs,\'bench_data_root_overrides\':inputs,\'bench_scenes\':[j[\'scene\'] for j in jobs],\n        \'full_run_seed\':int(options[\'full_run_seed\']),\'registration_seeds\':options[\'registration_seeds\'],\n        \'protocol\':\'one full native SLAM per scene -> shared frozen submaps -> single/multi registration -> same ICP\'}\n    write_json(Path(options[\'tools_dir\'])/\'single_multi_handoff.json\',value)\n    return value\n\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'phase\',choices=[\'prepare\',\'slam\',\'export\',\'handoff\']);parser.add_argument(\'params\')\n    args=parser.parse_args();options=json.loads(Path(args.params).read_text())\n    os.environ[\'HF_HOME\']=options[\'cache_root\'];os.environ[\'WANDB_MODE\']=\'disabled\'\n    {\'prepare\':prepare_jobs,\'slam\':run_jobs,\'export\':export_native,\'handoff\':handoff}[args.phase](options)\n\n\nif __name__==\'__main__\':main()\n'
FULL_SLAM_RUNNER = TOOLS_DIR/'full_slam_runner.py'
FULL_SLAM_RUNNER.write_text(FULL_SLAM_RUNNER_SOURCE)
FULL_SLAM_JOBS = TOOLS_DIR/'full_slam_jobs.json'
FULL_SLAM_PARAMS = TOOLS_DIR/'full_slam_params.json'
FULL_SLAM_PARAMS.write_text(json.dumps({'repo':str(REPO),'python':str(PY311),'cache_root':str(CACHE_ROOT),
    'tools_dir':str(TOOLS_DIR),'scenes':FULL_SLAM_SCENES,'full_run_seed':FULL_SLAM_SEED,
    'download_root':str(FULL_SLAM_DOWNLOAD_ROOT),'staging_root':str(FULL_SLAM_STAGING_ROOT),
    'output_base':str(FULL_SLAM_OUTPUT_BASE),'jobs_file':str(FULL_SLAM_JOBS),
    'local_scene_roots':FULL_SLAM_LOCAL_SCENE_ROOTS,'run_full_slam':RUN_FULL_SLAM,
    'registration_seeds':BENCH_SEEDS,'max_run_hours':FULL_SLAM_MAX_HOURS,
    'min_free_disk_gib':FULL_SLAM_MIN_FREE_DISK_GIB,'gpu_sample_seconds':FULL_SLAM_GPU_SAMPLE_SECONDS,
    'export_paper_assets':FULL_SLAM_EXPORT_PAPER_ASSETS,'strict_exports':FULL_SLAM_STRICT_EXPORTS},indent=2))
subprocess.run([str(PY311),str(FULL_SLAM_RUNNER),'prepare',str(FULL_SLAM_PARAMS)],cwd=REPO,check=True)
print('Jobs:',FULL_SLAM_JOBS)


## 8. Chạy full SLAM

In [ ]:
# Full tracking/mapping/frontend/backend/map refinement từ CLI của repo bạn.
subprocess.run([str(PY311),str(FULL_SLAM_RUNNER),'slam',str(FULL_SLAM_PARAMS)],cwd=REPO,check=True)
print('Full SLAM finished. Outputs:',FULL_SLAM_OUTPUT_BASE)


## 9. Metric và ảnh pipeline

In [ ]:
# Giữ metric native và chạy lại các exporter paper-assets V1/V2 từ bản của bạn.
subprocess.run([str(PY311),str(FULL_SLAM_RUNNER),'export',str(FULL_SLAM_PARAMS)],cwd=REPO,check=True)
print('Native summary:',FULL_SLAM_OUTPUT_BASE/'native_full_slam_summary.csv')
print('Export audit:',FULL_SLAM_OUTPUT_BASE/'native_evaluation_status.json')


## 10. Kết nối output với benchmark

In [ ]:
# Không cần có kết quả trước hoặc tự gán đường dẫn BENCH_RUNS.
subprocess.run([str(PY311),str(FULL_SLAM_RUNNER),'handoff',str(FULL_SLAM_PARAMS)],cwd=REPO,check=True)
handoff=json.loads((TOOLS_DIR/'single_multi_handoff.json').read_text())
BENCH_RUNS=handoff['bench_runs']
BENCH_DATA_ROOT_OVERRIDES=handoff['bench_data_root_overrides']
BENCH_SCENES=handoff['bench_scenes']
BENCH_PYTHON=str(PY311)
print('Automatic SLAM -> benchmark:',json.dumps(BENCH_RUNS,indent=2))


## 11. So sánh coarse Single-/Multi-view V3.3.1

In [ ]:
# ================================================================
# Single vs multi-view coarse registration + fixed ablations, current V3.3.1
# Uses the completed Office-0 SLAM output; no second SLAM run.
# The packaged cell contains unchanged helpers from the recent notebook.
# ================================================================
from pathlib import Path
import os, sys, json, time, math, hashlib, pickle, copy, inspect
import threading, subprocess, contextlib, io, zipfile, gc
from collections import OrderedDict, defaultdict
import numpy as np
import pandas as pd

# EDIT ONLY PATHS / RUN SELECTION BEFORE THE FIRST BENCHMARK.
BENCH_REPO = globals().get('BENCH_REPO', Path(globals().get('REPO', '/kaggle/working/3DGS-SLAM-TURTLEBOT4')))
BENCH_PYTHON = globals().get('BENCH_PYTHON', str(globals().get('PY311',sys.executable)))
BENCH_RUNS = globals().get('BENCH_RUNS', {})
BENCH_SEARCH_ROOTS = globals().get('BENCH_SEARCH_ROOTS', [Path('/kaggle/working/gslam_outputs')])
BENCH_SCENES = globals().get('BENCH_SCENES', ['office_0', 'apart_0', 'apart_1', 'apart_2'])
BENCH_DATA_ROOT_OVERRIDES = globals().get('BENCH_DATA_ROOT_OVERRIDES', {})
BENCH_SEEDS = globals().get('BENCH_SEEDS', [0, 1, 2])
BENCH_ICP_SEEDS = globals().get('BENCH_ICP_SEEDS', [0, 1, 2])
BENCH_K_VALUES = globals().get('BENCH_K_VALUES', [256, 512, 1024])
BENCH_PRIMARY_K = globals().get('BENCH_PRIMARY_K', 512)
BENCH_EXPECTED_TOTAL_LOOPS = globals().get('BENCH_EXPECTED_TOTAL_LOOPS', 21)
BENCH_SCOPE = globals().get('BENCH_SCOPE', 'detected')
BENCH_OUTPUT = globals().get('BENCH_OUTPUT', Path('/kaggle/working/single_multi_v331_benchmark'))
BENCH_IMPORT_RESULTS = globals().get('BENCH_IMPORT_RESULTS', [])
BENCH_BUILD_MISSING_CACHE = globals().get('BENCH_BUILD_MISSING_CACHE', True)
BENCH_RETRY_ERRORS = globals().get('BENCH_RETRY_ERRORS', True)
BENCH_STRICT_INPUT_HASH = globals().get('BENCH_STRICT_INPUT_HASH', True)
BENCH_RUN_ABLATIONS = globals().get('BENCH_RUN_ABLATIONS', True)
BENCH_RUN_FPFH = globals().get('BENCH_RUN_FPFH', True)
BENCH_EXPORT_PAIR_FIGURES = globals().get('BENCH_EXPORT_PAIR_FIGURES', True)
BENCH_GPU_DEVICE = globals().get('BENCH_GPU_DEVICE', 0)
BENCH_GPU_POLL_SECONDS = globals().get('BENCH_GPU_POLL_SECONDS', 0.2)
BENCH_VERBOSE = globals().get('BENCH_VERBOSE', False)
BENCH_MARGIN_POSITIVE = globals().get('BENCH_MARGIN_POSITIVE', 0.001)
# NEW experimental shape ablation, not a claimed existing V3.3.1 component.
BENCH_SCALE_LOG_THRESHOLD = globals().get('BENCH_SCALE_LOG_THRESHOLD', 0.35)
BENCH_AXIS_COS_THRESHOLD = globals().get('BENCH_AXIS_COS_THRESHOLD', 0.15)
BENCH_AXIS_ANISOTROPY_MIN = globals().get('BENCH_AXIS_ANISOTROPY_MIN', 1.2)
BENCH_QUATERNION_ORDER = globals().get('BENCH_QUATERNION_ORDER', 'wxyz')
BENCH_FPFH_VOXEL_M = globals().get('BENCH_FPFH_VOXEL_M', 0.05)
BENCH_FPFH_MAX_ITERS = globals().get('BENCH_FPFH_MAX_ITERS', 100000)
BENCH_FPFH_CONFIDENCE = globals().get('BENCH_FPFH_CONFIDENCE', 0.999)
BENCH_EXECUTE = globals().get('BENCH_EXECUTE', True)

_SB_HELPERS = '# Original notebook cell 46\ndef mv_find_submap_index(agent_id, frame_id):\n    agent_id, frame_id = int(agent_id), int(frame_id)\n    for i, sub in enumerate(mv_submaps[agent_id]):\n        if int(sub[\'submap_start_frame_id\']) <= frame_id <= int(sub[\'submap_end_frame_id\']):\n            return i\n    raise KeyError(f\'Không tìm thấy submap chứa agent={agent_id}, frame={frame_id}\')\n\ndef mv_frame_c2w(submap, frame_id):\n    offset = int(frame_id) - int(submap[\'submap_start_frame_id\'])\n    poses = submap[\'submap_c2ws\']\n    if offset < 0 or offset >= len(poses):\n        raise IndexError((frame_id, submap[\'submap_start_frame_id\'], submap[\'submap_end_frame_id\']))\n    pose = poses[offset]\n    if torch.is_tensor(pose):\n        pose = pose.detach().cpu().numpy()\n    return np.asarray(pose, dtype=np.float64)\n\n# Original notebook cell 47\ndef mv_to_pil(image):\n    if isinstance(image, Image.Image):\n        return image.convert(\'RGB\')\n    if torch.is_tensor(image):\n        arr = image.detach().cpu().numpy()\n    else:\n        arr = np.asarray(image)\n    arr = np.asarray(arr)\n    if arr.ndim != 3:\n        raise ValueError(f\'RGB image phải 3D, nhận {arr.shape}\')\n    if arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):\n        arr = np.moveaxis(arr, 0, -1)\n    if arr.shape[-1] == 1:\n        arr = np.repeat(arr, 3, axis=-1)\n    arr = arr[..., :3]\n    if np.issubdtype(arr.dtype, np.floating):\n        if np.nanmax(arr) <= 1.5:\n            arr = arr * 255.0\n    arr = np.nan_to_num(arr, nan=0.0, posinf=255.0, neginf=0.0)\n    arr = np.clip(arr, 0, 255).astype(np.uint8)\n    return Image.fromarray(arr, mode=\'RGB\')\n\ndef mv_depth_tensor(depth):\n    if torch.is_tensor(depth):\n        d = depth.detach().to(MV_DEVICE, dtype=torch.float32)\n        while d.ndim > 2 and 1 in d.shape:\n            d = d.squeeze()\n        if d.ndim == 3:\n            d = d[0] if d.shape[0] <= 4 else d[..., 0]\n        if d.ndim != 2:\n            raise ValueError(f\'Depth phải 2D sau squeeze, nhận {tuple(d.shape)}\')\n        return d\n    d = np.asarray(depth)\n    while d.ndim > 2 and 1 in d.shape:\n        d = np.squeeze(d)\n    if d.ndim == 3:\n        d = d[..., 0] if d.shape[-1] <= 4 else d[0]\n    if d.ndim != 2:\n        raise ValueError(f\'Depth phải 2D sau squeeze, nhận {d.shape}\')\n    return torch.as_tensor(d, device=MV_DEVICE, dtype=torch.float32)\n\n@torch.inference_mode()\ndef mv_extract_feature_map(image):\n    pil = mv_to_pil(image)\n    orig_w, orig_h = pil.size\n    tokens, (gh, gw), (patch_h_px, patch_w_px), meta = (\n        mv_feature_extractor.extract_patch_tokens(pil, return_metadata=True)\n    )\n    fmap = tokens.reshape(gh, gw, -1)\n    # tokens từ repo đã L2-normalize; normalize lại để chống sai số số học.\n    fmap = F.normalize(fmap.float(), dim=-1)\n    return fmap, {\n        \'orig_h\': int(orig_h), \'orig_w\': int(orig_w),\n        \'gh\': int(gh), \'gw\': int(gw),\n        \'patch_h_px\': float(patch_h_px), \'patch_w_px\': float(patch_w_px),\n        **meta,\n    }\n\n@torch.inference_mode()\ndef mv_bilinear_sample(fmap, meta, u, v):\n    gh, gw = int(meta[\'gh\']), int(meta[\'gw\'])\n    patch_h_px = float(meta[\'patch_h_px\'])\n    patch_w_px = float(meta[\'patch_w_px\'])\n\n    # Original pixel coordinate -> continuous patch-center coordinate.\n    gx = u / patch_w_px - 0.5\n    gy = v / patch_h_px - 0.5\n    xnorm = (2.0 * gx / (gw - 1) - 1.0) if gw > 1 else torch.zeros_like(gx)\n    ynorm = (2.0 * gy / (gh - 1) - 1.0) if gh > 1 else torch.zeros_like(gy)\n    grid = torch.stack([xnorm, ynorm], dim=-1).view(1, -1, 1, 2)\n    feat_chw = fmap.permute(2, 0, 1).unsqueeze(0)\n    sampled = F.grid_sample(\n        feat_chw, grid, mode=\'bilinear\', padding_mode=\'zeros\', align_corners=True\n    )[0, :, :, 0].T\n    return F.normalize(sampled, dim=-1)\n\n# Original notebook cell 48\ndef mv_restore_gaussians(submap_cpu):\n    # Use the same restore path as the repository.\n    model = load_3dgs(submap_cpu)\n    xyz = model.get_xyz().detach().float()\n    opacity = model.get_opacity().detach().float().reshape(-1)\n    if xyz.ndim != 2 or xyz.shape[-1] != 3:\n        raise RuntimeError(f\'Unexpected xyz shape: {tuple(xyz.shape)}\')\n    if opacity.shape[0] != xyz.shape[0]:\n        raise RuntimeError(f\'Unexpected opacity shape: {tuple(opacity.shape)} for xyz {tuple(xyz.shape)}\')\n    if xyz.device.type != \'cuda\':\n        xyz = xyz.to(MV_DEVICE)\n        opacity = opacity.to(MV_DEVICE)\n    return model, xyz, opacity\n\ndef mv_spatial_topk(xyz_np, score_np, k, voxel_size):\n    order = np.argsort(-score_np)\n    vox = np.floor(xyz_np / float(voxel_size)).astype(np.int64)\n    chosen, seen = [], set()\n    for idx in order:\n        key = tuple(vox[idx].tolist())\n        if key in seen:\n            continue\n        seen.add(key)\n        chosen.append(int(idx))\n        if len(chosen) >= k:\n            break\n    if len(chosen) < min(k, len(order)):\n        chosen_set = set(chosen)\n        for idx in order:\n            idx = int(idx)\n            if idx not in chosen_set:\n                chosen.append(idx)\n                chosen_set.add(idx)\n                if len(chosen) >= k:\n                    break\n    return np.asarray(chosen, dtype=np.int64)\n\ndef mv_cache_path(agent_id, submap_index):\n    p = MV_CACHE_ROOT / f\'agent_{int(agent_id)}\'\n    p.mkdir(parents=True, exist_ok=True)\n    return p / f\'{int(submap_index):06d}.pt\'\n\ndef mv_build_cache(agent_id, submap_index):\n    import time as _time\n    build_start = _time.perf_counter()\n    cache_path = mv_cache_path(agent_id, submap_index)\n    if cache_path.exists() and not MV_REBUILD_CACHE:\n        cache = torch.load(cache_path, map_location=\'cpu\', weights_only=False)\n        if cache.get(\'version\') == MV_CACHE_VERSION:\n            print(\'[cache hit]\', cache_path.name, \'| landmarks:\', len(cache[\'xyz\']))\n            return cache\n        print(\'[cache stale] rebuild\', cache_path.name, \'| version:\', cache.get(\'version\'))\n\n    submap_path = mv_submap_paths[int(agent_id)][int(submap_index)]\n    sub = torch.load(submap_path, map_location=\'cpu\', weights_only=False)\n    model, xyz, opacity = mv_restore_gaussians(sub)\n    n = int(xyz.shape[0])\n\n    # Multi-view accumulator.\n    sum_desc = torch.zeros((n, MV_EMBED_DIM), device=MV_DEVICE, dtype=torch.float32)\n    weight_sum = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\n    obs_count = torch.zeros(n, device=MV_DEVICE, dtype=torch.int32)\n    residual_sum = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\n\n    # Fair single-view ablation: keep the highest-quality ONE observation for each Gaussian.\n    # float16 saves memory; final matching converts back to float32 + L2 normalize.\n    best_weight = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\n    best_desc = torch.zeros((n, MV_EMBED_DIM), device=MV_DEVICE, dtype=torch.float16)\n    best_view_id = torch.full((n,), -1, device=MV_DEVICE, dtype=torch.int32)\n\n    keyframe_ids = sub[\'keyframe_ids\']\n    if torch.is_tensor(keyframe_ids):\n        keyframe_ids = keyframe_ids.detach().cpu().tolist()\n    else:\n        keyframe_ids = list(keyframe_ids)\n\n    K = np.asarray(mv_datasets[int(agent_id)].intrinsics, dtype=np.float64)\n    fx, fy, cx, cy = float(K[0, 0]), float(K[1, 1]), float(K[0, 2]), float(K[1, 2])\n\n    used_views = 0\n    for view_i, kf_id in enumerate(keyframe_ids):\n        kf_id = int(kf_id)\n        _, image, depth, _ = mv_datasets[int(agent_id)][kf_id]\n        fmap, meta = mv_extract_feature_map(image)\n        h, w = int(meta[\'orig_h\']), int(meta[\'orig_w\'])\n        depth_t = mv_depth_tensor(depth)\n        if tuple(depth_t.shape) != (h, w):\n            depth_t = F.interpolate(depth_t[None, None], size=(h, w), mode=\'nearest\')[0, 0]\n\n        pose_offset = kf_id - int(sub[\'submap_start_frame_id\'])\n        if pose_offset < 0 or pose_offset >= len(sub[\'submap_c2ws\']):\n            raise IndexError(\n                f\'agent {agent_id} submap {submap_index}: keyframe {kf_id} -> pose offset {pose_offset}, \'\n                f\'poses={len(sub["submap_c2ws"])}\'\n            )\n        c2w = sub[\'submap_c2ws\'][pose_offset]\n        c2w = torch.as_tensor(c2w, device=MV_DEVICE, dtype=torch.float32)\n        w2c = torch.linalg.inv(c2w)\n\n        cam = xyz @ w2c[:3, :3].T + w2c[:3, 3]\n        z = cam[:, 2]\n        z_safe = z.clamp_min(1e-6)\n        u = fx * cam[:, 0] / z_safe + cx\n        v = fy * cam[:, 1] / z_safe + cy\n\n        inside = (\n            torch.isfinite(cam).all(dim=1)\n            & torch.isfinite(opacity)\n            & (z > 0)\n            & (u >= 0) & (u <= w - 1)\n            & (v >= 0) & (v <= h - 1)\n            & (opacity >= MV_MIN_OPACITY)\n        )\n        candidate_idx = torch.where(inside)[0]\n        if candidate_idx.numel() == 0:\n            continue\n\n        ui = torch.round(u[candidate_idx]).long().clamp(0, w - 1)\n        vi = torch.round(v[candidate_idx]).long().clamp(0, h - 1)\n        observed_depth = depth_t[vi, ui]\n        residual = torch.abs(z[candidate_idx] - observed_depth)\n        depth_ok = (\n            (observed_depth > 0)\n            & torch.isfinite(observed_depth)\n            & torch.isfinite(residual)\n            & (residual <= MV_DEPTH_TOLERANCE_M)\n        )\n        valid_idx = candidate_idx[depth_ok]\n        if valid_idx.numel() == 0:\n            continue\n\n        residual_valid = residual[depth_ok]\n        desc = mv_bilinear_sample(fmap, meta, u[valid_idx], v[valid_idx])\n        weights = opacity[valid_idx] * torch.exp(\n            -residual_valid / max(MV_DEPTH_TOLERANCE_M, 1e-6)\n        )\n\n        # Multi-view accumulation.\n        sum_desc.index_add_(0, valid_idx, desc * weights[:, None])\n        weight_sum.index_add_(0, valid_idx, weights)\n        obs_count.index_add_(0, valid_idx, torch.ones_like(valid_idx, dtype=torch.int32))\n        residual_sum.index_add_(0, valid_idx, residual_valid)\n\n        # Single-best-view descriptor for the exact same Gaussian identity.\n        old_best = best_weight[valid_idx]\n        better = weights > old_best\n        if bool(better.any()):\n            update_idx = valid_idx[better]\n            best_weight[update_idx] = weights[better]\n            best_desc[update_idx] = desc[better].to(torch.float16)\n            best_view_id[update_idx] = int(kf_id)\n\n        used_views += 1\n        if view_i % 5 == 0 or view_i == len(keyframe_ids) - 1:\n            print(\n                f\'  agent {agent_id} submap {submap_index} view {view_i + 1}/{len(keyframe_ids)} \'\n                f\'valid={int(valid_idx.numel())}\'\n            )\n\n        del fmap, depth_t, desc\n\n    valid = (\n        (obs_count >= MV_MIN_OBSERVATIONS)\n        & (weight_sum > 1e-8)\n        & (best_weight > 0)\n        & (best_view_id >= 0)\n    )\n    valid_idx = torch.where(valid)[0]\n    if valid_idx.numel() < 3:\n        raise RuntimeError(\n            f\'agent {agent_id} submap {submap_index}: chỉ có {valid_idx.numel()} Gaussian hợp lệ.\'\n        )\n\n    summed = sum_desc[valid_idx]\n    ws = weight_sum[valid_idx]\n    descriptors_multi = F.normalize(summed, dim=-1)\n    stability = torch.linalg.norm(summed, dim=-1) / ws.clamp_min(1e-8)\n    mean_residual = residual_sum[valid_idx] / obs_count[valid_idx].float().clamp_min(1)\n    score = (\n        stability.clamp_min(0)\n        * torch.log1p(obs_count[valid_idx].float())\n        * (0.5 + 0.5 * opacity[valid_idx])\n        / (1.0 + mean_residual / max(MV_DEPTH_TOLERANCE_M, 1e-6))\n    )\n\n    # ONE shared selected landmark set for BOTH variants.\n    xyz_valid = xyz[valid_idx].detach().cpu().numpy()\n    selected_local = mv_spatial_topk(\n        xyz_valid,\n        score.detach().cpu().numpy(),\n        min(MV_MAX_LANDMARKS, len(xyz_valid)),\n        MV_VOXEL_SIZE_M,\n    )\n    selected_local_t = torch.as_tensor(selected_local, device=MV_DEVICE, dtype=torch.long)\n    selected_global = valid_idx[selected_local_t]\n\n    descriptor_multi = F.normalize(descriptors_multi[selected_local_t], dim=-1)\n    descriptor_single = F.normalize(best_desc[selected_global].float(), dim=-1)\n\n    cache = {\n        \'agent_id\': int(agent_id),\n        \'submap_index\': int(submap_index),\n        \'submap_start_frame_id\': int(sub[\'submap_start_frame_id\']),\n        \'submap_end_frame_id\': int(sub[\'submap_end_frame_id\']),\n        \'keyframe_ids\': torch.as_tensor(keyframe_ids, dtype=torch.int32),\n        \'gaussian_index\': selected_global.detach().cpu().int(),\n        \'xyz\': xyz[selected_global].detach().cpu().half(),\n        \'descriptor_single_best_view\': descriptor_single.detach().cpu().half(),\n        \'descriptor_multi_view\': descriptor_multi.detach().cpu().half(),\n        \'single_best_view_id\': best_view_id[selected_global].detach().cpu().int(),\n        \'single_best_weight\': best_weight[selected_global].detach().cpu().half(),\n        \'stability\': stability[selected_local_t].detach().cpu().half(),\n        \'observation_count\': obs_count[selected_global].detach().cpu().short(),\n        \'depth_residual_mean\': mean_residual[selected_local_t].detach().cpu().half(),\n        \'opacity\': opacity[selected_global].detach().cpu().half(),\n        \'score\': score[selected_local_t].detach().cpu().half(),\n        \'num_gaussians_total\': n,\n        \'num_gaussians_multiview\': int(valid_idx.numel()),\n        \'num_views_used\': int(used_views),\n        \'version\': MV_CACHE_VERSION,\n        \'build_time_s\': float(_time.perf_counter() - build_start),\n    }\n    torch.save(cache, cache_path)\n    print(\n        \'[cache built]\', cache_path,\n        \'| total=\', n,\n        \'| eligible=\', int(valid_idx.numel()),\n        \'| kept=\', len(cache[\'xyz\']),\n        \'| views=\', used_views,\n        \'| mean obs=\', f"{float(cache[\'observation_count\'].float().mean()):.2f}",\n        \'| mean rho=\', f"{float(cache[\'stability\'].float().mean()):.4f}",\n        \'| build_s=\', f"{cache[\'build_time_s\']:.1f}",\n    )\n\n    del model, sub, xyz, opacity, sum_desc, weight_sum, obs_count, residual_sum\n    del best_weight, best_desc, best_view_id\n    torch.cuda.empty_cache()\n    return cache\n\n# Original notebook cell 49\ndef mv_world_to_camera_points(points_xyz, c2w):\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\n    w2c = np.linalg.inv(np.asarray(c2w, dtype=np.float64))\n    return points_xyz @ w2c[:3, :3].T + w2c[:3, 3]\n\ndef mv_match_descriptors(src_desc, tgt_desc, src_conf=None, tgt_conf=None):\n    # Mirror logic của coarse_registration_gaussian_landmark trong repo:\n    # mutual-NN + margin nhỏ nhất ở hai chiều.\n    src = F.normalize(src_desc.float().to(MV_DEVICE), dim=-1)\n    tgt = F.normalize(tgt_desc.float().to(MV_DEVICE), dim=-1)\n    sim = src @ tgt.T\n\n    src_k = min(2, sim.shape[1])\n    tgt_k = min(2, sim.shape[0])\n    src_values, src_indices = sim.topk(k=src_k, dim=1)\n    tgt_values, tgt_indices = sim.topk(k=tgt_k, dim=0)\n\n    source_to_target = src_indices[:, 0]\n    target_to_source = tgt_indices[0, :]\n    src_ids = torch.arange(src.shape[0], device=MV_DEVICE)\n    mutual = target_to_source[source_to_target] == src_ids\n\n    matched_src = torch.nonzero(mutual).squeeze(1)\n    matched_tgt = source_to_target[matched_src]\n    match_similarity = src_values[matched_src, 0]\n    src_margin = (\n        src_values[matched_src, 0] - src_values[matched_src, 1]\n        if src_k == 2 else torch.full_like(match_similarity, float(\'inf\'))\n    )\n    tgt_margin = (\n        tgt_values[0, matched_tgt] - tgt_values[1, matched_tgt]\n        if tgt_k == 2 else torch.full_like(match_similarity, float(\'inf\'))\n    )\n    match_margin = torch.minimum(src_margin, tgt_margin)\n\n    keep = (\n        (match_similarity >= MV_MIN_SIMILARITY)\n        & (match_margin >= MV_MIN_MARGIN)\n    )\n    matched_src = matched_src[keep]\n    matched_tgt = matched_tgt[keep]\n    scores = match_similarity[keep]\n\n    # Stability chỉ dùng để rank/diagnostic, không đổi correspondence threshold baseline.\n    if src_conf is not None and tgt_conf is not None and len(matched_src):\n        sc = src_conf.float().to(MV_DEVICE)[matched_src].clamp(0, 1)\n        tc = tgt_conf.float().to(MV_DEVICE)[matched_tgt].clamp(0, 1)\n        confidence = torch.sqrt(sc * tc).clamp_min(1e-4)\n    else:\n        confidence = torch.ones_like(scores)\n\n    return (\n        matched_src.detach().cpu().numpy(),\n        matched_tgt.detach().cpu().numpy(),\n        scores.detach().cpu().numpy(),\n        confidence.detach().cpu().numpy(),\n        match_margin[keep].detach().cpu().numpy(),\n    )\n\ndef mv_geometric_compatibility(src_pts, tgt_pts):\n    # Dùng đúng helper của baseline repo, không adaptive threshold riêng.\n    return _rigidity_compatibility_mask(\n        src_pts,\n        tgt_pts,\n        MV_COMPATIBILITY_THRESHOLD_M,\n        MV_MIN_COMPATIBILITY_SUPPORT,\n    )\n\ndef mv_ransac_baseline(source, target, seed):\n    # Dùng đúng Open3D RANSAC + inlier refit của baseline.\n    try:\n        o3d.utility.random.seed(int(seed))\n    except Exception:\n        pass\n    T, n_inliers = _ransac_from_correspondences(\n        source,\n        target,\n        MV_RANSAC_DISTANCE_THRESHOLD_M,\n        refit_inliers=True,\n    )\n    return np.asarray(T, dtype=np.float64), int(n_inliers)\n\ndef mv_rotation_translation_error(T_est, T_gt):\n    Rerr = T_est[:3, :3].T @ T_gt[:3, :3]\n    c = np.clip((np.trace(Rerr) - 1.0) / 2.0, -1.0, 1.0)\n    r = float(np.degrees(np.arccos(c)))\n    t = float(np.linalg.norm(T_est[:3, 3] - T_gt[:3, 3]) * 100.0)\n    return r, t\n\n# Original notebook cell 50\ndef mv_gt_pose(agent_id, frame_id):\n    pose = mv_datasets[int(agent_id)][int(frame_id)][-1]\n    if torch.is_tensor(pose):\n        pose = pose.detach().cpu().numpy()\n    return np.asarray(pose, dtype=np.float64)\n\ndef mv_baseline_coarse_transform(loop):\n    # Historical baseline coarse transform; DO NOT use loop.transformation (post-ICP).\n    for name in (\'init_transformation\', \'coarse_transformation\', \'initial_transformation\'):\n        if hasattr(loop, name):\n            value = getattr(loop, name)\n            if value is not None:\n                arr = np.asarray(value, dtype=np.float64)\n                if arr.shape == (4, 4):\n                    return arr, name\n    return None, None\n\ndef mv_metric_row(sa, sf, ta, tf):\n    if _mv_metrics.empty:\n        return None\n    required = {\'source_agent_id\',\'source_frame_id\',\'target_agent_id\',\'target_frame_id\'}\n    if not required.issubset(_mv_metrics.columns):\n        return None\n    m = _mv_metrics[\n        (_mv_metrics.source_agent_id.astype(int) == int(sa))\n        & (_mv_metrics.source_frame_id.astype(int) == int(sf))\n        & (_mv_metrics.target_agent_id.astype(int) == int(ta))\n        & (_mv_metrics.target_frame_id.astype(int) == int(tf))\n    ]\n    if len(m) == 0:\n        return None\n    return m.iloc[0]\n\ndef mv_first_metric(row, names):\n    if row is None:\n        return np.nan\n    for name in names:\n        if name in row.index and pd.notna(row[name]):\n            return float(row[name])\n    return np.nan\n\ndef mv_descriptor_for_variant(cache, variant):\n    if variant == \'single_best_view\':\n        return cache[\'descriptor_single_best_view\']\n    if variant == \'multi_view\':\n        return cache[\'descriptor_multi_view\']\n    raise KeyError(variant)\n\ndef _safe_mean(x):\n    x = pd.Series(x).dropna()\n    return float(x.mean()) if len(x) else None\n\ndef _safe_std(x):\n    x = pd.Series(x).dropna()\n    return float(x.std(ddof=0)) if len(x) else None\n\n# Original notebook cell 58\ndef mv_world_to_camera_points(points_xyz, c2w):\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\n    w2c = np.linalg.inv(np.asarray(c2w, dtype=np.float64))\n    return points_xyz @ w2c[:3, :3].T + w2c[:3, 3]\n\ndef mv_rotation_translation_error(T_est, T_gt):\n    Rerr = T_est[:3, :3].T @ T_gt[:3, :3]\n    c = np.clip((np.trace(Rerr) - 1.0) / 2.0, -1.0, 1.0)\n    r = float(np.degrees(np.arccos(c)))\n    t = float(np.linalg.norm(T_est[:3, 3] - T_gt[:3, 3]) * 100.0)\n    return r, t\n\ndef a_match_descriptors(src_desc, tgt_desc, src_conf=None, tgt_conf=None):\n    # Exact logic from v4 Mục 13.5: mutual Top-1 + bidirectional Top1-Top2 margin.\n    src = F.normalize(src_desc.float().to(B_DEVICE), dim=-1)\n    tgt = F.normalize(tgt_desc.float().to(B_DEVICE), dim=-1)\n    sim = src @ tgt.T\n\n    src_k = min(2, sim.shape[1])\n    tgt_k = min(2, sim.shape[0])\n    src_values, src_indices = sim.topk(k=src_k, dim=1)\n    tgt_values, tgt_indices = sim.topk(k=tgt_k, dim=0)\n\n    source_to_target = src_indices[:, 0]\n    target_to_source = tgt_indices[0, :]\n    src_ids = torch.arange(src.shape[0], device=B_DEVICE)\n    mutual = target_to_source[source_to_target] == src_ids\n\n    matched_src = torch.nonzero(mutual).squeeze(1)\n    matched_tgt = source_to_target[matched_src]\n    match_similarity = src_values[matched_src, 0]\n    src_margin = (\n        src_values[matched_src, 0] - src_values[matched_src, 1]\n        if src_k == 2 else torch.full_like(match_similarity, float(\'inf\'))\n    )\n    tgt_margin = (\n        tgt_values[0, matched_tgt] - tgt_values[1, matched_tgt]\n        if tgt_k == 2 else torch.full_like(match_similarity, float(\'inf\'))\n    )\n    match_margin = torch.minimum(src_margin, tgt_margin)\n\n    keep = (\n        (match_similarity >= A_MIN_SIMILARITY)\n        & (match_margin >= A_MIN_MARGIN)\n    )\n    matched_src = matched_src[keep]\n    matched_tgt = matched_tgt[keep]\n    scores = match_similarity[keep]\n\n    if src_conf is not None and tgt_conf is not None and len(matched_src):\n        sc = src_conf.float().to(B_DEVICE)[matched_src].clamp(0, 1)\n        tc = tgt_conf.float().to(B_DEVICE)[matched_tgt].clamp(0, 1)\n        confidence = torch.sqrt(sc * tc).clamp_min(1e-4)\n    else:\n        confidence = torch.ones_like(scores)\n\n    return (\n        matched_src.detach().cpu().numpy(),\n        matched_tgt.detach().cpu().numpy(),\n        scores.detach().cpu().numpy(),\n        confidence.detach().cpu().numpy(),\n        match_margin[keep].detach().cpu().numpy(),\n    )\n\ndef a_geometric_compatibility(src_pts, tgt_pts):\n    return _rigidity_compatibility_mask(\n        src_pts, tgt_pts,\n        A_COMPATIBILITY_THRESHOLD_M,\n        A_MIN_COMPATIBILITY_SUPPORT,\n    )\n\ndef a_ransac(source, target, seed):\n    try:\n        o3d.utility.random.seed(int(seed))\n    except Exception:\n        pass\n    T, n_inliers = _ransac_from_correspondences(\n        source, target,\n        A_RANSAC_DISTANCE_THRESHOLD_M,\n        refit_inliers=True,\n    )\n    return np.asarray(T, dtype=np.float64), int(n_inliers)\n\ndef a_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\n    t0 = time.perf_counter()\n    src_xyz = mv_world_to_camera_points(scache[\'xyz\'].float().numpy(), src_c2w)\n    tgt_xyz = mv_world_to_camera_points(tcache[\'xyz\'].float().numpy(), tgt_c2w)\n\n    sm, tm, sim, stability_conf, margin = a_match_descriptors(\n        scache[\'descriptor_multi_view\'], tcache[\'descriptor_multi_view\'],\n        scache[\'stability\'], tcache[\'stability\']\n    )\n    out = {\n        \'T_est\': None,\n        \'n_semantic_matches\': int(len(sm)),\n        \'n_compatible_matches\': 0,\n        \'n_ransac_inliers\': 0,\n        \'ransac_inlier_ratio\': 0.0,\n        \'mean_match_similarity\': float(np.mean(sim)) if len(sim) else np.nan,\n        \'mean_match_margin\': float(np.mean(margin)) if len(margin) else np.nan,\n        \'mean_compatibility_support\': 0.0,\n        \'matcher_runtime_s\': np.nan,\n    }\n    if len(sm) >= 3:\n        src_m = src_xyz[sm]\n        tgt_m = tgt_xyz[tm]\n        keep, support = a_geometric_compatibility(src_m, tgt_m)\n        out[\'mean_compatibility_support\'] = float(np.mean(support)) if len(support) else 0.0\n        if int(keep.sum()) >= 3:\n            src_g, tgt_g = src_m[keep], tgt_m[keep]\n            out[\'n_compatible_matches\'] = int(keep.sum())\n            T, nin = a_ransac(src_g, tgt_g, seed)\n            out[\'n_ransac_inliers\'] = int(nin)\n            out[\'ransac_inlier_ratio\'] = float(nin / max(1, int(keep.sum())))\n            out[\'T_est\'] = T\n    out[\'matcher_runtime_s\'] = float(time.perf_counter() - t0)\n    return out\n\n# Original notebook cell 60\nV2_TOPK_MAX = 5\n\nV2_MARGIN_TOP1 = 0.060\n\nV2_MARGIN_TOP3 = 0.025\n\nV2_RIGIDITY_THRESHOLD_M = 0.03\n\nV2_MIN_SUPPORT = 8\n\nV2_MIN_CORRESPONDENCES = 8\n\nV2_COVERAGE_VOXEL_M = 0.12\n\nV2_MAX_PER_SOURCE_VOXEL = 2\n\nV2_MAX_PER_TARGET_VOXEL = 2\n\nV2_MAX_RANSAC_PAIRS = 96\n\nV2_FALLBACK_MAX_PAIRS = 160\n\nV2_RANSAC_THRESHOLD_M = 0.05\n\nV2_TIGHT_REFIT_THRESHOLD_M = 0.03\n\nV2_RANSAC_MAX_ITERS = 700\n\nV2_RANSAC_MIN_ITERS = 64\n\nV2_RANSAC_EARLY_INLIER_RATIO = 0.82\n\nV2_RANSAC_EARLY_MEDIAN_M = 0.020\n\nV2_DESCRIPTOR_KEY = \'descriptor_multi_view\'\n\nV2_SEEDS = list(A_SEEDS)\n\nV3_DESCRIPTOR_KEY = V2_DESCRIPTOR_KEY\n\nV3_TOPK_MAX = V2_TOPK_MAX\n\nV3_RIGIDITY_THRESHOLD_M = V2_RIGIDITY_THRESHOLD_M\n\nV3_MIN_SUPPORT = V2_MIN_SUPPORT\n\nV3_MIN_CORRESPONDENCES = V2_MIN_CORRESPONDENCES\n\nV3_COVERAGE_VOXEL_M = V2_COVERAGE_VOXEL_M\n\nV3_FAST_MAX_PAIRS = 96\n\nV3_FAST_MAX_PER_SOURCE_VOXEL = 2\n\nV3_FAST_MAX_PER_TARGET_VOXEL = 2\n\nV3_HARD_TOPK = 5\n\nV3_HARD_MAX_PAIRS = 192\n\nV3_HARD_MAX_PER_SOURCE_VOXEL = 3\n\nV3_HARD_MAX_PER_TARGET_VOXEL = 3\n\nV3_HARD_MAX_SOURCE_REUSE = 2\n\nV3_HARD_MAX_TARGET_REUSE = 2\n\nV3_HARD_TRIGGER_RATIO_5CM = 0.80\n\nV3_HARD_TRIGGER_RATIO_3CM = 0.70\n\nV3_RANSAC_THRESHOLD_M = V2_RANSAC_THRESHOLD_M\n\nV3_TIGHT_REFIT_THRESHOLD_M = V2_TIGHT_REFIT_THRESHOLD_M\n\nV3_FAST_RANSAC_MAX_ITERS = 700\n\nV3_FAST_RANSAC_MIN_ITERS = 64\n\nV3_FAST_EARLY_RATIO = 0.82\n\nV3_FAST_EARLY_MEDIAN_M = 0.020\n\nV3_HARD_RANSAC_MAX_ITERS = 1800\n\nV3_HARD_RANSAC_MIN_ITERS = 96\n\nV3_HARD_EARLY_RATIO = 0.78\n\nV3_HARD_EARLY_MEDIAN_M = 0.025\n\nV3_SCORE_WEIGHTS = {\n    \'weighted_ratio_5cm\': 0.50,\n    \'ratio_5cm\': 0.20,\n    \'ratio_3cm\': 0.20,\n    \'residual_quality\': 0.10,\n}\n\nV3_HARD_MIN_VALIDATION_SCORE_GAIN = 0.005\n\nV3_SEEDS = list(V2_SEEDS)\n\nV32_DESCRIPTOR_KEY = V3_DESCRIPTOR_KEY\n\nV32_MIN_CORRESPONDENCES = V3_MIN_CORRESPONDENCES\n\nV32_RIGIDITY_THRESHOLD_M = V3_RIGIDITY_THRESHOLD_M\n\nV32_MIN_SUPPORT = V3_MIN_SUPPORT\n\nV32_FAST_MAX_PAIRS = V3_FAST_MAX_PAIRS\n\nV32_FAST_MAX_PER_SOURCE_VOXEL = V3_FAST_MAX_PER_SOURCE_VOXEL\n\nV32_FAST_MAX_PER_TARGET_VOXEL = V3_FAST_MAX_PER_TARGET_VOXEL\n\nV32_FAST_RANSAC_MAX_ITERS = V3_FAST_RANSAC_MAX_ITERS\n\nV32_FAST_RANSAC_MIN_ITERS = V3_FAST_RANSAC_MIN_ITERS\n\nV32_FAST_EARLY_RATIO = V3_FAST_EARLY_RATIO\n\nV32_FAST_EARLY_MEDIAN_M = V3_FAST_EARLY_MEDIAN_M\n\nV32_HARD_TRIGGER_RATIO_5CM = V3_HARD_TRIGGER_RATIO_5CM\n\nV32_HARD_TRIGGER_RATIO_3CM = V3_HARD_TRIGGER_RATIO_3CM\n\nV32_HARD_TOPK = 5\n\nV32_HARD_PREFILTER_MAX_PAIRS = 180\n\nV32_HARD_MAX_PAIRS = 144\n\nV32_HARD_MAX_PER_SOURCE_VOXEL = 3\n\nV32_HARD_MAX_PER_TARGET_VOXEL = 3\n\nV32_MARGIN_FULL_SCALE = 0.060\n\nV32_HARD_QUALITY_WEIGHTS = {\n    \'base_confidence\': 0.60,\n    \'margin_quality\': 0.25,\n    \'rank_quality\': 0.15,\n}\n\nV32_LOCAL_INITIAL_GATE_M = 0.080\n\nV32_LOCAL_FINAL_GATE_M = 0.050\n\nV32_LOCAL_ROBUST_SCALE_M = 0.030\n\nV32_LOCAL_MIN_PAIRS = 12\n\nV32_LOCAL_IRLS_ITERS = 3\n\nV32_TRUST_MAX_ROT_DEG = 0.35\n\nV32_TRUST_MAX_TRANS_M = 0.010\n\nV32_LINESEARCH_FRACTIONS = (0.25, 0.50, 0.75, 1.00)\n\nV32_LOCAL_SCORE_WEIGHTS = {\n    \'weighted_ratio_5cm\': 0.30,\n    \'ratio_5cm\': 0.20,\n    \'ratio_3cm\': 0.35,\n    \'residual_quality\': 0.15,\n}\n\nV32_STEP_PENALTY_ROT = 0.0015\n\nV32_STEP_PENALTY_TRANS = 0.0015\n\nV32_MIN_REGULARIZED_GAIN = 0.0010\n\nV32_MAX_RATIO5_DROP = 0.010\n\nV32_MAX_RATIO3_DROP = 0.005\n\nV32_MAX_MEDIAN5_INCREASE_M = 0.0015\n\nV32_SEEDS = list(A_SEEDS)\n\n# Original notebook cell 61\ndef v2_np(x):\n    if torch.is_tensor(x):\n        x = x.detach().cpu().numpy()\n    return np.asarray(x)\n\ndef v2_normalize_desc(x):\n    x = v2_np(x).astype(np.float64, copy=False)\n    n = np.linalg.norm(x, axis=1, keepdims=True)\n    return x / np.maximum(n, 1e-12)\n\ndef v2_adaptive_k_from_row(sorted_scores):\n    """K=1 khi Top-1 rõ; K=3 khi vừa; K=5 khi mơ hồ."""\n    n = len(sorted_scores)\n    if n <= 1:\n        return 1\n    margin12 = float(sorted_scores[0] - sorted_scores[1])\n    if margin12 >= V2_MARGIN_TOP1:\n        return 1\n    if margin12 >= V2_MARGIN_TOP3:\n        return min(3, n)\n    return min(V2_TOPK_MAX, n)\n\ndef v2_build_adaptive_bidirectional(src_desc, tgt_desc):\n    src = v2_normalize_desc(src_desc)\n    tgt = v2_normalize_desc(tgt_desc)\n    sim = src @ tgt.T\n\n    ns, nt = sim.shape\n    src_lists = []\n    src_k_used = []\n    for i in range(ns):\n        order = np.argsort(-sim[i])[:min(V2_TOPK_MAX, nt)]\n        scores = sim[i, order]\n        k = v2_adaptive_k_from_row(scores)\n        src_lists.append(order[:k].tolist())\n        src_k_used.append(k)\n\n    tgt_lists = []\n    tgt_k_used = []\n    sim_t = sim.T\n    for j in range(nt):\n        order = np.argsort(-sim_t[j])[:min(V2_TOPK_MAX, ns)]\n        scores = sim_t[j, order]\n        k = v2_adaptive_k_from_row(scores)\n        tgt_lists.append(set(order[:k].tolist()))\n        tgt_k_used.append(k)\n\n    pairs, sims = [], []\n    for i, candidates in enumerate(src_lists):\n        for j in candidates:\n            j = int(j)\n            if i in tgt_lists[j]:\n                pairs.append((i, j))\n                sims.append(float(sim[i, j]))\n\n    if not pairs:\n        return (\n            np.empty((0, 2), dtype=np.int64),\n            np.empty((0,), dtype=np.float64),\n            np.asarray(src_k_used),\n            np.asarray(tgt_k_used),\n        )\n\n    return (\n        np.asarray(pairs, dtype=np.int64),\n        np.asarray(sims, dtype=np.float64),\n        np.asarray(src_k_used),\n        np.asarray(tgt_k_used),\n    )\n\ndef v2_pair_quality(scache, tcache, pairs, similarities, support):\n    pairs = np.asarray(pairs, dtype=np.int64)\n    if len(pairs) == 0:\n        return np.empty((0,), dtype=np.float64)\n\n    sidx, tidx = pairs[:, 0], pairs[:, 1]\n\n    s_stab = v2_np(scache[\'stability\']).astype(np.float64)[sidx]\n    t_stab = v2_np(tcache[\'stability\']).astype(np.float64)[tidx]\n    stability = np.sqrt(np.clip(s_stab * t_stab, 0.0, 1.0))\n\n    s_obs = v2_np(scache[\'observation_count\']).astype(np.float64)[sidx]\n    t_obs = v2_np(tcache[\'observation_count\']).astype(np.float64)[tidx]\n    obs = np.sqrt(np.log1p(s_obs) * np.log1p(t_obs))\n    obs /= max(np.log1p(10.0), 1e-12)\n    obs = np.clip(obs, 0.0, 1.0)\n\n    s_res = v2_np(scache[\'depth_residual_mean\']).astype(np.float64)[sidx]\n    t_res = v2_np(tcache[\'depth_residual_mean\']).astype(np.float64)[tidx]\n    depth_q = np.exp(-(s_res + t_res) / max(2.0 * MV_DEPTH_TOLERANCE_M, 1e-6))\n\n    s_op = v2_np(scache[\'opacity\']).astype(np.float64)[sidx]\n    t_op = v2_np(tcache[\'opacity\']).astype(np.float64)[tidx]\n    opacity_q = np.sqrt(np.clip(s_op * t_op, 0.0, 1.0))\n\n    support = np.asarray(support, dtype=np.float64)\n    support_q = support / max(float(np.max(support)) if len(support) else 1.0, 1.0)\n\n    similarity_q = np.clip(np.asarray(similarities, dtype=np.float64), 0.0, 1.0)\n    metadata_q = stability * depth_q * (0.5 + 0.5 * obs) * (0.5 + 0.5 * opacity_q)\n\n    # Development score; keep weights explicit for ablation.\n    confidence = 0.45 * similarity_q + 0.35 * support_q + 0.20 * metadata_q\n    return np.asarray(confidence, dtype=np.float64)\n\ndef v2_greedy_unique_by_confidence(pairs, confidence, support, min_support=V2_MIN_SUPPORT):\n    pairs = np.asarray(pairs, dtype=np.int64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    support = np.asarray(support, dtype=np.int64)\n\n    valid = np.flatnonzero(support >= int(min_support))\n    if len(valid) == 0:\n        return np.empty((0,2), np.int64), np.empty(0), np.empty(0, np.int64)\n\n    order = valid[np.argsort(-confidence[valid])]\n    used_s, used_t, keep = set(), set(), []\n    for idx in order:\n        s, t = map(int, pairs[idx])\n        if s in used_s or t in used_t:\n            continue\n        used_s.add(s); used_t.add(t); keep.append(int(idx))\n\n    keep = np.asarray(keep, dtype=np.int64)\n    return pairs[keep], confidence[keep], support[keep]\n\ndef v2_spatial_coverage_select(pairs, confidence, src_xyz, tgt_xyz):\n    """Select high-confidence pairs while limiting concentration per source/target voxel."""\n    pairs = np.asarray(pairs, dtype=np.int64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    if len(pairs) <= V2_MAX_RANSAC_PAIRS:\n        return pairs, confidence\n\n    order = np.argsort(-confidence)\n    src_vox = np.floor(src_xyz / V2_COVERAGE_VOXEL_M).astype(np.int64)\n    tgt_vox = np.floor(tgt_xyz / V2_COVERAGE_VOXEL_M).astype(np.int64)\n\n    src_count, tgt_count = {}, {}\n    keep = []\n    for idx in order:\n        s, t = map(int, pairs[idx])\n        sv = tuple(src_vox[s].tolist())\n        tv = tuple(tgt_vox[t].tolist())\n        if src_count.get(sv, 0) >= V2_MAX_PER_SOURCE_VOXEL:\n            continue\n        if tgt_count.get(tv, 0) >= V2_MAX_PER_TARGET_VOXEL:\n            continue\n        keep.append(int(idx))\n        src_count[sv] = src_count.get(sv, 0) + 1\n        tgt_count[tv] = tgt_count.get(tv, 0) + 1\n        if len(keep) >= V2_MAX_RANSAC_PAIRS:\n            break\n\n    # If coverage was too restrictive, fill by confidence up to fallback cap.\n    if len(keep) < V2_MIN_CORRESPONDENCES:\n        keep_set = set(keep)\n        for idx in order:\n            if int(idx) not in keep_set:\n                keep.append(int(idx)); keep_set.add(int(idx))\n            if len(keep) >= min(V2_FALLBACK_MAX_PAIRS, len(pairs)):\n                break\n\n    keep = np.asarray(keep, dtype=np.int64)\n    return pairs[keep], confidence[keep]\n\ndef v2_geometry_condition(points):\n    points = np.asarray(points, dtype=np.float64)\n    if len(points) < 3:\n        return {\'s1\': np.nan, \'s2_s1\': 0.0, \'s3_s1\': 0.0}\n    x = points - points.mean(axis=0, keepdims=True)\n    s = np.linalg.svd(x, compute_uv=False)\n    s = np.pad(s, (0, max(0, 3-len(s))), constant_values=0.0)[:3]\n    return {\n        \'s1\': float(s[0]),\n        \'s2_s1\': float(s[1] / max(s[0], 1e-12)),\n        \'s3_s1\': float(s[2] / max(s[0], 1e-12)),\n    }\n\n# Original notebook cell 62\ndef mv_topk_rigidity_support(candidate_pairs, src_xyz, tgt_xyz, threshold_m=0.03):\n    candidate_pairs = np.asarray(candidate_pairs, dtype=np.int64)\n    if len(candidate_pairs) == 0:\n        return np.empty((0,), dtype=np.int64)\n    src_pair_xyz = np.asarray(src_xyz)[candidate_pairs[:, 0]]\n    tgt_pair_xyz = np.asarray(tgt_xyz)[candidate_pairs[:, 1]]\n    src_dist = np.linalg.norm(src_pair_xyz[:, None, :] - src_pair_xyz[None, :, :], axis=2)\n    tgt_dist = np.linalg.norm(tgt_pair_xyz[:, None, :] - tgt_pair_xyz[None, :, :], axis=2)\n    compatible = np.abs(src_dist - tgt_dist) <= float(threshold_m)\n    np.fill_diagonal(compatible, False)\n    same_src = candidate_pairs[:, 0][:, None] == candidate_pairs[:, 0][None, :]\n    same_tgt = candidate_pairs[:, 1][:, None] == candidate_pairs[:, 1][None, :]\n    compatible[same_src | same_tgt] = False\n    return compatible.sum(axis=1).astype(np.int64)\n\n# Original notebook cell 63\ndef v2_weighted_kabsch(P, Q, w=None):\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    if len(P) < 3:\n        raise ValueError(\'Cần >= 3 correspondence.\')\n\n    if w is None:\n        w = np.ones(len(P), dtype=np.float64)\n    w = np.asarray(w, dtype=np.float64)\n    w = np.clip(w, 1e-8, None)\n    w = w / w.sum()\n\n    pc = np.sum(P * w[:, None], axis=0)\n    qc = np.sum(Q * w[:, None], axis=0)\n    X = P - pc\n    Y = Q - qc\n    H = (X * w[:, None]).T @ Y\n    U, _, Vt = np.linalg.svd(H)\n    R = Vt.T @ U.T\n    if np.linalg.det(R) < 0:\n        Vt[-1] *= -1.0\n        R = Vt.T @ U.T\n    t = qc - R @ pc\n\n    T = np.eye(4, dtype=np.float64)\n    T[:3,:3] = R\n    T[:3,3] = t\n    return T\n\ndef v2_transform_points(T, P):\n    return (T[:3,:3] @ np.asarray(P).T).T + T[:3,3][None,:]\n\ndef v2_non_degenerate_triplet(P):\n    a, b, c = np.asarray(P, dtype=np.float64)\n    area2 = np.linalg.norm(np.cross(b-a, c-a))\n    return bool(area2 > 1e-5)\n\ndef v2_prosac_ransac(P, Q, confidence, seed):\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    n = len(P)\n    if n < 3:\n        return None\n\n    order = np.argsort(-confidence)\n    P = P[order]; Q = Q[order]; confidence = confidence[order]\n    conf_prob = np.clip(confidence, 1e-8, None)\n    rng = np.random.default_rng(int(seed))\n\n    best = None\n    best_score = -np.inf\n    best_inliers = None\n    best_median = np.inf\n    start = time.perf_counter()\n\n    start_pool = min(max(12, 3), n)\n\n    for it in range(V2_RANSAC_MAX_ITERS):\n        frac = it / max(1, V2_RANSAC_MAX_ITERS - 1)\n        pool_n = min(n, start_pool + int(frac * max(0, n - start_pool)))\n        probs = conf_prob[:pool_n]\n        probs = probs / probs.sum()\n        sample = rng.choice(pool_n, size=3, replace=False, p=probs)\n\n        if not v2_non_degenerate_triplet(P[sample]) or not v2_non_degenerate_triplet(Q[sample]):\n            continue\n\n        try:\n            T = v2_weighted_kabsch(P[sample], Q[sample], confidence[sample])\n        except Exception:\n            continue\n\n        residual = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\n        inliers = residual <= V2_RANSAC_THRESHOLD_M\n        nin = int(inliers.sum())\n        if nin < 3:\n            continue\n\n        weighted_support = float(confidence[inliers].sum())\n        med = float(np.median(residual[inliers]))\n        score = weighted_support - 0.25 * med\n\n        if score > best_score or (np.isclose(score, best_score) and med < best_median):\n            best = T\n            best_score = score\n            best_inliers = inliers.copy()\n            best_median = med\n\n        if it + 1 >= V2_RANSAC_MIN_ITERS and best_inliers is not None:\n            ratio = float(best_inliers.mean())\n            if ratio >= V2_RANSAC_EARLY_INLIER_RATIO and best_median <= V2_RANSAC_EARLY_MEDIAN_M:\n                break\n\n    if best_inliers is None or int(best_inliers.sum()) < 3:\n        return None\n\n    # Weighted refit on 5 cm consensus.\n    T_refit = v2_weighted_kabsch(P[best_inliers], Q[best_inliers], confidence[best_inliers])\n    residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\n    inliers5 = residual <= V2_RANSAC_THRESHOLD_M\n\n    # Tight 3 cm refit when enough strong pairs remain.\n    tight = residual <= V2_TIGHT_REFIT_THRESHOLD_M\n    if int(tight.sum()) >= 6:\n        T_refit = v2_weighted_kabsch(P[tight], Q[tight], confidence[tight])\n        residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\n        inliers5 = residual <= V2_RANSAC_THRESHOLD_M\n        tight = residual <= V2_TIGHT_REFIT_THRESHOLD_M\n\n    return {\n        \'T\': T_refit,\n        \'n_inliers_5cm\': int(inliers5.sum()),\n        \'n_inliers_3cm\': int(tight.sum()),\n        \'inlier_ratio_5cm\': float(inliers5.mean()),\n        \'median_residual_m\': float(np.median(residual[inliers5])) if inliers5.any() else np.inf,\n        \'iterations\': int(it + 1),\n        \'runtime_s\': float(time.perf_counter() - start),\n    }\n\ndef v2_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\n    t0 = time.perf_counter()\n\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\'xyz\']).astype(np.float64), src_c2w)\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\'xyz\']).astype(np.float64), tgt_c2w)\n\n    pairs, similarities, src_k, tgt_k = v2_build_adaptive_bidirectional(\n        scache[V2_DESCRIPTOR_KEY], tcache[V2_DESCRIPTOR_KEY]\n    )\n\n    out = {\n        \'T_est\': None,\n        \'candidate_pairs\': int(len(pairs)),\n        \'mean_src_k\': float(np.mean(src_k)) if len(src_k) else np.nan,\n        \'mean_tgt_k\': float(np.mean(tgt_k)) if len(tgt_k) else np.nan,\n        \'rigidity_pairs\': 0,\n        \'unique_pairs\': 0,\n        \'coverage_pairs\': 0,\n        \'mean_confidence\': np.nan,\n        \'source_s2_s1\': np.nan,\n        \'source_s3_s1\': np.nan,\n        \'target_s2_s1\': np.nan,\n        \'target_s3_s1\': np.nan,\n        \'ransac_inliers_5cm\': 0,\n        \'ransac_inliers_3cm\': 0,\n        \'ransac_iterations\': 0,\n        \'ransac_runtime_s\': 0.0,\n        \'matcher_runtime_s\': np.nan,\n    }\n\n    if len(pairs) < V2_MIN_CORRESPONDENCES:\n        out[\'matcher_runtime_s\'] = time.perf_counter() - t0\n        return out\n\n    support = mv_topk_rigidity_support(\n        pairs, src_xyz, tgt_xyz, threshold_m=V2_RIGIDITY_THRESHOLD_M\n    )\n    out[\'rigidity_pairs\'] = int(np.sum(support >= V2_MIN_SUPPORT))\n\n    conf_all = v2_pair_quality(scache, tcache, pairs, similarities, support)\n    unique_pairs, unique_conf, unique_support = v2_greedy_unique_by_confidence(\n        pairs, conf_all, support, min_support=V2_MIN_SUPPORT\n    )\n    out[\'unique_pairs\'] = int(len(unique_pairs))\n\n    if len(unique_pairs) < V2_MIN_CORRESPONDENCES:\n        out[\'matcher_runtime_s\'] = time.perf_counter() - t0\n        return out\n\n    selected_pairs, selected_conf = v2_spatial_coverage_select(\n        unique_pairs, unique_conf, src_xyz, tgt_xyz\n    )\n    out[\'coverage_pairs\'] = int(len(selected_pairs))\n    out[\'mean_confidence\'] = float(np.mean(selected_conf)) if len(selected_conf) else np.nan\n\n    if len(selected_pairs) < V2_MIN_CORRESPONDENCES:\n        out[\'matcher_runtime_s\'] = time.perf_counter() - t0\n        return out\n\n    P = src_xyz[selected_pairs[:,0]]\n    Q = tgt_xyz[selected_pairs[:,1]]\n    gs = v2_geometry_condition(P)\n    gt = v2_geometry_condition(Q)\n    out[\'source_s2_s1\'], out[\'source_s3_s1\'] = gs[\'s2_s1\'], gs[\'s3_s1\']\n    out[\'target_s2_s1\'], out[\'target_s3_s1\'] = gt[\'s2_s1\'], gt[\'s3_s1\']\n\n    r = v2_prosac_ransac(P, Q, selected_conf, seed)\n    if r is None:\n        out[\'matcher_runtime_s\'] = time.perf_counter() - t0\n        return out\n\n    out[\'T_est\'] = r[\'T\']\n    out[\'ransac_inliers_5cm\'] = r[\'n_inliers_5cm\']\n    out[\'ransac_inliers_3cm\'] = r[\'n_inliers_3cm\']\n    out[\'ransac_iterations\'] = r[\'iterations\']\n    out[\'ransac_runtime_s\'] = r[\'runtime_s\']\n    out[\'matcher_runtime_s\'] = float(time.perf_counter() - t0)\n    return out\n\n# Original notebook cell 64\ndef v3_conf_stats(x, prefix=\'\'):\n    x = np.asarray(x, dtype=np.float64)\n    if len(x) == 0:\n        return {\n            f\'{prefix}confidence_mean\': np.nan,\n            f\'{prefix}confidence_std\': np.nan,\n            f\'{prefix}confidence_p10\': np.nan,\n            f\'{prefix}confidence_p50\': np.nan,\n            f\'{prefix}confidence_p90\': np.nan,\n        }\n    return {\n        f\'{prefix}confidence_mean\': float(np.mean(x)),\n        f\'{prefix}confidence_std\': float(np.std(x)),\n        f\'{prefix}confidence_p10\': float(np.percentile(x, 10)),\n        f\'{prefix}confidence_p50\': float(np.percentile(x, 50)),\n        f\'{prefix}confidence_p90\': float(np.percentile(x, 90)),\n    }\n\ndef v3_spatial_coverage_select(\n    pairs,\n    confidence,\n    support,\n    src_xyz,\n    tgt_xyz,\n    *,\n    max_pairs,\n    max_per_source_voxel,\n    max_per_target_voxel,\n    min_keep=V3_MIN_CORRESPONDENCES,\n):\n    """Always enforce source + target voxel coverage.\n\n    Important V2 fix:\n      V2 returned immediately when len(pairs) <= 96, so hard loops with\n      e.g. 85 pairs never received spatial balancing. V3 always executes\n      the voxel-selection pass.\n    """\n    pairs = np.asarray(pairs, dtype=np.int64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    support = np.asarray(support, dtype=np.int64)\n\n    if len(pairs) == 0:\n        info = {\n            \'input_pairs\': 0,\n            \'strict_coverage_pairs\': 0,\n            \'filled_pairs\': 0,\n            \'output_pairs\': 0,\n        }\n        return pairs, confidence, support, info\n\n    order = np.argsort(-confidence)\n    src_vox = np.floor(np.asarray(src_xyz) / V3_COVERAGE_VOXEL_M).astype(np.int64)\n    tgt_vox = np.floor(np.asarray(tgt_xyz) / V3_COVERAGE_VOXEL_M).astype(np.int64)\n\n    src_count, tgt_count = {}, {}\n    keep = []\n\n    for idx in order:\n        s, t = map(int, pairs[idx])\n        sv = tuple(src_vox[s].tolist())\n        tv = tuple(tgt_vox[t].tolist())\n\n        if src_count.get(sv, 0) >= int(max_per_source_voxel):\n            continue\n        if tgt_count.get(tv, 0) >= int(max_per_target_voxel):\n            continue\n\n        keep.append(int(idx))\n        src_count[sv] = src_count.get(sv, 0) + 1\n        tgt_count[tv] = tgt_count.get(tv, 0) + 1\n\n        if len(keep) >= min(int(max_pairs), len(pairs)):\n            break\n\n    strict_n = len(keep)\n\n    # Safety only: if strict coverage leaves too few pairs for rigid pose,\n    # fill by confidence until min_keep. Do NOT fill back to max_pairs.\n    if len(keep) < min(int(min_keep), len(pairs)):\n        keep_set = set(keep)\n        for idx in order:\n            idx = int(idx)\n            if idx in keep_set:\n                continue\n            keep.append(idx)\n            keep_set.add(idx)\n            if len(keep) >= min(int(min_keep), int(max_pairs), len(pairs)):\n                break\n\n    keep = np.asarray(keep, dtype=np.int64)\n    info = {\n        \'input_pairs\': int(len(pairs)),\n        \'strict_coverage_pairs\': int(strict_n),\n        \'filled_pairs\': int(max(0, len(keep) - strict_n)),\n        \'output_pairs\': int(len(keep)),\n    }\n    return pairs[keep], confidence[keep], support[keep], info\n\ndef v3_greedy_limited_reuse(\n    pairs,\n    confidence,\n    support,\n    *,\n    min_support=V3_MIN_SUPPORT,\n    max_source_reuse=V3_HARD_MAX_SOURCE_REUSE,\n    max_target_reuse=V3_HARD_MAX_TARGET_REUSE,\n):\n    """Relax strict 1-to-1 only for the hard fallback.\n\n    Each landmark can participate in at most a small number of pairs.\n    This expands the pool without allowing a single ambiguous landmark\n    to dominate RANSAC.\n    """\n    pairs = np.asarray(pairs, dtype=np.int64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    support = np.asarray(support, dtype=np.int64)\n\n    valid = np.flatnonzero(support >= int(min_support))\n    if len(valid) == 0:\n        return (\n            np.empty((0, 2), dtype=np.int64),\n            np.empty((0,), dtype=np.float64),\n            np.empty((0,), dtype=np.int64),\n        )\n\n    # Confidence primary; rigidity support as tie-breaker.\n    order_local = np.lexsort((-support[valid], -confidence[valid]))\n    order = valid[order_local]\n\n    src_use, tgt_use = {}, {}\n    seen = set()\n    keep = []\n\n    for idx in order:\n        s, t = map(int, pairs[idx])\n        key = (s, t)\n        if key in seen:\n            continue\n        if src_use.get(s, 0) >= int(max_source_reuse):\n            continue\n        if tgt_use.get(t, 0) >= int(max_target_reuse):\n            continue\n\n        keep.append(int(idx))\n        seen.add(key)\n        src_use[s] = src_use.get(s, 0) + 1\n        tgt_use[t] = tgt_use.get(t, 0) + 1\n\n    keep = np.asarray(keep, dtype=np.int64)\n    return pairs[keep], confidence[keep], support[keep]\n\ndef v3_geometry_diag(pairs, src_xyz, tgt_xyz, prefix):\n    if len(pairs) < 3:\n        return {\n            f\'{prefix}source_s2_s1\': np.nan,\n            f\'{prefix}source_s3_s1\': np.nan,\n            f\'{prefix}target_s2_s1\': np.nan,\n            f\'{prefix}target_s3_s1\': np.nan,\n        }\n    P = src_xyz[pairs[:, 0]]\n    Q = tgt_xyz[pairs[:, 1]]\n    gs = v2_geometry_condition(P)\n    gt = v2_geometry_condition(Q)\n    return {\n        f\'{prefix}source_s2_s1\': gs[\'s2_s1\'],\n        f\'{prefix}source_s3_s1\': gs[\'s3_s1\'],\n        f\'{prefix}target_s2_s1\': gt[\'s2_s1\'],\n        f\'{prefix}target_s3_s1\': gt[\'s3_s1\'],\n    }\n\n# Original notebook cell 65\ndef v3_consensus_stats(T, P, Q, confidence):\n    """Evaluate one transform with dimensionless, comparable terms."""\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n\n    if T is None or len(P) == 0:\n        return {\n            \'score\': -np.inf,\n            \'ratio_5cm\': 0.0,\n            \'ratio_3cm\': 0.0,\n            \'weighted_ratio_5cm\': 0.0,\n            \'median_5cm_m\': np.inf,\n            \'mean_5cm_m\': np.inf,\n            \'p90_all_m\': np.inf,\n            \'residual_quality\': 0.0,\n            \'n_inliers_5cm\': 0,\n            \'n_inliers_3cm\': 0,\n        }\n\n    residual = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\n    in5 = residual <= V3_RANSAC_THRESHOLD_M\n    in3 = residual <= V3_TIGHT_REFIT_THRESHOLD_M\n\n    ratio5 = float(np.mean(in5))\n    ratio3 = float(np.mean(in3))\n\n    conf = np.clip(confidence, 1e-8, None)\n    weighted_ratio5 = float(conf[in5].sum() / max(conf.sum(), 1e-12))\n\n    if np.any(in5):\n        median5 = float(np.median(residual[in5]))\n        mean5 = float(np.mean(residual[in5]))\n        residual_quality = float(\n            1.0 - np.clip(median5 / V3_RANSAC_THRESHOLD_M, 0.0, 1.0)\n        )\n    else:\n        median5 = np.inf\n        mean5 = np.inf\n        residual_quality = 0.0\n\n    score = (\n        V3_SCORE_WEIGHTS[\'weighted_ratio_5cm\'] * weighted_ratio5\n        + V3_SCORE_WEIGHTS[\'ratio_5cm\'] * ratio5\n        + V3_SCORE_WEIGHTS[\'ratio_3cm\'] * ratio3\n        + V3_SCORE_WEIGHTS[\'residual_quality\'] * residual_quality\n    )\n\n    return {\n        \'score\': float(score),\n        \'ratio_5cm\': ratio5,\n        \'ratio_3cm\': ratio3,\n        \'weighted_ratio_5cm\': weighted_ratio5,\n        \'median_5cm_m\': median5,\n        \'mean_5cm_m\': mean5,\n        \'p90_all_m\': float(np.percentile(residual, 90)) if len(residual) else np.inf,\n        \'residual_quality\': residual_quality,\n        \'n_inliers_5cm\': int(in5.sum()),\n        \'n_inliers_3cm\': int(in3.sum()),\n    }\n\ndef v3_prosac_ransac(\n    P,\n    Q,\n    confidence,\n    seed,\n    *,\n    max_iters,\n    min_iters,\n    early_ratio,\n    early_median_m,\n):\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    confidence = np.asarray(confidence, dtype=np.float64)\n    n = len(P)\n    if n < 3:\n        return None\n\n    order = np.argsort(-confidence)\n    P = P[order]\n    Q = Q[order]\n    confidence = confidence[order]\n\n    conf_prob = np.clip(confidence, 1e-8, None)\n    rng = np.random.default_rng(int(seed))\n\n    best_T = None\n    best_stats = None\n    best_score = -np.inf\n    best_median = np.inf\n    start = time.perf_counter()\n\n    start_pool = min(max(12, 3), n)\n    iterations_run = 0\n\n    for it in range(int(max_iters)):\n        iterations_run = it + 1\n        frac = it / max(1, int(max_iters) - 1)\n        pool_n = min(n, start_pool + int(frac * max(0, n - start_pool)))\n\n        probs = conf_prob[:pool_n]\n        probs = probs / probs.sum()\n        sample = rng.choice(pool_n, size=3, replace=False, p=probs)\n\n        if not v2_non_degenerate_triplet(P[sample]):\n            continue\n        if not v2_non_degenerate_triplet(Q[sample]):\n            continue\n\n        try:\n            T = v2_weighted_kabsch(P[sample], Q[sample], confidence[sample])\n        except Exception:\n            continue\n\n        stats = v3_consensus_stats(T, P, Q, confidence)\n        if stats[\'n_inliers_5cm\'] < 3:\n            continue\n\n        # New score: dimensionless [0,1]-like quality terms.\n        if (\n            stats[\'score\'] > best_score\n            or (\n                np.isclose(stats[\'score\'], best_score)\n                and stats[\'median_5cm_m\'] < best_median\n            )\n        ):\n            best_T = T\n            best_stats = stats\n            best_score = stats[\'score\']\n            best_median = stats[\'median_5cm_m\']\n\n        if iterations_run >= int(min_iters) and best_stats is not None:\n            if (\n                best_stats[\'ratio_5cm\'] >= float(early_ratio)\n                and best_stats[\'median_5cm_m\'] <= float(early_median_m)\n            ):\n                break\n\n    if best_T is None:\n        return None\n\n    # 5 cm weighted refit.\n    residual = np.linalg.norm(v2_transform_points(best_T, P) - Q, axis=1)\n    in5 = residual <= V3_RANSAC_THRESHOLD_M\n    if int(in5.sum()) >= 3:\n        T_refit = v2_weighted_kabsch(P[in5], Q[in5], confidence[in5])\n    else:\n        T_refit = best_T\n\n    # Tight 3 cm refit when enough strong pairs remain.\n    residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\n    in3 = residual <= V3_TIGHT_REFIT_THRESHOLD_M\n    if int(in3.sum()) >= 6:\n        T_refit = v2_weighted_kabsch(P[in3], Q[in3], confidence[in3])\n\n    final_stats = v3_consensus_stats(T_refit, P, Q, confidence)\n    final_stats.update({\n        \'T\': T_refit,\n        \'iterations\': int(iterations_run),\n        \'runtime_s\': float(time.perf_counter() - start),\n    })\n    return final_stats\n\ndef v3_should_trigger_fallback(ransac_result):\n    reasons = []\n    if ransac_result is None:\n        return True, [\'fast_ransac_failed\']\n\n    if int(ransac_result[\'iterations\']) >= int(V3_FAST_RANSAC_MAX_ITERS):\n        reasons.append(\'fast_hit_max_iters\')\n    if float(ransac_result[\'ratio_5cm\']) < float(V3_HARD_TRIGGER_RATIO_5CM):\n        reasons.append(\'fast_ratio5_low\')\n    if float(ransac_result[\'ratio_3cm\']) < float(V3_HARD_TRIGGER_RATIO_3CM):\n        reasons.append(\'fast_ratio3_low\')\n\n    return bool(reasons), reasons\n\n# Original notebook cell 66\ndef v32_build_hard_candidates(src_desc, tgt_desc, top_k=V32_HARD_TOPK):\n    """Build mutual Top-K pairs and ambiguity features in ONE similarity pass."""\n    src = v2_normalize_desc(src_desc)\n    tgt = v2_normalize_desc(tgt_desc)\n    sim = src @ tgt.T\n    ns, nt = sim.shape\n\n    if ns == 0 or nt == 0:\n        zf = np.empty((0,), dtype=np.float64)\n        zi = np.empty((0,), dtype=np.int64)\n        return np.empty((0, 2), dtype=np.int64), zf, {\n            \'src_margin12\': zf, \'tgt_margin12\': zf,\n            \'margin_quality\': zf, \'src_rank\': zi, \'tgt_rank\': zi,\n            \'rank_quality\': zf,\n        }\n\n    ks = min(max(2, int(top_k)), nt)\n    kt = min(max(2, int(top_k)), ns)\n\n    # Partial top-k, then sort only within top-k for exact ranks/margins.\n    src_idx = np.argpartition(-sim, kth=ks - 1, axis=1)[:, :ks]\n    src_val = np.take_along_axis(sim, src_idx, axis=1)\n    src_ord = np.argsort(-src_val, axis=1)\n    src_idx = np.take_along_axis(src_idx, src_ord, axis=1)\n    src_val = np.take_along_axis(src_val, src_ord, axis=1)\n\n    sim_t = sim.T\n    tgt_idx = np.argpartition(-sim_t, kth=kt - 1, axis=1)[:, :kt]\n    tgt_val = np.take_along_axis(sim_t, tgt_idx, axis=1)\n    tgt_ord = np.argsort(-tgt_val, axis=1)\n    tgt_idx = np.take_along_axis(tgt_idx, tgt_ord, axis=1)\n    tgt_val = np.take_along_axis(tgt_val, tgt_ord, axis=1)\n\n    src_margin_all = np.maximum(src_val[:, 0] - src_val[:, 1], 0.0)\n    tgt_margin_all = np.maximum(tgt_val[:, 0] - tgt_val[:, 1], 0.0)\n\n    # Rank lookup only for retained Top-K entries.\n    src_rank = np.full((ns, nt), int(top_k), dtype=np.int16)\n    tgt_rank = np.full((ns, nt), int(top_k), dtype=np.int16)\n    for r in range(ks):\n        src_rank[np.arange(ns), src_idx[:, r]] = r\n    for r in range(kt):\n        tgt_rank[tgt_idx[:, r], np.arange(nt)] = r\n\n    pairs = []\n    pair_sim = []\n    for s in range(ns):\n        for t in src_idx[s]:\n            t = int(t)\n            if tgt_rank[s, t] >= int(top_k):\n                continue\n            pairs.append((s, t))\n            pair_sim.append(float(sim[s, t]))\n\n    if not pairs:\n        zf = np.empty((0,), dtype=np.float64)\n        zi = np.empty((0,), dtype=np.int64)\n        return np.empty((0, 2), dtype=np.int64), zf, {\n            \'src_margin12\': zf, \'tgt_margin12\': zf,\n            \'margin_quality\': zf, \'src_rank\': zi, \'tgt_rank\': zi,\n            \'rank_quality\': zf,\n        }\n\n    pairs = np.asarray(pairs, dtype=np.int64)\n    pair_sim = np.asarray(pair_sim, dtype=np.float64)\n    sidx, tidx = pairs[:, 0], pairs[:, 1]\n\n    sm = src_margin_all[sidx]\n    tm = tgt_margin_all[tidx]\n    smq = np.clip(sm / max(V32_MARGIN_FULL_SCALE, 1e-12), 0.0, 1.0)\n    tmq = np.clip(tm / max(V32_MARGIN_FULL_SCALE, 1e-12), 0.0, 1.0)\n    margin_q = np.sqrt(smq * tmq)\n\n    sr = src_rank[sidx, tidx].astype(np.int64)\n    tr = tgt_rank[sidx, tidx].astype(np.int64)\n    denom = max(2.0 * (int(top_k) - 1), 1.0)\n    rank_q = np.clip(1.0 - (sr + tr) / denom, 0.0, 1.0)\n\n    ambiguity = {\n        \'src_margin12\': sm.astype(np.float64),\n        \'tgt_margin12\': tm.astype(np.float64),\n        \'margin_quality\': margin_q.astype(np.float64),\n        \'src_rank\': sr,\n        \'tgt_rank\': tr,\n        \'rank_quality\': rank_q.astype(np.float64),\n    }\n    return pairs, pair_sim, ambiguity\n\ndef v32_hard_quality(base_confidence, ambiguity):\n    base = np.clip(np.asarray(base_confidence, dtype=np.float64), 0.0, 1.0)\n    mq = np.clip(np.asarray(ambiguity[\'margin_quality\'], dtype=np.float64), 0.0, 1.0)\n    rq = np.clip(np.asarray(ambiguity[\'rank_quality\'], dtype=np.float64), 0.0, 1.0)\n    w = V32_HARD_QUALITY_WEIGHTS\n    return (\n        w[\'base_confidence\'] * base\n        + w[\'margin_quality\'] * mq\n        + w[\'rank_quality\'] * rq\n    )\n\ndef v32_greedy_unique_with_indices(pairs, quality, support, min_support=V32_MIN_SUPPORT):\n    pairs = np.asarray(pairs, dtype=np.int64)\n    quality = np.asarray(quality, dtype=np.float64)\n    support = np.asarray(support, dtype=np.int64)\n    valid = np.flatnonzero(support >= int(min_support))\n    if len(valid) == 0:\n        e = np.empty((0,), dtype=np.int64)\n        return np.empty((0, 2), dtype=np.int64), np.empty((0,)), np.empty((0,), dtype=np.int64), e\n\n    order_local = np.lexsort((-support[valid], -quality[valid]))\n    order = valid[order_local]\n    used_s, used_t, keep = set(), set(), []\n    for idx in order:\n        s, t = map(int, pairs[idx])\n        if s in used_s or t in used_t:\n            continue\n        used_s.add(s); used_t.add(t); keep.append(int(idx))\n    keep = np.asarray(keep, dtype=np.int64)\n    return pairs[keep], quality[keep], support[keep], keep\n\ndef v32_top_quality_prefilter(pairs, quality, support, source_indices, max_pairs):\n    pairs = np.asarray(pairs, dtype=np.int64)\n    quality = np.asarray(quality, dtype=np.float64)\n    support = np.asarray(support, dtype=np.int64)\n    source_indices = np.asarray(source_indices, dtype=np.int64)\n    if len(pairs) <= int(max_pairs):\n        return pairs, quality, support, source_indices\n    order = np.argsort(-quality)[:int(max_pairs)]\n    return pairs[order], quality[order], support[order], source_indices[order]\n\ndef v32_ambiguity_diag(ambiguity, idx, prefix=\'hard_\'):\n    idx = np.asarray(idx, dtype=np.int64)\n    if len(idx) == 0:\n        return {\n            f\'{prefix}src_margin12_mean\': np.nan,\n            f\'{prefix}tgt_margin12_mean\': np.nan,\n            f\'{prefix}margin_quality_mean\': np.nan,\n            f\'{prefix}rank_quality_mean\': np.nan,\n            f\'{prefix}mutual_top1_rate\': np.nan,\n        }\n    top1 = (ambiguity[\'src_rank\'][idx] == 0) & (ambiguity[\'tgt_rank\'][idx] == 0)\n    return {\n        f\'{prefix}src_margin12_mean\': float(np.mean(ambiguity[\'src_margin12\'][idx])),\n        f\'{prefix}tgt_margin12_mean\': float(np.mean(ambiguity[\'tgt_margin12\'][idx])),\n        f\'{prefix}margin_quality_mean\': float(np.mean(ambiguity[\'margin_quality\'][idx])),\n        f\'{prefix}rank_quality_mean\': float(np.mean(ambiguity[\'rank_quality\'][idx])),\n        f\'{prefix}mutual_top1_rate\': float(np.mean(top1)),\n    }\n\ndef v32_rotation_angle_deg(R):\n    R = np.asarray(R, dtype=np.float64)\n    c = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)\n    return float(np.degrees(np.arccos(c)))\n\ndef v32_fractional_rotation(R_delta, alpha):\n    """Return R_delta ** alpha for a small rotation using Rodrigues."""\n    R_delta = np.asarray(R_delta, dtype=np.float64)\n    c = np.clip((np.trace(R_delta) - 1.0) / 2.0, -1.0, 1.0)\n    theta = float(np.arccos(c))\n    if theta < 1e-10:\n        return np.eye(3, dtype=np.float64)\n\n    s = np.sin(theta)\n    if abs(s) < 1e-8:\n        # Trust-region deltas should never be near pi; safe SVD fallback.\n        U, _, Vt = np.linalg.svd((1.0 - alpha) * np.eye(3) + alpha * R_delta)\n        R = U @ Vt\n        if np.linalg.det(R) < 0:\n            U[:, -1] *= -1.0\n            R = U @ Vt\n        return R\n\n    axis = np.array([\n        R_delta[2, 1] - R_delta[1, 2],\n        R_delta[0, 2] - R_delta[2, 0],\n        R_delta[1, 0] - R_delta[0, 1],\n    ], dtype=np.float64) / (2.0 * s)\n    axis /= max(np.linalg.norm(axis), 1e-12)\n    a = float(alpha) * theta\n    K = np.array([\n        [0.0, -axis[2], axis[1]],\n        [axis[2], 0.0, -axis[0]],\n        [-axis[1], axis[0], 0.0],\n    ], dtype=np.float64)\n    return np.eye(3) + np.sin(a) * K + (1.0 - np.cos(a)) * (K @ K)\n\ndef v32_pose_delta(T_candidate, T_fast):\n    if T_candidate is None or T_fast is None:\n        return np.inf, np.inf\n    Rrel = T_candidate[:3, :3] @ T_fast[:3, :3].T\n    rot_deg = v32_rotation_angle_deg(Rrel)\n    trans_m = float(np.linalg.norm(T_candidate[:3, 3] - T_fast[:3, 3]))\n    return rot_deg, trans_m\n\ndef v32_blend_transform(T_fast, T_raw, alpha):\n    alpha = float(np.clip(alpha, 0.0, 1.0))\n    Rrel = T_raw[:3, :3] @ T_fast[:3, :3].T\n    Rstep = v32_fractional_rotation(Rrel, alpha)\n    T = np.eye(4, dtype=np.float64)\n    T[:3, :3] = Rstep @ T_fast[:3, :3]\n    T[:3, 3] = (1.0 - alpha) * T_fast[:3, 3] + alpha * T_raw[:3, 3]\n    return T\n\ndef v32_local_precision_score(stats):\n    if stats is None or not np.isfinite(stats.get(\'score\', np.nan)):\n        return -np.inf\n    w = V32_LOCAL_SCORE_WEIGHTS\n    return float(\n        w[\'weighted_ratio_5cm\'] * stats[\'weighted_ratio_5cm\']\n        + w[\'ratio_5cm\'] * stats[\'ratio_5cm\']\n        + w[\'ratio_3cm\'] * stats[\'ratio_3cm\']\n        + w[\'residual_quality\'] * stats[\'residual_quality\']\n    )\n\ndef v32_robust_local_raw_candidates(T_fast, P, Q, quality):\n    """Generate full / rotation-only / translation-only raw local candidates.\n\n    No random sampling is used. The current Fast pose defines a residual gate,\n    then Cauchy-like residual weights suppress remaining outliers.\n    """\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    quality = np.clip(np.asarray(quality, dtype=np.float64), 1e-8, None)\n\n    if T_fast is None or len(P) < V32_LOCAL_MIN_PAIRS:\n        return {}, {\'initial_gate_pairs\': 0, \'final_gate_pairs\': 0}\n\n    T_iter = np.asarray(T_fast, dtype=np.float64).copy()\n    initial_res = np.linalg.norm(v2_transform_points(T_fast, P) - Q, axis=1)\n    initial_gate = initial_res <= V32_LOCAL_INITIAL_GATE_M\n    if int(initial_gate.sum()) < V32_LOCAL_MIN_PAIRS:\n        return {}, {\n            \'initial_gate_pairs\': int(initial_gate.sum()),\n            \'final_gate_pairs\': int(initial_gate.sum()),\n            \'initial_residual_median_m\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\n        }\n\n    gate = initial_gate.copy()\n    last_w = None\n    for it in range(int(V32_LOCAL_IRLS_ITERS)):\n        res = np.linalg.norm(v2_transform_points(T_iter, P) - Q, axis=1)\n        frac = it / max(1, int(V32_LOCAL_IRLS_ITERS) - 1)\n        gate_thr = (\n            V32_LOCAL_INITIAL_GATE_M\n            + frac * (V32_LOCAL_FINAL_GATE_M - V32_LOCAL_INITIAL_GATE_M)\n        )\n        gate = res <= gate_thr\n        if int(gate.sum()) < V32_LOCAL_MIN_PAIRS:\n            break\n\n        robust = 1.0 / (1.0 + (res / max(V32_LOCAL_ROBUST_SCALE_M, 1e-12)) ** 2)\n        w = quality * robust\n        w[~gate] = 0.0\n        if int(np.sum(w > 0)) < V32_LOCAL_MIN_PAIRS:\n            break\n\n        try:\n            T_iter = v2_weighted_kabsch(P[gate], Q[gate], w[gate])\n            last_w = w\n        except Exception:\n            break\n\n    final_res = np.linalg.norm(v2_transform_points(T_iter, P) - Q, axis=1)\n    final_gate = final_res <= V32_LOCAL_FINAL_GATE_M\n    if int(final_gate.sum()) < V32_LOCAL_MIN_PAIRS:\n        final_gate = initial_gate.copy()\n\n    robust = 1.0 / (1.0 + (final_res / max(V32_LOCAL_ROBUST_SCALE_M, 1e-12)) ** 2)\n    final_w = quality * robust\n    final_w[~final_gate] = 0.0\n\n    if int(np.sum(final_w > 0)) < V32_LOCAL_MIN_PAIRS:\n        return {}, {\n            \'initial_gate_pairs\': int(initial_gate.sum()),\n            \'final_gate_pairs\': int(final_gate.sum()),\n            \'initial_residual_median_m\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\n            \'final_residual_median_m\': float(np.median(final_res[final_gate])) if np.any(final_gate) else np.nan,\n        }\n\n    # Full SE(3) candidate.\n    T_full = v2_weighted_kabsch(P[final_gate], Q[final_gate], final_w[final_gate])\n\n    # Rotation-only: use robust Kabsch rotation but preserve Fast translation.\n    T_rot = np.asarray(T_fast, dtype=np.float64).copy()\n    T_rot[:3, :3] = T_full[:3, :3]\n\n    # Translation-only: preserve Fast rotation and robustly estimate only t.\n    Rf = T_fast[:3, :3]\n    t_samples = Q[final_gate] - (Rf @ P[final_gate].T).T\n    wg = final_w[final_gate]\n    wg = wg / max(wg.sum(), 1e-12)\n    t_new = np.sum(t_samples * wg[:, None], axis=0)\n    T_trans = np.asarray(T_fast, dtype=np.float64).copy()\n    T_trans[:3, 3] = t_new\n\n    diag = {\n        \'initial_gate_pairs\': int(initial_gate.sum()),\n        \'final_gate_pairs\': int(final_gate.sum()),\n        \'initial_residual_median_m\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\n        \'final_residual_median_m\': float(np.median(final_res[final_gate])) if np.any(final_gate) else np.nan,\n    }\n    return {\n        \'full\': T_full,\n        \'rotation_only\': T_rot,\n        \'translation_only\': T_trans,\n    }, diag\n\ndef v32_select_local_candidate(T_fast, raw_candidates, VP, VQ, val_conf):\n    fast_stats = v3_consensus_stats(T_fast, VP, VQ, val_conf)\n    fast_precision = v32_local_precision_score(fast_stats)\n\n    best = {\n        \'T\': T_fast,\n        \'type\': \'fast\',\n        \'alpha\': 0.0,\n        \'stats\': fast_stats,\n        \'precision_score\': fast_precision,\n        \'regularized_score\': fast_precision,\n        \'gain\': 0.0,\n        \'delta_rot_deg\': 0.0,\n        \'delta_trans_m\': 0.0,\n        \'accepted\': False,\n    }\n\n    for ctype, T_raw in raw_candidates.items():\n        raw_rot, raw_trans = v32_pose_delta(T_raw, T_fast)\n        alpha_cap = 1.0\n        if np.isfinite(raw_rot) and raw_rot > 1e-12:\n            alpha_cap = min(alpha_cap, V32_TRUST_MAX_ROT_DEG / raw_rot)\n        if np.isfinite(raw_trans) and raw_trans > 1e-12:\n            alpha_cap = min(alpha_cap, V32_TRUST_MAX_TRANS_M / raw_trans)\n        alpha_cap = float(np.clip(alpha_cap, 0.0, 1.0))\n        if alpha_cap <= 1e-6:\n            continue\n\n        for frac in V32_LINESEARCH_FRACTIONS:\n            alpha = float(frac) * alpha_cap\n            T_cand = v32_blend_transform(T_fast, T_raw, alpha)\n            stats = v3_consensus_stats(T_cand, VP, VQ, val_conf)\n            precision = v32_local_precision_score(stats)\n            d_rot, d_trans = v32_pose_delta(T_cand, T_fast)\n\n            rot_norm = d_rot / max(V32_TRUST_MAX_ROT_DEG, 1e-12)\n            trans_norm = d_trans / max(V32_TRUST_MAX_TRANS_M, 1e-12)\n            reg_score = (\n                precision\n                - V32_STEP_PENALTY_ROT * rot_norm\n                - V32_STEP_PENALTY_TRANS * trans_norm\n            )\n            gain = reg_score - fast_precision\n\n            # Precision/consensus guards. These are GT-free.\n            guards_ok = (\n                stats[\'ratio_5cm\'] >= fast_stats[\'ratio_5cm\'] - V32_MAX_RATIO5_DROP\n                and stats[\'ratio_3cm\'] >= fast_stats[\'ratio_3cm\'] - V32_MAX_RATIO3_DROP\n                and stats[\'median_5cm_m\'] <= fast_stats[\'median_5cm_m\'] + V32_MAX_MEDIAN5_INCREASE_M\n            )\n\n            if (\n                guards_ok\n                and gain >= V32_MIN_REGULARIZED_GAIN\n                and reg_score > best[\'regularized_score\']\n            ):\n                best = {\n                    \'T\': T_cand,\n                    \'type\': ctype,\n                    \'alpha\': alpha,\n                    \'stats\': stats,\n                    \'precision_score\': precision,\n                    \'regularized_score\': reg_score,\n                    \'gain\': gain,\n                    \'delta_rot_deg\': d_rot,\n                    \'delta_trans_m\': d_trans,\n                    \'accepted\': True,\n                }\n\n    return best, fast_stats, fast_precision\n\n# Original notebook cell 67\ndef v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\n    t0 = time.perf_counter()\n\n    src_xyz = mv_world_to_camera_points(\n        v2_np(scache[\'xyz\']).astype(np.float64), src_c2w\n    )\n    tgt_xyz = mv_world_to_camera_points(\n        v2_np(tcache[\'xyz\']).astype(np.float64), tgt_c2w\n    )\n\n    out = {\n        \'T_est\': None,\n        \'T_fast_debug\': None,\n        \'T_local_debug\': None,\n        \'selected_stage\': \'failed\',\n        \'fallback_triggered\': False,\n        \'fallback_reason\': \'\',\n        \'fallback_selected\': False,\n        \'local_selected_type\': \'none\',\n        \'local_selected_alpha\': 0.0,\n        \'mean_src_k\': np.nan,\n        \'mean_tgt_k\': np.nan,\n        \'fast_candidate_pairs\': 0,\n        \'fast_rigidity_pairs\': 0,\n        \'fast_unique_pairs\': 0,\n        \'fast_coverage_strict_pairs\': 0,\n        \'fast_coverage_filled_pairs\': 0,\n        \'fast_selected_pairs\': 0,\n        \'hard_candidate_pairs\': 0,\n        \'hard_rigidity_pairs\': 0,\n        \'hard_validation_pairs\': 0,\n        \'hard_unique_pairs\': 0,\n        \'hard_prefilter_pairs\': 0,\n        \'hard_coverage_strict_pairs\': 0,\n        \'hard_coverage_filled_pairs\': 0,\n        \'hard_selected_pairs\': 0,\n        \'local_initial_gate_pairs\': 0,\n        \'local_final_gate_pairs\': 0,\n        \'fast_validation_score\': np.nan,\n        \'fast_local_precision_score\': np.nan,\n        \'local_validation_score\': np.nan,\n        \'local_precision_score\': np.nan,\n        \'local_regularized_score_gain\': np.nan,\n        \'local_delta_rot_deg\': 0.0,\n        \'local_delta_trans_m\': 0.0,\n        \'matcher_runtime_s\': np.nan,\n    }\n\n    # ------------------------------------------------------------\n    # FAST PATH - same behavior as V3/V3.1.\n    # ------------------------------------------------------------\n    fast_pairs, fast_sim, src_k, tgt_k = v2_build_adaptive_bidirectional(\n        scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY]\n    )\n    out[\'fast_candidate_pairs\'] = int(len(fast_pairs))\n    out[\'mean_src_k\'] = float(np.mean(src_k)) if len(src_k) else np.nan\n    out[\'mean_tgt_k\'] = float(np.mean(tgt_k)) if len(tgt_k) else np.nan\n\n    fast_r = None\n    fast_unique_pairs = np.empty((0, 2), dtype=np.int64)\n\n    if len(fast_pairs) >= V32_MIN_CORRESPONDENCES:\n        fast_support = mv_topk_rigidity_support(\n            fast_pairs, src_xyz, tgt_xyz, threshold_m=V32_RIGIDITY_THRESHOLD_M\n        )\n        out[\'fast_rigidity_pairs\'] = int(np.sum(fast_support >= V32_MIN_SUPPORT))\n        fast_conf_all = v2_pair_quality(\n            scache, tcache, fast_pairs, fast_sim, fast_support\n        )\n        fast_unique_pairs, fast_unique_conf, fast_unique_support = (\n            v2_greedy_unique_by_confidence(\n                fast_pairs, fast_conf_all, fast_support,\n                min_support=V32_MIN_SUPPORT,\n            )\n        )\n        out[\'fast_unique_pairs\'] = int(len(fast_unique_pairs))\n        out.update(v3_conf_stats(fast_unique_conf, \'fast_unique_\'))\n\n        if len(fast_unique_pairs) >= V32_MIN_CORRESPONDENCES:\n            fp, fc, fs, f_cov = v3_spatial_coverage_select(\n                fast_unique_pairs, fast_unique_conf, fast_unique_support,\n                src_xyz, tgt_xyz,\n                max_pairs=V32_FAST_MAX_PAIRS,\n                max_per_source_voxel=V32_FAST_MAX_PER_SOURCE_VOXEL,\n                max_per_target_voxel=V32_FAST_MAX_PER_TARGET_VOXEL,\n            )\n            out[\'fast_coverage_strict_pairs\'] = f_cov[\'strict_coverage_pairs\']\n            out[\'fast_coverage_filled_pairs\'] = f_cov[\'filled_pairs\']\n            out[\'fast_selected_pairs\'] = int(len(fp))\n            out.update(v3_conf_stats(fc, \'fast_selected_\'))\n            out.update(v3_geometry_diag(fp, src_xyz, tgt_xyz, \'fast_\'))\n\n            if len(fp) >= V32_MIN_CORRESPONDENCES:\n                fast_r = v3_prosac_ransac(\n                    src_xyz[fp[:, 0]], tgt_xyz[fp[:, 1]], fc, seed,\n                    max_iters=V32_FAST_RANSAC_MAX_ITERS,\n                    min_iters=V32_FAST_RANSAC_MIN_ITERS,\n                    early_ratio=V32_FAST_EARLY_RATIO,\n                    early_median_m=V32_FAST_EARLY_MEDIAN_M,\n                )\n\n    if fast_r is not None:\n        out[\'T_fast_debug\'] = fast_r[\'T\']\n        for k in [\n            \'score\', \'ratio_5cm\', \'ratio_3cm\', \'weighted_ratio_5cm\',\n            \'median_5cm_m\', \'mean_5cm_m\', \'p90_all_m\', \'residual_quality\',\n            \'n_inliers_5cm\', \'n_inliers_3cm\', \'iterations\', \'runtime_s\',\n        ]:\n            out[f\'fast_ransac_{k}\'] = fast_r[k]\n    else:\n        for k in [\n            \'score\', \'ratio_5cm\', \'ratio_3cm\', \'weighted_ratio_5cm\',\n            \'median_5cm_m\', \'mean_5cm_m\', \'p90_all_m\', \'residual_quality\',\n        ]:\n            out[f\'fast_ransac_{k}\'] = np.nan\n        out[\'fast_ransac_n_inliers_5cm\'] = 0\n        out[\'fast_ransac_n_inliers_3cm\'] = 0\n        out[\'fast_ransac_iterations\'] = 0\n        out[\'fast_ransac_runtime_s\'] = 0.0\n\n    # Same hard-case detector.\n    reasons = []\n    if fast_r is None:\n        trigger = True\n        reasons = [\'fast_ransac_failed\']\n    else:\n        if int(fast_r[\'iterations\']) >= int(V32_FAST_RANSAC_MAX_ITERS):\n            reasons.append(\'fast_hit_max_iters\')\n        if float(fast_r[\'ratio_5cm\']) < float(V32_HARD_TRIGGER_RATIO_5CM):\n            reasons.append(\'fast_ratio5_low\')\n        if float(fast_r[\'ratio_3cm\']) < float(V32_HARD_TRIGGER_RATIO_3CM):\n            reasons.append(\'fast_ratio3_low\')\n        trigger = bool(reasons)\n\n    if len(fast_unique_pairs) < V32_MIN_CORRESPONDENCES:\n        trigger = True\n        reasons = list(dict.fromkeys(reasons + [\'fast_too_few_unique_pairs\']))\n\n    out[\'fallback_triggered\'] = bool(trigger)\n    out[\'fallback_reason\'] = \'|\'.join(reasons)\n\n    selected_T = fast_r[\'T\'] if fast_r is not None else None\n    selected_stage = \'fast\' if selected_T is not None else \'failed\'\n\n    # ------------------------------------------------------------\n    # HARD LOCAL PATH - no hard RANSAC.\n    # ------------------------------------------------------------\n    local_choice = None\n    fast_validation = None\n\n    if trigger:\n        hard_pairs, hard_sim, ambiguity = v32_build_hard_candidates(\n            scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY],\n            top_k=V32_HARD_TOPK,\n        )\n        out[\'hard_candidate_pairs\'] = int(len(hard_pairs))\n\n        if len(hard_pairs) >= V32_MIN_CORRESPONDENCES:\n            hard_support = mv_topk_rigidity_support(\n                hard_pairs, src_xyz, tgt_xyz,\n                threshold_m=V32_RIGIDITY_THRESHOLD_M,\n            )\n            out[\'hard_rigidity_pairs\'] = int(np.sum(hard_support >= V32_MIN_SUPPORT))\n\n            hard_base_conf = v2_pair_quality(\n                scache, tcache, hard_pairs, hard_sim, hard_support\n            )\n            hard_quality_all = v32_hard_quality(hard_base_conf, ambiguity)\n\n            # Common validation pool uses base confidence, not the local score.\n            val_pairs, val_conf, val_support = v2_greedy_unique_by_confidence(\n                hard_pairs, hard_base_conf, hard_support,\n                min_support=V32_MIN_SUPPORT,\n            )\n            out[\'hard_validation_pairs\'] = int(len(val_pairs))\n\n            hu_pairs, hu_q, hu_support, hu_idx = v32_greedy_unique_with_indices(\n                hard_pairs, hard_quality_all, hard_support,\n                min_support=V32_MIN_SUPPORT,\n            )\n            out[\'hard_unique_pairs\'] = int(len(hu_pairs))\n            out.update(v3_conf_stats(hu_q, \'hard_unique_quality_\'))\n            out.update(v32_ambiguity_diag(ambiguity, hu_idx, \'hard_unique_\'))\n\n            hp0, hq0, hs0, hidx0 = v32_top_quality_prefilter(\n                hu_pairs, hu_q, hu_support, hu_idx,\n                max_pairs=V32_HARD_PREFILTER_MAX_PAIRS,\n            )\n            out[\'hard_prefilter_pairs\'] = int(len(hp0))\n            out.update(v32_ambiguity_diag(ambiguity, hidx0, \'hard_prefilter_\'))\n\n            if len(hp0) >= V32_MIN_CORRESPONDENCES:\n                hp, hq, hs, h_cov = v3_spatial_coverage_select(\n                    hp0, hq0, hs0, src_xyz, tgt_xyz,\n                    max_pairs=V32_HARD_MAX_PAIRS,\n                    max_per_source_voxel=V32_HARD_MAX_PER_SOURCE_VOXEL,\n                    max_per_target_voxel=V32_HARD_MAX_PER_TARGET_VOXEL,\n                )\n                out[\'hard_coverage_strict_pairs\'] = h_cov[\'strict_coverage_pairs\']\n                out[\'hard_coverage_filled_pairs\'] = h_cov[\'filled_pairs\']\n                out[\'hard_selected_pairs\'] = int(len(hp))\n                out.update(v3_conf_stats(hq, \'hard_selected_quality_\'))\n                out.update(v3_geometry_diag(hp, src_xyz, tgt_xyz, \'hard_\'))\n\n                pair_to_idx = {\n                    (int(s), int(t)): int(idx)\n                    for (s, t), idx in zip(hp0.tolist(), hidx0.tolist())\n                }\n                hp_idx = np.asarray(\n                    [pair_to_idx[(int(s), int(t))] for s, t in hp],\n                    dtype=np.int64,\n                ) if len(hp) else np.empty((0,), dtype=np.int64)\n                out.update(v32_ambiguity_diag(ambiguity, hp_idx, \'hard_selected_\'))\n\n                # Local refinement only makes sense when Fast succeeded.\n                if fast_r is not None and len(hp) >= V32_LOCAL_MIN_PAIRS:\n                    HP = src_xyz[hp[:, 0]]\n                    HQ = tgt_xyz[hp[:, 1]]\n                    raw_candidates, local_diag = v32_robust_local_raw_candidates(\n                        fast_r[\'T\'], HP, HQ, hq\n                    )\n                    out[\'local_initial_gate_pairs\'] = local_diag.get(\'initial_gate_pairs\', 0)\n                    out[\'local_final_gate_pairs\'] = local_diag.get(\'final_gate_pairs\', 0)\n                    out[\'local_initial_residual_median_m\'] = local_diag.get(\'initial_residual_median_m\', np.nan)\n                    out[\'local_final_residual_median_m\'] = local_diag.get(\'final_residual_median_m\', np.nan)\n\n                    if len(val_pairs) >= V32_MIN_CORRESPONDENCES and raw_candidates:\n                        VP = src_xyz[val_pairs[:, 0]]\n                        VQ = tgt_xyz[val_pairs[:, 1]]\n                        local_choice, fast_validation, fast_precision = v32_select_local_candidate(\n                            fast_r[\'T\'], raw_candidates, VP, VQ, val_conf\n                        )\n                        out[\'fast_validation_score\'] = fast_validation[\'score\']\n                        out[\'fast_local_precision_score\'] = fast_precision\n                        out[\'local_validation_score\'] = local_choice[\'stats\'][\'score\']\n                        out[\'local_precision_score\'] = local_choice[\'precision_score\']\n                        out[\'local_regularized_score_gain\'] = local_choice[\'gain\']\n                        out[\'local_delta_rot_deg\'] = local_choice[\'delta_rot_deg\']\n                        out[\'local_delta_trans_m\'] = local_choice[\'delta_trans_m\']\n                        out[\'local_selected_type\'] = local_choice[\'type\']\n                        out[\'local_selected_alpha\'] = local_choice[\'alpha\']\n\n                        if local_choice[\'accepted\']:\n                            selected_T = local_choice[\'T\']\n                            selected_stage = \'local\'\n                            out[\'fallback_selected\'] = True\n                            out[\'T_local_debug\'] = selected_T\n\n    out.setdefault(\'local_initial_residual_median_m\', np.nan)\n    out.setdefault(\'local_final_residual_median_m\', np.nan)\n\n    out[\'selected_stage\'] = selected_stage\n    out[\'T_est\'] = selected_T\n\n    # Fast RANSAC remains the only random robust estimation stage in V3.2.\n    if fast_r is not None:\n        out[\'selected_pairs\'] = int(out[\'fast_selected_pairs\'])\n        out[\'selected_ransac_score\'] = float(fast_r[\'score\'])\n        out[\'selected_ransac_iterations\'] = int(fast_r[\'iterations\'])\n        out[\'selected_ransac_runtime_s\'] = float(fast_r[\'runtime_s\'])\n    else:\n        out[\'selected_pairs\'] = 0\n        out[\'selected_ransac_score\'] = np.nan\n        out[\'selected_ransac_iterations\'] = 0\n        out[\'selected_ransac_runtime_s\'] = 0.0\n\n    # Report consensus of the actually selected pose on the validation pool\n    # when available; otherwise fall back to Fast RANSAC\'s own pool metrics.\n    if trigger and fast_r is not None and out[\'hard_validation_pairs\'] >= V32_MIN_CORRESPONDENCES:\n        # Rebuild validation arrays only if they exist in this scope.\n        if \'val_pairs\' in locals() and len(val_pairs) >= V32_MIN_CORRESPONDENCES:\n            VP = src_xyz[val_pairs[:, 0]]\n            VQ = tgt_xyz[val_pairs[:, 1]]\n            selected_stats = v3_consensus_stats(selected_T, VP, VQ, val_conf)\n            out[\'selected_inlier_ratio_5cm\'] = float(selected_stats[\'ratio_5cm\'])\n            out[\'selected_inlier_ratio_3cm\'] = float(selected_stats[\'ratio_3cm\'])\n            out[\'selected_weighted_ratio_5cm\'] = float(selected_stats[\'weighted_ratio_5cm\'])\n            out[\'selected_median_5cm_m\'] = float(selected_stats[\'median_5cm_m\'])\n            out[\'selected_validation_score\'] = float(selected_stats[\'score\'])\n        else:\n            out[\'selected_inlier_ratio_5cm\'] = float(fast_r[\'ratio_5cm\'])\n            out[\'selected_inlier_ratio_3cm\'] = float(fast_r[\'ratio_3cm\'])\n            out[\'selected_weighted_ratio_5cm\'] = float(fast_r[\'weighted_ratio_5cm\'])\n            out[\'selected_median_5cm_m\'] = float(fast_r[\'median_5cm_m\'])\n            out[\'selected_validation_score\'] = np.nan\n    elif fast_r is not None:\n        out[\'selected_inlier_ratio_5cm\'] = float(fast_r[\'ratio_5cm\'])\n        out[\'selected_inlier_ratio_3cm\'] = float(fast_r[\'ratio_3cm\'])\n        out[\'selected_weighted_ratio_5cm\'] = float(fast_r[\'weighted_ratio_5cm\'])\n        out[\'selected_median_5cm_m\'] = float(fast_r[\'median_5cm_m\'])\n        out[\'selected_validation_score\'] = np.nan\n    else:\n        out[\'selected_inlier_ratio_5cm\'] = 0.0\n        out[\'selected_inlier_ratio_3cm\'] = 0.0\n        out[\'selected_weighted_ratio_5cm\'] = 0.0\n        out[\'selected_median_5cm_m\'] = np.nan\n        out[\'selected_validation_score\'] = np.nan\n\n    out[\'matcher_runtime_s\'] = float(time.perf_counter() - t0)\n    return out\n\n# Original notebook cell 69\nICP_DISTANCE_THRESHOLD_M = 0.005\n\nICP_MAX_ITER = 500\n\nICP_REL_FITNESS = 1e-9\n\nICP_REL_RMSE = 1e-9\n\ndef _as_np_pose(x):\n    if torch.is_tensor(x):\n        x = x.detach().cpu().numpy()\n    return np.asarray(x, dtype=np.float64)\n\ndef loop_to_submap_start_transform(T_loop, source_sub, target_sub, source_frame, target_frame):\n    Cs0 = _as_np_pose(source_sub[\'submap_c2ws\'][0])\n    Ct0 = _as_np_pose(target_sub[\'submap_c2ws\'][0])\n    CsL = mv_frame_c2w(source_sub, source_frame)\n    CtL = mv_frame_c2w(target_sub, target_frame)\n    # PsL = inv(CsL) Cs0 Ps0; PtL = inv(CtL) Ct0 Pt0\n    return np.linalg.inv(Ct0) @ CtL @ np.asarray(T_loop) @ np.linalg.inv(CsL) @ Cs0\n\ndef submap_start_to_loop_transform(T_start, source_sub, target_sub, source_frame, target_frame):\n    Cs0 = _as_np_pose(source_sub[\'submap_c2ws\'][0])\n    Ct0 = _as_np_pose(target_sub[\'submap_c2ws\'][0])\n    CsL = mv_frame_c2w(source_sub, source_frame)\n    CtL = mv_frame_c2w(target_sub, target_frame)\n    As = np.linalg.inv(CsL) @ Cs0\n    At = np.linalg.inv(CtL) @ Ct0\n    return At @ np.asarray(T_start) @ np.linalg.inv(As)\n\ndef transform_delta(T_a, T_b):\n    D = np.asarray(T_a) @ np.linalg.inv(np.asarray(T_b))\n    c = np.clip((np.trace(D[:3, :3]) - 1.0) / 2.0, -1.0, 1.0)\n    rot_deg = float(np.degrees(np.arccos(c)))\n    trans_cm = float(np.linalg.norm(D[:3, 3]) * 100.0)\n    return rot_deg, trans_cm\n\ndef dense_cloud_for_submap(agent_id, submap_index):\n    key = (int(agent_id), int(submap_index))\n    if key in _dense_cloud_cache:\n        return _dense_cloud_cache[key]\n    sub = mv_submaps[key[0]][key[1]]\n    t0 = time.perf_counter()\n    color, depth = repo_utils.get_rgbd(sub)\n    cloud = repo_utils.rgbd2ptcloud(\n        color, depth, sub[\'intrinsics\'], np.eye(4)\n    )\n    cloud.estimate_normals()\n    info = {\n        \'points\': int(len(cloud.points)),\n        \'build_time_s\': float(time.perf_counter() - t0),\n    }\n    _dense_cloud_cache[key] = (cloud, info)\n    print(\'[dense cloud]\', key, \'| points=\', info[\'points\'], \'| build_s=\', f"{info[\'build_time_s\']:.2f}")\n    return _dense_cloud_cache[key]\n\ndef exact_production_icp(source_cloud, target_cloud, T_init):\n    initial_eval = o3d.pipelines.registration.evaluate_registration(\n        source_cloud, target_cloud, ICP_DISTANCE_THRESHOLD_M, np.asarray(T_init)\n    )\n    t0 = time.perf_counter()\n    result = o3d.pipelines.registration.registration_icp(\n        source_cloud,\n        target_cloud,\n        ICP_DISTANCE_THRESHOLD_M,\n        np.asarray(T_init),\n        o3d.pipelines.registration.TransformationEstimationPointToPlane(),\n        o3d.pipelines.registration.ICPConvergenceCriteria(\n            max_iteration=ICP_MAX_ITER,\n            relative_fitness=ICP_REL_FITNESS,\n            relative_rmse=ICP_REL_RMSE,\n        ),\n    )\n    return {\n        \'T_final\': np.asarray(result.transformation, dtype=np.float64),\n        \'initial_fitness_5mm\': float(initial_eval.fitness),\n        \'initial_rmse_5mm_m\': float(initial_eval.inlier_rmse),\n        \'final_fitness_5mm\': float(result.fitness),\n        \'final_rmse_5mm_m\': float(result.inlier_rmse),\n        \'runtime_s\': float(time.perf_counter() - t0),\n    }\n\n# Original notebook cell 71\nV33_HOLDOUT_FRACTION = 0.30\n\nV33_HOLDOUT_VOXEL_M = 0.12\n\nV33_MIN_TRAIN_PAIRS = 18\n\nV33_MIN_VALIDATION_PAIRS = 8\n\nV33_IRLS_ITERS = 4\n\nV33_ROBUST_SCALE_M = 0.030\n\nV33_INITIAL_GATE_M = 0.080\n\nV33_FINAL_GATE_M = 0.050\n\nV33_LEVERAGE_STRENGTH = 0.30\n\nV33_OBS_MID_EIGEN_RATIO_MIN = 0.050\n\nV33_OBS_RMS_SPREAD_MIN_M = 0.080\n\nV33_TRUST_MAX_ROT_DEG = float(V32_TRUST_MAX_ROT_DEG)\n\nV33_TRUST_MAX_TRANS_M = float(V32_TRUST_MAX_TRANS_M)\n\nV33_LINESEARCH_FRACTIONS = (0.25, 0.50, 0.75, 1.00)\n\nV33_MIN_REGULARIZED_GAIN = 0.0015\n\nV33_MAX_RATIO5_DROP = 0.005\n\nV33_MAX_RATIO3_DROP = 0.005\n\nV33_MAX_MEDIAN5_INCREASE_M = 0.0010\n\nV33_STEP_PENALTY_ROT = 0.0010\n\nV33_STEP_PENALTY_TRANS = 0.0010\n\nV33_CYCLE_PENALTY_ROT = 0.0010\n\nV33_CYCLE_PENALTY_TRANS = 0.0010\n\nV33_HARD_RANSAC_MAX_ITERS = int(V3_HARD_RANSAC_MAX_ITERS)\n\nV33_HARD_RANSAC_MIN_ITERS = int(V3_HARD_RANSAC_MIN_ITERS)\n\nV33_HARD_EARLY_RATIO = float(V3_HARD_EARLY_RATIO)\n\nV33_HARD_EARLY_MEDIAN_M = float(V3_HARD_EARLY_MEDIAN_M)\n\n# Original notebook cell 73\ndef v33_group_key(pair, src_xyz, tgt_xyz, voxel_m=V33_HOLDOUT_VOXEL_M):\n    s, t = map(int, pair)\n    sv = np.floor(np.asarray(src_xyz[s], dtype=np.float64) / float(voxel_m)).astype(np.int64)\n    tv = np.floor(np.asarray(tgt_xyz[t], dtype=np.float64) / float(voxel_m)).astype(np.int64)\n    return tuple(sv.tolist() + tv.tolist())\n\ndef v33_group_hash(key, seed):\n    payload = (str(int(seed)) + \'|\' + \',\'.join(map(str, key))).encode(\'utf-8\')\n    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), \'little\')\n\ndef v33_spatial_holdout_split(pairs, src_xyz, tgt_xyz, seed):\n    pairs = np.asarray(pairs, dtype=np.int64)\n    n = len(pairs)\n    if n < V33_MIN_TRAIN_PAIRS + V33_MIN_VALIDATION_PAIRS:\n        return None, None, {\'reason\': \'too_few_pairs\', \'n\': n}\n\n    groups = {}\n    for idx, pair in enumerate(pairs):\n        key = v33_group_key(pair, src_xyz, tgt_xyz)\n        groups.setdefault(key, []).append(idx)\n\n    group_items = []\n    for key, idxs in groups.items():\n        group_items.append((v33_group_hash(key, seed), key, np.asarray(idxs, dtype=np.int64)))\n    group_items.sort(key=lambda x: x[0])\n\n    target_val = max(V33_MIN_VALIDATION_PAIRS, int(round(V33_HOLDOUT_FRACTION * n)))\n    target_val = min(target_val, n - V33_MIN_TRAIN_PAIRS)\n\n    val = []\n    train = []\n    val_count = 0\n    for _, _, idxs in group_items:\n        # Fill validation first by whole spatial groups; keep enough room for train.\n        can_take_val = (val_count < target_val) and (n - (val_count + len(idxs)) >= V33_MIN_TRAIN_PAIRS)\n        if can_take_val:\n            val.extend(idxs.tolist())\n            val_count += len(idxs)\n        else:\n            train.extend(idxs.tolist())\n\n    # Repair only if group granularity produced too small a split.\n    if len(val) < V33_MIN_VALIDATION_PAIRS:\n        movable = [x for x in group_items if any(i in train for i in x[2].tolist())]\n        for _, _, idxs in movable:\n            if len(train) - len(idxs) < V33_MIN_TRAIN_PAIRS:\n                continue\n            for i in idxs.tolist():\n                if i in train:\n                    train.remove(i)\n                    val.append(i)\n            if len(val) >= V33_MIN_VALIDATION_PAIRS:\n                break\n\n    train = np.asarray(sorted(set(train)), dtype=np.int64)\n    val = np.asarray(sorted(set(val)), dtype=np.int64)\n    if len(train) < V33_MIN_TRAIN_PAIRS or len(val) < V33_MIN_VALIDATION_PAIRS:\n        return None, None, {\n            \'reason\': \'split_failed\', \'n\': n,\n            \'train\': int(len(train)), \'val\': int(len(val)),\n            \'groups\': int(len(groups)),\n        }\n\n    return train, val, {\n        \'reason\': \'ok\', \'n\': n,\n        \'train\': int(len(train)), \'val\': int(len(val)),\n        \'groups\': int(len(groups)),\n        \'val_fraction\': float(len(val) / n),\n    }\n\ndef v33_weighted_covariance(X, w):\n    X = np.asarray(X, dtype=np.float64)\n    w = np.clip(np.asarray(w, dtype=np.float64), 1e-12, None)\n    w = w / max(w.sum(), 1e-12)\n    c = np.sum(X * w[:, None], axis=0)\n    X0 = X - c\n    C = (X0 * w[:, None]).T @ X0\n    return C, c\n\ndef v33_observability(P, Q, w):\n    Cp, cp = v33_weighted_covariance(P, w)\n    Cq, cq = v33_weighted_covariance(Q, w)\n    ep = np.clip(np.linalg.eigvalsh(Cp), 0.0, None)\n    eq = np.clip(np.linalg.eigvalsh(Cq), 0.0, None)\n    p_mid = float(ep[-2] / max(ep[-1], 1e-12)) if len(ep) >= 2 else 0.0\n    q_mid = float(eq[-2] / max(eq[-1], 1e-12)) if len(eq) >= 2 else 0.0\n    p_spread = float(np.sqrt(max(np.trace(Cp), 0.0)))\n    q_spread = float(np.sqrt(max(np.trace(Cq), 0.0)))\n    mid_ratio = min(p_mid, q_mid)\n    rms_spread = min(p_spread, q_spread)\n    rot_observable = bool(\n        mid_ratio >= V33_OBS_MID_EIGEN_RATIO_MIN\n        and rms_spread >= V33_OBS_RMS_SPREAD_MIN_M\n    )\n    return {\n        \'src_eigs\': ep, \'tgt_eigs\': eq,\n        \'mid_eigen_ratio\': mid_ratio,\n        \'rms_spread_m\': rms_spread,\n        \'rotation_observable\': rot_observable,\n    }\n\ndef v33_leverage_weights(P, Q, base_w):\n    P = np.asarray(P, dtype=np.float64)\n    Q = np.asarray(Q, dtype=np.float64)\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\n    _, cp = v33_weighted_covariance(P, base_w)\n    _, cq = v33_weighted_covariance(Q, base_w)\n    rp = np.linalg.norm(P - cp, axis=1)\n    rq = np.linalg.norm(Q - cq, axis=1)\n    scale_p = max(float(np.percentile(rp, 75)), 1e-6)\n    scale_q = max(float(np.percentile(rq, 75)), 1e-6)\n    leverage = 0.5 * (rp / scale_p + rq / scale_q)\n    leverage = np.clip(leverage, 0.25, 2.0)\n    multiplier = (1.0 - V33_LEVERAGE_STRENGTH) + V33_LEVERAGE_STRENGTH * leverage\n    return base_w * multiplier, leverage\n\n# Original notebook cell 75\ndef v33_cauchy_weights(residual_m, scale_m=V33_ROBUST_SCALE_M):\n    residual_m = np.asarray(residual_m, dtype=np.float64)\n    x = residual_m / max(float(scale_m), 1e-9)\n    return 1.0 / (1.0 + x * x)\n\ndef v33_gate_threshold(iter_idx):\n    frac = iter_idx / max(V33_IRLS_ITERS - 1, 1)\n    return V33_INITIAL_GATE_M + frac * (V33_FINAL_GATE_M - V33_INITIAL_GATE_M)\n\ndef v33_irls_full(T_init, P, Q, base_w):\n    T = np.asarray(T_init, dtype=np.float64).copy()\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\n    for it in range(V33_IRLS_ITERS):\n        res = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\n        gate = res <= v33_gate_threshold(it)\n        if int(gate.sum()) < 6:\n            break\n        w = base_w * v33_cauchy_weights(res)\n        try:\n            T = v2_weighted_kabsch(P[gate], Q[gate], w[gate])\n        except Exception:\n            break\n    return T\n\ndef v33_irls_translation(T_init, P, Q, base_w):\n    T = np.asarray(T_init, dtype=np.float64).copy()\n    R = T[:3, :3].copy()\n    t = T[:3, 3].copy()\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\n    for it in range(V33_IRLS_ITERS):\n        pred = P @ R.T + t\n        res = np.linalg.norm(pred - Q, axis=1)\n        gate = res <= v33_gate_threshold(it)\n        if int(gate.sum()) < 6:\n            break\n        w = base_w * v33_cauchy_weights(res)\n        wg = w[gate]\n        rhs = Q[gate] - P[gate] @ R.T\n        t = np.sum(rhs * wg[:, None], axis=0) / max(wg.sum(), 1e-12)\n    Tout = np.eye(4, dtype=np.float64)\n    Tout[:3, :3] = R\n    Tout[:3, 3] = t\n    return Tout\n\ndef v33_weighted_rotation_fixed_translation(P, Q, t_fixed, w):\n    P = np.asarray(P, dtype=np.float64)\n    Q0 = np.asarray(Q, dtype=np.float64) - np.asarray(t_fixed, dtype=np.float64)[None, :]\n    w = np.clip(np.asarray(w, dtype=np.float64), 1e-12, None)\n    H = (P * w[:, None]).T @ Q0\n    U, _, Vt = np.linalg.svd(H)\n    R = Vt.T @ U.T\n    if np.linalg.det(R) < 0:\n        Vt[-1, :] *= -1.0\n        R = Vt.T @ U.T\n    return R\n\ndef v33_irls_rotation(T_init, P, Q, base_w):\n    T = np.asarray(T_init, dtype=np.float64).copy()\n    t = T[:3, 3].copy()\n    R = T[:3, :3].copy()\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\n    for it in range(V33_IRLS_ITERS):\n        pred = P @ R.T + t\n        res = np.linalg.norm(pred - Q, axis=1)\n        gate = res <= v33_gate_threshold(it)\n        if int(gate.sum()) < 6:\n            break\n        w = base_w * v33_cauchy_weights(res)\n        R = v33_weighted_rotation_fixed_translation(P[gate], Q[gate], t, w[gate])\n    Tout = np.eye(4, dtype=np.float64)\n    Tout[:3, :3] = R\n    Tout[:3, 3] = t\n    return Tout\n\ndef v33_alpha_cap(T_fast, T_raw):\n    dr, dt = v32_pose_delta(T_raw, T_fast)\n    cap = 1.0\n    if np.isfinite(dr) and dr > 1e-12:\n        cap = min(cap, V33_TRUST_MAX_ROT_DEG / dr)\n    if np.isfinite(dt) and dt > 1e-12:\n        cap = min(cap, V33_TRUST_MAX_TRANS_M / dt)\n    return float(np.clip(cap, 0.0, 1.0)), float(dr), float(dt)\n\ndef v33_validation_score(T, P, Q, w, T_cycle_reference=None, T_fast=None):\n    stats = v3_consensus_stats(T, P, Q, w)\n    precision = v32_local_precision_score(stats)\n    if T_fast is None:\n        step_r = step_t = 0.0\n    else:\n        step_r, step_t = v32_pose_delta(T, T_fast)\n\n    cycle_r = cycle_t = 0.0\n    if T_cycle_reference is not None:\n        cycle_r, cycle_t = v32_pose_delta(T, T_cycle_reference)\n\n    step_penalty = (\n        V33_STEP_PENALTY_ROT * min(step_r / max(V33_TRUST_MAX_ROT_DEG, 1e-9), 2.0)\n        + V33_STEP_PENALTY_TRANS * min(step_t / max(V33_TRUST_MAX_TRANS_M, 1e-9), 2.0)\n    )\n    cycle_penalty = (\n        V33_CYCLE_PENALTY_ROT * min(cycle_r / 1.0, 2.0)\n        + V33_CYCLE_PENALTY_TRANS * min(cycle_t / 0.03, 2.0)\n    )\n    regularized = float(precision - step_penalty - cycle_penalty)\n    return {\n        \'stats\': stats,\n        \'precision_score\': float(precision),\n        \'regularized_score\': regularized,\n        \'step_rot_deg\': float(step_r),\n        \'step_trans_m\': float(step_t),\n        \'cycle_rot_deg\': float(cycle_r),\n        \'cycle_trans_m\': float(cycle_t),\n    }\n\ndef v33_candidate_acceptable(base_eval, cand_eval):\n    b = base_eval[\'stats\']; c = cand_eval[\'stats\']\n    if not np.isfinite(cand_eval[\'regularized_score\']):\n        return False\n    if cand_eval[\'regularized_score\'] < base_eval[\'regularized_score\'] + V33_MIN_REGULARIZED_GAIN:\n        return False\n    if c[\'ratio_5cm\'] < b[\'ratio_5cm\'] - V33_MAX_RATIO5_DROP:\n        return False\n    if c[\'ratio_3cm\'] < b[\'ratio_3cm\'] - V33_MAX_RATIO3_DROP:\n        return False\n    if np.isfinite(b[\'median_5cm_m\']) and np.isfinite(c[\'median_5cm_m\']):\n        if c[\'median_5cm_m\'] > b[\'median_5cm_m\'] + V33_MAX_MEDIAN5_INCREASE_M:\n            return False\n    return True\n\n# Original notebook cell 77\ndef v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz):\n    hard_pairs, hard_sim, ambiguity = v32_build_hard_candidates(\n        scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY],\n        top_k=V32_HARD_TOPK,\n    )\n    if len(hard_pairs) < V32_MIN_CORRESPONDENCES:\n        return None\n\n    hard_support = mv_topk_rigidity_support(\n        hard_pairs, src_xyz, tgt_xyz,\n        threshold_m=V32_RIGIDITY_THRESHOLD_M,\n    )\n    hard_base_conf = v2_pair_quality(\n        scache, tcache, hard_pairs, hard_sim, hard_support\n    )\n    hard_quality_all = v32_hard_quality(hard_base_conf, ambiguity)\n\n    hu_pairs, hu_q, hu_support, hu_idx = v32_greedy_unique_with_indices(\n        hard_pairs, hard_quality_all, hard_support,\n        min_support=V32_MIN_SUPPORT,\n    )\n    if len(hu_pairs) < V32_MIN_CORRESPONDENCES:\n        return None\n\n    hp0, hq0, hs0, hidx0 = v32_top_quality_prefilter(\n        hu_pairs, hu_q, hu_support, hu_idx,\n        max_pairs=V32_HARD_PREFILTER_MAX_PAIRS,\n    )\n    if len(hp0) < V32_MIN_CORRESPONDENCES:\n        return None\n\n    hp, hq, hs, cov = v3_spatial_coverage_select(\n        hp0, hq0, hs0, src_xyz, tgt_xyz,\n        max_pairs=V32_HARD_MAX_PAIRS,\n        max_per_source_voxel=V32_HARD_MAX_PER_SOURCE_VOXEL,\n        max_per_target_voxel=V32_HARD_MAX_PER_TARGET_VOXEL,\n    )\n    if len(hp) < V32_MIN_CORRESPONDENCES:\n        return None\n\n    return {\n        \'pairs\': hp, \'quality\': hq, \'support\': hs,\n        \'candidate_pairs\': int(len(hard_pairs)),\n        \'selected_pairs\': int(len(hp)),\n        \'coverage\': cov,\n    }\n\ndef v33_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\n    start = time.perf_counter()\n    base = v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed)\n    out = dict(base)\n    out[\'method\'] = \'Bplus_mv_v3_3\'\n    out[\'v33_used_holdout\'] = False\n    out[\'v33_selected_type\'] = \'v32_passthrough\'\n    out[\'v33_validation_gain\'] = 0.0\n    out[\'v33_train_pairs\'] = 0\n    out[\'v33_validation_pairs\'] = 0\n    out[\'v33_holdout_groups\'] = 0\n    out[\'v33_rotation_observable\'] = False\n    out[\'v33_mid_eigen_ratio\'] = np.nan\n    out[\'v33_rms_spread_m\'] = np.nan\n    out[\'v33_cycle_rot_deg\'] = np.nan\n    out[\'v33_cycle_trans_m\'] = np.nan\n\n    # Easy case remains exactly V3.2.\n    if not bool(base.get(\'fallback_triggered\', False)):\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\'xyz\']).astype(np.float64), src_c2w)\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\'xyz\']).astype(np.float64), tgt_c2w)\n    pool = v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz)\n    if pool is None:\n        out[\'v33_selected_type\'] = \'v32_fallback_no_pool\'\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    hp = pool[\'pairs\']; hq = pool[\'quality\']\n    train_idx, val_idx, split_diag = v33_spatial_holdout_split(hp, src_xyz, tgt_xyz, seed)\n    if train_idx is None:\n        out[\'v33_selected_type\'] = \'v32_fallback_split_failed\'\n        out[\'v33_split_reason\'] = split_diag.get(\'reason\')\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    out[\'v33_used_holdout\'] = True\n    out[\'v33_train_pairs\'] = int(len(train_idx))\n    out[\'v33_validation_pairs\'] = int(len(val_idx))\n    out[\'v33_holdout_groups\'] = int(split_diag.get(\'groups\', 0))\n\n    TP = src_xyz[hp[train_idx, 0]]\n    TQ = tgt_xyz[hp[train_idx, 1]]\n    TW0 = np.clip(hq[train_idx], 1e-8, None)\n    VP = src_xyz[hp[val_idx, 0]]\n    VQ = tgt_xyz[hp[val_idx, 1]]\n    VW = np.clip(hq[val_idx], 1e-8, None)\n\n    TW, leverage = v33_leverage_weights(TP, TQ, TW0)\n    obs = v33_observability(TP, TQ, TW)\n    out[\'v33_rotation_observable\'] = bool(obs[\'rotation_observable\'])\n    out[\'v33_mid_eigen_ratio\'] = float(obs[\'mid_eigen_ratio\'])\n    out[\'v33_rms_spread_m\'] = float(obs[\'rms_spread_m\'])\n    out[\'v33_leverage_mean\'] = float(np.mean(leverage))\n    out[\'v33_leverage_p90\'] = float(np.percentile(leverage, 90))\n\n    T_fast = base.get(\'T_fast_debug\')\n    # Rare case: Fast failed. Build a robust anchor on TRAIN only.\n    if T_fast is None:\n        hard_anchor = v3_prosac_ransac(\n            TP, TQ, TW, seed,\n            max_iters=V33_HARD_RANSAC_MAX_ITERS,\n            min_iters=V33_HARD_RANSAC_MIN_ITERS,\n            early_ratio=V33_HARD_EARLY_RATIO,\n            early_median_m=V33_HARD_EARLY_MEDIAN_M,\n        )\n        if hard_anchor is None:\n            out[\'v33_selected_type\'] = \'v32_fallback_anchor_failed\'\n            out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n            return out\n        T_fast = hard_anchor[\'T\']\n        out[\'v33_anchor_from_hard_train_ransac\'] = True\n    else:\n        out[\'v33_anchor_from_hard_train_ransac\'] = False\n\n    # Reverse train-only fit for a soft bidirectional consistency reference.\n    try:\n        T_reverse = v33_irls_full(np.linalg.inv(T_fast), TQ, TP, TW)\n        T_cycle_reference = np.linalg.inv(T_reverse)\n    except Exception:\n        T_cycle_reference = None\n\n    base_eval = v33_validation_score(\n        T_fast, VP, VQ, VW,\n        T_cycle_reference=T_cycle_reference,\n        T_fast=T_fast,\n    )\n    out[\'v33_fast_validation_score\'] = float(base_eval[\'regularized_score\'])\n\n    raw = {\n        \'translation_irls\': v33_irls_translation(T_fast, TP, TQ, TW),\n    }\n    # Only allow hypotheses that change rotation when the geometry supports it.\n    if obs[\'rotation_observable\']:\n        raw[\'full_irls\'] = v33_irls_full(T_fast, TP, TQ, TW)\n        raw[\'rotation_irls\'] = v33_irls_rotation(T_fast, TP, TQ, TW)\n\n    candidates = []\n    for kind, T_raw in raw.items():\n        cap, raw_dr, raw_dt = v33_alpha_cap(T_fast, T_raw)\n        if cap <= 0:\n            continue\n        for frac in V33_LINESEARCH_FRACTIONS:\n            alpha = float(frac) * cap\n            if alpha <= 0:\n                continue\n            T = v32_blend_transform(T_fast, T_raw, alpha)\n            ev = v33_validation_score(\n                T, VP, VQ, VW,\n                T_cycle_reference=T_cycle_reference,\n                T_fast=T_fast,\n            )\n            accepted = v33_candidate_acceptable(base_eval, ev)\n            candidates.append({\n                \'type\': kind, \'alpha\': alpha,\n                \'T\': T, \'eval\': ev, \'accepted\': bool(accepted),\n                \'raw_delta_rot_deg\': raw_dr,\n                \'raw_delta_trans_m\': raw_dt,\n            })\n\n    accepted = [c for c in candidates if c[\'accepted\']]\n    if accepted:\n        best = max(accepted, key=lambda c: c[\'eval\'][\'regularized_score\'])\n        out[\'T_est\'] = best[\'T\']\n        out[\'selected_stage\'] = \'v33_holdout_local\'\n        out[\'fallback_selected\'] = True\n        out[\'v33_selected_type\'] = best[\'type\']\n        out[\'v33_selected_alpha\'] = float(best[\'alpha\'])\n        out[\'v33_validation_score\'] = float(best[\'eval\'][\'regularized_score\'])\n        out[\'v33_validation_gain\'] = float(\n            best[\'eval\'][\'regularized_score\'] - base_eval[\'regularized_score\']\n        )\n        out[\'v33_cycle_rot_deg\'] = float(best[\'eval\'][\'cycle_rot_deg\'])\n        out[\'v33_cycle_trans_m\'] = float(best[\'eval\'][\'cycle_trans_m\'])\n        out[\'v33_step_rot_deg\'] = float(best[\'eval\'][\'step_rot_deg\'])\n        out[\'v33_step_trans_m\'] = float(best[\'eval\'][\'step_trans_m\'])\n    else:\n        # Critical safety behavior: when hold-out does not prove a gain, keep FAST.\n        out[\'T_est\'] = T_fast\n        out[\'selected_stage\'] = \'v33_keep_fast\'\n        out[\'fallback_selected\'] = False\n        out[\'v33_selected_type\'] = \'keep_fast\'\n        out[\'v33_selected_alpha\'] = 0.0\n        out[\'v33_validation_score\'] = float(base_eval[\'regularized_score\'])\n        out[\'v33_validation_gain\'] = 0.0\n        out[\'v33_cycle_rot_deg\'] = float(base_eval[\'cycle_rot_deg\'])\n        out[\'v33_cycle_trans_m\'] = float(base_eval[\'cycle_trans_m\'])\n        out[\'v33_step_rot_deg\'] = 0.0\n        out[\'v33_step_trans_m\'] = 0.0\n\n    # Recompute selected consensus on the SAME hard pool for diagnostics only.\n    selected_stats = v3_consensus_stats(\n        out[\'T_est\'], src_xyz[hp[:,0]], tgt_xyz[hp[:,1]], hq\n    )\n    out[\'selected_pairs\'] = int(len(hp))\n    out[\'selected_inlier_ratio_5cm\'] = float(selected_stats[\'ratio_5cm\'])\n    out[\'selected_inlier_ratio_3cm\'] = float(selected_stats[\'ratio_3cm\'])\n    out[\'selected_validation_score\'] = float(selected_stats[\'score\'])\n    out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n    return out\n\n# Original notebook cell 79\nV331_MIN_GAIN_TRANSLATION = 0.015\n\nV331_MIN_GAIN_FAST_RESCUE = 0.015\n\nV331_MIN_GAIN_ROTATION = 0.020\n\nV331_MIN_GAIN_FULL = 0.020\n\nV331_ROTATION_ENABLE_MID_RATIO = 0.45\n\nV331_ROTATION_ENABLE_SPREAD_M = 0.10\n\nV331_MAX_FROM_V32_ROT_DEG = float(V32_TRUST_MAX_ROT_DEG)\n\nV331_MAX_FROM_V32_TRANS_M = max(float(V32_TRUST_MAX_TRANS_M), 0.010)\n\nV331_MAX_RATIO5_DROP = 0.002\n\nV331_MAX_RATIO3_DROP = 0.002\n\nV331_MAX_MEDIAN5_INCREASE_M = 0.0005\n\nV331_MAX_CYCLE_ROT_INCREASE_DEG = 0.20\n\nV331_MAX_CYCLE_TRANS_INCREASE_M = 0.005\n\nV331_LINESEARCH_FRACTIONS = tuple(V33_LINESEARCH_FRACTIONS)\n\n# Original notebook cell 81\ndef v331_min_gain(kind):\n    if kind.startswith(\'translation\'):\n        return V331_MIN_GAIN_TRANSLATION\n    if kind == \'fast_rescue\':\n        return V331_MIN_GAIN_FAST_RESCUE\n    if kind.startswith(\'rotation\'):\n        return V331_MIN_GAIN_ROTATION\n    if kind.startswith(\'full\'):\n        return V331_MIN_GAIN_FULL\n    return max(V331_MIN_GAIN_TRANSLATION, V331_MIN_GAIN_ROTATION)\n\ndef v331_rotation_allowed(obs):\n    return bool(\n        float(obs[\'mid_eigen_ratio\']) >= V331_ROTATION_ENABLE_MID_RATIO\n        and float(obs[\'rms_spread_m\']) >= V331_ROTATION_ENABLE_SPREAD_M\n    )\n\ndef v331_validation_eval(T, P, Q, w, T_cycle_reference, T_v32):\n    # Reuse V3.3 score, but step penalty is now relative to V3.2 selected pose.\n    return v33_validation_score(\n        T, P, Q, w,\n        T_cycle_reference=T_cycle_reference,\n        T_fast=T_v32,\n    )\n\ndef v331_candidate_acceptable(base_eval, cand_eval, kind, T_candidate, T_v32):\n    b = base_eval[\'stats\']\n    c = cand_eval[\'stats\']\n    if not np.isfinite(cand_eval[\'regularized_score\']):\n        return False, \'nonfinite_score\'\n\n    gain = float(cand_eval[\'regularized_score\'] - base_eval[\'regularized_score\'])\n    if gain < v331_min_gain(kind):\n        return False, \'gain_too_small\'\n\n    # Trust region is measured from the already accepted V3.2 solution.\n    dr, dt = v32_pose_delta(T_candidate, T_v32)\n    if dr > V331_MAX_FROM_V32_ROT_DEG + 1e-12:\n        return False, \'delta_rot_from_v32\'\n    if dt > V331_MAX_FROM_V32_TRANS_M + 1e-12:\n        return False, \'delta_trans_from_v32\'\n\n    if c[\'ratio_5cm\'] < b[\'ratio_5cm\'] - V331_MAX_RATIO5_DROP:\n        return False, \'ratio5_drop\'\n    if c[\'ratio_3cm\'] < b[\'ratio_3cm\'] - V331_MAX_RATIO3_DROP:\n        return False, \'ratio3_drop\'\n\n    if np.isfinite(b[\'median_5cm_m\']) and np.isfinite(c[\'median_5cm_m\']):\n        if c[\'median_5cm_m\'] > b[\'median_5cm_m\'] + V331_MAX_MEDIAN5_INCREASE_M:\n            return False, \'median5_regression\'\n\n    if np.isfinite(base_eval[\'cycle_rot_deg\']) and np.isfinite(cand_eval[\'cycle_rot_deg\']):\n        if cand_eval[\'cycle_rot_deg\'] > base_eval[\'cycle_rot_deg\'] + V331_MAX_CYCLE_ROT_INCREASE_DEG:\n            return False, \'cycle_rot_regression\'\n    if np.isfinite(base_eval[\'cycle_trans_m\']) and np.isfinite(cand_eval[\'cycle_trans_m\']):\n        if cand_eval[\'cycle_trans_m\'] > base_eval[\'cycle_trans_m\'] + V331_MAX_CYCLE_TRANS_INCREASE_M:\n            return False, \'cycle_trans_regression\'\n\n    return True, \'accepted\'\n\n# Original notebook cell 83\ndef v331_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\n    start = time.perf_counter()\n    base = v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed)\n    out = dict(base)\n    out[\'method\'] = \'Bsafe_mv_v3_3_1\'\n    out[\'v331_used_holdout\'] = False\n    out[\'v331_selected_type\'] = \'v32_passthrough\'\n    out[\'v331_validation_gain\'] = 0.0\n    out[\'v331_train_pairs\'] = 0\n    out[\'v331_validation_pairs\'] = 0\n    out[\'v331_rotation_allowed\'] = False\n    out[\'v331_mid_eigen_ratio\'] = np.nan\n    out[\'v331_rms_spread_m\'] = np.nan\n    out[\'v331_candidate_count\'] = 0\n    out[\'v331_accepted_count\'] = 0\n    out[\'v331_reject_reasons\'] = \'\'\n    out[\'v331_base_stage\'] = base.get(\'selected_stage\')\n    out[\'v331_base_selected_type\'] = base.get(\'local_selected_type\')\n\n    # Easy case is exactly V3.2.\n    if not bool(base.get(\'fallback_triggered\', False)):\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    T_v32 = base.get(\'T_est\')\n    if T_v32 is None:\n        out[\'v331_selected_type\'] = \'v32_failed_no_pose\'\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n    T_v32 = np.asarray(T_v32, dtype=np.float64)\n\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\'xyz\']).astype(np.float64), src_c2w)\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\'xyz\']).astype(np.float64), tgt_c2w)\n    pool = v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz)\n    if pool is None:\n        out[\'v331_selected_type\'] = \'keep_v32_no_pool\'\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    hp = pool[\'pairs\']\n    hq = pool[\'quality\']\n    train_idx, val_idx, split_diag = v33_spatial_holdout_split(hp, src_xyz, tgt_xyz, seed)\n    if train_idx is None:\n        out[\'v331_selected_type\'] = \'keep_v32_split_failed\'\n        out[\'v331_split_reason\'] = split_diag.get(\'reason\')\n        out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n        return out\n\n    out[\'v331_used_holdout\'] = True\n    out[\'v331_train_pairs\'] = int(len(train_idx))\n    out[\'v331_validation_pairs\'] = int(len(val_idx))\n\n    TP = src_xyz[hp[train_idx, 0]]\n    TQ = tgt_xyz[hp[train_idx, 1]]\n    TW0 = np.clip(hq[train_idx], 1e-8, None)\n    VP = src_xyz[hp[val_idx, 0]]\n    VQ = tgt_xyz[hp[val_idx, 1]]\n    VW = np.clip(hq[val_idx], 1e-8, None)\n\n    TW, leverage = v33_leverage_weights(TP, TQ, TW0)\n    obs = v33_observability(TP, TQ, TW)\n    rot_allowed = v331_rotation_allowed(obs)\n    out[\'v331_rotation_allowed\'] = bool(rot_allowed)\n    out[\'v331_mid_eigen_ratio\'] = float(obs[\'mid_eigen_ratio\'])\n    out[\'v331_rms_spread_m\'] = float(obs[\'rms_spread_m\'])\n\n    T_fast = base.get(\'T_fast_debug\')\n    if T_fast is None:\n        T_fast = T_v32.copy()\n    else:\n        T_fast = np.asarray(T_fast, dtype=np.float64)\n\n    # Reverse TRAIN-only fit provides a cycle-consistency reference.\n    try:\n        T_reverse = v33_irls_full(np.linalg.inv(T_v32), TQ, TP, TW)\n        T_cycle_reference = np.linalg.inv(T_reverse)\n    except Exception:\n        T_cycle_reference = None\n\n    base_eval = v331_validation_eval(\n        T_v32, VP, VQ, VW,\n        T_cycle_reference=T_cycle_reference,\n        T_v32=T_v32,\n    )\n    out[\'v331_v32_validation_score\'] = float(base_eval[\'regularized_score\'])\n\n    # Raw candidates. V3.2 selected is not in this list because it is the baseline.\n    raw_specs = []\n    dr_fast, dt_fast = v32_pose_delta(T_fast, T_v32)\n    if dr_fast > 1e-10 or dt_fast > 1e-10:\n        raw_specs.append((\'fast_rescue\', T_fast, T_fast, False))\n\n    raw_specs.append((\n        \'translation_fast\',\n        v33_irls_translation(T_fast, TP, TQ, TW),\n        T_fast,\n        True,\n    ))\n    raw_specs.append((\n        \'translation_v32\',\n        v33_irls_translation(T_v32, TP, TQ, TW),\n        T_v32,\n        True,\n    ))\n\n    if rot_allowed:\n        raw_specs.append((\n            \'rotation_fast\',\n            v33_irls_rotation(T_fast, TP, TQ, TW),\n            T_fast,\n            True,\n        ))\n        raw_specs.append((\n            \'full_fast\',\n            v33_irls_full(T_fast, TP, TQ, TW),\n            T_fast,\n            True,\n        ))\n\n    candidates = []\n    reject_reasons = {}\n    for kind, T_raw, anchor, use_linesearch in raw_specs:\n        if kind == \'fast_rescue\':\n            trial_list = [(1.0, np.asarray(T_raw, dtype=np.float64))]\n        else:\n            cap, _, _ = v33_alpha_cap(anchor, T_raw)\n            trial_list = []\n            if cap > 0:\n                for frac in V331_LINESEARCH_FRACTIONS:\n                    alpha = float(frac) * float(cap)\n                    if alpha > 0:\n                        trial_list.append((alpha, v32_blend_transform(anchor, T_raw, alpha)))\n\n        for alpha, T in trial_list:\n            ev = v331_validation_eval(\n                T, VP, VQ, VW,\n                T_cycle_reference=T_cycle_reference,\n                T_v32=T_v32,\n            )\n            accepted, reason = v331_candidate_acceptable(\n                base_eval, ev, kind, T, T_v32\n            )\n            reject_reasons[reason] = reject_reasons.get(reason, 0) + 1\n            candidates.append({\n                \'type\': kind,\n                \'alpha\': float(alpha),\n                \'T\': T,\n                \'eval\': ev,\n                \'accepted\': bool(accepted),\n                \'reason\': reason,\n                \'gain\': float(ev[\'regularized_score\'] - base_eval[\'regularized_score\']),\n            })\n\n    out[\'v331_candidate_count\'] = int(len(candidates))\n    accepted = [c for c in candidates if c[\'accepted\']]\n    out[\'v331_accepted_count\'] = int(len(accepted))\n    out[\'v331_reject_reasons\'] = \'|\'.join(\n        f\'{k}:{v}\' for k, v in sorted(reject_reasons.items())\n    )\n\n    if accepted:\n        best = max(accepted, key=lambda c: c[\'eval\'][\'regularized_score\'])\n        out[\'T_est\'] = best[\'T\']\n        out[\'selected_stage\'] = \'v331_safe_holdout\'\n        out[\'fallback_selected\'] = True\n        out[\'v331_selected_type\'] = best[\'type\']\n        out[\'v331_selected_alpha\'] = float(best[\'alpha\'])\n        out[\'v331_validation_gain\'] = float(best[\'gain\'])\n        out[\'v331_validation_score\'] = float(best[\'eval\'][\'regularized_score\'])\n        out[\'v331_cycle_rot_deg\'] = float(best[\'eval\'][\'cycle_rot_deg\'])\n        out[\'v331_cycle_trans_m\'] = float(best[\'eval\'][\'cycle_trans_m\'])\n        out[\'v331_step_from_v32_rot_deg\'] = float(best[\'eval\'][\'step_rot_deg\'])\n        out[\'v331_step_from_v32_trans_m\'] = float(best[\'eval\'][\'step_trans_m\'])\n    else:\n        # Safety invariant: preserve the exact V3.2 selected solution.\n        out[\'T_est\'] = T_v32\n        out[\'selected_stage\'] = \'v331_keep_v32\'\n        out[\'fallback_selected\'] = bool(base.get(\'fallback_selected\', False))\n        out[\'v331_selected_type\'] = \'keep_v32_selected\'\n        out[\'v331_selected_alpha\'] = 0.0\n        out[\'v331_validation_gain\'] = 0.0\n        out[\'v331_validation_score\'] = float(base_eval[\'regularized_score\'])\n        out[\'v331_cycle_rot_deg\'] = float(base_eval[\'cycle_rot_deg\'])\n        out[\'v331_cycle_trans_m\'] = float(base_eval[\'cycle_trans_m\'])\n        out[\'v331_step_from_v32_rot_deg\'] = 0.0\n        out[\'v331_step_from_v32_trans_m\'] = 0.0\n\n    selected_stats = v3_consensus_stats(\n        out[\'T_est\'], src_xyz[hp[:, 0]], tgt_xyz[hp[:, 1]], hq\n    )\n    out[\'selected_pairs\'] = int(len(hp))\n    out[\'selected_inlier_ratio_5cm\'] = float(selected_stats[\'ratio_5cm\'])\n    out[\'selected_inlier_ratio_3cm\'] = float(selected_stats[\'ratio_3cm\'])\n    out[\'selected_validation_score\'] = float(selected_stats[\'score\'])\n    out[\'matcher_runtime_s\'] = float(time.perf_counter() - start)\n    return out\n'
_SB_PROVENANCE = {'source_name': 'mv-slam_apart_1_heldout_v331(1).ipynb', 'source_sha256': '420a7a87b20421e9416af654b4d3becbbbf374015903dd77339d44ce7407c15c', 'helper_sha256': 'bf4430d906da180d6482d223f0e3f4648e40015148dee26751ff433f29908c8f', 'selected_original_cells': [46, 47, 48, 49, 50, 58, 60, 61, 62, 63, 64, 65, 66, 67, 69, 71, 73, 75, 77, 79, 81, 83], 'function_sha256': {'mv_find_submap_index': 'b6121be2ba7c4906007edb41426580cff6be66180eee12531dd6e71f8216da90', 'mv_frame_c2w': '885dede389cdd20ffcd5bb47691b01a293017c4b56cf7490723cf32e83dcb312', 'mv_to_pil': '82ecffffb663bb0352d1dbffab4b4731338e490145e18942ec4443bf11c83639', 'mv_depth_tensor': '489f6feccd5a93c77c1b16328df0fdc15434a654284fe910f30aa5372fd32430', 'mv_extract_feature_map': '47a85425d4921dc331ddf7c9724a29f74eaeee37df5845b08dd7877e8993a0f2', 'mv_bilinear_sample': '3c1a1f477221bf9dd635bf857b58b1d5f4e28739789db69c74396a740eb04d1f', 'mv_restore_gaussians': 'cd2291813692990c2f61d29b2b3fd0954feb50dd0b427905d7c760785a776540', 'mv_spatial_topk': '99e875a201f52e224d56a34381f0ebf493df9c7d7f3b3842b5b6fa0a168fae82', 'mv_cache_path': 'b59b56f2328476052db2c4af44fcff971788dc8661b1a76fdd683b4ed12ca05d', 'mv_build_cache': '2db98cedef8aceb39bddaa8ac236326099faa0d08f624f2f9465b04b22816319', 'mv_world_to_camera_points': '758176c6923d1773f60e5c2abe322af9b65c9207ab63f21df421735c359ee931', 'mv_match_descriptors': '9da40aa9ce14f38e99157100ad3372f39a2f7f9c2a089f45934bb14f3a67473d', 'mv_geometric_compatibility': '3d82a6f0efc7bbcaeac3d00e738305e0b8f41877aed350ec0a50601b0ff88cd6', 'mv_ransac_baseline': '1455d008b254b5bb917a47461f7a1ac2dc1ec6fd24e8cbe8ecc3689a8c5c011c', 'mv_rotation_translation_error': '43e90ac9e8baca08d62e1c57ddd098c4bc963029fb10b9e4442f97f5c5462b3b', 'mv_gt_pose': '1ba7390a85908815d153030a66ce43211bdeebe2d0194b89ea650bf22d23dd5c', 'mv_baseline_coarse_transform': '25dabeef8b54dff2207a38cb02a1af7d42e0dc364951796d2996f6b5ff947c51', 'mv_metric_row': '17127afb6d6f54e83402d9aceb4a3854ea203f463f11895ee0b6f03fafd15eb2', 'mv_first_metric': '3ecf5a7a35fad19b660977bdd54e322a422409f61b1a730f5369c48ee55da420', 'mv_descriptor_for_variant': 'c083260274e9c5bbbf11620b2e0b53a1f58457e1a13c0a181bdda8b43e3284e8', '_safe_mean': '2fe0204fb5f6fc234eb53359c5148b696a4d140b8befbfb893f889c2c37c2a5f', '_safe_std': 'b71510c6ad6e6dd76c6a0d79521d2d1a4a9c34e22c0fb2c3b48403dd1b9a5371', 'a_match_descriptors': '5688ff7afc6d0f0cdb3e56fcbd9e81058dfdfa8135179177cd0eea6dce25f154', 'a_geometric_compatibility': 'cfaf781c8d593453be3531459e10cabc6f1ec011b85a935ff5fcf0406e4be818', 'a_ransac': 'b977a55bd9dbb156833c16ae015c64d02cc2804ab5f945b8d7e98e99ee3cf32c', 'a_registration_once': '1f84f748081c9b1e421cf245101a51633beb92621657e6cfaf7f20aed2f2b23b', 'v2_np': '5ac5c6049f19e07e9b3a494c0714231ce1f7e8f55218c7e8f66a4ff8615c74aa', 'v2_normalize_desc': '685376be31751cb2e9a13e3b465f3c3c244a7d55ad4d4bda78f9ae0faec4f0c2', 'v2_adaptive_k_from_row': '38a5b1ee34acf1a9439bd2006ebbd5469e33402c71eebfa6f106f5be16b4dec0', 'v2_build_adaptive_bidirectional': 'c6415663e1d85385632c4cb46e3a7778c8f782762e420b6b5b3023a4a5a12555', 'v2_pair_quality': '85d8cd2e59a001fbd2a7d25dd53a9e14ba6deef762d9fe8d45e4cb303fe02c8e', 'v2_greedy_unique_by_confidence': '5667bb770a47080f8a063c4b64cb09a73e5261d7eaf4b2c2856c2e97bdb94d3f', 'v2_spatial_coverage_select': 'a518bfdcbfbbfb9a5198271dd7e450590b8c29aa3781b681bf9acf6462a26a6f', 'v2_geometry_condition': '6705691ba5e76eaf1f0a8194cc8e22f4279d2e7233848dcec5c33a5bb029ccff', 'mv_topk_rigidity_support': 'c19740ec850961079895b2b61649f8d9a98e0394aa97ef22c4666d185fd674c0', 'v2_weighted_kabsch': '7f048c6bb4cc22c29ff4b3b1e3e45bf5a4117a4fa780ece418906807bed885e2', 'v2_transform_points': 'a43eecd684b86adeb221e8bb8736b203160cca0a6f9a94095ae2154c196bc730', 'v2_non_degenerate_triplet': 'c92b35eef1c987db31f2647baa00bddc0ed521d3b89b0f8a5c00cdeb76e1b373', 'v2_prosac_ransac': '057327ee19f00f399ebbe6731844beb121b62c1f70350b58acd245098e41700b', 'v2_registration_once': 'f3950988ff0cbe57331bc38d1fde1b48e1532f7e3064651fd06c0b92dcc2a495', 'v3_conf_stats': '0a94e4a37b089ba146b1e69d0f27542c4f97050f1e7638e536fcc04eaba2de3a', 'v3_spatial_coverage_select': '77692b4fa988118456ae50581eb84d774587ffccf0b875d87045c34795f10db9', 'v3_greedy_limited_reuse': '1efffd2867df58b1d9e8b703abf144fd8ef41f9fa748eae52fc095e8a683805f', 'v3_geometry_diag': '581f4f436c4c2bfdaf7b4160a90376d9596a10b0618920f1350072fc34c75778', 'v3_consensus_stats': 'fcffc4eea851a77de2412b0388925f9612f0a88534433fca9befa4b73da5973d', 'v3_prosac_ransac': 'dd0af2930943a642915d8f66f959fa94fd377fb984428f2f3e23cffe41bedd37', 'v3_should_trigger_fallback': 'f9c3781420f911f48bb47d3d0e5021d3d90bd5b1e3d58eefa5588e97b2512465', 'v32_build_hard_candidates': '7924fb4674589f2806fafeec068ca92078e7d09d6680fed5ec2e24e738c32b47', 'v32_hard_quality': 'b625b3d385d5232259c0964ec481040dad8e4d9224af7fffc5e906cd85f8c036', 'v32_greedy_unique_with_indices': '774a7591d43ed092591af586c8a784e159c275a765b147860ccc7e0997ba9341', 'v32_top_quality_prefilter': '002280df6d22ce491c9f6b70f4c8fe1e80a84cdb5742482687bb58e876ebb2c2', 'v32_ambiguity_diag': '12809884e8efafd92c8feb017ff54a193a6832584759c39b5149534ca80645fa', 'v32_rotation_angle_deg': '394cf36b8fa5de86f867ac46af734e28159acf7bcdce62424acdaa972e502ffb', 'v32_fractional_rotation': '8f238c18bd509f1a90e58a181c48b83a194fc4edbb385c5d51b214e67601ab98', 'v32_pose_delta': '749c015b7a81490ffd6fe1f456add3d4872988fe3879ac1646b0899a7ef3786d', 'v32_blend_transform': 'b2aa9147336b856d8d0afe92068441a5f026465a88c7a6ed4453de8ea8075bc7', 'v32_local_precision_score': '0adfded284c793b845750b85ef602bb796b1a88aebcda3363aa9c3058d28e594', 'v32_robust_local_raw_candidates': 'adb58b3b539ef91d0574675e4d3305727e92b34d19fed6aee47e2c7f60d1e8d2', 'v32_select_local_candidate': '6602dbdd2084e3a507dd97bf607f2a76a661106784f2c4aa14aa509143787832', 'v32_registration_once': '490c35a02cc2ec0dbe7753a06479748ffec2438eaa6de0fbf8554d61d66b3fa4', '_as_np_pose': '0a3427c9c8278ad75e12c0ca625c85e8c42ca66ecc784de3577e05b7e7bfb300', 'loop_to_submap_start_transform': '468d9d2c51eff76c9d84d4c6212e4a6378f53ab47d7eeae562460a9beac43162', 'submap_start_to_loop_transform': 'e5462468f9e43d4650c1bbbe79820cff44f5190a33863a8257fd5226ded02ace', 'transform_delta': '28385475d49f233f0f4a67ad51943efaefb952256d9fb5884d11c29652ce9477', 'dense_cloud_for_submap': '1bc41b3d66e11f2f6c3574fc89c4985be28b8825959df1ff5b2268915a6349df', 'exact_production_icp': 'ed3d9bcb29abe777e3ca935350c24eb01c6673f9cad58d81106f94a492cae07a', 'v33_group_key': 'ef401610b904e7653ac7475d274d08a826482c7e4066f4047509cb1a526c8f1a', 'v33_group_hash': '555e5d6596c4fc735ed71b5395a55ad2503960c47a76048d163ca5636cf08cd8', 'v33_spatial_holdout_split': 'e91c66b3350dc741f272fed88c4d6b2cefee9dcc09b2a769d48af93bb5eac453', 'v33_weighted_covariance': 'cae1b802f6de16970d056a7bcf29ef56df659252f704e20af568c706e0adb0b5', 'v33_observability': '782d227074d2fbef4d47e545619f59f4eb209f9e559071366d0d3b7aef0ae0fd', 'v33_leverage_weights': '8036ed90d7c2d365c6a9299d9bf09f6dc055c5f2630c9aa49cbc24fde06e5f60', 'v33_cauchy_weights': 'e15319e464e3301f4e12515c4e2a03bfc3f1a50a4644bc0d7b2523154f330b7d', 'v33_gate_threshold': 'e7bf45ae7d39eba5ef5b97a7338401922f668db3e014d8732b0bf70d01a5504a', 'v33_irls_full': '5cf02962b8499a79fd93b10e3d2b6087ce18737c19f53862627341a622748531', 'v33_irls_translation': 'fec17ef9818680c0cf9ebcb9ed60c575fda37f2f58680441d6a35c9b6d9ccbd2', 'v33_weighted_rotation_fixed_translation': '85a6f84952649eb7551cdca346c091bea62cb8724fd0660d787c15460f20c189', 'v33_irls_rotation': '3d772e35a853757230fdac1bc1f53dc98e8ef2d17cbd03d4a9c6e3a18076fe76', 'v33_alpha_cap': 'b34c20595a88946dbae3b733fb98ec3d95b3497dbcf530b75a1890e5e9091f41', 'v33_validation_score': '566f47f35e958b9970f158f0f5cec654d71ebb43e32911129dc12898670b0eb9', 'v33_candidate_acceptable': '416ea0628cbef8299fd9837638f6dfe8588b805409d288578e87842ad58efe8e', 'v33_prepare_hard_pool': '00950e076a0c8700001fe0f920b4713122cd7ed65cc30528ccf34e7657387b02', 'v33_registration_once': '4254d9894dc2cb48db70f61b9c3b92a921b93987d0a91dd5af1fd45e3a458c33', 'v331_min_gain': 'd814c9ec61249ccb9ccd4a34811ed9935fee295c9e9b1c158341001054c6974f', 'v331_rotation_allowed': '8ccc1c207d08f0beb3a9db4d68950013719daf5c7ef5a0f1f25040c85085b646', 'v331_validation_eval': '2bf343a32f91f9c9c0916fc934ac973943d5544ec60da8a37cfa78a3614cd3f2', 'v331_candidate_acceptable': 'ecfa623379463705bcea5cc61d6232c4770e559772e3a7546e73f7945dffee42', 'v331_registration_once': 'd1269355e4a500d3184a83ba149ab065e36b057441716cf98c97bb699d772a96'}}
_SB_DRIVER_SHA256 = '97c86bd56b677d81a01a96a9c764d9fd63e9a84d0888400ae9da3f6a3b016a57'
_SB_WORKER_SOURCE = '# ================================================================\n# Single vs multi-view coarse registration + fixed ablations, current V3.3.1\n# Uses the completed Office-0 SLAM output; no second SLAM run.\n# The packaged cell contains unchanged helpers from the recent notebook.\n# ================================================================\nfrom pathlib import Path\nimport os, sys, json, time, math, hashlib, pickle, copy, inspect\nimport threading, subprocess, contextlib, io, zipfile, gc\nfrom collections import OrderedDict, defaultdict\nimport numpy as np\nimport pandas as pd\n\n# EDIT ONLY PATHS / RUN SELECTION BEFORE THE FIRST BENCHMARK.\nBENCH_REPO = globals().get(\'BENCH_REPO\', Path(globals().get(\'REPO\', \'/kaggle/working/3DGS-SLAM-TURTLEBOT4\')))\nBENCH_PYTHON = globals().get(\'BENCH_PYTHON\', str(globals().get(\'PY311\',sys.executable)))\nBENCH_RUNS = globals().get(\'BENCH_RUNS\', {})\nBENCH_SEARCH_ROOTS = globals().get(\'BENCH_SEARCH_ROOTS\', [Path(\'/kaggle/working/gslam_outputs\')])\nBENCH_SCENES = globals().get(\'BENCH_SCENES\', [\'office_0\', \'apart_0\', \'apart_1\', \'apart_2\'])\nBENCH_DATA_ROOT_OVERRIDES = globals().get(\'BENCH_DATA_ROOT_OVERRIDES\', {})\nBENCH_SEEDS = globals().get(\'BENCH_SEEDS\', [0, 1, 2])\nBENCH_ICP_SEEDS = globals().get(\'BENCH_ICP_SEEDS\', [0, 1, 2])\nBENCH_K_VALUES = globals().get(\'BENCH_K_VALUES\', [256, 512, 1024])\nBENCH_PRIMARY_K = globals().get(\'BENCH_PRIMARY_K\', 512)\nBENCH_EXPECTED_TOTAL_LOOPS = globals().get(\'BENCH_EXPECTED_TOTAL_LOOPS\', 21)\nBENCH_SCOPE = globals().get(\'BENCH_SCOPE\', \'detected\')\nBENCH_OUTPUT = globals().get(\'BENCH_OUTPUT\', Path(\'/kaggle/working/single_multi_v331_benchmark\'))\nBENCH_IMPORT_RESULTS = globals().get(\'BENCH_IMPORT_RESULTS\', [])\nBENCH_BUILD_MISSING_CACHE = globals().get(\'BENCH_BUILD_MISSING_CACHE\', True)\nBENCH_RETRY_ERRORS = globals().get(\'BENCH_RETRY_ERRORS\', True)\nBENCH_STRICT_INPUT_HASH = globals().get(\'BENCH_STRICT_INPUT_HASH\', True)\nBENCH_RUN_ABLATIONS = globals().get(\'BENCH_RUN_ABLATIONS\', True)\nBENCH_RUN_FPFH = globals().get(\'BENCH_RUN_FPFH\', True)\nBENCH_EXPORT_PAIR_FIGURES = globals().get(\'BENCH_EXPORT_PAIR_FIGURES\', True)\nBENCH_GPU_DEVICE = globals().get(\'BENCH_GPU_DEVICE\', 0)\nBENCH_GPU_POLL_SECONDS = globals().get(\'BENCH_GPU_POLL_SECONDS\', 0.2)\nBENCH_VERBOSE = globals().get(\'BENCH_VERBOSE\', False)\nBENCH_MARGIN_POSITIVE = globals().get(\'BENCH_MARGIN_POSITIVE\', 0.001)\n# NEW experimental shape ablation, not a claimed existing V3.3.1 component.\nBENCH_SCALE_LOG_THRESHOLD = globals().get(\'BENCH_SCALE_LOG_THRESHOLD\', 0.35)\nBENCH_AXIS_COS_THRESHOLD = globals().get(\'BENCH_AXIS_COS_THRESHOLD\', 0.15)\nBENCH_AXIS_ANISOTROPY_MIN = globals().get(\'BENCH_AXIS_ANISOTROPY_MIN\', 1.2)\nBENCH_QUATERNION_ORDER = globals().get(\'BENCH_QUATERNION_ORDER\', \'wxyz\')\nBENCH_FPFH_VOXEL_M = globals().get(\'BENCH_FPFH_VOXEL_M\', 0.05)\nBENCH_FPFH_MAX_ITERS = globals().get(\'BENCH_FPFH_MAX_ITERS\', 100000)\nBENCH_FPFH_CONFIDENCE = globals().get(\'BENCH_FPFH_CONFIDENCE\', 0.999)\nBENCH_EXECUTE = globals().get(\'BENCH_EXECUTE\', True)\n\n_SB_HELPERS = \'# Original notebook cell 46\\ndef mv_find_submap_index(agent_id, frame_id):\\n    agent_id, frame_id = int(agent_id), int(frame_id)\\n    for i, sub in enumerate(mv_submaps[agent_id]):\\n        if int(sub[\\\'submap_start_frame_id\\\']) <= frame_id <= int(sub[\\\'submap_end_frame_id\\\']):\\n            return i\\n    raise KeyError(f\\\'Không tìm thấy submap chứa agent={agent_id}, frame={frame_id}\\\')\\n\\ndef mv_frame_c2w(submap, frame_id):\\n    offset = int(frame_id) - int(submap[\\\'submap_start_frame_id\\\'])\\n    poses = submap[\\\'submap_c2ws\\\']\\n    if offset < 0 or offset >= len(poses):\\n        raise IndexError((frame_id, submap[\\\'submap_start_frame_id\\\'], submap[\\\'submap_end_frame_id\\\']))\\n    pose = poses[offset]\\n    if torch.is_tensor(pose):\\n        pose = pose.detach().cpu().numpy()\\n    return np.asarray(pose, dtype=np.float64)\\n\\n# Original notebook cell 47\\ndef mv_to_pil(image):\\n    if isinstance(image, Image.Image):\\n        return image.convert(\\\'RGB\\\')\\n    if torch.is_tensor(image):\\n        arr = image.detach().cpu().numpy()\\n    else:\\n        arr = np.asarray(image)\\n    arr = np.asarray(arr)\\n    if arr.ndim != 3:\\n        raise ValueError(f\\\'RGB image phải 3D, nhận {arr.shape}\\\')\\n    if arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):\\n        arr = np.moveaxis(arr, 0, -1)\\n    if arr.shape[-1] == 1:\\n        arr = np.repeat(arr, 3, axis=-1)\\n    arr = arr[..., :3]\\n    if np.issubdtype(arr.dtype, np.floating):\\n        if np.nanmax(arr) <= 1.5:\\n            arr = arr * 255.0\\n    arr = np.nan_to_num(arr, nan=0.0, posinf=255.0, neginf=0.0)\\n    arr = np.clip(arr, 0, 255).astype(np.uint8)\\n    return Image.fromarray(arr, mode=\\\'RGB\\\')\\n\\ndef mv_depth_tensor(depth):\\n    if torch.is_tensor(depth):\\n        d = depth.detach().to(MV_DEVICE, dtype=torch.float32)\\n        while d.ndim > 2 and 1 in d.shape:\\n            d = d.squeeze()\\n        if d.ndim == 3:\\n            d = d[0] if d.shape[0] <= 4 else d[..., 0]\\n        if d.ndim != 2:\\n            raise ValueError(f\\\'Depth phải 2D sau squeeze, nhận {tuple(d.shape)}\\\')\\n        return d\\n    d = np.asarray(depth)\\n    while d.ndim > 2 and 1 in d.shape:\\n        d = np.squeeze(d)\\n    if d.ndim == 3:\\n        d = d[..., 0] if d.shape[-1] <= 4 else d[0]\\n    if d.ndim != 2:\\n        raise ValueError(f\\\'Depth phải 2D sau squeeze, nhận {d.shape}\\\')\\n    return torch.as_tensor(d, device=MV_DEVICE, dtype=torch.float32)\\n\\n@torch.inference_mode()\\ndef mv_extract_feature_map(image):\\n    pil = mv_to_pil(image)\\n    orig_w, orig_h = pil.size\\n    tokens, (gh, gw), (patch_h_px, patch_w_px), meta = (\\n        mv_feature_extractor.extract_patch_tokens(pil, return_metadata=True)\\n    )\\n    fmap = tokens.reshape(gh, gw, -1)\\n    # tokens từ repo đã L2-normalize; normalize lại để chống sai số số học.\\n    fmap = F.normalize(fmap.float(), dim=-1)\\n    return fmap, {\\n        \\\'orig_h\\\': int(orig_h), \\\'orig_w\\\': int(orig_w),\\n        \\\'gh\\\': int(gh), \\\'gw\\\': int(gw),\\n        \\\'patch_h_px\\\': float(patch_h_px), \\\'patch_w_px\\\': float(patch_w_px),\\n        **meta,\\n    }\\n\\n@torch.inference_mode()\\ndef mv_bilinear_sample(fmap, meta, u, v):\\n    gh, gw = int(meta[\\\'gh\\\']), int(meta[\\\'gw\\\'])\\n    patch_h_px = float(meta[\\\'patch_h_px\\\'])\\n    patch_w_px = float(meta[\\\'patch_w_px\\\'])\\n\\n    # Original pixel coordinate -> continuous patch-center coordinate.\\n    gx = u / patch_w_px - 0.5\\n    gy = v / patch_h_px - 0.5\\n    xnorm = (2.0 * gx / (gw - 1) - 1.0) if gw > 1 else torch.zeros_like(gx)\\n    ynorm = (2.0 * gy / (gh - 1) - 1.0) if gh > 1 else torch.zeros_like(gy)\\n    grid = torch.stack([xnorm, ynorm], dim=-1).view(1, -1, 1, 2)\\n    feat_chw = fmap.permute(2, 0, 1).unsqueeze(0)\\n    sampled = F.grid_sample(\\n        feat_chw, grid, mode=\\\'bilinear\\\', padding_mode=\\\'zeros\\\', align_corners=True\\n    )[0, :, :, 0].T\\n    return F.normalize(sampled, dim=-1)\\n\\n# Original notebook cell 48\\ndef mv_restore_gaussians(submap_cpu):\\n    # Use the same restore path as the repository.\\n    model = load_3dgs(submap_cpu)\\n    xyz = model.get_xyz().detach().float()\\n    opacity = model.get_opacity().detach().float().reshape(-1)\\n    if xyz.ndim != 2 or xyz.shape[-1] != 3:\\n        raise RuntimeError(f\\\'Unexpected xyz shape: {tuple(xyz.shape)}\\\')\\n    if opacity.shape[0] != xyz.shape[0]:\\n        raise RuntimeError(f\\\'Unexpected opacity shape: {tuple(opacity.shape)} for xyz {tuple(xyz.shape)}\\\')\\n    if xyz.device.type != \\\'cuda\\\':\\n        xyz = xyz.to(MV_DEVICE)\\n        opacity = opacity.to(MV_DEVICE)\\n    return model, xyz, opacity\\n\\ndef mv_spatial_topk(xyz_np, score_np, k, voxel_size):\\n    order = np.argsort(-score_np)\\n    vox = np.floor(xyz_np / float(voxel_size)).astype(np.int64)\\n    chosen, seen = [], set()\\n    for idx in order:\\n        key = tuple(vox[idx].tolist())\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        chosen.append(int(idx))\\n        if len(chosen) >= k:\\n            break\\n    if len(chosen) < min(k, len(order)):\\n        chosen_set = set(chosen)\\n        for idx in order:\\n            idx = int(idx)\\n            if idx not in chosen_set:\\n                chosen.append(idx)\\n                chosen_set.add(idx)\\n                if len(chosen) >= k:\\n                    break\\n    return np.asarray(chosen, dtype=np.int64)\\n\\ndef mv_cache_path(agent_id, submap_index):\\n    p = MV_CACHE_ROOT / f\\\'agent_{int(agent_id)}\\\'\\n    p.mkdir(parents=True, exist_ok=True)\\n    return p / f\\\'{int(submap_index):06d}.pt\\\'\\n\\ndef mv_build_cache(agent_id, submap_index):\\n    import time as _time\\n    build_start = _time.perf_counter()\\n    cache_path = mv_cache_path(agent_id, submap_index)\\n    if cache_path.exists() and not MV_REBUILD_CACHE:\\n        cache = torch.load(cache_path, map_location=\\\'cpu\\\', weights_only=False)\\n        if cache.get(\\\'version\\\') == MV_CACHE_VERSION:\\n            print(\\\'[cache hit]\\\', cache_path.name, \\\'| landmarks:\\\', len(cache[\\\'xyz\\\']))\\n            return cache\\n        print(\\\'[cache stale] rebuild\\\', cache_path.name, \\\'| version:\\\', cache.get(\\\'version\\\'))\\n\\n    submap_path = mv_submap_paths[int(agent_id)][int(submap_index)]\\n    sub = torch.load(submap_path, map_location=\\\'cpu\\\', weights_only=False)\\n    model, xyz, opacity = mv_restore_gaussians(sub)\\n    n = int(xyz.shape[0])\\n\\n    # Multi-view accumulator.\\n    sum_desc = torch.zeros((n, MV_EMBED_DIM), device=MV_DEVICE, dtype=torch.float32)\\n    weight_sum = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\\n    obs_count = torch.zeros(n, device=MV_DEVICE, dtype=torch.int32)\\n    residual_sum = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\\n\\n    # Fair single-view ablation: keep the highest-quality ONE observation for each Gaussian.\\n    # float16 saves memory; final matching converts back to float32 + L2 normalize.\\n    best_weight = torch.zeros(n, device=MV_DEVICE, dtype=torch.float32)\\n    best_desc = torch.zeros((n, MV_EMBED_DIM), device=MV_DEVICE, dtype=torch.float16)\\n    best_view_id = torch.full((n,), -1, device=MV_DEVICE, dtype=torch.int32)\\n\\n    keyframe_ids = sub[\\\'keyframe_ids\\\']\\n    if torch.is_tensor(keyframe_ids):\\n        keyframe_ids = keyframe_ids.detach().cpu().tolist()\\n    else:\\n        keyframe_ids = list(keyframe_ids)\\n\\n    K = np.asarray(mv_datasets[int(agent_id)].intrinsics, dtype=np.float64)\\n    fx, fy, cx, cy = float(K[0, 0]), float(K[1, 1]), float(K[0, 2]), float(K[1, 2])\\n\\n    used_views = 0\\n    for view_i, kf_id in enumerate(keyframe_ids):\\n        kf_id = int(kf_id)\\n        _, image, depth, _ = mv_datasets[int(agent_id)][kf_id]\\n        fmap, meta = mv_extract_feature_map(image)\\n        h, w = int(meta[\\\'orig_h\\\']), int(meta[\\\'orig_w\\\'])\\n        depth_t = mv_depth_tensor(depth)\\n        if tuple(depth_t.shape) != (h, w):\\n            depth_t = F.interpolate(depth_t[None, None], size=(h, w), mode=\\\'nearest\\\')[0, 0]\\n\\n        pose_offset = kf_id - int(sub[\\\'submap_start_frame_id\\\'])\\n        if pose_offset < 0 or pose_offset >= len(sub[\\\'submap_c2ws\\\']):\\n            raise IndexError(\\n                f\\\'agent {agent_id} submap {submap_index}: keyframe {kf_id} -> pose offset {pose_offset}, \\\'\\n                f\\\'poses={len(sub["submap_c2ws"])}\\\'\\n            )\\n        c2w = sub[\\\'submap_c2ws\\\'][pose_offset]\\n        c2w = torch.as_tensor(c2w, device=MV_DEVICE, dtype=torch.float32)\\n        w2c = torch.linalg.inv(c2w)\\n\\n        cam = xyz @ w2c[:3, :3].T + w2c[:3, 3]\\n        z = cam[:, 2]\\n        z_safe = z.clamp_min(1e-6)\\n        u = fx * cam[:, 0] / z_safe + cx\\n        v = fy * cam[:, 1] / z_safe + cy\\n\\n        inside = (\\n            torch.isfinite(cam).all(dim=1)\\n            & torch.isfinite(opacity)\\n            & (z > 0)\\n            & (u >= 0) & (u <= w - 1)\\n            & (v >= 0) & (v <= h - 1)\\n            & (opacity >= MV_MIN_OPACITY)\\n        )\\n        candidate_idx = torch.where(inside)[0]\\n        if candidate_idx.numel() == 0:\\n            continue\\n\\n        ui = torch.round(u[candidate_idx]).long().clamp(0, w - 1)\\n        vi = torch.round(v[candidate_idx]).long().clamp(0, h - 1)\\n        observed_depth = depth_t[vi, ui]\\n        residual = torch.abs(z[candidate_idx] - observed_depth)\\n        depth_ok = (\\n            (observed_depth > 0)\\n            & torch.isfinite(observed_depth)\\n            & torch.isfinite(residual)\\n            & (residual <= MV_DEPTH_TOLERANCE_M)\\n        )\\n        valid_idx = candidate_idx[depth_ok]\\n        if valid_idx.numel() == 0:\\n            continue\\n\\n        residual_valid = residual[depth_ok]\\n        desc = mv_bilinear_sample(fmap, meta, u[valid_idx], v[valid_idx])\\n        weights = opacity[valid_idx] * torch.exp(\\n            -residual_valid / max(MV_DEPTH_TOLERANCE_M, 1e-6)\\n        )\\n\\n        # Multi-view accumulation.\\n        sum_desc.index_add_(0, valid_idx, desc * weights[:, None])\\n        weight_sum.index_add_(0, valid_idx, weights)\\n        obs_count.index_add_(0, valid_idx, torch.ones_like(valid_idx, dtype=torch.int32))\\n        residual_sum.index_add_(0, valid_idx, residual_valid)\\n\\n        # Single-best-view descriptor for the exact same Gaussian identity.\\n        old_best = best_weight[valid_idx]\\n        better = weights > old_best\\n        if bool(better.any()):\\n            update_idx = valid_idx[better]\\n            best_weight[update_idx] = weights[better]\\n            best_desc[update_idx] = desc[better].to(torch.float16)\\n            best_view_id[update_idx] = int(kf_id)\\n\\n        used_views += 1\\n        if view_i % 5 == 0 or view_i == len(keyframe_ids) - 1:\\n            print(\\n                f\\\'  agent {agent_id} submap {submap_index} view {view_i + 1}/{len(keyframe_ids)} \\\'\\n                f\\\'valid={int(valid_idx.numel())}\\\'\\n            )\\n\\n        del fmap, depth_t, desc\\n\\n    valid = (\\n        (obs_count >= MV_MIN_OBSERVATIONS)\\n        & (weight_sum > 1e-8)\\n        & (best_weight > 0)\\n        & (best_view_id >= 0)\\n    )\\n    valid_idx = torch.where(valid)[0]\\n    if valid_idx.numel() < 3:\\n        raise RuntimeError(\\n            f\\\'agent {agent_id} submap {submap_index}: chỉ có {valid_idx.numel()} Gaussian hợp lệ.\\\'\\n        )\\n\\n    summed = sum_desc[valid_idx]\\n    ws = weight_sum[valid_idx]\\n    descriptors_multi = F.normalize(summed, dim=-1)\\n    stability = torch.linalg.norm(summed, dim=-1) / ws.clamp_min(1e-8)\\n    mean_residual = residual_sum[valid_idx] / obs_count[valid_idx].float().clamp_min(1)\\n    score = (\\n        stability.clamp_min(0)\\n        * torch.log1p(obs_count[valid_idx].float())\\n        * (0.5 + 0.5 * opacity[valid_idx])\\n        / (1.0 + mean_residual / max(MV_DEPTH_TOLERANCE_M, 1e-6))\\n    )\\n\\n    # ONE shared selected landmark set for BOTH variants.\\n    xyz_valid = xyz[valid_idx].detach().cpu().numpy()\\n    selected_local = mv_spatial_topk(\\n        xyz_valid,\\n        score.detach().cpu().numpy(),\\n        min(MV_MAX_LANDMARKS, len(xyz_valid)),\\n        MV_VOXEL_SIZE_M,\\n    )\\n    selected_local_t = torch.as_tensor(selected_local, device=MV_DEVICE, dtype=torch.long)\\n    selected_global = valid_idx[selected_local_t]\\n\\n    descriptor_multi = F.normalize(descriptors_multi[selected_local_t], dim=-1)\\n    descriptor_single = F.normalize(best_desc[selected_global].float(), dim=-1)\\n\\n    cache = {\\n        \\\'agent_id\\\': int(agent_id),\\n        \\\'submap_index\\\': int(submap_index),\\n        \\\'submap_start_frame_id\\\': int(sub[\\\'submap_start_frame_id\\\']),\\n        \\\'submap_end_frame_id\\\': int(sub[\\\'submap_end_frame_id\\\']),\\n        \\\'keyframe_ids\\\': torch.as_tensor(keyframe_ids, dtype=torch.int32),\\n        \\\'gaussian_index\\\': selected_global.detach().cpu().int(),\\n        \\\'xyz\\\': xyz[selected_global].detach().cpu().half(),\\n        \\\'descriptor_single_best_view\\\': descriptor_single.detach().cpu().half(),\\n        \\\'descriptor_multi_view\\\': descriptor_multi.detach().cpu().half(),\\n        \\\'single_best_view_id\\\': best_view_id[selected_global].detach().cpu().int(),\\n        \\\'single_best_weight\\\': best_weight[selected_global].detach().cpu().half(),\\n        \\\'stability\\\': stability[selected_local_t].detach().cpu().half(),\\n        \\\'observation_count\\\': obs_count[selected_global].detach().cpu().short(),\\n        \\\'depth_residual_mean\\\': mean_residual[selected_local_t].detach().cpu().half(),\\n        \\\'opacity\\\': opacity[selected_global].detach().cpu().half(),\\n        \\\'score\\\': score[selected_local_t].detach().cpu().half(),\\n        \\\'num_gaussians_total\\\': n,\\n        \\\'num_gaussians_multiview\\\': int(valid_idx.numel()),\\n        \\\'num_views_used\\\': int(used_views),\\n        \\\'version\\\': MV_CACHE_VERSION,\\n        \\\'build_time_s\\\': float(_time.perf_counter() - build_start),\\n    }\\n    torch.save(cache, cache_path)\\n    print(\\n        \\\'[cache built]\\\', cache_path,\\n        \\\'| total=\\\', n,\\n        \\\'| eligible=\\\', int(valid_idx.numel()),\\n        \\\'| kept=\\\', len(cache[\\\'xyz\\\']),\\n        \\\'| views=\\\', used_views,\\n        \\\'| mean obs=\\\', f"{float(cache[\\\'observation_count\\\'].float().mean()):.2f}",\\n        \\\'| mean rho=\\\', f"{float(cache[\\\'stability\\\'].float().mean()):.4f}",\\n        \\\'| build_s=\\\', f"{cache[\\\'build_time_s\\\']:.1f}",\\n    )\\n\\n    del model, sub, xyz, opacity, sum_desc, weight_sum, obs_count, residual_sum\\n    del best_weight, best_desc, best_view_id\\n    torch.cuda.empty_cache()\\n    return cache\\n\\n# Original notebook cell 49\\ndef mv_world_to_camera_points(points_xyz, c2w):\\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\\n    w2c = np.linalg.inv(np.asarray(c2w, dtype=np.float64))\\n    return points_xyz @ w2c[:3, :3].T + w2c[:3, 3]\\n\\ndef mv_match_descriptors(src_desc, tgt_desc, src_conf=None, tgt_conf=None):\\n    # Mirror logic của coarse_registration_gaussian_landmark trong repo:\\n    # mutual-NN + margin nhỏ nhất ở hai chiều.\\n    src = F.normalize(src_desc.float().to(MV_DEVICE), dim=-1)\\n    tgt = F.normalize(tgt_desc.float().to(MV_DEVICE), dim=-1)\\n    sim = src @ tgt.T\\n\\n    src_k = min(2, sim.shape[1])\\n    tgt_k = min(2, sim.shape[0])\\n    src_values, src_indices = sim.topk(k=src_k, dim=1)\\n    tgt_values, tgt_indices = sim.topk(k=tgt_k, dim=0)\\n\\n    source_to_target = src_indices[:, 0]\\n    target_to_source = tgt_indices[0, :]\\n    src_ids = torch.arange(src.shape[0], device=MV_DEVICE)\\n    mutual = target_to_source[source_to_target] == src_ids\\n\\n    matched_src = torch.nonzero(mutual).squeeze(1)\\n    matched_tgt = source_to_target[matched_src]\\n    match_similarity = src_values[matched_src, 0]\\n    src_margin = (\\n        src_values[matched_src, 0] - src_values[matched_src, 1]\\n        if src_k == 2 else torch.full_like(match_similarity, float(\\\'inf\\\'))\\n    )\\n    tgt_margin = (\\n        tgt_values[0, matched_tgt] - tgt_values[1, matched_tgt]\\n        if tgt_k == 2 else torch.full_like(match_similarity, float(\\\'inf\\\'))\\n    )\\n    match_margin = torch.minimum(src_margin, tgt_margin)\\n\\n    keep = (\\n        (match_similarity >= MV_MIN_SIMILARITY)\\n        & (match_margin >= MV_MIN_MARGIN)\\n    )\\n    matched_src = matched_src[keep]\\n    matched_tgt = matched_tgt[keep]\\n    scores = match_similarity[keep]\\n\\n    # Stability chỉ dùng để rank/diagnostic, không đổi correspondence threshold baseline.\\n    if src_conf is not None and tgt_conf is not None and len(matched_src):\\n        sc = src_conf.float().to(MV_DEVICE)[matched_src].clamp(0, 1)\\n        tc = tgt_conf.float().to(MV_DEVICE)[matched_tgt].clamp(0, 1)\\n        confidence = torch.sqrt(sc * tc).clamp_min(1e-4)\\n    else:\\n        confidence = torch.ones_like(scores)\\n\\n    return (\\n        matched_src.detach().cpu().numpy(),\\n        matched_tgt.detach().cpu().numpy(),\\n        scores.detach().cpu().numpy(),\\n        confidence.detach().cpu().numpy(),\\n        match_margin[keep].detach().cpu().numpy(),\\n    )\\n\\ndef mv_geometric_compatibility(src_pts, tgt_pts):\\n    # Dùng đúng helper của baseline repo, không adaptive threshold riêng.\\n    return _rigidity_compatibility_mask(\\n        src_pts,\\n        tgt_pts,\\n        MV_COMPATIBILITY_THRESHOLD_M,\\n        MV_MIN_COMPATIBILITY_SUPPORT,\\n    )\\n\\ndef mv_ransac_baseline(source, target, seed):\\n    # Dùng đúng Open3D RANSAC + inlier refit của baseline.\\n    try:\\n        o3d.utility.random.seed(int(seed))\\n    except Exception:\\n        pass\\n    T, n_inliers = _ransac_from_correspondences(\\n        source,\\n        target,\\n        MV_RANSAC_DISTANCE_THRESHOLD_M,\\n        refit_inliers=True,\\n    )\\n    return np.asarray(T, dtype=np.float64), int(n_inliers)\\n\\ndef mv_rotation_translation_error(T_est, T_gt):\\n    Rerr = T_est[:3, :3].T @ T_gt[:3, :3]\\n    c = np.clip((np.trace(Rerr) - 1.0) / 2.0, -1.0, 1.0)\\n    r = float(np.degrees(np.arccos(c)))\\n    t = float(np.linalg.norm(T_est[:3, 3] - T_gt[:3, 3]) * 100.0)\\n    return r, t\\n\\n# Original notebook cell 50\\ndef mv_gt_pose(agent_id, frame_id):\\n    pose = mv_datasets[int(agent_id)][int(frame_id)][-1]\\n    if torch.is_tensor(pose):\\n        pose = pose.detach().cpu().numpy()\\n    return np.asarray(pose, dtype=np.float64)\\n\\ndef mv_baseline_coarse_transform(loop):\\n    # Historical baseline coarse transform; DO NOT use loop.transformation (post-ICP).\\n    for name in (\\\'init_transformation\\\', \\\'coarse_transformation\\\', \\\'initial_transformation\\\'):\\n        if hasattr(loop, name):\\n            value = getattr(loop, name)\\n            if value is not None:\\n                arr = np.asarray(value, dtype=np.float64)\\n                if arr.shape == (4, 4):\\n                    return arr, name\\n    return None, None\\n\\ndef mv_metric_row(sa, sf, ta, tf):\\n    if _mv_metrics.empty:\\n        return None\\n    required = {\\\'source_agent_id\\\',\\\'source_frame_id\\\',\\\'target_agent_id\\\',\\\'target_frame_id\\\'}\\n    if not required.issubset(_mv_metrics.columns):\\n        return None\\n    m = _mv_metrics[\\n        (_mv_metrics.source_agent_id.astype(int) == int(sa))\\n        & (_mv_metrics.source_frame_id.astype(int) == int(sf))\\n        & (_mv_metrics.target_agent_id.astype(int) == int(ta))\\n        & (_mv_metrics.target_frame_id.astype(int) == int(tf))\\n    ]\\n    if len(m) == 0:\\n        return None\\n    return m.iloc[0]\\n\\ndef mv_first_metric(row, names):\\n    if row is None:\\n        return np.nan\\n    for name in names:\\n        if name in row.index and pd.notna(row[name]):\\n            return float(row[name])\\n    return np.nan\\n\\ndef mv_descriptor_for_variant(cache, variant):\\n    if variant == \\\'single_best_view\\\':\\n        return cache[\\\'descriptor_single_best_view\\\']\\n    if variant == \\\'multi_view\\\':\\n        return cache[\\\'descriptor_multi_view\\\']\\n    raise KeyError(variant)\\n\\ndef _safe_mean(x):\\n    x = pd.Series(x).dropna()\\n    return float(x.mean()) if len(x) else None\\n\\ndef _safe_std(x):\\n    x = pd.Series(x).dropna()\\n    return float(x.std(ddof=0)) if len(x) else None\\n\\n# Original notebook cell 58\\ndef mv_world_to_camera_points(points_xyz, c2w):\\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\\n    w2c = np.linalg.inv(np.asarray(c2w, dtype=np.float64))\\n    return points_xyz @ w2c[:3, :3].T + w2c[:3, 3]\\n\\ndef mv_rotation_translation_error(T_est, T_gt):\\n    Rerr = T_est[:3, :3].T @ T_gt[:3, :3]\\n    c = np.clip((np.trace(Rerr) - 1.0) / 2.0, -1.0, 1.0)\\n    r = float(np.degrees(np.arccos(c)))\\n    t = float(np.linalg.norm(T_est[:3, 3] - T_gt[:3, 3]) * 100.0)\\n    return r, t\\n\\ndef a_match_descriptors(src_desc, tgt_desc, src_conf=None, tgt_conf=None):\\n    # Exact logic from v4 Mục 13.5: mutual Top-1 + bidirectional Top1-Top2 margin.\\n    src = F.normalize(src_desc.float().to(B_DEVICE), dim=-1)\\n    tgt = F.normalize(tgt_desc.float().to(B_DEVICE), dim=-1)\\n    sim = src @ tgt.T\\n\\n    src_k = min(2, sim.shape[1])\\n    tgt_k = min(2, sim.shape[0])\\n    src_values, src_indices = sim.topk(k=src_k, dim=1)\\n    tgt_values, tgt_indices = sim.topk(k=tgt_k, dim=0)\\n\\n    source_to_target = src_indices[:, 0]\\n    target_to_source = tgt_indices[0, :]\\n    src_ids = torch.arange(src.shape[0], device=B_DEVICE)\\n    mutual = target_to_source[source_to_target] == src_ids\\n\\n    matched_src = torch.nonzero(mutual).squeeze(1)\\n    matched_tgt = source_to_target[matched_src]\\n    match_similarity = src_values[matched_src, 0]\\n    src_margin = (\\n        src_values[matched_src, 0] - src_values[matched_src, 1]\\n        if src_k == 2 else torch.full_like(match_similarity, float(\\\'inf\\\'))\\n    )\\n    tgt_margin = (\\n        tgt_values[0, matched_tgt] - tgt_values[1, matched_tgt]\\n        if tgt_k == 2 else torch.full_like(match_similarity, float(\\\'inf\\\'))\\n    )\\n    match_margin = torch.minimum(src_margin, tgt_margin)\\n\\n    keep = (\\n        (match_similarity >= A_MIN_SIMILARITY)\\n        & (match_margin >= A_MIN_MARGIN)\\n    )\\n    matched_src = matched_src[keep]\\n    matched_tgt = matched_tgt[keep]\\n    scores = match_similarity[keep]\\n\\n    if src_conf is not None and tgt_conf is not None and len(matched_src):\\n        sc = src_conf.float().to(B_DEVICE)[matched_src].clamp(0, 1)\\n        tc = tgt_conf.float().to(B_DEVICE)[matched_tgt].clamp(0, 1)\\n        confidence = torch.sqrt(sc * tc).clamp_min(1e-4)\\n    else:\\n        confidence = torch.ones_like(scores)\\n\\n    return (\\n        matched_src.detach().cpu().numpy(),\\n        matched_tgt.detach().cpu().numpy(),\\n        scores.detach().cpu().numpy(),\\n        confidence.detach().cpu().numpy(),\\n        match_margin[keep].detach().cpu().numpy(),\\n    )\\n\\ndef a_geometric_compatibility(src_pts, tgt_pts):\\n    return _rigidity_compatibility_mask(\\n        src_pts, tgt_pts,\\n        A_COMPATIBILITY_THRESHOLD_M,\\n        A_MIN_COMPATIBILITY_SUPPORT,\\n    )\\n\\ndef a_ransac(source, target, seed):\\n    try:\\n        o3d.utility.random.seed(int(seed))\\n    except Exception:\\n        pass\\n    T, n_inliers = _ransac_from_correspondences(\\n        source, target,\\n        A_RANSAC_DISTANCE_THRESHOLD_M,\\n        refit_inliers=True,\\n    )\\n    return np.asarray(T, dtype=np.float64), int(n_inliers)\\n\\ndef a_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\\n    t0 = time.perf_counter()\\n    src_xyz = mv_world_to_camera_points(scache[\\\'xyz\\\'].float().numpy(), src_c2w)\\n    tgt_xyz = mv_world_to_camera_points(tcache[\\\'xyz\\\'].float().numpy(), tgt_c2w)\\n\\n    sm, tm, sim, stability_conf, margin = a_match_descriptors(\\n        scache[\\\'descriptor_multi_view\\\'], tcache[\\\'descriptor_multi_view\\\'],\\n        scache[\\\'stability\\\'], tcache[\\\'stability\\\']\\n    )\\n    out = {\\n        \\\'T_est\\\': None,\\n        \\\'n_semantic_matches\\\': int(len(sm)),\\n        \\\'n_compatible_matches\\\': 0,\\n        \\\'n_ransac_inliers\\\': 0,\\n        \\\'ransac_inlier_ratio\\\': 0.0,\\n        \\\'mean_match_similarity\\\': float(np.mean(sim)) if len(sim) else np.nan,\\n        \\\'mean_match_margin\\\': float(np.mean(margin)) if len(margin) else np.nan,\\n        \\\'mean_compatibility_support\\\': 0.0,\\n        \\\'matcher_runtime_s\\\': np.nan,\\n    }\\n    if len(sm) >= 3:\\n        src_m = src_xyz[sm]\\n        tgt_m = tgt_xyz[tm]\\n        keep, support = a_geometric_compatibility(src_m, tgt_m)\\n        out[\\\'mean_compatibility_support\\\'] = float(np.mean(support)) if len(support) else 0.0\\n        if int(keep.sum()) >= 3:\\n            src_g, tgt_g = src_m[keep], tgt_m[keep]\\n            out[\\\'n_compatible_matches\\\'] = int(keep.sum())\\n            T, nin = a_ransac(src_g, tgt_g, seed)\\n            out[\\\'n_ransac_inliers\\\'] = int(nin)\\n            out[\\\'ransac_inlier_ratio\\\'] = float(nin / max(1, int(keep.sum())))\\n            out[\\\'T_est\\\'] = T\\n    out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - t0)\\n    return out\\n\\n# Original notebook cell 60\\nV2_TOPK_MAX = 5\\n\\nV2_MARGIN_TOP1 = 0.060\\n\\nV2_MARGIN_TOP3 = 0.025\\n\\nV2_RIGIDITY_THRESHOLD_M = 0.03\\n\\nV2_MIN_SUPPORT = 8\\n\\nV2_MIN_CORRESPONDENCES = 8\\n\\nV2_COVERAGE_VOXEL_M = 0.12\\n\\nV2_MAX_PER_SOURCE_VOXEL = 2\\n\\nV2_MAX_PER_TARGET_VOXEL = 2\\n\\nV2_MAX_RANSAC_PAIRS = 96\\n\\nV2_FALLBACK_MAX_PAIRS = 160\\n\\nV2_RANSAC_THRESHOLD_M = 0.05\\n\\nV2_TIGHT_REFIT_THRESHOLD_M = 0.03\\n\\nV2_RANSAC_MAX_ITERS = 700\\n\\nV2_RANSAC_MIN_ITERS = 64\\n\\nV2_RANSAC_EARLY_INLIER_RATIO = 0.82\\n\\nV2_RANSAC_EARLY_MEDIAN_M = 0.020\\n\\nV2_DESCRIPTOR_KEY = \\\'descriptor_multi_view\\\'\\n\\nV2_SEEDS = list(A_SEEDS)\\n\\nV3_DESCRIPTOR_KEY = V2_DESCRIPTOR_KEY\\n\\nV3_TOPK_MAX = V2_TOPK_MAX\\n\\nV3_RIGIDITY_THRESHOLD_M = V2_RIGIDITY_THRESHOLD_M\\n\\nV3_MIN_SUPPORT = V2_MIN_SUPPORT\\n\\nV3_MIN_CORRESPONDENCES = V2_MIN_CORRESPONDENCES\\n\\nV3_COVERAGE_VOXEL_M = V2_COVERAGE_VOXEL_M\\n\\nV3_FAST_MAX_PAIRS = 96\\n\\nV3_FAST_MAX_PER_SOURCE_VOXEL = 2\\n\\nV3_FAST_MAX_PER_TARGET_VOXEL = 2\\n\\nV3_HARD_TOPK = 5\\n\\nV3_HARD_MAX_PAIRS = 192\\n\\nV3_HARD_MAX_PER_SOURCE_VOXEL = 3\\n\\nV3_HARD_MAX_PER_TARGET_VOXEL = 3\\n\\nV3_HARD_MAX_SOURCE_REUSE = 2\\n\\nV3_HARD_MAX_TARGET_REUSE = 2\\n\\nV3_HARD_TRIGGER_RATIO_5CM = 0.80\\n\\nV3_HARD_TRIGGER_RATIO_3CM = 0.70\\n\\nV3_RANSAC_THRESHOLD_M = V2_RANSAC_THRESHOLD_M\\n\\nV3_TIGHT_REFIT_THRESHOLD_M = V2_TIGHT_REFIT_THRESHOLD_M\\n\\nV3_FAST_RANSAC_MAX_ITERS = 700\\n\\nV3_FAST_RANSAC_MIN_ITERS = 64\\n\\nV3_FAST_EARLY_RATIO = 0.82\\n\\nV3_FAST_EARLY_MEDIAN_M = 0.020\\n\\nV3_HARD_RANSAC_MAX_ITERS = 1800\\n\\nV3_HARD_RANSAC_MIN_ITERS = 96\\n\\nV3_HARD_EARLY_RATIO = 0.78\\n\\nV3_HARD_EARLY_MEDIAN_M = 0.025\\n\\nV3_SCORE_WEIGHTS = {\\n    \\\'weighted_ratio_5cm\\\': 0.50,\\n    \\\'ratio_5cm\\\': 0.20,\\n    \\\'ratio_3cm\\\': 0.20,\\n    \\\'residual_quality\\\': 0.10,\\n}\\n\\nV3_HARD_MIN_VALIDATION_SCORE_GAIN = 0.005\\n\\nV3_SEEDS = list(V2_SEEDS)\\n\\nV32_DESCRIPTOR_KEY = V3_DESCRIPTOR_KEY\\n\\nV32_MIN_CORRESPONDENCES = V3_MIN_CORRESPONDENCES\\n\\nV32_RIGIDITY_THRESHOLD_M = V3_RIGIDITY_THRESHOLD_M\\n\\nV32_MIN_SUPPORT = V3_MIN_SUPPORT\\n\\nV32_FAST_MAX_PAIRS = V3_FAST_MAX_PAIRS\\n\\nV32_FAST_MAX_PER_SOURCE_VOXEL = V3_FAST_MAX_PER_SOURCE_VOXEL\\n\\nV32_FAST_MAX_PER_TARGET_VOXEL = V3_FAST_MAX_PER_TARGET_VOXEL\\n\\nV32_FAST_RANSAC_MAX_ITERS = V3_FAST_RANSAC_MAX_ITERS\\n\\nV32_FAST_RANSAC_MIN_ITERS = V3_FAST_RANSAC_MIN_ITERS\\n\\nV32_FAST_EARLY_RATIO = V3_FAST_EARLY_RATIO\\n\\nV32_FAST_EARLY_MEDIAN_M = V3_FAST_EARLY_MEDIAN_M\\n\\nV32_HARD_TRIGGER_RATIO_5CM = V3_HARD_TRIGGER_RATIO_5CM\\n\\nV32_HARD_TRIGGER_RATIO_3CM = V3_HARD_TRIGGER_RATIO_3CM\\n\\nV32_HARD_TOPK = 5\\n\\nV32_HARD_PREFILTER_MAX_PAIRS = 180\\n\\nV32_HARD_MAX_PAIRS = 144\\n\\nV32_HARD_MAX_PER_SOURCE_VOXEL = 3\\n\\nV32_HARD_MAX_PER_TARGET_VOXEL = 3\\n\\nV32_MARGIN_FULL_SCALE = 0.060\\n\\nV32_HARD_QUALITY_WEIGHTS = {\\n    \\\'base_confidence\\\': 0.60,\\n    \\\'margin_quality\\\': 0.25,\\n    \\\'rank_quality\\\': 0.15,\\n}\\n\\nV32_LOCAL_INITIAL_GATE_M = 0.080\\n\\nV32_LOCAL_FINAL_GATE_M = 0.050\\n\\nV32_LOCAL_ROBUST_SCALE_M = 0.030\\n\\nV32_LOCAL_MIN_PAIRS = 12\\n\\nV32_LOCAL_IRLS_ITERS = 3\\n\\nV32_TRUST_MAX_ROT_DEG = 0.35\\n\\nV32_TRUST_MAX_TRANS_M = 0.010\\n\\nV32_LINESEARCH_FRACTIONS = (0.25, 0.50, 0.75, 1.00)\\n\\nV32_LOCAL_SCORE_WEIGHTS = {\\n    \\\'weighted_ratio_5cm\\\': 0.30,\\n    \\\'ratio_5cm\\\': 0.20,\\n    \\\'ratio_3cm\\\': 0.35,\\n    \\\'residual_quality\\\': 0.15,\\n}\\n\\nV32_STEP_PENALTY_ROT = 0.0015\\n\\nV32_STEP_PENALTY_TRANS = 0.0015\\n\\nV32_MIN_REGULARIZED_GAIN = 0.0010\\n\\nV32_MAX_RATIO5_DROP = 0.010\\n\\nV32_MAX_RATIO3_DROP = 0.005\\n\\nV32_MAX_MEDIAN5_INCREASE_M = 0.0015\\n\\nV32_SEEDS = list(A_SEEDS)\\n\\n# Original notebook cell 61\\ndef v2_np(x):\\n    if torch.is_tensor(x):\\n        x = x.detach().cpu().numpy()\\n    return np.asarray(x)\\n\\ndef v2_normalize_desc(x):\\n    x = v2_np(x).astype(np.float64, copy=False)\\n    n = np.linalg.norm(x, axis=1, keepdims=True)\\n    return x / np.maximum(n, 1e-12)\\n\\ndef v2_adaptive_k_from_row(sorted_scores):\\n    """K=1 khi Top-1 rõ; K=3 khi vừa; K=5 khi mơ hồ."""\\n    n = len(sorted_scores)\\n    if n <= 1:\\n        return 1\\n    margin12 = float(sorted_scores[0] - sorted_scores[1])\\n    if margin12 >= V2_MARGIN_TOP1:\\n        return 1\\n    if margin12 >= V2_MARGIN_TOP3:\\n        return min(3, n)\\n    return min(V2_TOPK_MAX, n)\\n\\ndef v2_build_adaptive_bidirectional(src_desc, tgt_desc):\\n    src = v2_normalize_desc(src_desc)\\n    tgt = v2_normalize_desc(tgt_desc)\\n    sim = src @ tgt.T\\n\\n    ns, nt = sim.shape\\n    src_lists = []\\n    src_k_used = []\\n    for i in range(ns):\\n        order = np.argsort(-sim[i])[:min(V2_TOPK_MAX, nt)]\\n        scores = sim[i, order]\\n        k = v2_adaptive_k_from_row(scores)\\n        src_lists.append(order[:k].tolist())\\n        src_k_used.append(k)\\n\\n    tgt_lists = []\\n    tgt_k_used = []\\n    sim_t = sim.T\\n    for j in range(nt):\\n        order = np.argsort(-sim_t[j])[:min(V2_TOPK_MAX, ns)]\\n        scores = sim_t[j, order]\\n        k = v2_adaptive_k_from_row(scores)\\n        tgt_lists.append(set(order[:k].tolist()))\\n        tgt_k_used.append(k)\\n\\n    pairs, sims = [], []\\n    for i, candidates in enumerate(src_lists):\\n        for j in candidates:\\n            j = int(j)\\n            if i in tgt_lists[j]:\\n                pairs.append((i, j))\\n                sims.append(float(sim[i, j]))\\n\\n    if not pairs:\\n        return (\\n            np.empty((0, 2), dtype=np.int64),\\n            np.empty((0,), dtype=np.float64),\\n            np.asarray(src_k_used),\\n            np.asarray(tgt_k_used),\\n        )\\n\\n    return (\\n        np.asarray(pairs, dtype=np.int64),\\n        np.asarray(sims, dtype=np.float64),\\n        np.asarray(src_k_used),\\n        np.asarray(tgt_k_used),\\n    )\\n\\ndef v2_pair_quality(scache, tcache, pairs, similarities, support):\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    if len(pairs) == 0:\\n        return np.empty((0,), dtype=np.float64)\\n\\n    sidx, tidx = pairs[:, 0], pairs[:, 1]\\n\\n    s_stab = v2_np(scache[\\\'stability\\\']).astype(np.float64)[sidx]\\n    t_stab = v2_np(tcache[\\\'stability\\\']).astype(np.float64)[tidx]\\n    stability = np.sqrt(np.clip(s_stab * t_stab, 0.0, 1.0))\\n\\n    s_obs = v2_np(scache[\\\'observation_count\\\']).astype(np.float64)[sidx]\\n    t_obs = v2_np(tcache[\\\'observation_count\\\']).astype(np.float64)[tidx]\\n    obs = np.sqrt(np.log1p(s_obs) * np.log1p(t_obs))\\n    obs /= max(np.log1p(10.0), 1e-12)\\n    obs = np.clip(obs, 0.0, 1.0)\\n\\n    s_res = v2_np(scache[\\\'depth_residual_mean\\\']).astype(np.float64)[sidx]\\n    t_res = v2_np(tcache[\\\'depth_residual_mean\\\']).astype(np.float64)[tidx]\\n    depth_q = np.exp(-(s_res + t_res) / max(2.0 * MV_DEPTH_TOLERANCE_M, 1e-6))\\n\\n    s_op = v2_np(scache[\\\'opacity\\\']).astype(np.float64)[sidx]\\n    t_op = v2_np(tcache[\\\'opacity\\\']).astype(np.float64)[tidx]\\n    opacity_q = np.sqrt(np.clip(s_op * t_op, 0.0, 1.0))\\n\\n    support = np.asarray(support, dtype=np.float64)\\n    support_q = support / max(float(np.max(support)) if len(support) else 1.0, 1.0)\\n\\n    similarity_q = np.clip(np.asarray(similarities, dtype=np.float64), 0.0, 1.0)\\n    metadata_q = stability * depth_q * (0.5 + 0.5 * obs) * (0.5 + 0.5 * opacity_q)\\n\\n    # Development score; keep weights explicit for ablation.\\n    confidence = 0.45 * similarity_q + 0.35 * support_q + 0.20 * metadata_q\\n    return np.asarray(confidence, dtype=np.float64)\\n\\ndef v2_greedy_unique_by_confidence(pairs, confidence, support, min_support=V2_MIN_SUPPORT):\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    support = np.asarray(support, dtype=np.int64)\\n\\n    valid = np.flatnonzero(support >= int(min_support))\\n    if len(valid) == 0:\\n        return np.empty((0,2), np.int64), np.empty(0), np.empty(0, np.int64)\\n\\n    order = valid[np.argsort(-confidence[valid])]\\n    used_s, used_t, keep = set(), set(), []\\n    for idx in order:\\n        s, t = map(int, pairs[idx])\\n        if s in used_s or t in used_t:\\n            continue\\n        used_s.add(s); used_t.add(t); keep.append(int(idx))\\n\\n    keep = np.asarray(keep, dtype=np.int64)\\n    return pairs[keep], confidence[keep], support[keep]\\n\\ndef v2_spatial_coverage_select(pairs, confidence, src_xyz, tgt_xyz):\\n    """Select high-confidence pairs while limiting concentration per source/target voxel."""\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    if len(pairs) <= V2_MAX_RANSAC_PAIRS:\\n        return pairs, confidence\\n\\n    order = np.argsort(-confidence)\\n    src_vox = np.floor(src_xyz / V2_COVERAGE_VOXEL_M).astype(np.int64)\\n    tgt_vox = np.floor(tgt_xyz / V2_COVERAGE_VOXEL_M).astype(np.int64)\\n\\n    src_count, tgt_count = {}, {}\\n    keep = []\\n    for idx in order:\\n        s, t = map(int, pairs[idx])\\n        sv = tuple(src_vox[s].tolist())\\n        tv = tuple(tgt_vox[t].tolist())\\n        if src_count.get(sv, 0) >= V2_MAX_PER_SOURCE_VOXEL:\\n            continue\\n        if tgt_count.get(tv, 0) >= V2_MAX_PER_TARGET_VOXEL:\\n            continue\\n        keep.append(int(idx))\\n        src_count[sv] = src_count.get(sv, 0) + 1\\n        tgt_count[tv] = tgt_count.get(tv, 0) + 1\\n        if len(keep) >= V2_MAX_RANSAC_PAIRS:\\n            break\\n\\n    # If coverage was too restrictive, fill by confidence up to fallback cap.\\n    if len(keep) < V2_MIN_CORRESPONDENCES:\\n        keep_set = set(keep)\\n        for idx in order:\\n            if int(idx) not in keep_set:\\n                keep.append(int(idx)); keep_set.add(int(idx))\\n            if len(keep) >= min(V2_FALLBACK_MAX_PAIRS, len(pairs)):\\n                break\\n\\n    keep = np.asarray(keep, dtype=np.int64)\\n    return pairs[keep], confidence[keep]\\n\\ndef v2_geometry_condition(points):\\n    points = np.asarray(points, dtype=np.float64)\\n    if len(points) < 3:\\n        return {\\\'s1\\\': np.nan, \\\'s2_s1\\\': 0.0, \\\'s3_s1\\\': 0.0}\\n    x = points - points.mean(axis=0, keepdims=True)\\n    s = np.linalg.svd(x, compute_uv=False)\\n    s = np.pad(s, (0, max(0, 3-len(s))), constant_values=0.0)[:3]\\n    return {\\n        \\\'s1\\\': float(s[0]),\\n        \\\'s2_s1\\\': float(s[1] / max(s[0], 1e-12)),\\n        \\\'s3_s1\\\': float(s[2] / max(s[0], 1e-12)),\\n    }\\n\\n# Original notebook cell 62\\ndef mv_topk_rigidity_support(candidate_pairs, src_xyz, tgt_xyz, threshold_m=0.03):\\n    candidate_pairs = np.asarray(candidate_pairs, dtype=np.int64)\\n    if len(candidate_pairs) == 0:\\n        return np.empty((0,), dtype=np.int64)\\n    src_pair_xyz = np.asarray(src_xyz)[candidate_pairs[:, 0]]\\n    tgt_pair_xyz = np.asarray(tgt_xyz)[candidate_pairs[:, 1]]\\n    src_dist = np.linalg.norm(src_pair_xyz[:, None, :] - src_pair_xyz[None, :, :], axis=2)\\n    tgt_dist = np.linalg.norm(tgt_pair_xyz[:, None, :] - tgt_pair_xyz[None, :, :], axis=2)\\n    compatible = np.abs(src_dist - tgt_dist) <= float(threshold_m)\\n    np.fill_diagonal(compatible, False)\\n    same_src = candidate_pairs[:, 0][:, None] == candidate_pairs[:, 0][None, :]\\n    same_tgt = candidate_pairs[:, 1][:, None] == candidate_pairs[:, 1][None, :]\\n    compatible[same_src | same_tgt] = False\\n    return compatible.sum(axis=1).astype(np.int64)\\n\\n# Original notebook cell 63\\ndef v2_weighted_kabsch(P, Q, w=None):\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    if len(P) < 3:\\n        raise ValueError(\\\'Cần >= 3 correspondence.\\\')\\n\\n    if w is None:\\n        w = np.ones(len(P), dtype=np.float64)\\n    w = np.asarray(w, dtype=np.float64)\\n    w = np.clip(w, 1e-8, None)\\n    w = w / w.sum()\\n\\n    pc = np.sum(P * w[:, None], axis=0)\\n    qc = np.sum(Q * w[:, None], axis=0)\\n    X = P - pc\\n    Y = Q - qc\\n    H = (X * w[:, None]).T @ Y\\n    U, _, Vt = np.linalg.svd(H)\\n    R = Vt.T @ U.T\\n    if np.linalg.det(R) < 0:\\n        Vt[-1] *= -1.0\\n        R = Vt.T @ U.T\\n    t = qc - R @ pc\\n\\n    T = np.eye(4, dtype=np.float64)\\n    T[:3,:3] = R\\n    T[:3,3] = t\\n    return T\\n\\ndef v2_transform_points(T, P):\\n    return (T[:3,:3] @ np.asarray(P).T).T + T[:3,3][None,:]\\n\\ndef v2_non_degenerate_triplet(P):\\n    a, b, c = np.asarray(P, dtype=np.float64)\\n    area2 = np.linalg.norm(np.cross(b-a, c-a))\\n    return bool(area2 > 1e-5)\\n\\ndef v2_prosac_ransac(P, Q, confidence, seed):\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    n = len(P)\\n    if n < 3:\\n        return None\\n\\n    order = np.argsort(-confidence)\\n    P = P[order]; Q = Q[order]; confidence = confidence[order]\\n    conf_prob = np.clip(confidence, 1e-8, None)\\n    rng = np.random.default_rng(int(seed))\\n\\n    best = None\\n    best_score = -np.inf\\n    best_inliers = None\\n    best_median = np.inf\\n    start = time.perf_counter()\\n\\n    start_pool = min(max(12, 3), n)\\n\\n    for it in range(V2_RANSAC_MAX_ITERS):\\n        frac = it / max(1, V2_RANSAC_MAX_ITERS - 1)\\n        pool_n = min(n, start_pool + int(frac * max(0, n - start_pool)))\\n        probs = conf_prob[:pool_n]\\n        probs = probs / probs.sum()\\n        sample = rng.choice(pool_n, size=3, replace=False, p=probs)\\n\\n        if not v2_non_degenerate_triplet(P[sample]) or not v2_non_degenerate_triplet(Q[sample]):\\n            continue\\n\\n        try:\\n            T = v2_weighted_kabsch(P[sample], Q[sample], confidence[sample])\\n        except Exception:\\n            continue\\n\\n        residual = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\\n        inliers = residual <= V2_RANSAC_THRESHOLD_M\\n        nin = int(inliers.sum())\\n        if nin < 3:\\n            continue\\n\\n        weighted_support = float(confidence[inliers].sum())\\n        med = float(np.median(residual[inliers]))\\n        score = weighted_support - 0.25 * med\\n\\n        if score > best_score or (np.isclose(score, best_score) and med < best_median):\\n            best = T\\n            best_score = score\\n            best_inliers = inliers.copy()\\n            best_median = med\\n\\n        if it + 1 >= V2_RANSAC_MIN_ITERS and best_inliers is not None:\\n            ratio = float(best_inliers.mean())\\n            if ratio >= V2_RANSAC_EARLY_INLIER_RATIO and best_median <= V2_RANSAC_EARLY_MEDIAN_M:\\n                break\\n\\n    if best_inliers is None or int(best_inliers.sum()) < 3:\\n        return None\\n\\n    # Weighted refit on 5 cm consensus.\\n    T_refit = v2_weighted_kabsch(P[best_inliers], Q[best_inliers], confidence[best_inliers])\\n    residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\\n    inliers5 = residual <= V2_RANSAC_THRESHOLD_M\\n\\n    # Tight 3 cm refit when enough strong pairs remain.\\n    tight = residual <= V2_TIGHT_REFIT_THRESHOLD_M\\n    if int(tight.sum()) >= 6:\\n        T_refit = v2_weighted_kabsch(P[tight], Q[tight], confidence[tight])\\n        residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\\n        inliers5 = residual <= V2_RANSAC_THRESHOLD_M\\n        tight = residual <= V2_TIGHT_REFIT_THRESHOLD_M\\n\\n    return {\\n        \\\'T\\\': T_refit,\\n        \\\'n_inliers_5cm\\\': int(inliers5.sum()),\\n        \\\'n_inliers_3cm\\\': int(tight.sum()),\\n        \\\'inlier_ratio_5cm\\\': float(inliers5.mean()),\\n        \\\'median_residual_m\\\': float(np.median(residual[inliers5])) if inliers5.any() else np.inf,\\n        \\\'iterations\\\': int(it + 1),\\n        \\\'runtime_s\\\': float(time.perf_counter() - start),\\n    }\\n\\ndef v2_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\\n    t0 = time.perf_counter()\\n\\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\\\'xyz\\\']).astype(np.float64), src_c2w)\\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\\\'xyz\\\']).astype(np.float64), tgt_c2w)\\n\\n    pairs, similarities, src_k, tgt_k = v2_build_adaptive_bidirectional(\\n        scache[V2_DESCRIPTOR_KEY], tcache[V2_DESCRIPTOR_KEY]\\n    )\\n\\n    out = {\\n        \\\'T_est\\\': None,\\n        \\\'candidate_pairs\\\': int(len(pairs)),\\n        \\\'mean_src_k\\\': float(np.mean(src_k)) if len(src_k) else np.nan,\\n        \\\'mean_tgt_k\\\': float(np.mean(tgt_k)) if len(tgt_k) else np.nan,\\n        \\\'rigidity_pairs\\\': 0,\\n        \\\'unique_pairs\\\': 0,\\n        \\\'coverage_pairs\\\': 0,\\n        \\\'mean_confidence\\\': np.nan,\\n        \\\'source_s2_s1\\\': np.nan,\\n        \\\'source_s3_s1\\\': np.nan,\\n        \\\'target_s2_s1\\\': np.nan,\\n        \\\'target_s3_s1\\\': np.nan,\\n        \\\'ransac_inliers_5cm\\\': 0,\\n        \\\'ransac_inliers_3cm\\\': 0,\\n        \\\'ransac_iterations\\\': 0,\\n        \\\'ransac_runtime_s\\\': 0.0,\\n        \\\'matcher_runtime_s\\\': np.nan,\\n    }\\n\\n    if len(pairs) < V2_MIN_CORRESPONDENCES:\\n        out[\\\'matcher_runtime_s\\\'] = time.perf_counter() - t0\\n        return out\\n\\n    support = mv_topk_rigidity_support(\\n        pairs, src_xyz, tgt_xyz, threshold_m=V2_RIGIDITY_THRESHOLD_M\\n    )\\n    out[\\\'rigidity_pairs\\\'] = int(np.sum(support >= V2_MIN_SUPPORT))\\n\\n    conf_all = v2_pair_quality(scache, tcache, pairs, similarities, support)\\n    unique_pairs, unique_conf, unique_support = v2_greedy_unique_by_confidence(\\n        pairs, conf_all, support, min_support=V2_MIN_SUPPORT\\n    )\\n    out[\\\'unique_pairs\\\'] = int(len(unique_pairs))\\n\\n    if len(unique_pairs) < V2_MIN_CORRESPONDENCES:\\n        out[\\\'matcher_runtime_s\\\'] = time.perf_counter() - t0\\n        return out\\n\\n    selected_pairs, selected_conf = v2_spatial_coverage_select(\\n        unique_pairs, unique_conf, src_xyz, tgt_xyz\\n    )\\n    out[\\\'coverage_pairs\\\'] = int(len(selected_pairs))\\n    out[\\\'mean_confidence\\\'] = float(np.mean(selected_conf)) if len(selected_conf) else np.nan\\n\\n    if len(selected_pairs) < V2_MIN_CORRESPONDENCES:\\n        out[\\\'matcher_runtime_s\\\'] = time.perf_counter() - t0\\n        return out\\n\\n    P = src_xyz[selected_pairs[:,0]]\\n    Q = tgt_xyz[selected_pairs[:,1]]\\n    gs = v2_geometry_condition(P)\\n    gt = v2_geometry_condition(Q)\\n    out[\\\'source_s2_s1\\\'], out[\\\'source_s3_s1\\\'] = gs[\\\'s2_s1\\\'], gs[\\\'s3_s1\\\']\\n    out[\\\'target_s2_s1\\\'], out[\\\'target_s3_s1\\\'] = gt[\\\'s2_s1\\\'], gt[\\\'s3_s1\\\']\\n\\n    r = v2_prosac_ransac(P, Q, selected_conf, seed)\\n    if r is None:\\n        out[\\\'matcher_runtime_s\\\'] = time.perf_counter() - t0\\n        return out\\n\\n    out[\\\'T_est\\\'] = r[\\\'T\\\']\\n    out[\\\'ransac_inliers_5cm\\\'] = r[\\\'n_inliers_5cm\\\']\\n    out[\\\'ransac_inliers_3cm\\\'] = r[\\\'n_inliers_3cm\\\']\\n    out[\\\'ransac_iterations\\\'] = r[\\\'iterations\\\']\\n    out[\\\'ransac_runtime_s\\\'] = r[\\\'runtime_s\\\']\\n    out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - t0)\\n    return out\\n\\n# Original notebook cell 64\\ndef v3_conf_stats(x, prefix=\\\'\\\'):\\n    x = np.asarray(x, dtype=np.float64)\\n    if len(x) == 0:\\n        return {\\n            f\\\'{prefix}confidence_mean\\\': np.nan,\\n            f\\\'{prefix}confidence_std\\\': np.nan,\\n            f\\\'{prefix}confidence_p10\\\': np.nan,\\n            f\\\'{prefix}confidence_p50\\\': np.nan,\\n            f\\\'{prefix}confidence_p90\\\': np.nan,\\n        }\\n    return {\\n        f\\\'{prefix}confidence_mean\\\': float(np.mean(x)),\\n        f\\\'{prefix}confidence_std\\\': float(np.std(x)),\\n        f\\\'{prefix}confidence_p10\\\': float(np.percentile(x, 10)),\\n        f\\\'{prefix}confidence_p50\\\': float(np.percentile(x, 50)),\\n        f\\\'{prefix}confidence_p90\\\': float(np.percentile(x, 90)),\\n    }\\n\\ndef v3_spatial_coverage_select(\\n    pairs,\\n    confidence,\\n    support,\\n    src_xyz,\\n    tgt_xyz,\\n    *,\\n    max_pairs,\\n    max_per_source_voxel,\\n    max_per_target_voxel,\\n    min_keep=V3_MIN_CORRESPONDENCES,\\n):\\n    """Always enforce source + target voxel coverage.\\n\\n    Important V2 fix:\\n      V2 returned immediately when len(pairs) <= 96, so hard loops with\\n      e.g. 85 pairs never received spatial balancing. V3 always executes\\n      the voxel-selection pass.\\n    """\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    support = np.asarray(support, dtype=np.int64)\\n\\n    if len(pairs) == 0:\\n        info = {\\n            \\\'input_pairs\\\': 0,\\n            \\\'strict_coverage_pairs\\\': 0,\\n            \\\'filled_pairs\\\': 0,\\n            \\\'output_pairs\\\': 0,\\n        }\\n        return pairs, confidence, support, info\\n\\n    order = np.argsort(-confidence)\\n    src_vox = np.floor(np.asarray(src_xyz) / V3_COVERAGE_VOXEL_M).astype(np.int64)\\n    tgt_vox = np.floor(np.asarray(tgt_xyz) / V3_COVERAGE_VOXEL_M).astype(np.int64)\\n\\n    src_count, tgt_count = {}, {}\\n    keep = []\\n\\n    for idx in order:\\n        s, t = map(int, pairs[idx])\\n        sv = tuple(src_vox[s].tolist())\\n        tv = tuple(tgt_vox[t].tolist())\\n\\n        if src_count.get(sv, 0) >= int(max_per_source_voxel):\\n            continue\\n        if tgt_count.get(tv, 0) >= int(max_per_target_voxel):\\n            continue\\n\\n        keep.append(int(idx))\\n        src_count[sv] = src_count.get(sv, 0) + 1\\n        tgt_count[tv] = tgt_count.get(tv, 0) + 1\\n\\n        if len(keep) >= min(int(max_pairs), len(pairs)):\\n            break\\n\\n    strict_n = len(keep)\\n\\n    # Safety only: if strict coverage leaves too few pairs for rigid pose,\\n    # fill by confidence until min_keep. Do NOT fill back to max_pairs.\\n    if len(keep) < min(int(min_keep), len(pairs)):\\n        keep_set = set(keep)\\n        for idx in order:\\n            idx = int(idx)\\n            if idx in keep_set:\\n                continue\\n            keep.append(idx)\\n            keep_set.add(idx)\\n            if len(keep) >= min(int(min_keep), int(max_pairs), len(pairs)):\\n                break\\n\\n    keep = np.asarray(keep, dtype=np.int64)\\n    info = {\\n        \\\'input_pairs\\\': int(len(pairs)),\\n        \\\'strict_coverage_pairs\\\': int(strict_n),\\n        \\\'filled_pairs\\\': int(max(0, len(keep) - strict_n)),\\n        \\\'output_pairs\\\': int(len(keep)),\\n    }\\n    return pairs[keep], confidence[keep], support[keep], info\\n\\ndef v3_greedy_limited_reuse(\\n    pairs,\\n    confidence,\\n    support,\\n    *,\\n    min_support=V3_MIN_SUPPORT,\\n    max_source_reuse=V3_HARD_MAX_SOURCE_REUSE,\\n    max_target_reuse=V3_HARD_MAX_TARGET_REUSE,\\n):\\n    """Relax strict 1-to-1 only for the hard fallback.\\n\\n    Each landmark can participate in at most a small number of pairs.\\n    This expands the pool without allowing a single ambiguous landmark\\n    to dominate RANSAC.\\n    """\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    support = np.asarray(support, dtype=np.int64)\\n\\n    valid = np.flatnonzero(support >= int(min_support))\\n    if len(valid) == 0:\\n        return (\\n            np.empty((0, 2), dtype=np.int64),\\n            np.empty((0,), dtype=np.float64),\\n            np.empty((0,), dtype=np.int64),\\n        )\\n\\n    # Confidence primary; rigidity support as tie-breaker.\\n    order_local = np.lexsort((-support[valid], -confidence[valid]))\\n    order = valid[order_local]\\n\\n    src_use, tgt_use = {}, {}\\n    seen = set()\\n    keep = []\\n\\n    for idx in order:\\n        s, t = map(int, pairs[idx])\\n        key = (s, t)\\n        if key in seen:\\n            continue\\n        if src_use.get(s, 0) >= int(max_source_reuse):\\n            continue\\n        if tgt_use.get(t, 0) >= int(max_target_reuse):\\n            continue\\n\\n        keep.append(int(idx))\\n        seen.add(key)\\n        src_use[s] = src_use.get(s, 0) + 1\\n        tgt_use[t] = tgt_use.get(t, 0) + 1\\n\\n    keep = np.asarray(keep, dtype=np.int64)\\n    return pairs[keep], confidence[keep], support[keep]\\n\\ndef v3_geometry_diag(pairs, src_xyz, tgt_xyz, prefix):\\n    if len(pairs) < 3:\\n        return {\\n            f\\\'{prefix}source_s2_s1\\\': np.nan,\\n            f\\\'{prefix}source_s3_s1\\\': np.nan,\\n            f\\\'{prefix}target_s2_s1\\\': np.nan,\\n            f\\\'{prefix}target_s3_s1\\\': np.nan,\\n        }\\n    P = src_xyz[pairs[:, 0]]\\n    Q = tgt_xyz[pairs[:, 1]]\\n    gs = v2_geometry_condition(P)\\n    gt = v2_geometry_condition(Q)\\n    return {\\n        f\\\'{prefix}source_s2_s1\\\': gs[\\\'s2_s1\\\'],\\n        f\\\'{prefix}source_s3_s1\\\': gs[\\\'s3_s1\\\'],\\n        f\\\'{prefix}target_s2_s1\\\': gt[\\\'s2_s1\\\'],\\n        f\\\'{prefix}target_s3_s1\\\': gt[\\\'s3_s1\\\'],\\n    }\\n\\n# Original notebook cell 65\\ndef v3_consensus_stats(T, P, Q, confidence):\\n    """Evaluate one transform with dimensionless, comparable terms."""\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n\\n    if T is None or len(P) == 0:\\n        return {\\n            \\\'score\\\': -np.inf,\\n            \\\'ratio_5cm\\\': 0.0,\\n            \\\'ratio_3cm\\\': 0.0,\\n            \\\'weighted_ratio_5cm\\\': 0.0,\\n            \\\'median_5cm_m\\\': np.inf,\\n            \\\'mean_5cm_m\\\': np.inf,\\n            \\\'p90_all_m\\\': np.inf,\\n            \\\'residual_quality\\\': 0.0,\\n            \\\'n_inliers_5cm\\\': 0,\\n            \\\'n_inliers_3cm\\\': 0,\\n        }\\n\\n    residual = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\\n    in5 = residual <= V3_RANSAC_THRESHOLD_M\\n    in3 = residual <= V3_TIGHT_REFIT_THRESHOLD_M\\n\\n    ratio5 = float(np.mean(in5))\\n    ratio3 = float(np.mean(in3))\\n\\n    conf = np.clip(confidence, 1e-8, None)\\n    weighted_ratio5 = float(conf[in5].sum() / max(conf.sum(), 1e-12))\\n\\n    if np.any(in5):\\n        median5 = float(np.median(residual[in5]))\\n        mean5 = float(np.mean(residual[in5]))\\n        residual_quality = float(\\n            1.0 - np.clip(median5 / V3_RANSAC_THRESHOLD_M, 0.0, 1.0)\\n        )\\n    else:\\n        median5 = np.inf\\n        mean5 = np.inf\\n        residual_quality = 0.0\\n\\n    score = (\\n        V3_SCORE_WEIGHTS[\\\'weighted_ratio_5cm\\\'] * weighted_ratio5\\n        + V3_SCORE_WEIGHTS[\\\'ratio_5cm\\\'] * ratio5\\n        + V3_SCORE_WEIGHTS[\\\'ratio_3cm\\\'] * ratio3\\n        + V3_SCORE_WEIGHTS[\\\'residual_quality\\\'] * residual_quality\\n    )\\n\\n    return {\\n        \\\'score\\\': float(score),\\n        \\\'ratio_5cm\\\': ratio5,\\n        \\\'ratio_3cm\\\': ratio3,\\n        \\\'weighted_ratio_5cm\\\': weighted_ratio5,\\n        \\\'median_5cm_m\\\': median5,\\n        \\\'mean_5cm_m\\\': mean5,\\n        \\\'p90_all_m\\\': float(np.percentile(residual, 90)) if len(residual) else np.inf,\\n        \\\'residual_quality\\\': residual_quality,\\n        \\\'n_inliers_5cm\\\': int(in5.sum()),\\n        \\\'n_inliers_3cm\\\': int(in3.sum()),\\n    }\\n\\ndef v3_prosac_ransac(\\n    P,\\n    Q,\\n    confidence,\\n    seed,\\n    *,\\n    max_iters,\\n    min_iters,\\n    early_ratio,\\n    early_median_m,\\n):\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    confidence = np.asarray(confidence, dtype=np.float64)\\n    n = len(P)\\n    if n < 3:\\n        return None\\n\\n    order = np.argsort(-confidence)\\n    P = P[order]\\n    Q = Q[order]\\n    confidence = confidence[order]\\n\\n    conf_prob = np.clip(confidence, 1e-8, None)\\n    rng = np.random.default_rng(int(seed))\\n\\n    best_T = None\\n    best_stats = None\\n    best_score = -np.inf\\n    best_median = np.inf\\n    start = time.perf_counter()\\n\\n    start_pool = min(max(12, 3), n)\\n    iterations_run = 0\\n\\n    for it in range(int(max_iters)):\\n        iterations_run = it + 1\\n        frac = it / max(1, int(max_iters) - 1)\\n        pool_n = min(n, start_pool + int(frac * max(0, n - start_pool)))\\n\\n        probs = conf_prob[:pool_n]\\n        probs = probs / probs.sum()\\n        sample = rng.choice(pool_n, size=3, replace=False, p=probs)\\n\\n        if not v2_non_degenerate_triplet(P[sample]):\\n            continue\\n        if not v2_non_degenerate_triplet(Q[sample]):\\n            continue\\n\\n        try:\\n            T = v2_weighted_kabsch(P[sample], Q[sample], confidence[sample])\\n        except Exception:\\n            continue\\n\\n        stats = v3_consensus_stats(T, P, Q, confidence)\\n        if stats[\\\'n_inliers_5cm\\\'] < 3:\\n            continue\\n\\n        # New score: dimensionless [0,1]-like quality terms.\\n        if (\\n            stats[\\\'score\\\'] > best_score\\n            or (\\n                np.isclose(stats[\\\'score\\\'], best_score)\\n                and stats[\\\'median_5cm_m\\\'] < best_median\\n            )\\n        ):\\n            best_T = T\\n            best_stats = stats\\n            best_score = stats[\\\'score\\\']\\n            best_median = stats[\\\'median_5cm_m\\\']\\n\\n        if iterations_run >= int(min_iters) and best_stats is not None:\\n            if (\\n                best_stats[\\\'ratio_5cm\\\'] >= float(early_ratio)\\n                and best_stats[\\\'median_5cm_m\\\'] <= float(early_median_m)\\n            ):\\n                break\\n\\n    if best_T is None:\\n        return None\\n\\n    # 5 cm weighted refit.\\n    residual = np.linalg.norm(v2_transform_points(best_T, P) - Q, axis=1)\\n    in5 = residual <= V3_RANSAC_THRESHOLD_M\\n    if int(in5.sum()) >= 3:\\n        T_refit = v2_weighted_kabsch(P[in5], Q[in5], confidence[in5])\\n    else:\\n        T_refit = best_T\\n\\n    # Tight 3 cm refit when enough strong pairs remain.\\n    residual = np.linalg.norm(v2_transform_points(T_refit, P) - Q, axis=1)\\n    in3 = residual <= V3_TIGHT_REFIT_THRESHOLD_M\\n    if int(in3.sum()) >= 6:\\n        T_refit = v2_weighted_kabsch(P[in3], Q[in3], confidence[in3])\\n\\n    final_stats = v3_consensus_stats(T_refit, P, Q, confidence)\\n    final_stats.update({\\n        \\\'T\\\': T_refit,\\n        \\\'iterations\\\': int(iterations_run),\\n        \\\'runtime_s\\\': float(time.perf_counter() - start),\\n    })\\n    return final_stats\\n\\ndef v3_should_trigger_fallback(ransac_result):\\n    reasons = []\\n    if ransac_result is None:\\n        return True, [\\\'fast_ransac_failed\\\']\\n\\n    if int(ransac_result[\\\'iterations\\\']) >= int(V3_FAST_RANSAC_MAX_ITERS):\\n        reasons.append(\\\'fast_hit_max_iters\\\')\\n    if float(ransac_result[\\\'ratio_5cm\\\']) < float(V3_HARD_TRIGGER_RATIO_5CM):\\n        reasons.append(\\\'fast_ratio5_low\\\')\\n    if float(ransac_result[\\\'ratio_3cm\\\']) < float(V3_HARD_TRIGGER_RATIO_3CM):\\n        reasons.append(\\\'fast_ratio3_low\\\')\\n\\n    return bool(reasons), reasons\\n\\n# Original notebook cell 66\\ndef v32_build_hard_candidates(src_desc, tgt_desc, top_k=V32_HARD_TOPK):\\n    """Build mutual Top-K pairs and ambiguity features in ONE similarity pass."""\\n    src = v2_normalize_desc(src_desc)\\n    tgt = v2_normalize_desc(tgt_desc)\\n    sim = src @ tgt.T\\n    ns, nt = sim.shape\\n\\n    if ns == 0 or nt == 0:\\n        zf = np.empty((0,), dtype=np.float64)\\n        zi = np.empty((0,), dtype=np.int64)\\n        return np.empty((0, 2), dtype=np.int64), zf, {\\n            \\\'src_margin12\\\': zf, \\\'tgt_margin12\\\': zf,\\n            \\\'margin_quality\\\': zf, \\\'src_rank\\\': zi, \\\'tgt_rank\\\': zi,\\n            \\\'rank_quality\\\': zf,\\n        }\\n\\n    ks = min(max(2, int(top_k)), nt)\\n    kt = min(max(2, int(top_k)), ns)\\n\\n    # Partial top-k, then sort only within top-k for exact ranks/margins.\\n    src_idx = np.argpartition(-sim, kth=ks - 1, axis=1)[:, :ks]\\n    src_val = np.take_along_axis(sim, src_idx, axis=1)\\n    src_ord = np.argsort(-src_val, axis=1)\\n    src_idx = np.take_along_axis(src_idx, src_ord, axis=1)\\n    src_val = np.take_along_axis(src_val, src_ord, axis=1)\\n\\n    sim_t = sim.T\\n    tgt_idx = np.argpartition(-sim_t, kth=kt - 1, axis=1)[:, :kt]\\n    tgt_val = np.take_along_axis(sim_t, tgt_idx, axis=1)\\n    tgt_ord = np.argsort(-tgt_val, axis=1)\\n    tgt_idx = np.take_along_axis(tgt_idx, tgt_ord, axis=1)\\n    tgt_val = np.take_along_axis(tgt_val, tgt_ord, axis=1)\\n\\n    src_margin_all = np.maximum(src_val[:, 0] - src_val[:, 1], 0.0)\\n    tgt_margin_all = np.maximum(tgt_val[:, 0] - tgt_val[:, 1], 0.0)\\n\\n    # Rank lookup only for retained Top-K entries.\\n    src_rank = np.full((ns, nt), int(top_k), dtype=np.int16)\\n    tgt_rank = np.full((ns, nt), int(top_k), dtype=np.int16)\\n    for r in range(ks):\\n        src_rank[np.arange(ns), src_idx[:, r]] = r\\n    for r in range(kt):\\n        tgt_rank[tgt_idx[:, r], np.arange(nt)] = r\\n\\n    pairs = []\\n    pair_sim = []\\n    for s in range(ns):\\n        for t in src_idx[s]:\\n            t = int(t)\\n            if tgt_rank[s, t] >= int(top_k):\\n                continue\\n            pairs.append((s, t))\\n            pair_sim.append(float(sim[s, t]))\\n\\n    if not pairs:\\n        zf = np.empty((0,), dtype=np.float64)\\n        zi = np.empty((0,), dtype=np.int64)\\n        return np.empty((0, 2), dtype=np.int64), zf, {\\n            \\\'src_margin12\\\': zf, \\\'tgt_margin12\\\': zf,\\n            \\\'margin_quality\\\': zf, \\\'src_rank\\\': zi, \\\'tgt_rank\\\': zi,\\n            \\\'rank_quality\\\': zf,\\n        }\\n\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    pair_sim = np.asarray(pair_sim, dtype=np.float64)\\n    sidx, tidx = pairs[:, 0], pairs[:, 1]\\n\\n    sm = src_margin_all[sidx]\\n    tm = tgt_margin_all[tidx]\\n    smq = np.clip(sm / max(V32_MARGIN_FULL_SCALE, 1e-12), 0.0, 1.0)\\n    tmq = np.clip(tm / max(V32_MARGIN_FULL_SCALE, 1e-12), 0.0, 1.0)\\n    margin_q = np.sqrt(smq * tmq)\\n\\n    sr = src_rank[sidx, tidx].astype(np.int64)\\n    tr = tgt_rank[sidx, tidx].astype(np.int64)\\n    denom = max(2.0 * (int(top_k) - 1), 1.0)\\n    rank_q = np.clip(1.0 - (sr + tr) / denom, 0.0, 1.0)\\n\\n    ambiguity = {\\n        \\\'src_margin12\\\': sm.astype(np.float64),\\n        \\\'tgt_margin12\\\': tm.astype(np.float64),\\n        \\\'margin_quality\\\': margin_q.astype(np.float64),\\n        \\\'src_rank\\\': sr,\\n        \\\'tgt_rank\\\': tr,\\n        \\\'rank_quality\\\': rank_q.astype(np.float64),\\n    }\\n    return pairs, pair_sim, ambiguity\\n\\ndef v32_hard_quality(base_confidence, ambiguity):\\n    base = np.clip(np.asarray(base_confidence, dtype=np.float64), 0.0, 1.0)\\n    mq = np.clip(np.asarray(ambiguity[\\\'margin_quality\\\'], dtype=np.float64), 0.0, 1.0)\\n    rq = np.clip(np.asarray(ambiguity[\\\'rank_quality\\\'], dtype=np.float64), 0.0, 1.0)\\n    w = V32_HARD_QUALITY_WEIGHTS\\n    return (\\n        w[\\\'base_confidence\\\'] * base\\n        + w[\\\'margin_quality\\\'] * mq\\n        + w[\\\'rank_quality\\\'] * rq\\n    )\\n\\ndef v32_greedy_unique_with_indices(pairs, quality, support, min_support=V32_MIN_SUPPORT):\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    quality = np.asarray(quality, dtype=np.float64)\\n    support = np.asarray(support, dtype=np.int64)\\n    valid = np.flatnonzero(support >= int(min_support))\\n    if len(valid) == 0:\\n        e = np.empty((0,), dtype=np.int64)\\n        return np.empty((0, 2), dtype=np.int64), np.empty((0,)), np.empty((0,), dtype=np.int64), e\\n\\n    order_local = np.lexsort((-support[valid], -quality[valid]))\\n    order = valid[order_local]\\n    used_s, used_t, keep = set(), set(), []\\n    for idx in order:\\n        s, t = map(int, pairs[idx])\\n        if s in used_s or t in used_t:\\n            continue\\n        used_s.add(s); used_t.add(t); keep.append(int(idx))\\n    keep = np.asarray(keep, dtype=np.int64)\\n    return pairs[keep], quality[keep], support[keep], keep\\n\\ndef v32_top_quality_prefilter(pairs, quality, support, source_indices, max_pairs):\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    quality = np.asarray(quality, dtype=np.float64)\\n    support = np.asarray(support, dtype=np.int64)\\n    source_indices = np.asarray(source_indices, dtype=np.int64)\\n    if len(pairs) <= int(max_pairs):\\n        return pairs, quality, support, source_indices\\n    order = np.argsort(-quality)[:int(max_pairs)]\\n    return pairs[order], quality[order], support[order], source_indices[order]\\n\\ndef v32_ambiguity_diag(ambiguity, idx, prefix=\\\'hard_\\\'):\\n    idx = np.asarray(idx, dtype=np.int64)\\n    if len(idx) == 0:\\n        return {\\n            f\\\'{prefix}src_margin12_mean\\\': np.nan,\\n            f\\\'{prefix}tgt_margin12_mean\\\': np.nan,\\n            f\\\'{prefix}margin_quality_mean\\\': np.nan,\\n            f\\\'{prefix}rank_quality_mean\\\': np.nan,\\n            f\\\'{prefix}mutual_top1_rate\\\': np.nan,\\n        }\\n    top1 = (ambiguity[\\\'src_rank\\\'][idx] == 0) & (ambiguity[\\\'tgt_rank\\\'][idx] == 0)\\n    return {\\n        f\\\'{prefix}src_margin12_mean\\\': float(np.mean(ambiguity[\\\'src_margin12\\\'][idx])),\\n        f\\\'{prefix}tgt_margin12_mean\\\': float(np.mean(ambiguity[\\\'tgt_margin12\\\'][idx])),\\n        f\\\'{prefix}margin_quality_mean\\\': float(np.mean(ambiguity[\\\'margin_quality\\\'][idx])),\\n        f\\\'{prefix}rank_quality_mean\\\': float(np.mean(ambiguity[\\\'rank_quality\\\'][idx])),\\n        f\\\'{prefix}mutual_top1_rate\\\': float(np.mean(top1)),\\n    }\\n\\ndef v32_rotation_angle_deg(R):\\n    R = np.asarray(R, dtype=np.float64)\\n    c = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)\\n    return float(np.degrees(np.arccos(c)))\\n\\ndef v32_fractional_rotation(R_delta, alpha):\\n    """Return R_delta ** alpha for a small rotation using Rodrigues."""\\n    R_delta = np.asarray(R_delta, dtype=np.float64)\\n    c = np.clip((np.trace(R_delta) - 1.0) / 2.0, -1.0, 1.0)\\n    theta = float(np.arccos(c))\\n    if theta < 1e-10:\\n        return np.eye(3, dtype=np.float64)\\n\\n    s = np.sin(theta)\\n    if abs(s) < 1e-8:\\n        # Trust-region deltas should never be near pi; safe SVD fallback.\\n        U, _, Vt = np.linalg.svd((1.0 - alpha) * np.eye(3) + alpha * R_delta)\\n        R = U @ Vt\\n        if np.linalg.det(R) < 0:\\n            U[:, -1] *= -1.0\\n            R = U @ Vt\\n        return R\\n\\n    axis = np.array([\\n        R_delta[2, 1] - R_delta[1, 2],\\n        R_delta[0, 2] - R_delta[2, 0],\\n        R_delta[1, 0] - R_delta[0, 1],\\n    ], dtype=np.float64) / (2.0 * s)\\n    axis /= max(np.linalg.norm(axis), 1e-12)\\n    a = float(alpha) * theta\\n    K = np.array([\\n        [0.0, -axis[2], axis[1]],\\n        [axis[2], 0.0, -axis[0]],\\n        [-axis[1], axis[0], 0.0],\\n    ], dtype=np.float64)\\n    return np.eye(3) + np.sin(a) * K + (1.0 - np.cos(a)) * (K @ K)\\n\\ndef v32_pose_delta(T_candidate, T_fast):\\n    if T_candidate is None or T_fast is None:\\n        return np.inf, np.inf\\n    Rrel = T_candidate[:3, :3] @ T_fast[:3, :3].T\\n    rot_deg = v32_rotation_angle_deg(Rrel)\\n    trans_m = float(np.linalg.norm(T_candidate[:3, 3] - T_fast[:3, 3]))\\n    return rot_deg, trans_m\\n\\ndef v32_blend_transform(T_fast, T_raw, alpha):\\n    alpha = float(np.clip(alpha, 0.0, 1.0))\\n    Rrel = T_raw[:3, :3] @ T_fast[:3, :3].T\\n    Rstep = v32_fractional_rotation(Rrel, alpha)\\n    T = np.eye(4, dtype=np.float64)\\n    T[:3, :3] = Rstep @ T_fast[:3, :3]\\n    T[:3, 3] = (1.0 - alpha) * T_fast[:3, 3] + alpha * T_raw[:3, 3]\\n    return T\\n\\ndef v32_local_precision_score(stats):\\n    if stats is None or not np.isfinite(stats.get(\\\'score\\\', np.nan)):\\n        return -np.inf\\n    w = V32_LOCAL_SCORE_WEIGHTS\\n    return float(\\n        w[\\\'weighted_ratio_5cm\\\'] * stats[\\\'weighted_ratio_5cm\\\']\\n        + w[\\\'ratio_5cm\\\'] * stats[\\\'ratio_5cm\\\']\\n        + w[\\\'ratio_3cm\\\'] * stats[\\\'ratio_3cm\\\']\\n        + w[\\\'residual_quality\\\'] * stats[\\\'residual_quality\\\']\\n    )\\n\\ndef v32_robust_local_raw_candidates(T_fast, P, Q, quality):\\n    """Generate full / rotation-only / translation-only raw local candidates.\\n\\n    No random sampling is used. The current Fast pose defines a residual gate,\\n    then Cauchy-like residual weights suppress remaining outliers.\\n    """\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    quality = np.clip(np.asarray(quality, dtype=np.float64), 1e-8, None)\\n\\n    if T_fast is None or len(P) < V32_LOCAL_MIN_PAIRS:\\n        return {}, {\\\'initial_gate_pairs\\\': 0, \\\'final_gate_pairs\\\': 0}\\n\\n    T_iter = np.asarray(T_fast, dtype=np.float64).copy()\\n    initial_res = np.linalg.norm(v2_transform_points(T_fast, P) - Q, axis=1)\\n    initial_gate = initial_res <= V32_LOCAL_INITIAL_GATE_M\\n    if int(initial_gate.sum()) < V32_LOCAL_MIN_PAIRS:\\n        return {}, {\\n            \\\'initial_gate_pairs\\\': int(initial_gate.sum()),\\n            \\\'final_gate_pairs\\\': int(initial_gate.sum()),\\n            \\\'initial_residual_median_m\\\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\\n        }\\n\\n    gate = initial_gate.copy()\\n    last_w = None\\n    for it in range(int(V32_LOCAL_IRLS_ITERS)):\\n        res = np.linalg.norm(v2_transform_points(T_iter, P) - Q, axis=1)\\n        frac = it / max(1, int(V32_LOCAL_IRLS_ITERS) - 1)\\n        gate_thr = (\\n            V32_LOCAL_INITIAL_GATE_M\\n            + frac * (V32_LOCAL_FINAL_GATE_M - V32_LOCAL_INITIAL_GATE_M)\\n        )\\n        gate = res <= gate_thr\\n        if int(gate.sum()) < V32_LOCAL_MIN_PAIRS:\\n            break\\n\\n        robust = 1.0 / (1.0 + (res / max(V32_LOCAL_ROBUST_SCALE_M, 1e-12)) ** 2)\\n        w = quality * robust\\n        w[~gate] = 0.0\\n        if int(np.sum(w > 0)) < V32_LOCAL_MIN_PAIRS:\\n            break\\n\\n        try:\\n            T_iter = v2_weighted_kabsch(P[gate], Q[gate], w[gate])\\n            last_w = w\\n        except Exception:\\n            break\\n\\n    final_res = np.linalg.norm(v2_transform_points(T_iter, P) - Q, axis=1)\\n    final_gate = final_res <= V32_LOCAL_FINAL_GATE_M\\n    if int(final_gate.sum()) < V32_LOCAL_MIN_PAIRS:\\n        final_gate = initial_gate.copy()\\n\\n    robust = 1.0 / (1.0 + (final_res / max(V32_LOCAL_ROBUST_SCALE_M, 1e-12)) ** 2)\\n    final_w = quality * robust\\n    final_w[~final_gate] = 0.0\\n\\n    if int(np.sum(final_w > 0)) < V32_LOCAL_MIN_PAIRS:\\n        return {}, {\\n            \\\'initial_gate_pairs\\\': int(initial_gate.sum()),\\n            \\\'final_gate_pairs\\\': int(final_gate.sum()),\\n            \\\'initial_residual_median_m\\\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\\n            \\\'final_residual_median_m\\\': float(np.median(final_res[final_gate])) if np.any(final_gate) else np.nan,\\n        }\\n\\n    # Full SE(3) candidate.\\n    T_full = v2_weighted_kabsch(P[final_gate], Q[final_gate], final_w[final_gate])\\n\\n    # Rotation-only: use robust Kabsch rotation but preserve Fast translation.\\n    T_rot = np.asarray(T_fast, dtype=np.float64).copy()\\n    T_rot[:3, :3] = T_full[:3, :3]\\n\\n    # Translation-only: preserve Fast rotation and robustly estimate only t.\\n    Rf = T_fast[:3, :3]\\n    t_samples = Q[final_gate] - (Rf @ P[final_gate].T).T\\n    wg = final_w[final_gate]\\n    wg = wg / max(wg.sum(), 1e-12)\\n    t_new = np.sum(t_samples * wg[:, None], axis=0)\\n    T_trans = np.asarray(T_fast, dtype=np.float64).copy()\\n    T_trans[:3, 3] = t_new\\n\\n    diag = {\\n        \\\'initial_gate_pairs\\\': int(initial_gate.sum()),\\n        \\\'final_gate_pairs\\\': int(final_gate.sum()),\\n        \\\'initial_residual_median_m\\\': float(np.median(initial_res[initial_gate])) if np.any(initial_gate) else np.nan,\\n        \\\'final_residual_median_m\\\': float(np.median(final_res[final_gate])) if np.any(final_gate) else np.nan,\\n    }\\n    return {\\n        \\\'full\\\': T_full,\\n        \\\'rotation_only\\\': T_rot,\\n        \\\'translation_only\\\': T_trans,\\n    }, diag\\n\\ndef v32_select_local_candidate(T_fast, raw_candidates, VP, VQ, val_conf):\\n    fast_stats = v3_consensus_stats(T_fast, VP, VQ, val_conf)\\n    fast_precision = v32_local_precision_score(fast_stats)\\n\\n    best = {\\n        \\\'T\\\': T_fast,\\n        \\\'type\\\': \\\'fast\\\',\\n        \\\'alpha\\\': 0.0,\\n        \\\'stats\\\': fast_stats,\\n        \\\'precision_score\\\': fast_precision,\\n        \\\'regularized_score\\\': fast_precision,\\n        \\\'gain\\\': 0.0,\\n        \\\'delta_rot_deg\\\': 0.0,\\n        \\\'delta_trans_m\\\': 0.0,\\n        \\\'accepted\\\': False,\\n    }\\n\\n    for ctype, T_raw in raw_candidates.items():\\n        raw_rot, raw_trans = v32_pose_delta(T_raw, T_fast)\\n        alpha_cap = 1.0\\n        if np.isfinite(raw_rot) and raw_rot > 1e-12:\\n            alpha_cap = min(alpha_cap, V32_TRUST_MAX_ROT_DEG / raw_rot)\\n        if np.isfinite(raw_trans) and raw_trans > 1e-12:\\n            alpha_cap = min(alpha_cap, V32_TRUST_MAX_TRANS_M / raw_trans)\\n        alpha_cap = float(np.clip(alpha_cap, 0.0, 1.0))\\n        if alpha_cap <= 1e-6:\\n            continue\\n\\n        for frac in V32_LINESEARCH_FRACTIONS:\\n            alpha = float(frac) * alpha_cap\\n            T_cand = v32_blend_transform(T_fast, T_raw, alpha)\\n            stats = v3_consensus_stats(T_cand, VP, VQ, val_conf)\\n            precision = v32_local_precision_score(stats)\\n            d_rot, d_trans = v32_pose_delta(T_cand, T_fast)\\n\\n            rot_norm = d_rot / max(V32_TRUST_MAX_ROT_DEG, 1e-12)\\n            trans_norm = d_trans / max(V32_TRUST_MAX_TRANS_M, 1e-12)\\n            reg_score = (\\n                precision\\n                - V32_STEP_PENALTY_ROT * rot_norm\\n                - V32_STEP_PENALTY_TRANS * trans_norm\\n            )\\n            gain = reg_score - fast_precision\\n\\n            # Precision/consensus guards. These are GT-free.\\n            guards_ok = (\\n                stats[\\\'ratio_5cm\\\'] >= fast_stats[\\\'ratio_5cm\\\'] - V32_MAX_RATIO5_DROP\\n                and stats[\\\'ratio_3cm\\\'] >= fast_stats[\\\'ratio_3cm\\\'] - V32_MAX_RATIO3_DROP\\n                and stats[\\\'median_5cm_m\\\'] <= fast_stats[\\\'median_5cm_m\\\'] + V32_MAX_MEDIAN5_INCREASE_M\\n            )\\n\\n            if (\\n                guards_ok\\n                and gain >= V32_MIN_REGULARIZED_GAIN\\n                and reg_score > best[\\\'regularized_score\\\']\\n            ):\\n                best = {\\n                    \\\'T\\\': T_cand,\\n                    \\\'type\\\': ctype,\\n                    \\\'alpha\\\': alpha,\\n                    \\\'stats\\\': stats,\\n                    \\\'precision_score\\\': precision,\\n                    \\\'regularized_score\\\': reg_score,\\n                    \\\'gain\\\': gain,\\n                    \\\'delta_rot_deg\\\': d_rot,\\n                    \\\'delta_trans_m\\\': d_trans,\\n                    \\\'accepted\\\': True,\\n                }\\n\\n    return best, fast_stats, fast_precision\\n\\n# Original notebook cell 67\\ndef v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\\n    t0 = time.perf_counter()\\n\\n    src_xyz = mv_world_to_camera_points(\\n        v2_np(scache[\\\'xyz\\\']).astype(np.float64), src_c2w\\n    )\\n    tgt_xyz = mv_world_to_camera_points(\\n        v2_np(tcache[\\\'xyz\\\']).astype(np.float64), tgt_c2w\\n    )\\n\\n    out = {\\n        \\\'T_est\\\': None,\\n        \\\'T_fast_debug\\\': None,\\n        \\\'T_local_debug\\\': None,\\n        \\\'selected_stage\\\': \\\'failed\\\',\\n        \\\'fallback_triggered\\\': False,\\n        \\\'fallback_reason\\\': \\\'\\\',\\n        \\\'fallback_selected\\\': False,\\n        \\\'local_selected_type\\\': \\\'none\\\',\\n        \\\'local_selected_alpha\\\': 0.0,\\n        \\\'mean_src_k\\\': np.nan,\\n        \\\'mean_tgt_k\\\': np.nan,\\n        \\\'fast_candidate_pairs\\\': 0,\\n        \\\'fast_rigidity_pairs\\\': 0,\\n        \\\'fast_unique_pairs\\\': 0,\\n        \\\'fast_coverage_strict_pairs\\\': 0,\\n        \\\'fast_coverage_filled_pairs\\\': 0,\\n        \\\'fast_selected_pairs\\\': 0,\\n        \\\'hard_candidate_pairs\\\': 0,\\n        \\\'hard_rigidity_pairs\\\': 0,\\n        \\\'hard_validation_pairs\\\': 0,\\n        \\\'hard_unique_pairs\\\': 0,\\n        \\\'hard_prefilter_pairs\\\': 0,\\n        \\\'hard_coverage_strict_pairs\\\': 0,\\n        \\\'hard_coverage_filled_pairs\\\': 0,\\n        \\\'hard_selected_pairs\\\': 0,\\n        \\\'local_initial_gate_pairs\\\': 0,\\n        \\\'local_final_gate_pairs\\\': 0,\\n        \\\'fast_validation_score\\\': np.nan,\\n        \\\'fast_local_precision_score\\\': np.nan,\\n        \\\'local_validation_score\\\': np.nan,\\n        \\\'local_precision_score\\\': np.nan,\\n        \\\'local_regularized_score_gain\\\': np.nan,\\n        \\\'local_delta_rot_deg\\\': 0.0,\\n        \\\'local_delta_trans_m\\\': 0.0,\\n        \\\'matcher_runtime_s\\\': np.nan,\\n    }\\n\\n    # ------------------------------------------------------------\\n    # FAST PATH - same behavior as V3/V3.1.\\n    # ------------------------------------------------------------\\n    fast_pairs, fast_sim, src_k, tgt_k = v2_build_adaptive_bidirectional(\\n        scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY]\\n    )\\n    out[\\\'fast_candidate_pairs\\\'] = int(len(fast_pairs))\\n    out[\\\'mean_src_k\\\'] = float(np.mean(src_k)) if len(src_k) else np.nan\\n    out[\\\'mean_tgt_k\\\'] = float(np.mean(tgt_k)) if len(tgt_k) else np.nan\\n\\n    fast_r = None\\n    fast_unique_pairs = np.empty((0, 2), dtype=np.int64)\\n\\n    if len(fast_pairs) >= V32_MIN_CORRESPONDENCES:\\n        fast_support = mv_topk_rigidity_support(\\n            fast_pairs, src_xyz, tgt_xyz, threshold_m=V32_RIGIDITY_THRESHOLD_M\\n        )\\n        out[\\\'fast_rigidity_pairs\\\'] = int(np.sum(fast_support >= V32_MIN_SUPPORT))\\n        fast_conf_all = v2_pair_quality(\\n            scache, tcache, fast_pairs, fast_sim, fast_support\\n        )\\n        fast_unique_pairs, fast_unique_conf, fast_unique_support = (\\n            v2_greedy_unique_by_confidence(\\n                fast_pairs, fast_conf_all, fast_support,\\n                min_support=V32_MIN_SUPPORT,\\n            )\\n        )\\n        out[\\\'fast_unique_pairs\\\'] = int(len(fast_unique_pairs))\\n        out.update(v3_conf_stats(fast_unique_conf, \\\'fast_unique_\\\'))\\n\\n        if len(fast_unique_pairs) >= V32_MIN_CORRESPONDENCES:\\n            fp, fc, fs, f_cov = v3_spatial_coverage_select(\\n                fast_unique_pairs, fast_unique_conf, fast_unique_support,\\n                src_xyz, tgt_xyz,\\n                max_pairs=V32_FAST_MAX_PAIRS,\\n                max_per_source_voxel=V32_FAST_MAX_PER_SOURCE_VOXEL,\\n                max_per_target_voxel=V32_FAST_MAX_PER_TARGET_VOXEL,\\n            )\\n            out[\\\'fast_coverage_strict_pairs\\\'] = f_cov[\\\'strict_coverage_pairs\\\']\\n            out[\\\'fast_coverage_filled_pairs\\\'] = f_cov[\\\'filled_pairs\\\']\\n            out[\\\'fast_selected_pairs\\\'] = int(len(fp))\\n            out.update(v3_conf_stats(fc, \\\'fast_selected_\\\'))\\n            out.update(v3_geometry_diag(fp, src_xyz, tgt_xyz, \\\'fast_\\\'))\\n\\n            if len(fp) >= V32_MIN_CORRESPONDENCES:\\n                fast_r = v3_prosac_ransac(\\n                    src_xyz[fp[:, 0]], tgt_xyz[fp[:, 1]], fc, seed,\\n                    max_iters=V32_FAST_RANSAC_MAX_ITERS,\\n                    min_iters=V32_FAST_RANSAC_MIN_ITERS,\\n                    early_ratio=V32_FAST_EARLY_RATIO,\\n                    early_median_m=V32_FAST_EARLY_MEDIAN_M,\\n                )\\n\\n    if fast_r is not None:\\n        out[\\\'T_fast_debug\\\'] = fast_r[\\\'T\\\']\\n        for k in [\\n            \\\'score\\\', \\\'ratio_5cm\\\', \\\'ratio_3cm\\\', \\\'weighted_ratio_5cm\\\',\\n            \\\'median_5cm_m\\\', \\\'mean_5cm_m\\\', \\\'p90_all_m\\\', \\\'residual_quality\\\',\\n            \\\'n_inliers_5cm\\\', \\\'n_inliers_3cm\\\', \\\'iterations\\\', \\\'runtime_s\\\',\\n        ]:\\n            out[f\\\'fast_ransac_{k}\\\'] = fast_r[k]\\n    else:\\n        for k in [\\n            \\\'score\\\', \\\'ratio_5cm\\\', \\\'ratio_3cm\\\', \\\'weighted_ratio_5cm\\\',\\n            \\\'median_5cm_m\\\', \\\'mean_5cm_m\\\', \\\'p90_all_m\\\', \\\'residual_quality\\\',\\n        ]:\\n            out[f\\\'fast_ransac_{k}\\\'] = np.nan\\n        out[\\\'fast_ransac_n_inliers_5cm\\\'] = 0\\n        out[\\\'fast_ransac_n_inliers_3cm\\\'] = 0\\n        out[\\\'fast_ransac_iterations\\\'] = 0\\n        out[\\\'fast_ransac_runtime_s\\\'] = 0.0\\n\\n    # Same hard-case detector.\\n    reasons = []\\n    if fast_r is None:\\n        trigger = True\\n        reasons = [\\\'fast_ransac_failed\\\']\\n    else:\\n        if int(fast_r[\\\'iterations\\\']) >= int(V32_FAST_RANSAC_MAX_ITERS):\\n            reasons.append(\\\'fast_hit_max_iters\\\')\\n        if float(fast_r[\\\'ratio_5cm\\\']) < float(V32_HARD_TRIGGER_RATIO_5CM):\\n            reasons.append(\\\'fast_ratio5_low\\\')\\n        if float(fast_r[\\\'ratio_3cm\\\']) < float(V32_HARD_TRIGGER_RATIO_3CM):\\n            reasons.append(\\\'fast_ratio3_low\\\')\\n        trigger = bool(reasons)\\n\\n    if len(fast_unique_pairs) < V32_MIN_CORRESPONDENCES:\\n        trigger = True\\n        reasons = list(dict.fromkeys(reasons + [\\\'fast_too_few_unique_pairs\\\']))\\n\\n    out[\\\'fallback_triggered\\\'] = bool(trigger)\\n    out[\\\'fallback_reason\\\'] = \\\'|\\\'.join(reasons)\\n\\n    selected_T = fast_r[\\\'T\\\'] if fast_r is not None else None\\n    selected_stage = \\\'fast\\\' if selected_T is not None else \\\'failed\\\'\\n\\n    # ------------------------------------------------------------\\n    # HARD LOCAL PATH - no hard RANSAC.\\n    # ------------------------------------------------------------\\n    local_choice = None\\n    fast_validation = None\\n\\n    if trigger:\\n        hard_pairs, hard_sim, ambiguity = v32_build_hard_candidates(\\n            scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY],\\n            top_k=V32_HARD_TOPK,\\n        )\\n        out[\\\'hard_candidate_pairs\\\'] = int(len(hard_pairs))\\n\\n        if len(hard_pairs) >= V32_MIN_CORRESPONDENCES:\\n            hard_support = mv_topk_rigidity_support(\\n                hard_pairs, src_xyz, tgt_xyz,\\n                threshold_m=V32_RIGIDITY_THRESHOLD_M,\\n            )\\n            out[\\\'hard_rigidity_pairs\\\'] = int(np.sum(hard_support >= V32_MIN_SUPPORT))\\n\\n            hard_base_conf = v2_pair_quality(\\n                scache, tcache, hard_pairs, hard_sim, hard_support\\n            )\\n            hard_quality_all = v32_hard_quality(hard_base_conf, ambiguity)\\n\\n            # Common validation pool uses base confidence, not the local score.\\n            val_pairs, val_conf, val_support = v2_greedy_unique_by_confidence(\\n                hard_pairs, hard_base_conf, hard_support,\\n                min_support=V32_MIN_SUPPORT,\\n            )\\n            out[\\\'hard_validation_pairs\\\'] = int(len(val_pairs))\\n\\n            hu_pairs, hu_q, hu_support, hu_idx = v32_greedy_unique_with_indices(\\n                hard_pairs, hard_quality_all, hard_support,\\n                min_support=V32_MIN_SUPPORT,\\n            )\\n            out[\\\'hard_unique_pairs\\\'] = int(len(hu_pairs))\\n            out.update(v3_conf_stats(hu_q, \\\'hard_unique_quality_\\\'))\\n            out.update(v32_ambiguity_diag(ambiguity, hu_idx, \\\'hard_unique_\\\'))\\n\\n            hp0, hq0, hs0, hidx0 = v32_top_quality_prefilter(\\n                hu_pairs, hu_q, hu_support, hu_idx,\\n                max_pairs=V32_HARD_PREFILTER_MAX_PAIRS,\\n            )\\n            out[\\\'hard_prefilter_pairs\\\'] = int(len(hp0))\\n            out.update(v32_ambiguity_diag(ambiguity, hidx0, \\\'hard_prefilter_\\\'))\\n\\n            if len(hp0) >= V32_MIN_CORRESPONDENCES:\\n                hp, hq, hs, h_cov = v3_spatial_coverage_select(\\n                    hp0, hq0, hs0, src_xyz, tgt_xyz,\\n                    max_pairs=V32_HARD_MAX_PAIRS,\\n                    max_per_source_voxel=V32_HARD_MAX_PER_SOURCE_VOXEL,\\n                    max_per_target_voxel=V32_HARD_MAX_PER_TARGET_VOXEL,\\n                )\\n                out[\\\'hard_coverage_strict_pairs\\\'] = h_cov[\\\'strict_coverage_pairs\\\']\\n                out[\\\'hard_coverage_filled_pairs\\\'] = h_cov[\\\'filled_pairs\\\']\\n                out[\\\'hard_selected_pairs\\\'] = int(len(hp))\\n                out.update(v3_conf_stats(hq, \\\'hard_selected_quality_\\\'))\\n                out.update(v3_geometry_diag(hp, src_xyz, tgt_xyz, \\\'hard_\\\'))\\n\\n                pair_to_idx = {\\n                    (int(s), int(t)): int(idx)\\n                    for (s, t), idx in zip(hp0.tolist(), hidx0.tolist())\\n                }\\n                hp_idx = np.asarray(\\n                    [pair_to_idx[(int(s), int(t))] for s, t in hp],\\n                    dtype=np.int64,\\n                ) if len(hp) else np.empty((0,), dtype=np.int64)\\n                out.update(v32_ambiguity_diag(ambiguity, hp_idx, \\\'hard_selected_\\\'))\\n\\n                # Local refinement only makes sense when Fast succeeded.\\n                if fast_r is not None and len(hp) >= V32_LOCAL_MIN_PAIRS:\\n                    HP = src_xyz[hp[:, 0]]\\n                    HQ = tgt_xyz[hp[:, 1]]\\n                    raw_candidates, local_diag = v32_robust_local_raw_candidates(\\n                        fast_r[\\\'T\\\'], HP, HQ, hq\\n                    )\\n                    out[\\\'local_initial_gate_pairs\\\'] = local_diag.get(\\\'initial_gate_pairs\\\', 0)\\n                    out[\\\'local_final_gate_pairs\\\'] = local_diag.get(\\\'final_gate_pairs\\\', 0)\\n                    out[\\\'local_initial_residual_median_m\\\'] = local_diag.get(\\\'initial_residual_median_m\\\', np.nan)\\n                    out[\\\'local_final_residual_median_m\\\'] = local_diag.get(\\\'final_residual_median_m\\\', np.nan)\\n\\n                    if len(val_pairs) >= V32_MIN_CORRESPONDENCES and raw_candidates:\\n                        VP = src_xyz[val_pairs[:, 0]]\\n                        VQ = tgt_xyz[val_pairs[:, 1]]\\n                        local_choice, fast_validation, fast_precision = v32_select_local_candidate(\\n                            fast_r[\\\'T\\\'], raw_candidates, VP, VQ, val_conf\\n                        )\\n                        out[\\\'fast_validation_score\\\'] = fast_validation[\\\'score\\\']\\n                        out[\\\'fast_local_precision_score\\\'] = fast_precision\\n                        out[\\\'local_validation_score\\\'] = local_choice[\\\'stats\\\'][\\\'score\\\']\\n                        out[\\\'local_precision_score\\\'] = local_choice[\\\'precision_score\\\']\\n                        out[\\\'local_regularized_score_gain\\\'] = local_choice[\\\'gain\\\']\\n                        out[\\\'local_delta_rot_deg\\\'] = local_choice[\\\'delta_rot_deg\\\']\\n                        out[\\\'local_delta_trans_m\\\'] = local_choice[\\\'delta_trans_m\\\']\\n                        out[\\\'local_selected_type\\\'] = local_choice[\\\'type\\\']\\n                        out[\\\'local_selected_alpha\\\'] = local_choice[\\\'alpha\\\']\\n\\n                        if local_choice[\\\'accepted\\\']:\\n                            selected_T = local_choice[\\\'T\\\']\\n                            selected_stage = \\\'local\\\'\\n                            out[\\\'fallback_selected\\\'] = True\\n                            out[\\\'T_local_debug\\\'] = selected_T\\n\\n    out.setdefault(\\\'local_initial_residual_median_m\\\', np.nan)\\n    out.setdefault(\\\'local_final_residual_median_m\\\', np.nan)\\n\\n    out[\\\'selected_stage\\\'] = selected_stage\\n    out[\\\'T_est\\\'] = selected_T\\n\\n    # Fast RANSAC remains the only random robust estimation stage in V3.2.\\n    if fast_r is not None:\\n        out[\\\'selected_pairs\\\'] = int(out[\\\'fast_selected_pairs\\\'])\\n        out[\\\'selected_ransac_score\\\'] = float(fast_r[\\\'score\\\'])\\n        out[\\\'selected_ransac_iterations\\\'] = int(fast_r[\\\'iterations\\\'])\\n        out[\\\'selected_ransac_runtime_s\\\'] = float(fast_r[\\\'runtime_s\\\'])\\n    else:\\n        out[\\\'selected_pairs\\\'] = 0\\n        out[\\\'selected_ransac_score\\\'] = np.nan\\n        out[\\\'selected_ransac_iterations\\\'] = 0\\n        out[\\\'selected_ransac_runtime_s\\\'] = 0.0\\n\\n    # Report consensus of the actually selected pose on the validation pool\\n    # when available; otherwise fall back to Fast RANSAC\\\'s own pool metrics.\\n    if trigger and fast_r is not None and out[\\\'hard_validation_pairs\\\'] >= V32_MIN_CORRESPONDENCES:\\n        # Rebuild validation arrays only if they exist in this scope.\\n        if \\\'val_pairs\\\' in locals() and len(val_pairs) >= V32_MIN_CORRESPONDENCES:\\n            VP = src_xyz[val_pairs[:, 0]]\\n            VQ = tgt_xyz[val_pairs[:, 1]]\\n            selected_stats = v3_consensus_stats(selected_T, VP, VQ, val_conf)\\n            out[\\\'selected_inlier_ratio_5cm\\\'] = float(selected_stats[\\\'ratio_5cm\\\'])\\n            out[\\\'selected_inlier_ratio_3cm\\\'] = float(selected_stats[\\\'ratio_3cm\\\'])\\n            out[\\\'selected_weighted_ratio_5cm\\\'] = float(selected_stats[\\\'weighted_ratio_5cm\\\'])\\n            out[\\\'selected_median_5cm_m\\\'] = float(selected_stats[\\\'median_5cm_m\\\'])\\n            out[\\\'selected_validation_score\\\'] = float(selected_stats[\\\'score\\\'])\\n        else:\\n            out[\\\'selected_inlier_ratio_5cm\\\'] = float(fast_r[\\\'ratio_5cm\\\'])\\n            out[\\\'selected_inlier_ratio_3cm\\\'] = float(fast_r[\\\'ratio_3cm\\\'])\\n            out[\\\'selected_weighted_ratio_5cm\\\'] = float(fast_r[\\\'weighted_ratio_5cm\\\'])\\n            out[\\\'selected_median_5cm_m\\\'] = float(fast_r[\\\'median_5cm_m\\\'])\\n            out[\\\'selected_validation_score\\\'] = np.nan\\n    elif fast_r is not None:\\n        out[\\\'selected_inlier_ratio_5cm\\\'] = float(fast_r[\\\'ratio_5cm\\\'])\\n        out[\\\'selected_inlier_ratio_3cm\\\'] = float(fast_r[\\\'ratio_3cm\\\'])\\n        out[\\\'selected_weighted_ratio_5cm\\\'] = float(fast_r[\\\'weighted_ratio_5cm\\\'])\\n        out[\\\'selected_median_5cm_m\\\'] = float(fast_r[\\\'median_5cm_m\\\'])\\n        out[\\\'selected_validation_score\\\'] = np.nan\\n    else:\\n        out[\\\'selected_inlier_ratio_5cm\\\'] = 0.0\\n        out[\\\'selected_inlier_ratio_3cm\\\'] = 0.0\\n        out[\\\'selected_weighted_ratio_5cm\\\'] = 0.0\\n        out[\\\'selected_median_5cm_m\\\'] = np.nan\\n        out[\\\'selected_validation_score\\\'] = np.nan\\n\\n    out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - t0)\\n    return out\\n\\n# Original notebook cell 69\\nICP_DISTANCE_THRESHOLD_M = 0.005\\n\\nICP_MAX_ITER = 500\\n\\nICP_REL_FITNESS = 1e-9\\n\\nICP_REL_RMSE = 1e-9\\n\\ndef _as_np_pose(x):\\n    if torch.is_tensor(x):\\n        x = x.detach().cpu().numpy()\\n    return np.asarray(x, dtype=np.float64)\\n\\ndef loop_to_submap_start_transform(T_loop, source_sub, target_sub, source_frame, target_frame):\\n    Cs0 = _as_np_pose(source_sub[\\\'submap_c2ws\\\'][0])\\n    Ct0 = _as_np_pose(target_sub[\\\'submap_c2ws\\\'][0])\\n    CsL = mv_frame_c2w(source_sub, source_frame)\\n    CtL = mv_frame_c2w(target_sub, target_frame)\\n    # PsL = inv(CsL) Cs0 Ps0; PtL = inv(CtL) Ct0 Pt0\\n    return np.linalg.inv(Ct0) @ CtL @ np.asarray(T_loop) @ np.linalg.inv(CsL) @ Cs0\\n\\ndef submap_start_to_loop_transform(T_start, source_sub, target_sub, source_frame, target_frame):\\n    Cs0 = _as_np_pose(source_sub[\\\'submap_c2ws\\\'][0])\\n    Ct0 = _as_np_pose(target_sub[\\\'submap_c2ws\\\'][0])\\n    CsL = mv_frame_c2w(source_sub, source_frame)\\n    CtL = mv_frame_c2w(target_sub, target_frame)\\n    As = np.linalg.inv(CsL) @ Cs0\\n    At = np.linalg.inv(CtL) @ Ct0\\n    return At @ np.asarray(T_start) @ np.linalg.inv(As)\\n\\ndef transform_delta(T_a, T_b):\\n    D = np.asarray(T_a) @ np.linalg.inv(np.asarray(T_b))\\n    c = np.clip((np.trace(D[:3, :3]) - 1.0) / 2.0, -1.0, 1.0)\\n    rot_deg = float(np.degrees(np.arccos(c)))\\n    trans_cm = float(np.linalg.norm(D[:3, 3]) * 100.0)\\n    return rot_deg, trans_cm\\n\\ndef dense_cloud_for_submap(agent_id, submap_index):\\n    key = (int(agent_id), int(submap_index))\\n    if key in _dense_cloud_cache:\\n        return _dense_cloud_cache[key]\\n    sub = mv_submaps[key[0]][key[1]]\\n    t0 = time.perf_counter()\\n    color, depth = repo_utils.get_rgbd(sub)\\n    cloud = repo_utils.rgbd2ptcloud(\\n        color, depth, sub[\\\'intrinsics\\\'], np.eye(4)\\n    )\\n    cloud.estimate_normals()\\n    info = {\\n        \\\'points\\\': int(len(cloud.points)),\\n        \\\'build_time_s\\\': float(time.perf_counter() - t0),\\n    }\\n    _dense_cloud_cache[key] = (cloud, info)\\n    print(\\\'[dense cloud]\\\', key, \\\'| points=\\\', info[\\\'points\\\'], \\\'| build_s=\\\', f"{info[\\\'build_time_s\\\']:.2f}")\\n    return _dense_cloud_cache[key]\\n\\ndef exact_production_icp(source_cloud, target_cloud, T_init):\\n    initial_eval = o3d.pipelines.registration.evaluate_registration(\\n        source_cloud, target_cloud, ICP_DISTANCE_THRESHOLD_M, np.asarray(T_init)\\n    )\\n    t0 = time.perf_counter()\\n    result = o3d.pipelines.registration.registration_icp(\\n        source_cloud,\\n        target_cloud,\\n        ICP_DISTANCE_THRESHOLD_M,\\n        np.asarray(T_init),\\n        o3d.pipelines.registration.TransformationEstimationPointToPlane(),\\n        o3d.pipelines.registration.ICPConvergenceCriteria(\\n            max_iteration=ICP_MAX_ITER,\\n            relative_fitness=ICP_REL_FITNESS,\\n            relative_rmse=ICP_REL_RMSE,\\n        ),\\n    )\\n    return {\\n        \\\'T_final\\\': np.asarray(result.transformation, dtype=np.float64),\\n        \\\'initial_fitness_5mm\\\': float(initial_eval.fitness),\\n        \\\'initial_rmse_5mm_m\\\': float(initial_eval.inlier_rmse),\\n        \\\'final_fitness_5mm\\\': float(result.fitness),\\n        \\\'final_rmse_5mm_m\\\': float(result.inlier_rmse),\\n        \\\'runtime_s\\\': float(time.perf_counter() - t0),\\n    }\\n\\n# Original notebook cell 71\\nV33_HOLDOUT_FRACTION = 0.30\\n\\nV33_HOLDOUT_VOXEL_M = 0.12\\n\\nV33_MIN_TRAIN_PAIRS = 18\\n\\nV33_MIN_VALIDATION_PAIRS = 8\\n\\nV33_IRLS_ITERS = 4\\n\\nV33_ROBUST_SCALE_M = 0.030\\n\\nV33_INITIAL_GATE_M = 0.080\\n\\nV33_FINAL_GATE_M = 0.050\\n\\nV33_LEVERAGE_STRENGTH = 0.30\\n\\nV33_OBS_MID_EIGEN_RATIO_MIN = 0.050\\n\\nV33_OBS_RMS_SPREAD_MIN_M = 0.080\\n\\nV33_TRUST_MAX_ROT_DEG = float(V32_TRUST_MAX_ROT_DEG)\\n\\nV33_TRUST_MAX_TRANS_M = float(V32_TRUST_MAX_TRANS_M)\\n\\nV33_LINESEARCH_FRACTIONS = (0.25, 0.50, 0.75, 1.00)\\n\\nV33_MIN_REGULARIZED_GAIN = 0.0015\\n\\nV33_MAX_RATIO5_DROP = 0.005\\n\\nV33_MAX_RATIO3_DROP = 0.005\\n\\nV33_MAX_MEDIAN5_INCREASE_M = 0.0010\\n\\nV33_STEP_PENALTY_ROT = 0.0010\\n\\nV33_STEP_PENALTY_TRANS = 0.0010\\n\\nV33_CYCLE_PENALTY_ROT = 0.0010\\n\\nV33_CYCLE_PENALTY_TRANS = 0.0010\\n\\nV33_HARD_RANSAC_MAX_ITERS = int(V3_HARD_RANSAC_MAX_ITERS)\\n\\nV33_HARD_RANSAC_MIN_ITERS = int(V3_HARD_RANSAC_MIN_ITERS)\\n\\nV33_HARD_EARLY_RATIO = float(V3_HARD_EARLY_RATIO)\\n\\nV33_HARD_EARLY_MEDIAN_M = float(V3_HARD_EARLY_MEDIAN_M)\\n\\n# Original notebook cell 73\\ndef v33_group_key(pair, src_xyz, tgt_xyz, voxel_m=V33_HOLDOUT_VOXEL_M):\\n    s, t = map(int, pair)\\n    sv = np.floor(np.asarray(src_xyz[s], dtype=np.float64) / float(voxel_m)).astype(np.int64)\\n    tv = np.floor(np.asarray(tgt_xyz[t], dtype=np.float64) / float(voxel_m)).astype(np.int64)\\n    return tuple(sv.tolist() + tv.tolist())\\n\\ndef v33_group_hash(key, seed):\\n    payload = (str(int(seed)) + \\\'|\\\' + \\\',\\\'.join(map(str, key))).encode(\\\'utf-8\\\')\\n    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), \\\'little\\\')\\n\\ndef v33_spatial_holdout_split(pairs, src_xyz, tgt_xyz, seed):\\n    pairs = np.asarray(pairs, dtype=np.int64)\\n    n = len(pairs)\\n    if n < V33_MIN_TRAIN_PAIRS + V33_MIN_VALIDATION_PAIRS:\\n        return None, None, {\\\'reason\\\': \\\'too_few_pairs\\\', \\\'n\\\': n}\\n\\n    groups = {}\\n    for idx, pair in enumerate(pairs):\\n        key = v33_group_key(pair, src_xyz, tgt_xyz)\\n        groups.setdefault(key, []).append(idx)\\n\\n    group_items = []\\n    for key, idxs in groups.items():\\n        group_items.append((v33_group_hash(key, seed), key, np.asarray(idxs, dtype=np.int64)))\\n    group_items.sort(key=lambda x: x[0])\\n\\n    target_val = max(V33_MIN_VALIDATION_PAIRS, int(round(V33_HOLDOUT_FRACTION * n)))\\n    target_val = min(target_val, n - V33_MIN_TRAIN_PAIRS)\\n\\n    val = []\\n    train = []\\n    val_count = 0\\n    for _, _, idxs in group_items:\\n        # Fill validation first by whole spatial groups; keep enough room for train.\\n        can_take_val = (val_count < target_val) and (n - (val_count + len(idxs)) >= V33_MIN_TRAIN_PAIRS)\\n        if can_take_val:\\n            val.extend(idxs.tolist())\\n            val_count += len(idxs)\\n        else:\\n            train.extend(idxs.tolist())\\n\\n    # Repair only if group granularity produced too small a split.\\n    if len(val) < V33_MIN_VALIDATION_PAIRS:\\n        movable = [x for x in group_items if any(i in train for i in x[2].tolist())]\\n        for _, _, idxs in movable:\\n            if len(train) - len(idxs) < V33_MIN_TRAIN_PAIRS:\\n                continue\\n            for i in idxs.tolist():\\n                if i in train:\\n                    train.remove(i)\\n                    val.append(i)\\n            if len(val) >= V33_MIN_VALIDATION_PAIRS:\\n                break\\n\\n    train = np.asarray(sorted(set(train)), dtype=np.int64)\\n    val = np.asarray(sorted(set(val)), dtype=np.int64)\\n    if len(train) < V33_MIN_TRAIN_PAIRS or len(val) < V33_MIN_VALIDATION_PAIRS:\\n        return None, None, {\\n            \\\'reason\\\': \\\'split_failed\\\', \\\'n\\\': n,\\n            \\\'train\\\': int(len(train)), \\\'val\\\': int(len(val)),\\n            \\\'groups\\\': int(len(groups)),\\n        }\\n\\n    return train, val, {\\n        \\\'reason\\\': \\\'ok\\\', \\\'n\\\': n,\\n        \\\'train\\\': int(len(train)), \\\'val\\\': int(len(val)),\\n        \\\'groups\\\': int(len(groups)),\\n        \\\'val_fraction\\\': float(len(val) / n),\\n    }\\n\\ndef v33_weighted_covariance(X, w):\\n    X = np.asarray(X, dtype=np.float64)\\n    w = np.clip(np.asarray(w, dtype=np.float64), 1e-12, None)\\n    w = w / max(w.sum(), 1e-12)\\n    c = np.sum(X * w[:, None], axis=0)\\n    X0 = X - c\\n    C = (X0 * w[:, None]).T @ X0\\n    return C, c\\n\\ndef v33_observability(P, Q, w):\\n    Cp, cp = v33_weighted_covariance(P, w)\\n    Cq, cq = v33_weighted_covariance(Q, w)\\n    ep = np.clip(np.linalg.eigvalsh(Cp), 0.0, None)\\n    eq = np.clip(np.linalg.eigvalsh(Cq), 0.0, None)\\n    p_mid = float(ep[-2] / max(ep[-1], 1e-12)) if len(ep) >= 2 else 0.0\\n    q_mid = float(eq[-2] / max(eq[-1], 1e-12)) if len(eq) >= 2 else 0.0\\n    p_spread = float(np.sqrt(max(np.trace(Cp), 0.0)))\\n    q_spread = float(np.sqrt(max(np.trace(Cq), 0.0)))\\n    mid_ratio = min(p_mid, q_mid)\\n    rms_spread = min(p_spread, q_spread)\\n    rot_observable = bool(\\n        mid_ratio >= V33_OBS_MID_EIGEN_RATIO_MIN\\n        and rms_spread >= V33_OBS_RMS_SPREAD_MIN_M\\n    )\\n    return {\\n        \\\'src_eigs\\\': ep, \\\'tgt_eigs\\\': eq,\\n        \\\'mid_eigen_ratio\\\': mid_ratio,\\n        \\\'rms_spread_m\\\': rms_spread,\\n        \\\'rotation_observable\\\': rot_observable,\\n    }\\n\\ndef v33_leverage_weights(P, Q, base_w):\\n    P = np.asarray(P, dtype=np.float64)\\n    Q = np.asarray(Q, dtype=np.float64)\\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\\n    _, cp = v33_weighted_covariance(P, base_w)\\n    _, cq = v33_weighted_covariance(Q, base_w)\\n    rp = np.linalg.norm(P - cp, axis=1)\\n    rq = np.linalg.norm(Q - cq, axis=1)\\n    scale_p = max(float(np.percentile(rp, 75)), 1e-6)\\n    scale_q = max(float(np.percentile(rq, 75)), 1e-6)\\n    leverage = 0.5 * (rp / scale_p + rq / scale_q)\\n    leverage = np.clip(leverage, 0.25, 2.0)\\n    multiplier = (1.0 - V33_LEVERAGE_STRENGTH) + V33_LEVERAGE_STRENGTH * leverage\\n    return base_w * multiplier, leverage\\n\\n# Original notebook cell 75\\ndef v33_cauchy_weights(residual_m, scale_m=V33_ROBUST_SCALE_M):\\n    residual_m = np.asarray(residual_m, dtype=np.float64)\\n    x = residual_m / max(float(scale_m), 1e-9)\\n    return 1.0 / (1.0 + x * x)\\n\\ndef v33_gate_threshold(iter_idx):\\n    frac = iter_idx / max(V33_IRLS_ITERS - 1, 1)\\n    return V33_INITIAL_GATE_M + frac * (V33_FINAL_GATE_M - V33_INITIAL_GATE_M)\\n\\ndef v33_irls_full(T_init, P, Q, base_w):\\n    T = np.asarray(T_init, dtype=np.float64).copy()\\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\\n    for it in range(V33_IRLS_ITERS):\\n        res = np.linalg.norm(v2_transform_points(T, P) - Q, axis=1)\\n        gate = res <= v33_gate_threshold(it)\\n        if int(gate.sum()) < 6:\\n            break\\n        w = base_w * v33_cauchy_weights(res)\\n        try:\\n            T = v2_weighted_kabsch(P[gate], Q[gate], w[gate])\\n        except Exception:\\n            break\\n    return T\\n\\ndef v33_irls_translation(T_init, P, Q, base_w):\\n    T = np.asarray(T_init, dtype=np.float64).copy()\\n    R = T[:3, :3].copy()\\n    t = T[:3, 3].copy()\\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\\n    for it in range(V33_IRLS_ITERS):\\n        pred = P @ R.T + t\\n        res = np.linalg.norm(pred - Q, axis=1)\\n        gate = res <= v33_gate_threshold(it)\\n        if int(gate.sum()) < 6:\\n            break\\n        w = base_w * v33_cauchy_weights(res)\\n        wg = w[gate]\\n        rhs = Q[gate] - P[gate] @ R.T\\n        t = np.sum(rhs * wg[:, None], axis=0) / max(wg.sum(), 1e-12)\\n    Tout = np.eye(4, dtype=np.float64)\\n    Tout[:3, :3] = R\\n    Tout[:3, 3] = t\\n    return Tout\\n\\ndef v33_weighted_rotation_fixed_translation(P, Q, t_fixed, w):\\n    P = np.asarray(P, dtype=np.float64)\\n    Q0 = np.asarray(Q, dtype=np.float64) - np.asarray(t_fixed, dtype=np.float64)[None, :]\\n    w = np.clip(np.asarray(w, dtype=np.float64), 1e-12, None)\\n    H = (P * w[:, None]).T @ Q0\\n    U, _, Vt = np.linalg.svd(H)\\n    R = Vt.T @ U.T\\n    if np.linalg.det(R) < 0:\\n        Vt[-1, :] *= -1.0\\n        R = Vt.T @ U.T\\n    return R\\n\\ndef v33_irls_rotation(T_init, P, Q, base_w):\\n    T = np.asarray(T_init, dtype=np.float64).copy()\\n    t = T[:3, 3].copy()\\n    R = T[:3, :3].copy()\\n    base_w = np.clip(np.asarray(base_w, dtype=np.float64), 1e-8, None)\\n    for it in range(V33_IRLS_ITERS):\\n        pred = P @ R.T + t\\n        res = np.linalg.norm(pred - Q, axis=1)\\n        gate = res <= v33_gate_threshold(it)\\n        if int(gate.sum()) < 6:\\n            break\\n        w = base_w * v33_cauchy_weights(res)\\n        R = v33_weighted_rotation_fixed_translation(P[gate], Q[gate], t, w[gate])\\n    Tout = np.eye(4, dtype=np.float64)\\n    Tout[:3, :3] = R\\n    Tout[:3, 3] = t\\n    return Tout\\n\\ndef v33_alpha_cap(T_fast, T_raw):\\n    dr, dt = v32_pose_delta(T_raw, T_fast)\\n    cap = 1.0\\n    if np.isfinite(dr) and dr > 1e-12:\\n        cap = min(cap, V33_TRUST_MAX_ROT_DEG / dr)\\n    if np.isfinite(dt) and dt > 1e-12:\\n        cap = min(cap, V33_TRUST_MAX_TRANS_M / dt)\\n    return float(np.clip(cap, 0.0, 1.0)), float(dr), float(dt)\\n\\ndef v33_validation_score(T, P, Q, w, T_cycle_reference=None, T_fast=None):\\n    stats = v3_consensus_stats(T, P, Q, w)\\n    precision = v32_local_precision_score(stats)\\n    if T_fast is None:\\n        step_r = step_t = 0.0\\n    else:\\n        step_r, step_t = v32_pose_delta(T, T_fast)\\n\\n    cycle_r = cycle_t = 0.0\\n    if T_cycle_reference is not None:\\n        cycle_r, cycle_t = v32_pose_delta(T, T_cycle_reference)\\n\\n    step_penalty = (\\n        V33_STEP_PENALTY_ROT * min(step_r / max(V33_TRUST_MAX_ROT_DEG, 1e-9), 2.0)\\n        + V33_STEP_PENALTY_TRANS * min(step_t / max(V33_TRUST_MAX_TRANS_M, 1e-9), 2.0)\\n    )\\n    cycle_penalty = (\\n        V33_CYCLE_PENALTY_ROT * min(cycle_r / 1.0, 2.0)\\n        + V33_CYCLE_PENALTY_TRANS * min(cycle_t / 0.03, 2.0)\\n    )\\n    regularized = float(precision - step_penalty - cycle_penalty)\\n    return {\\n        \\\'stats\\\': stats,\\n        \\\'precision_score\\\': float(precision),\\n        \\\'regularized_score\\\': regularized,\\n        \\\'step_rot_deg\\\': float(step_r),\\n        \\\'step_trans_m\\\': float(step_t),\\n        \\\'cycle_rot_deg\\\': float(cycle_r),\\n        \\\'cycle_trans_m\\\': float(cycle_t),\\n    }\\n\\ndef v33_candidate_acceptable(base_eval, cand_eval):\\n    b = base_eval[\\\'stats\\\']; c = cand_eval[\\\'stats\\\']\\n    if not np.isfinite(cand_eval[\\\'regularized_score\\\']):\\n        return False\\n    if cand_eval[\\\'regularized_score\\\'] < base_eval[\\\'regularized_score\\\'] + V33_MIN_REGULARIZED_GAIN:\\n        return False\\n    if c[\\\'ratio_5cm\\\'] < b[\\\'ratio_5cm\\\'] - V33_MAX_RATIO5_DROP:\\n        return False\\n    if c[\\\'ratio_3cm\\\'] < b[\\\'ratio_3cm\\\'] - V33_MAX_RATIO3_DROP:\\n        return False\\n    if np.isfinite(b[\\\'median_5cm_m\\\']) and np.isfinite(c[\\\'median_5cm_m\\\']):\\n        if c[\\\'median_5cm_m\\\'] > b[\\\'median_5cm_m\\\'] + V33_MAX_MEDIAN5_INCREASE_M:\\n            return False\\n    return True\\n\\n# Original notebook cell 77\\ndef v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz):\\n    hard_pairs, hard_sim, ambiguity = v32_build_hard_candidates(\\n        scache[V32_DESCRIPTOR_KEY], tcache[V32_DESCRIPTOR_KEY],\\n        top_k=V32_HARD_TOPK,\\n    )\\n    if len(hard_pairs) < V32_MIN_CORRESPONDENCES:\\n        return None\\n\\n    hard_support = mv_topk_rigidity_support(\\n        hard_pairs, src_xyz, tgt_xyz,\\n        threshold_m=V32_RIGIDITY_THRESHOLD_M,\\n    )\\n    hard_base_conf = v2_pair_quality(\\n        scache, tcache, hard_pairs, hard_sim, hard_support\\n    )\\n    hard_quality_all = v32_hard_quality(hard_base_conf, ambiguity)\\n\\n    hu_pairs, hu_q, hu_support, hu_idx = v32_greedy_unique_with_indices(\\n        hard_pairs, hard_quality_all, hard_support,\\n        min_support=V32_MIN_SUPPORT,\\n    )\\n    if len(hu_pairs) < V32_MIN_CORRESPONDENCES:\\n        return None\\n\\n    hp0, hq0, hs0, hidx0 = v32_top_quality_prefilter(\\n        hu_pairs, hu_q, hu_support, hu_idx,\\n        max_pairs=V32_HARD_PREFILTER_MAX_PAIRS,\\n    )\\n    if len(hp0) < V32_MIN_CORRESPONDENCES:\\n        return None\\n\\n    hp, hq, hs, cov = v3_spatial_coverage_select(\\n        hp0, hq0, hs0, src_xyz, tgt_xyz,\\n        max_pairs=V32_HARD_MAX_PAIRS,\\n        max_per_source_voxel=V32_HARD_MAX_PER_SOURCE_VOXEL,\\n        max_per_target_voxel=V32_HARD_MAX_PER_TARGET_VOXEL,\\n    )\\n    if len(hp) < V32_MIN_CORRESPONDENCES:\\n        return None\\n\\n    return {\\n        \\\'pairs\\\': hp, \\\'quality\\\': hq, \\\'support\\\': hs,\\n        \\\'candidate_pairs\\\': int(len(hard_pairs)),\\n        \\\'selected_pairs\\\': int(len(hp)),\\n        \\\'coverage\\\': cov,\\n    }\\n\\ndef v33_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\\n    start = time.perf_counter()\\n    base = v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed)\\n    out = dict(base)\\n    out[\\\'method\\\'] = \\\'Bplus_mv_v3_3\\\'\\n    out[\\\'v33_used_holdout\\\'] = False\\n    out[\\\'v33_selected_type\\\'] = \\\'v32_passthrough\\\'\\n    out[\\\'v33_validation_gain\\\'] = 0.0\\n    out[\\\'v33_train_pairs\\\'] = 0\\n    out[\\\'v33_validation_pairs\\\'] = 0\\n    out[\\\'v33_holdout_groups\\\'] = 0\\n    out[\\\'v33_rotation_observable\\\'] = False\\n    out[\\\'v33_mid_eigen_ratio\\\'] = np.nan\\n    out[\\\'v33_rms_spread_m\\\'] = np.nan\\n    out[\\\'v33_cycle_rot_deg\\\'] = np.nan\\n    out[\\\'v33_cycle_trans_m\\\'] = np.nan\\n\\n    # Easy case remains exactly V3.2.\\n    if not bool(base.get(\\\'fallback_triggered\\\', False)):\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\\\'xyz\\\']).astype(np.float64), src_c2w)\\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\\\'xyz\\\']).astype(np.float64), tgt_c2w)\\n    pool = v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz)\\n    if pool is None:\\n        out[\\\'v33_selected_type\\\'] = \\\'v32_fallback_no_pool\\\'\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    hp = pool[\\\'pairs\\\']; hq = pool[\\\'quality\\\']\\n    train_idx, val_idx, split_diag = v33_spatial_holdout_split(hp, src_xyz, tgt_xyz, seed)\\n    if train_idx is None:\\n        out[\\\'v33_selected_type\\\'] = \\\'v32_fallback_split_failed\\\'\\n        out[\\\'v33_split_reason\\\'] = split_diag.get(\\\'reason\\\')\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    out[\\\'v33_used_holdout\\\'] = True\\n    out[\\\'v33_train_pairs\\\'] = int(len(train_idx))\\n    out[\\\'v33_validation_pairs\\\'] = int(len(val_idx))\\n    out[\\\'v33_holdout_groups\\\'] = int(split_diag.get(\\\'groups\\\', 0))\\n\\n    TP = src_xyz[hp[train_idx, 0]]\\n    TQ = tgt_xyz[hp[train_idx, 1]]\\n    TW0 = np.clip(hq[train_idx], 1e-8, None)\\n    VP = src_xyz[hp[val_idx, 0]]\\n    VQ = tgt_xyz[hp[val_idx, 1]]\\n    VW = np.clip(hq[val_idx], 1e-8, None)\\n\\n    TW, leverage = v33_leverage_weights(TP, TQ, TW0)\\n    obs = v33_observability(TP, TQ, TW)\\n    out[\\\'v33_rotation_observable\\\'] = bool(obs[\\\'rotation_observable\\\'])\\n    out[\\\'v33_mid_eigen_ratio\\\'] = float(obs[\\\'mid_eigen_ratio\\\'])\\n    out[\\\'v33_rms_spread_m\\\'] = float(obs[\\\'rms_spread_m\\\'])\\n    out[\\\'v33_leverage_mean\\\'] = float(np.mean(leverage))\\n    out[\\\'v33_leverage_p90\\\'] = float(np.percentile(leverage, 90))\\n\\n    T_fast = base.get(\\\'T_fast_debug\\\')\\n    # Rare case: Fast failed. Build a robust anchor on TRAIN only.\\n    if T_fast is None:\\n        hard_anchor = v3_prosac_ransac(\\n            TP, TQ, TW, seed,\\n            max_iters=V33_HARD_RANSAC_MAX_ITERS,\\n            min_iters=V33_HARD_RANSAC_MIN_ITERS,\\n            early_ratio=V33_HARD_EARLY_RATIO,\\n            early_median_m=V33_HARD_EARLY_MEDIAN_M,\\n        )\\n        if hard_anchor is None:\\n            out[\\\'v33_selected_type\\\'] = \\\'v32_fallback_anchor_failed\\\'\\n            out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n            return out\\n        T_fast = hard_anchor[\\\'T\\\']\\n        out[\\\'v33_anchor_from_hard_train_ransac\\\'] = True\\n    else:\\n        out[\\\'v33_anchor_from_hard_train_ransac\\\'] = False\\n\\n    # Reverse train-only fit for a soft bidirectional consistency reference.\\n    try:\\n        T_reverse = v33_irls_full(np.linalg.inv(T_fast), TQ, TP, TW)\\n        T_cycle_reference = np.linalg.inv(T_reverse)\\n    except Exception:\\n        T_cycle_reference = None\\n\\n    base_eval = v33_validation_score(\\n        T_fast, VP, VQ, VW,\\n        T_cycle_reference=T_cycle_reference,\\n        T_fast=T_fast,\\n    )\\n    out[\\\'v33_fast_validation_score\\\'] = float(base_eval[\\\'regularized_score\\\'])\\n\\n    raw = {\\n        \\\'translation_irls\\\': v33_irls_translation(T_fast, TP, TQ, TW),\\n    }\\n    # Only allow hypotheses that change rotation when the geometry supports it.\\n    if obs[\\\'rotation_observable\\\']:\\n        raw[\\\'full_irls\\\'] = v33_irls_full(T_fast, TP, TQ, TW)\\n        raw[\\\'rotation_irls\\\'] = v33_irls_rotation(T_fast, TP, TQ, TW)\\n\\n    candidates = []\\n    for kind, T_raw in raw.items():\\n        cap, raw_dr, raw_dt = v33_alpha_cap(T_fast, T_raw)\\n        if cap <= 0:\\n            continue\\n        for frac in V33_LINESEARCH_FRACTIONS:\\n            alpha = float(frac) * cap\\n            if alpha <= 0:\\n                continue\\n            T = v32_blend_transform(T_fast, T_raw, alpha)\\n            ev = v33_validation_score(\\n                T, VP, VQ, VW,\\n                T_cycle_reference=T_cycle_reference,\\n                T_fast=T_fast,\\n            )\\n            accepted = v33_candidate_acceptable(base_eval, ev)\\n            candidates.append({\\n                \\\'type\\\': kind, \\\'alpha\\\': alpha,\\n                \\\'T\\\': T, \\\'eval\\\': ev, \\\'accepted\\\': bool(accepted),\\n                \\\'raw_delta_rot_deg\\\': raw_dr,\\n                \\\'raw_delta_trans_m\\\': raw_dt,\\n            })\\n\\n    accepted = [c for c in candidates if c[\\\'accepted\\\']]\\n    if accepted:\\n        best = max(accepted, key=lambda c: c[\\\'eval\\\'][\\\'regularized_score\\\'])\\n        out[\\\'T_est\\\'] = best[\\\'T\\\']\\n        out[\\\'selected_stage\\\'] = \\\'v33_holdout_local\\\'\\n        out[\\\'fallback_selected\\\'] = True\\n        out[\\\'v33_selected_type\\\'] = best[\\\'type\\\']\\n        out[\\\'v33_selected_alpha\\\'] = float(best[\\\'alpha\\\'])\\n        out[\\\'v33_validation_score\\\'] = float(best[\\\'eval\\\'][\\\'regularized_score\\\'])\\n        out[\\\'v33_validation_gain\\\'] = float(\\n            best[\\\'eval\\\'][\\\'regularized_score\\\'] - base_eval[\\\'regularized_score\\\']\\n        )\\n        out[\\\'v33_cycle_rot_deg\\\'] = float(best[\\\'eval\\\'][\\\'cycle_rot_deg\\\'])\\n        out[\\\'v33_cycle_trans_m\\\'] = float(best[\\\'eval\\\'][\\\'cycle_trans_m\\\'])\\n        out[\\\'v33_step_rot_deg\\\'] = float(best[\\\'eval\\\'][\\\'step_rot_deg\\\'])\\n        out[\\\'v33_step_trans_m\\\'] = float(best[\\\'eval\\\'][\\\'step_trans_m\\\'])\\n    else:\\n        # Critical safety behavior: when hold-out does not prove a gain, keep FAST.\\n        out[\\\'T_est\\\'] = T_fast\\n        out[\\\'selected_stage\\\'] = \\\'v33_keep_fast\\\'\\n        out[\\\'fallback_selected\\\'] = False\\n        out[\\\'v33_selected_type\\\'] = \\\'keep_fast\\\'\\n        out[\\\'v33_selected_alpha\\\'] = 0.0\\n        out[\\\'v33_validation_score\\\'] = float(base_eval[\\\'regularized_score\\\'])\\n        out[\\\'v33_validation_gain\\\'] = 0.0\\n        out[\\\'v33_cycle_rot_deg\\\'] = float(base_eval[\\\'cycle_rot_deg\\\'])\\n        out[\\\'v33_cycle_trans_m\\\'] = float(base_eval[\\\'cycle_trans_m\\\'])\\n        out[\\\'v33_step_rot_deg\\\'] = 0.0\\n        out[\\\'v33_step_trans_m\\\'] = 0.0\\n\\n    # Recompute selected consensus on the SAME hard pool for diagnostics only.\\n    selected_stats = v3_consensus_stats(\\n        out[\\\'T_est\\\'], src_xyz[hp[:,0]], tgt_xyz[hp[:,1]], hq\\n    )\\n    out[\\\'selected_pairs\\\'] = int(len(hp))\\n    out[\\\'selected_inlier_ratio_5cm\\\'] = float(selected_stats[\\\'ratio_5cm\\\'])\\n    out[\\\'selected_inlier_ratio_3cm\\\'] = float(selected_stats[\\\'ratio_3cm\\\'])\\n    out[\\\'selected_validation_score\\\'] = float(selected_stats[\\\'score\\\'])\\n    out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n    return out\\n\\n# Original notebook cell 79\\nV331_MIN_GAIN_TRANSLATION = 0.015\\n\\nV331_MIN_GAIN_FAST_RESCUE = 0.015\\n\\nV331_MIN_GAIN_ROTATION = 0.020\\n\\nV331_MIN_GAIN_FULL = 0.020\\n\\nV331_ROTATION_ENABLE_MID_RATIO = 0.45\\n\\nV331_ROTATION_ENABLE_SPREAD_M = 0.10\\n\\nV331_MAX_FROM_V32_ROT_DEG = float(V32_TRUST_MAX_ROT_DEG)\\n\\nV331_MAX_FROM_V32_TRANS_M = max(float(V32_TRUST_MAX_TRANS_M), 0.010)\\n\\nV331_MAX_RATIO5_DROP = 0.002\\n\\nV331_MAX_RATIO3_DROP = 0.002\\n\\nV331_MAX_MEDIAN5_INCREASE_M = 0.0005\\n\\nV331_MAX_CYCLE_ROT_INCREASE_DEG = 0.20\\n\\nV331_MAX_CYCLE_TRANS_INCREASE_M = 0.005\\n\\nV331_LINESEARCH_FRACTIONS = tuple(V33_LINESEARCH_FRACTIONS)\\n\\n# Original notebook cell 81\\ndef v331_min_gain(kind):\\n    if kind.startswith(\\\'translation\\\'):\\n        return V331_MIN_GAIN_TRANSLATION\\n    if kind == \\\'fast_rescue\\\':\\n        return V331_MIN_GAIN_FAST_RESCUE\\n    if kind.startswith(\\\'rotation\\\'):\\n        return V331_MIN_GAIN_ROTATION\\n    if kind.startswith(\\\'full\\\'):\\n        return V331_MIN_GAIN_FULL\\n    return max(V331_MIN_GAIN_TRANSLATION, V331_MIN_GAIN_ROTATION)\\n\\ndef v331_rotation_allowed(obs):\\n    return bool(\\n        float(obs[\\\'mid_eigen_ratio\\\']) >= V331_ROTATION_ENABLE_MID_RATIO\\n        and float(obs[\\\'rms_spread_m\\\']) >= V331_ROTATION_ENABLE_SPREAD_M\\n    )\\n\\ndef v331_validation_eval(T, P, Q, w, T_cycle_reference, T_v32):\\n    # Reuse V3.3 score, but step penalty is now relative to V3.2 selected pose.\\n    return v33_validation_score(\\n        T, P, Q, w,\\n        T_cycle_reference=T_cycle_reference,\\n        T_fast=T_v32,\\n    )\\n\\ndef v331_candidate_acceptable(base_eval, cand_eval, kind, T_candidate, T_v32):\\n    b = base_eval[\\\'stats\\\']\\n    c = cand_eval[\\\'stats\\\']\\n    if not np.isfinite(cand_eval[\\\'regularized_score\\\']):\\n        return False, \\\'nonfinite_score\\\'\\n\\n    gain = float(cand_eval[\\\'regularized_score\\\'] - base_eval[\\\'regularized_score\\\'])\\n    if gain < v331_min_gain(kind):\\n        return False, \\\'gain_too_small\\\'\\n\\n    # Trust region is measured from the already accepted V3.2 solution.\\n    dr, dt = v32_pose_delta(T_candidate, T_v32)\\n    if dr > V331_MAX_FROM_V32_ROT_DEG + 1e-12:\\n        return False, \\\'delta_rot_from_v32\\\'\\n    if dt > V331_MAX_FROM_V32_TRANS_M + 1e-12:\\n        return False, \\\'delta_trans_from_v32\\\'\\n\\n    if c[\\\'ratio_5cm\\\'] < b[\\\'ratio_5cm\\\'] - V331_MAX_RATIO5_DROP:\\n        return False, \\\'ratio5_drop\\\'\\n    if c[\\\'ratio_3cm\\\'] < b[\\\'ratio_3cm\\\'] - V331_MAX_RATIO3_DROP:\\n        return False, \\\'ratio3_drop\\\'\\n\\n    if np.isfinite(b[\\\'median_5cm_m\\\']) and np.isfinite(c[\\\'median_5cm_m\\\']):\\n        if c[\\\'median_5cm_m\\\'] > b[\\\'median_5cm_m\\\'] + V331_MAX_MEDIAN5_INCREASE_M:\\n            return False, \\\'median5_regression\\\'\\n\\n    if np.isfinite(base_eval[\\\'cycle_rot_deg\\\']) and np.isfinite(cand_eval[\\\'cycle_rot_deg\\\']):\\n        if cand_eval[\\\'cycle_rot_deg\\\'] > base_eval[\\\'cycle_rot_deg\\\'] + V331_MAX_CYCLE_ROT_INCREASE_DEG:\\n            return False, \\\'cycle_rot_regression\\\'\\n    if np.isfinite(base_eval[\\\'cycle_trans_m\\\']) and np.isfinite(cand_eval[\\\'cycle_trans_m\\\']):\\n        if cand_eval[\\\'cycle_trans_m\\\'] > base_eval[\\\'cycle_trans_m\\\'] + V331_MAX_CYCLE_TRANS_INCREASE_M:\\n            return False, \\\'cycle_trans_regression\\\'\\n\\n    return True, \\\'accepted\\\'\\n\\n# Original notebook cell 83\\ndef v331_registration_once(scache, tcache, src_c2w, tgt_c2w, seed):\\n    start = time.perf_counter()\\n    base = v32_registration_once(scache, tcache, src_c2w, tgt_c2w, seed)\\n    out = dict(base)\\n    out[\\\'method\\\'] = \\\'Bsafe_mv_v3_3_1\\\'\\n    out[\\\'v331_used_holdout\\\'] = False\\n    out[\\\'v331_selected_type\\\'] = \\\'v32_passthrough\\\'\\n    out[\\\'v331_validation_gain\\\'] = 0.0\\n    out[\\\'v331_train_pairs\\\'] = 0\\n    out[\\\'v331_validation_pairs\\\'] = 0\\n    out[\\\'v331_rotation_allowed\\\'] = False\\n    out[\\\'v331_mid_eigen_ratio\\\'] = np.nan\\n    out[\\\'v331_rms_spread_m\\\'] = np.nan\\n    out[\\\'v331_candidate_count\\\'] = 0\\n    out[\\\'v331_accepted_count\\\'] = 0\\n    out[\\\'v331_reject_reasons\\\'] = \\\'\\\'\\n    out[\\\'v331_base_stage\\\'] = base.get(\\\'selected_stage\\\')\\n    out[\\\'v331_base_selected_type\\\'] = base.get(\\\'local_selected_type\\\')\\n\\n    # Easy case is exactly V3.2.\\n    if not bool(base.get(\\\'fallback_triggered\\\', False)):\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    T_v32 = base.get(\\\'T_est\\\')\\n    if T_v32 is None:\\n        out[\\\'v331_selected_type\\\'] = \\\'v32_failed_no_pose\\\'\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n    T_v32 = np.asarray(T_v32, dtype=np.float64)\\n\\n    src_xyz = mv_world_to_camera_points(v2_np(scache[\\\'xyz\\\']).astype(np.float64), src_c2w)\\n    tgt_xyz = mv_world_to_camera_points(v2_np(tcache[\\\'xyz\\\']).astype(np.float64), tgt_c2w)\\n    pool = v33_prepare_hard_pool(scache, tcache, src_xyz, tgt_xyz)\\n    if pool is None:\\n        out[\\\'v331_selected_type\\\'] = \\\'keep_v32_no_pool\\\'\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    hp = pool[\\\'pairs\\\']\\n    hq = pool[\\\'quality\\\']\\n    train_idx, val_idx, split_diag = v33_spatial_holdout_split(hp, src_xyz, tgt_xyz, seed)\\n    if train_idx is None:\\n        out[\\\'v331_selected_type\\\'] = \\\'keep_v32_split_failed\\\'\\n        out[\\\'v331_split_reason\\\'] = split_diag.get(\\\'reason\\\')\\n        out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n        return out\\n\\n    out[\\\'v331_used_holdout\\\'] = True\\n    out[\\\'v331_train_pairs\\\'] = int(len(train_idx))\\n    out[\\\'v331_validation_pairs\\\'] = int(len(val_idx))\\n\\n    TP = src_xyz[hp[train_idx, 0]]\\n    TQ = tgt_xyz[hp[train_idx, 1]]\\n    TW0 = np.clip(hq[train_idx], 1e-8, None)\\n    VP = src_xyz[hp[val_idx, 0]]\\n    VQ = tgt_xyz[hp[val_idx, 1]]\\n    VW = np.clip(hq[val_idx], 1e-8, None)\\n\\n    TW, leverage = v33_leverage_weights(TP, TQ, TW0)\\n    obs = v33_observability(TP, TQ, TW)\\n    rot_allowed = v331_rotation_allowed(obs)\\n    out[\\\'v331_rotation_allowed\\\'] = bool(rot_allowed)\\n    out[\\\'v331_mid_eigen_ratio\\\'] = float(obs[\\\'mid_eigen_ratio\\\'])\\n    out[\\\'v331_rms_spread_m\\\'] = float(obs[\\\'rms_spread_m\\\'])\\n\\n    T_fast = base.get(\\\'T_fast_debug\\\')\\n    if T_fast is None:\\n        T_fast = T_v32.copy()\\n    else:\\n        T_fast = np.asarray(T_fast, dtype=np.float64)\\n\\n    # Reverse TRAIN-only fit provides a cycle-consistency reference.\\n    try:\\n        T_reverse = v33_irls_full(np.linalg.inv(T_v32), TQ, TP, TW)\\n        T_cycle_reference = np.linalg.inv(T_reverse)\\n    except Exception:\\n        T_cycle_reference = None\\n\\n    base_eval = v331_validation_eval(\\n        T_v32, VP, VQ, VW,\\n        T_cycle_reference=T_cycle_reference,\\n        T_v32=T_v32,\\n    )\\n    out[\\\'v331_v32_validation_score\\\'] = float(base_eval[\\\'regularized_score\\\'])\\n\\n    # Raw candidates. V3.2 selected is not in this list because it is the baseline.\\n    raw_specs = []\\n    dr_fast, dt_fast = v32_pose_delta(T_fast, T_v32)\\n    if dr_fast > 1e-10 or dt_fast > 1e-10:\\n        raw_specs.append((\\\'fast_rescue\\\', T_fast, T_fast, False))\\n\\n    raw_specs.append((\\n        \\\'translation_fast\\\',\\n        v33_irls_translation(T_fast, TP, TQ, TW),\\n        T_fast,\\n        True,\\n    ))\\n    raw_specs.append((\\n        \\\'translation_v32\\\',\\n        v33_irls_translation(T_v32, TP, TQ, TW),\\n        T_v32,\\n        True,\\n    ))\\n\\n    if rot_allowed:\\n        raw_specs.append((\\n            \\\'rotation_fast\\\',\\n            v33_irls_rotation(T_fast, TP, TQ, TW),\\n            T_fast,\\n            True,\\n        ))\\n        raw_specs.append((\\n            \\\'full_fast\\\',\\n            v33_irls_full(T_fast, TP, TQ, TW),\\n            T_fast,\\n            True,\\n        ))\\n\\n    candidates = []\\n    reject_reasons = {}\\n    for kind, T_raw, anchor, use_linesearch in raw_specs:\\n        if kind == \\\'fast_rescue\\\':\\n            trial_list = [(1.0, np.asarray(T_raw, dtype=np.float64))]\\n        else:\\n            cap, _, _ = v33_alpha_cap(anchor, T_raw)\\n            trial_list = []\\n            if cap > 0:\\n                for frac in V331_LINESEARCH_FRACTIONS:\\n                    alpha = float(frac) * float(cap)\\n                    if alpha > 0:\\n                        trial_list.append((alpha, v32_blend_transform(anchor, T_raw, alpha)))\\n\\n        for alpha, T in trial_list:\\n            ev = v331_validation_eval(\\n                T, VP, VQ, VW,\\n                T_cycle_reference=T_cycle_reference,\\n                T_v32=T_v32,\\n            )\\n            accepted, reason = v331_candidate_acceptable(\\n                base_eval, ev, kind, T, T_v32\\n            )\\n            reject_reasons[reason] = reject_reasons.get(reason, 0) + 1\\n            candidates.append({\\n                \\\'type\\\': kind,\\n                \\\'alpha\\\': float(alpha),\\n                \\\'T\\\': T,\\n                \\\'eval\\\': ev,\\n                \\\'accepted\\\': bool(accepted),\\n                \\\'reason\\\': reason,\\n                \\\'gain\\\': float(ev[\\\'regularized_score\\\'] - base_eval[\\\'regularized_score\\\']),\\n            })\\n\\n    out[\\\'v331_candidate_count\\\'] = int(len(candidates))\\n    accepted = [c for c in candidates if c[\\\'accepted\\\']]\\n    out[\\\'v331_accepted_count\\\'] = int(len(accepted))\\n    out[\\\'v331_reject_reasons\\\'] = \\\'|\\\'.join(\\n        f\\\'{k}:{v}\\\' for k, v in sorted(reject_reasons.items())\\n    )\\n\\n    if accepted:\\n        best = max(accepted, key=lambda c: c[\\\'eval\\\'][\\\'regularized_score\\\'])\\n        out[\\\'T_est\\\'] = best[\\\'T\\\']\\n        out[\\\'selected_stage\\\'] = \\\'v331_safe_holdout\\\'\\n        out[\\\'fallback_selected\\\'] = True\\n        out[\\\'v331_selected_type\\\'] = best[\\\'type\\\']\\n        out[\\\'v331_selected_alpha\\\'] = float(best[\\\'alpha\\\'])\\n        out[\\\'v331_validation_gain\\\'] = float(best[\\\'gain\\\'])\\n        out[\\\'v331_validation_score\\\'] = float(best[\\\'eval\\\'][\\\'regularized_score\\\'])\\n        out[\\\'v331_cycle_rot_deg\\\'] = float(best[\\\'eval\\\'][\\\'cycle_rot_deg\\\'])\\n        out[\\\'v331_cycle_trans_m\\\'] = float(best[\\\'eval\\\'][\\\'cycle_trans_m\\\'])\\n        out[\\\'v331_step_from_v32_rot_deg\\\'] = float(best[\\\'eval\\\'][\\\'step_rot_deg\\\'])\\n        out[\\\'v331_step_from_v32_trans_m\\\'] = float(best[\\\'eval\\\'][\\\'step_trans_m\\\'])\\n    else:\\n        # Safety invariant: preserve the exact V3.2 selected solution.\\n        out[\\\'T_est\\\'] = T_v32\\n        out[\\\'selected_stage\\\'] = \\\'v331_keep_v32\\\'\\n        out[\\\'fallback_selected\\\'] = bool(base.get(\\\'fallback_selected\\\', False))\\n        out[\\\'v331_selected_type\\\'] = \\\'keep_v32_selected\\\'\\n        out[\\\'v331_selected_alpha\\\'] = 0.0\\n        out[\\\'v331_validation_gain\\\'] = 0.0\\n        out[\\\'v331_validation_score\\\'] = float(base_eval[\\\'regularized_score\\\'])\\n        out[\\\'v331_cycle_rot_deg\\\'] = float(base_eval[\\\'cycle_rot_deg\\\'])\\n        out[\\\'v331_cycle_trans_m\\\'] = float(base_eval[\\\'cycle_trans_m\\\'])\\n        out[\\\'v331_step_from_v32_rot_deg\\\'] = 0.0\\n        out[\\\'v331_step_from_v32_trans_m\\\'] = 0.0\\n\\n    selected_stats = v3_consensus_stats(\\n        out[\\\'T_est\\\'], src_xyz[hp[:, 0]], tgt_xyz[hp[:, 1]], hq\\n    )\\n    out[\\\'selected_pairs\\\'] = int(len(hp))\\n    out[\\\'selected_inlier_ratio_5cm\\\'] = float(selected_stats[\\\'ratio_5cm\\\'])\\n    out[\\\'selected_inlier_ratio_3cm\\\'] = float(selected_stats[\\\'ratio_3cm\\\'])\\n    out[\\\'selected_validation_score\\\'] = float(selected_stats[\\\'score\\\'])\\n    out[\\\'matcher_runtime_s\\\'] = float(time.perf_counter() - start)\\n    return out\\n\'\n_SB_PROVENANCE = {\'source_name\': \'mv-slam_apart_1_heldout_v331(1).ipynb\', \'source_sha256\': \'420a7a87b20421e9416af654b4d3becbbbf374015903dd77339d44ce7407c15c\', \'helper_sha256\': \'bf4430d906da180d6482d223f0e3f4648e40015148dee26751ff433f29908c8f\', \'selected_original_cells\': [46, 47, 48, 49, 50, 58, 60, 61, 62, 63, 64, 65, 66, 67, 69, 71, 73, 75, 77, 79, 81, 83], \'function_sha256\': {\'mv_find_submap_index\': \'b6121be2ba7c4906007edb41426580cff6be66180eee12531dd6e71f8216da90\', \'mv_frame_c2w\': \'885dede389cdd20ffcd5bb47691b01a293017c4b56cf7490723cf32e83dcb312\', \'mv_to_pil\': \'82ecffffb663bb0352d1dbffab4b4731338e490145e18942ec4443bf11c83639\', \'mv_depth_tensor\': \'489f6feccd5a93c77c1b16328df0fdc15434a654284fe910f30aa5372fd32430\', \'mv_extract_feature_map\': \'47a85425d4921dc331ddf7c9724a29f74eaeee37df5845b08dd7877e8993a0f2\', \'mv_bilinear_sample\': \'3c1a1f477221bf9dd635bf857b58b1d5f4e28739789db69c74396a740eb04d1f\', \'mv_restore_gaussians\': \'cd2291813692990c2f61d29b2b3fd0954feb50dd0b427905d7c760785a776540\', \'mv_spatial_topk\': \'99e875a201f52e224d56a34381f0ebf493df9c7d7f3b3842b5b6fa0a168fae82\', \'mv_cache_path\': \'b59b56f2328476052db2c4af44fcff971788dc8661b1a76fdd683b4ed12ca05d\', \'mv_build_cache\': \'2db98cedef8aceb39bddaa8ac236326099faa0d08f624f2f9465b04b22816319\', \'mv_world_to_camera_points\': \'758176c6923d1773f60e5c2abe322af9b65c9207ab63f21df421735c359ee931\', \'mv_match_descriptors\': \'9da40aa9ce14f38e99157100ad3372f39a2f7f9c2a089f45934bb14f3a67473d\', \'mv_geometric_compatibility\': \'3d82a6f0efc7bbcaeac3d00e738305e0b8f41877aed350ec0a50601b0ff88cd6\', \'mv_ransac_baseline\': \'1455d008b254b5bb917a47461f7a1ac2dc1ec6fd24e8cbe8ecc3689a8c5c011c\', \'mv_rotation_translation_error\': \'43e90ac9e8baca08d62e1c57ddd098c4bc963029fb10b9e4442f97f5c5462b3b\', \'mv_gt_pose\': \'1ba7390a85908815d153030a66ce43211bdeebe2d0194b89ea650bf22d23dd5c\', \'mv_baseline_coarse_transform\': \'25dabeef8b54dff2207a38cb02a1af7d42e0dc364951796d2996f6b5ff947c51\', \'mv_metric_row\': \'17127afb6d6f54e83402d9aceb4a3854ea203f463f11895ee0b6f03fafd15eb2\', \'mv_first_metric\': \'3ecf5a7a35fad19b660977bdd54e322a422409f61b1a730f5369c48ee55da420\', \'mv_descriptor_for_variant\': \'c083260274e9c5bbbf11620b2e0b53a1f58457e1a13c0a181bdda8b43e3284e8\', \'_safe_mean\': \'2fe0204fb5f6fc234eb53359c5148b696a4d140b8befbfb893f889c2c37c2a5f\', \'_safe_std\': \'b71510c6ad6e6dd76c6a0d79521d2d1a4a9c34e22c0fb2c3b48403dd1b9a5371\', \'a_match_descriptors\': \'5688ff7afc6d0f0cdb3e56fcbd9e81058dfdfa8135179177cd0eea6dce25f154\', \'a_geometric_compatibility\': \'cfaf781c8d593453be3531459e10cabc6f1ec011b85a935ff5fcf0406e4be818\', \'a_ransac\': \'b977a55bd9dbb156833c16ae015c64d02cc2804ab5f945b8d7e98e99ee3cf32c\', \'a_registration_once\': \'1f84f748081c9b1e421cf245101a51633beb92621657e6cfaf7f20aed2f2b23b\', \'v2_np\': \'5ac5c6049f19e07e9b3a494c0714231ce1f7e8f55218c7e8f66a4ff8615c74aa\', \'v2_normalize_desc\': \'685376be31751cb2e9a13e3b465f3c3c244a7d55ad4d4bda78f9ae0faec4f0c2\', \'v2_adaptive_k_from_row\': \'38a5b1ee34acf1a9439bd2006ebbd5469e33402c71eebfa6f106f5be16b4dec0\', \'v2_build_adaptive_bidirectional\': \'c6415663e1d85385632c4cb46e3a7778c8f782762e420b6b5b3023a4a5a12555\', \'v2_pair_quality\': \'85d8cd2e59a001fbd2a7d25dd53a9e14ba6deef762d9fe8d45e4cb303fe02c8e\', \'v2_greedy_unique_by_confidence\': \'5667bb770a47080f8a063c4b64cb09a73e5261d7eaf4b2c2856c2e97bdb94d3f\', \'v2_spatial_coverage_select\': \'a518bfdcbfbbfb9a5198271dd7e450590b8c29aa3781b681bf9acf6462a26a6f\', \'v2_geometry_condition\': \'6705691ba5e76eaf1f0a8194cc8e22f4279d2e7233848dcec5c33a5bb029ccff\', \'mv_topk_rigidity_support\': \'c19740ec850961079895b2b61649f8d9a98e0394aa97ef22c4666d185fd674c0\', \'v2_weighted_kabsch\': \'7f048c6bb4cc22c29ff4b3b1e3e45bf5a4117a4fa780ece418906807bed885e2\', \'v2_transform_points\': \'a43eecd684b86adeb221e8bb8736b203160cca0a6f9a94095ae2154c196bc730\', \'v2_non_degenerate_triplet\': \'c92b35eef1c987db31f2647baa00bddc0ed521d3b89b0f8a5c00cdeb76e1b373\', \'v2_prosac_ransac\': \'057327ee19f00f399ebbe6731844beb121b62c1f70350b58acd245098e41700b\', \'v2_registration_once\': \'f3950988ff0cbe57331bc38d1fde1b48e1532f7e3064651fd06c0b92dcc2a495\', \'v3_conf_stats\': \'0a94e4a37b089ba146b1e69d0f27542c4f97050f1e7638e536fcc04eaba2de3a\', \'v3_spatial_coverage_select\': \'77692b4fa988118456ae50581eb84d774587ffccf0b875d87045c34795f10db9\', \'v3_greedy_limited_reuse\': \'1efffd2867df58b1d9e8b703abf144fd8ef41f9fa748eae52fc095e8a683805f\', \'v3_geometry_diag\': \'581f4f436c4c2bfdaf7b4160a90376d9596a10b0618920f1350072fc34c75778\', \'v3_consensus_stats\': \'fcffc4eea851a77de2412b0388925f9612f0a88534433fca9befa4b73da5973d\', \'v3_prosac_ransac\': \'dd0af2930943a642915d8f66f959fa94fd377fb984428f2f3e23cffe41bedd37\', \'v3_should_trigger_fallback\': \'f9c3781420f911f48bb47d3d0e5021d3d90bd5b1e3d58eefa5588e97b2512465\', \'v32_build_hard_candidates\': \'7924fb4674589f2806fafeec068ca92078e7d09d6680fed5ec2e24e738c32b47\', \'v32_hard_quality\': \'b625b3d385d5232259c0964ec481040dad8e4d9224af7fffc5e906cd85f8c036\', \'v32_greedy_unique_with_indices\': \'774a7591d43ed092591af586c8a784e159c275a765b147860ccc7e0997ba9341\', \'v32_top_quality_prefilter\': \'002280df6d22ce491c9f6b70f4c8fe1e80a84cdb5742482687bb58e876ebb2c2\', \'v32_ambiguity_diag\': \'12809884e8efafd92c8feb017ff54a193a6832584759c39b5149534ca80645fa\', \'v32_rotation_angle_deg\': \'394cf36b8fa5de86f867ac46af734e28159acf7bcdce62424acdaa972e502ffb\', \'v32_fractional_rotation\': \'8f238c18bd509f1a90e58a181c48b83a194fc4edbb385c5d51b214e67601ab98\', \'v32_pose_delta\': \'749c015b7a81490ffd6fe1f456add3d4872988fe3879ac1646b0899a7ef3786d\', \'v32_blend_transform\': \'b2aa9147336b856d8d0afe92068441a5f026465a88c7a6ed4453de8ea8075bc7\', \'v32_local_precision_score\': \'0adfded284c793b845750b85ef602bb796b1a88aebcda3363aa9c3058d28e594\', \'v32_robust_local_raw_candidates\': \'adb58b3b539ef91d0574675e4d3305727e92b34d19fed6aee47e2c7f60d1e8d2\', \'v32_select_local_candidate\': \'6602dbdd2084e3a507dd97bf607f2a76a661106784f2c4aa14aa509143787832\', \'v32_registration_once\': \'490c35a02cc2ec0dbe7753a06479748ffec2438eaa6de0fbf8554d61d66b3fa4\', \'_as_np_pose\': \'0a3427c9c8278ad75e12c0ca625c85e8c42ca66ecc784de3577e05b7e7bfb300\', \'loop_to_submap_start_transform\': \'468d9d2c51eff76c9d84d4c6212e4a6378f53ab47d7eeae562460a9beac43162\', \'submap_start_to_loop_transform\': \'e5462468f9e43d4650c1bbbe79820cff44f5190a33863a8257fd5226ded02ace\', \'transform_delta\': \'28385475d49f233f0f4a67ad51943efaefb952256d9fb5884d11c29652ce9477\', \'dense_cloud_for_submap\': \'1bc41b3d66e11f2f6c3574fc89c4985be28b8825959df1ff5b2268915a6349df\', \'exact_production_icp\': \'ed3d9bcb29abe777e3ca935350c24eb01c6673f9cad58d81106f94a492cae07a\', \'v33_group_key\': \'ef401610b904e7653ac7475d274d08a826482c7e4066f4047509cb1a526c8f1a\', \'v33_group_hash\': \'555e5d6596c4fc735ed71b5395a55ad2503960c47a76048d163ca5636cf08cd8\', \'v33_spatial_holdout_split\': \'e91c66b3350dc741f272fed88c4d6b2cefee9dcc09b2a769d48af93bb5eac453\', \'v33_weighted_covariance\': \'cae1b802f6de16970d056a7bcf29ef56df659252f704e20af568c706e0adb0b5\', \'v33_observability\': \'782d227074d2fbef4d47e545619f59f4eb209f9e559071366d0d3b7aef0ae0fd\', \'v33_leverage_weights\': \'8036ed90d7c2d365c6a9299d9bf09f6dc055c5f2630c9aa49cbc24fde06e5f60\', \'v33_cauchy_weights\': \'e15319e464e3301f4e12515c4e2a03bfc3f1a50a4644bc0d7b2523154f330b7d\', \'v33_gate_threshold\': \'e7bf45ae7d39eba5ef5b97a7338401922f668db3e014d8732b0bf70d01a5504a\', \'v33_irls_full\': \'5cf02962b8499a79fd93b10e3d2b6087ce18737c19f53862627341a622748531\', \'v33_irls_translation\': \'fec17ef9818680c0cf9ebcb9ed60c575fda37f2f58680441d6a35c9b6d9ccbd2\', \'v33_weighted_rotation_fixed_translation\': \'85a6f84952649eb7551cdca346c091bea62cb8724fd0660d787c15460f20c189\', \'v33_irls_rotation\': \'3d772e35a853757230fdac1bc1f53dc98e8ef2d17cbd03d4a9c6e3a18076fe76\', \'v33_alpha_cap\': \'b34c20595a88946dbae3b733fb98ec3d95b3497dbcf530b75a1890e5e9091f41\', \'v33_validation_score\': \'566f47f35e958b9970f158f0f5cec654d71ebb43e32911129dc12898670b0eb9\', \'v33_candidate_acceptable\': \'416ea0628cbef8299fd9837638f6dfe8588b805409d288578e87842ad58efe8e\', \'v33_prepare_hard_pool\': \'00950e076a0c8700001fe0f920b4713122cd7ed65cc30528ccf34e7657387b02\', \'v33_registration_once\': \'4254d9894dc2cb48db70f61b9c3b92a921b93987d0a91dd5af1fd45e3a458c33\', \'v331_min_gain\': \'d814c9ec61249ccb9ccd4a34811ed9935fee295c9e9b1c158341001054c6974f\', \'v331_rotation_allowed\': \'8ccc1c207d08f0beb3a9db4d68950013719daf5c7ef5a0f1f25040c85085b646\', \'v331_validation_eval\': \'2bf343a32f91f9c9c0916fc934ac973943d5544ec60da8a37cfa78a3614cd3f2\', \'v331_candidate_acceptable\': \'ecfa623379463705bcea5cc61d6232c4770e559772e3a7546e73f7945dffee42\', \'v331_registration_once\': \'d1269355e4a500d3184a83ba149ab065e36b057441716cf98c97bb699d772a96\'}}\n_SB_DRIVER_SHA256 = \'97c86bd56b677d81a01a96a9c764d9fd63e9a84d0888400ae9da3f6a3b016a57\'\n_SB_WORKER_SOURCE = \'\'\n_SB_CELL_VERSION = \'sv-mv-v331-complete-2-py311\'\n\ndef sb_jsonable(x):\n    if isinstance(x, dict): return {str(k): sb_jsonable(v) for k,v in x.items()}\n    if isinstance(x, (list,tuple)): return [sb_jsonable(v) for v in x]\n    if isinstance(x, Path): return str(x)\n    if isinstance(x, np.ndarray): return sb_jsonable(x.tolist())\n    if isinstance(x, np.generic): return sb_jsonable(x.item())\n    if isinstance(x, float) and not np.isfinite(x): return None\n    return x\n\ndef sb_write_json(path, value):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(sb_jsonable(value),ensure_ascii=False,indent=2,allow_nan=False))\n    tmp.replace(path)\n\ndef sb_digest(value):\n    return hashlib.sha256(json.dumps(sb_jsonable(value),sort_keys=True).encode()).hexdigest()\n\ndef sb_file_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef sb_cache_content_hash(cache):\n    # Tensor content, not torch.save archive bytes; serialization IDs can change.\n    h=hashlib.sha256()\n    keys=[\'xyz\',\'gaussian_index\',\'descriptor_multi_view\',\'descriptor_single_best_view\',\'stability\',\n          \'observation_count\',\'depth_residual_mean\',\'opacity\',\'score\',\'scale_sorted\',\'rotation_axes_sorted\',\'rotation_axis_valid\']\n    for key in keys:\n        if key not in cache: continue\n        value=cache[key]\n        if hasattr(value,\'detach\'): value=value.detach().cpu().numpy()\n        value=np.ascontiguousarray(value)\n        h.update(key.encode()); h.update(str(value.dtype).encode()); h.update(str(value.shape).encode())\n        h.update(value.tobytes())\n    return h.hexdigest()\n\ndef sb_pose(x):\n    if hasattr(x,\'detach\'): x=x.detach().cpu().numpy()\n    a=np.asarray(x,dtype=np.float64)\n    if a.shape!=(4,4) or not np.isfinite(a).all(): raise ValueError(\'Invalid SE(3) matrix\')\n    if not np.allclose(a[3],[0,0,0,1],atol=1e-5): raise ValueError(\'Invalid homogeneous row\')\n    if not np.allclose(a[:3,:3].T@a[:3,:3],np.eye(3),atol=1e-3) or np.linalg.det(a[:3,:3])<0:\n        raise ValueError(\'Invalid rigid rotation\')\n    return a\n\ndef sb_error(T, gt):\n    T,gt=sb_pose(T),sb_pose(gt)\n    angle=np.degrees(np.arccos(np.clip((np.trace(T[:3,:3].T@gt[:3,:3])-1)/2,-1,1)))\n    return float(angle),float(100*np.linalg.norm(T[:3,3]-gt[:3,3]))\n\ndef sb_loop_key(loop):\n    return tuple(int(getattr(loop,k)) for k in [\'source_agent_id\',\'source_frame_id\',\'target_agent_id\',\'target_frame_id\'])\n\ndef sb_residual(T,P,Q):\n    return np.linalg.norm(np.asarray(P)@T[:3,:3].T+T[:3,3]-np.asarray(Q),axis=1)\n\ndef sb_loop_to_start(T,cs0,ct0,csl,ctl):\n    return np.linalg.inv(ct0)@ctl@T@np.linalg.inv(csl)@cs0\n\ndef sb_start_to_loop(T,cs0,ct0,csl,ctl):\n    return np.linalg.inv(ctl)@ct0@T@np.linalg.inv(cs0)@csl\n\ndef sb_slice_cache(cache,k,view):\n    n=min(int(k),len(cache[\'xyz\'])); out={}\n    for name,val in cache.items():\n        if (hasattr(val,\'shape\') and len(val.shape)>0 and len(val)==len(cache[\'xyz\'])\n                and name not in {\'keyframe_ids\'}): out[name]=val[:n]\n        else: out[name]=val\n    key={\'single_best_view\':\'descriptor_single_best_view\',\'multi_view\':\'descriptor_multi_view\'}[view]\n    out[\'descriptor_multi_view\']=out[key]  # private alias, all frozen paths see the chosen variant\n    out[\'benchmark_variant\']=view\n    return out\n\ndef sb_quaternion_rotation(q,order=\'wxyz\'):\n    q=np.asarray(q,dtype=np.float64)\n    if order==\'xyzw\': q=q[:,[3,0,1,2]]\n    elif order!=\'wxyz\': raise ValueError(order)\n    q=q/np.maximum(np.linalg.norm(q,axis=1,keepdims=True),1e-12)\n    w,x,y,z=q.T\n    R=np.stack([1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y),\n                2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x),\n                2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)],axis=1).reshape(-1,3,3)\n    return R\n\ndef sb_quaternion_axes(q,scales,order=\'wxyz\'):\n    R=sb_quaternion_rotation(q,order)\n    axes=R[np.arange(len(R)),:,np.argmax(scales,axis=1)]\n    return axes/np.maximum(np.linalg.norm(axes,axis=1,keepdims=True),1e-12)\n\ndef sb_shape_from_checkpoint(sub,indices):\n    # Exact raw-field convention already used by the recent Gaussian-glyph export.\n    def arr(v): return v.detach().cpu().numpy() if hasattr(v,\'detach\') else np.asarray(v)\n    params=sub[\'gaussian_model_params\']; idx=arr(indices).astype(np.int64)\n    scales=np.exp(arr(params[\'scaling\'])[idx].astype(np.float64))\n    quat=arr(params[\'rotation\'])[idx].astype(np.float64)\n    if scales.shape!=(len(idx),3) or quat.shape!=(len(idx),4) or not np.isfinite(scales).all() or not np.isfinite(quat).all() or not np.all(scales>0) or np.any(np.linalg.norm(quat,axis=1)<1e-12):\n        raise ValueError(\'Invalid Gaussian scaling/rotation fields\')\n    sorted_scale=np.sort(scales,axis=1)\n    R=sb_quaternion_rotation(quat,BENCH_QUATERNION_ORDER)\n    order=np.argsort(-scales,axis=1)\n    axes=np.take_along_axis(R,order[:,None,:],axis=2).astype(np.float32)\n    desc_scale=np.take_along_axis(scales,order,axis=1)\n    gap01=desc_scale[:,0]/desc_scale[:,1]>=BENCH_AXIS_ANISOTROPY_MIN\n    gap12=desc_scale[:,1]/desc_scale[:,2]>=BENCH_AXIS_ANISOTROPY_MIN\n    return {\'scale_sorted\':sorted_scale,\'rotation_axes_sorted\':axes,\n        \'rotation_axis_valid\':np.column_stack([gap01,gap01&gap12,gap12]),\n        \'principal_axis\':axes[:,:,0],\'axis_valid\':gap01}\n\ndef sb_shape_support(pairs,P,Q,sc,tc,threshold):\n    pairs=np.asarray(pairs,dtype=np.int64)\n    if len(pairs)==0: return np.empty(0,dtype=np.int64)\n    i,j=pairs.T; pp=np.asarray(P)[i]; qq=np.asarray(Q)[j]\n    ss=np.asarray(sc[\'scale_sorted\'])[i]; ts=np.asarray(tc[\'scale_sorted\'])[j]\n    scale_ok=np.max(np.abs(np.log(np.maximum(ss,1e-12)/np.maximum(ts,1e-12))),axis=1)<=BENCH_SCALE_LOG_THRESHOLD\n    sa=np.asarray(sc[\'rotation_axes_sorted\'])[i];ta=np.asarray(tc[\'rotation_axes_sorted\'])[j]\n    usable=np.asarray(sc[\'rotation_axis_valid\'])[i]&np.asarray(tc[\'rotation_axis_valid\'])[j]\n    support=np.zeros(len(pairs),dtype=np.int64)\n    # Stream all pair relationships. abs(R_i^T R_j) is invariant under a common\n    # rigid rotation and independent Gaussian-axis sign flips; skip ambiguous axes.\n    for start in range(0,len(pairs),256):\n        stop=min(start+256,len(pairs));sl=slice(start,stop)\n        compatible=np.abs(np.linalg.norm(pp[sl,None]-pp[None,:],axis=2)-\n                          np.linalg.norm(qq[sl,None]-qq[None,:],axis=2))<=threshold\n        compatible &= scale_ok[sl,None]&scale_ok[None,:]\n        for a in range(3):\n            for b in range(3):\n                use=usable[sl,a,None]&usable[None,:,b]\n                ds=np.abs(sa[sl,:,a]@sa[:,:,b].T);dt=np.abs(ta[sl,:,a]@ta[:,:,b].T)\n                compatible &= (~use)|(np.abs(ds-dt)<=BENCH_AXIS_COS_THRESHOLD)\n        compatible &= (i[sl,None]!=i[None,:])&(j[sl,None]!=j[None,:])\n        support[sl]=compatible.sum(axis=1)\n    return support\n\ndef sb_specs():\n    specs=[]\n    for method in [\'A_mnn\',\'B_v32\',\'B_v331\']:\n        for view in [\'single_best_view\',\'multi_view\']:\n            specs.append(dict(method=method,view=view,k=BENCH_PRIMARY_K,margin=0.0,geometry=\'distance\'))\n    if BENCH_RUN_ABLATIONS:\n        for k in BENCH_K_VALUES:\n            if k==BENCH_PRIMARY_K: continue\n            for view in [\'single_best_view\',\'multi_view\']:\n                specs.append(dict(method=\'B_v331\',view=view,k=k,margin=0.0,geometry=\'distance\'))\n        specs.append(dict(method=\'A_mnn\',view=\'multi_view\',k=BENCH_PRIMARY_K,\n                          margin=BENCH_MARGIN_POSITIVE,geometry=\'distance\'))\n        specs.append(dict(method=\'B_v331\',view=\'multi_view\',k=BENCH_PRIMARY_K,\n                          margin=0.0,geometry=\'distance_scale_relative_axis\'))\n    if BENCH_RUN_FPFH:\n        specs.append(dict(method=\'FPFH_reference\',view=\'geometry\',k=0,margin=0.0,geometry=\'fpfh\'))\n    for s in specs:\n        s[\'id\']=f"{s[\'method\']}_{s[\'view\']}_K{s[\'k\']}_m{s[\'margin\']:g}_{s[\'geometry\']}"\n    return specs\n\nclass SBMonitor:\n    """PyTorch allocator peak, plus sampled device memory from nvidia-smi.\n    Device memory includes other processes; it is not mislabeled as process memory.\n    """\n    def __enter__(self):\n        self.samples=[]; self.stop=threading.Event(); self.torch_base=None\n        if sb_torch.cuda.is_available():\n            sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)\n            self.torch_base=sb_torch.cuda.memory_allocated(BENCH_GPU_DEVICE)/2**20\n            sb_torch.cuda.reset_peak_memory_stats(BENCH_GPU_DEVICE)\n            self.thread=threading.Thread(target=self.poll,daemon=True); self.thread.start()\n        return self\n    def poll(self):\n        while not self.stop.is_set():\n            try:\n                p=subprocess.run([\'nvidia-smi\',\'-i\',str(BENCH_GPU_DEVICE),\'--query-gpu=memory.used\',\n                    \'--format=csv,noheader,nounits\'],capture_output=True,text=True,timeout=2)\n                if p.returncode==0: self.samples.append(float(p.stdout.strip().splitlines()[0]))\n            except Exception: pass\n            self.stop.wait(BENCH_GPU_POLL_SECONDS)\n    def __exit__(self,*exc):\n        self.stop.set()\n        if hasattr(self,\'thread\'): self.thread.join(timeout=3)\n        if self.torch_base is not None:\n            sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)\n            peak=sb_torch.cuda.max_memory_allocated(BENCH_GPU_DEVICE)/2**20\n        else: peak=None\n        self.metrics={\'gpu_peak_torch_allocated_mb\':peak,\n            \'gpu_peak_torch_incremental_mb\':None if peak is None else max(0,peak-self.torch_base),\n            \'gpu_peak_device_used_mb\':max(self.samples) if self.samples else None,\n            \'gpu_device_samples\':len(self.samples),\'gpu_peak_device_is_sampled\':True}\n\ndef sb_sync():\n    if sb_torch.cuda.is_available(): sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)\n\nclass SBSubmaps:\n    def __init__(self,paths): self.paths=paths; self.memo=OrderedDict()\n    def __getitem__(self,i):\n        i=int(i)\n        if i not in self.memo:\n            self.memo[i]=sb_torch.load(self.paths[i],map_location=\'cpu\',weights_only=False)\n            if len(self.memo)>2: self.memo.popitem(last=False)\n        self.memo.move_to_end(i); return self.memo[i]\n    def __len__(self): return len(self.paths)\n\nclass SBContext:\n    def __init__(self,scene,root,output):\n        self.scene=scene; self.root=Path(root); self.output=Path(output); self.output.mkdir(parents=True,exist_ok=True)\n        self.cfg=sb_load_config(str(self.root/\'config.yaml\'))\n        self.data_root=Path(BENCH_DATA_ROOT_OVERRIDES.get(scene,self.cfg[\'data\'][\'input_path\']))\n        if not self.data_root.exists(): raise FileNotFoundError(f\'{scene}: RGB-D not found: {self.data_root}; set BENCH_DATA_ROOT_OVERRIDES\')\n        self.paths={}; self.submaps={}; self.headers={}; self.datasets={}; self.dense_memo=OrderedDict(); self.fpfh_memo=OrderedDict()\n        self.dense_rows={}; self.fpfh_rows={}\n        dirs=sorted(p for p in self.data_root.iterdir() if p.is_dir() and not p.name.startswith(\'.\'))\n        for aid in map(int,self.cfg[\'data\'][\'agent_ids\']):\n            exact=[p for p in dirs if p.name.lower() in {f\'agent_{aid}\',f\'agent{aid}\'}]\n            path=exact[0] if len(exact)==1 else (dirs[aid] if 0<=aid<len(dirs) else None)\n            if path is None: raise RuntimeError(f\'Cannot resolve agent {aid} input folder\')\n            dc={**self.cfg[\'data\'],\'input_path\':str(path),**self.cfg[\'cam\'],**self.cfg[\'submap\']}\n            self.datasets[aid]=sb_get_dataset(self.cfg[\'dataset_name\'])(dc)\n            paths=sorted((self.root/f\'agent_{aid}\'/\'submaps\').glob(\'*.ckpt\'))\n            if not paths: raise FileNotFoundError(f\'{scene}: missing finalized submaps for agent {aid}\')\n            self.paths[aid]=paths; self.submaps[aid]=SBSubmaps(paths); self.headers[aid]=[]\n            for p in paths:\n                sub=sb_torch.load(p,map_location=\'cpu\',weights_only=False)\n                self.headers[aid].append({k:sub[k] for k in [\'submap_start_frame_id\',\'submap_end_frame_id\',\'submap_c2ws\',\'keyframe_ids\']})\n                del sub\n        candidate=self.root/\'detected_loops.pkl\'\n        if not candidate.exists(): raise FileNotFoundError(\'Requires pre-registration detected_loops.pkl; loops.pkl is not a substitute\')\n        with candidate.open(\'rb\') as f: all_loops=pickle.load(f)\n        self.loops=[l for l in all_loops if int(l.source_agent_id)!=int(l.target_agent_id)]\n        keys=[sb_loop_key(l) for l in self.loops]\n        if len(set(keys))!=len(keys): raise RuntimeError(\'Duplicate candidate keys; resolve duplicates before paired evaluation\')\n        self.admitted=set()\n        if (self.root/\'filtered_loops.pkl\').exists():\n            with (self.root/\'filtered_loops.pkl\').open(\'rb\') as f:\n                self.admitted={sb_loop_key(l) for l in pickle.load(f) if int(l.source_agent_id)!=int(l.target_agent_id)}\n        if BENCH_SCOPE==\'paper_common\': self.loops=[l for l in self.loops if sb_loop_key(l) in self.admitted]\n        elif BENCH_SCOPE!=\'detected\': raise ValueError(BENCH_SCOPE)\n        if not self.loops: raise RuntimeError(\'No inter-agent candidates in selected fixed scope\')\n        self.required=set()\n        for l in self.loops:\n            sa,sf,ta,tf=sb_loop_key(l); self.required.update([(sa,self.index(sa,sf)),(ta,self.index(ta,tf))])\n        fingerprints=[]\n        for aid,sid in sorted(self.required):\n            p=self.paths[aid][sid]; st=p.stat()\n            fingerprints.append({\'agent\':aid,\'submap\':sid,\'name\':p.name,\'bytes\':st.st_size,\'mtime_ns\':st.st_mtime_ns,\n                 \'sha256\':sb_file_hash(p) if BENCH_STRICT_INPUT_HASH else None})\n        self.input_fingerprint={\'config_sha256\':sb_file_hash(self.root/\'config.yaml\'),\n            \'candidates_sha256\':sb_file_hash(candidate),\'submaps\':fingerprints,\'data_root\':str(self.data_root),\n            \'baseline_admission_sha256\':sb_file_hash(self.root/\'filtered_loops.pkl\') if (self.root/\'filtered_loops.pkl\').exists() else None,\n            \'loop_keys\':[sb_loop_key(l) for l in self.loops]}\n        # RGB-D paths/size/mtime invalidate a cached descriptor if mounted inputs change.\n        self.input_fingerprint[\'dataset_inventory\']=sb_digest([\n            (str(p.relative_to(self.data_root)),p.stat().st_size,p.stat().st_mtime_ns)\n            for p in sorted(self.data_root.rglob(\'*\')) if p.is_file()])\n        self.input_hash=sb_digest(self.input_fingerprint)\n        self.caches={}; self.cache_rows=[]; self.ns=None\n    def index(self,aid,frame):\n        for i,h in enumerate(self.headers[int(aid)]):\n            if int(h[\'submap_start_frame_id\'])<=int(frame)<=int(h[\'submap_end_frame_id\']): return i\n        raise KeyError((aid,frame))\n    def c2w(self,aid,sid,frame):\n        h=self.headers[int(aid)][int(sid)]; off=int(frame)-int(h[\'submap_start_frame_id\'])\n        if off<0 or off>=len(h[\'submap_c2ws\']): raise IndexError((aid,sid,frame))\n        return sb_pose(h[\'submap_c2ws\'][off])\n    def frame_info(self,loop):\n        sa,sf,ta,tf=sb_loop_key(loop); si,ti=self.index(sa,sf),self.index(ta,tf)\n        cs=self.c2w(sa,si,sf); ct=self.c2w(ta,ti,tf)\n        cs0=self.c2w(sa,si,int(self.headers[sa][si][\'submap_start_frame_id\']))\n        ct0=self.c2w(ta,ti,int(self.headers[ta][ti][\'submap_start_frame_id\']))\n        return sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0\n    def gt(self,loop):\n        sa,sf,ta,tf=sb_loop_key(loop)\n        def get(a,f):\n            d=self.datasets[a]\n            # Match recent mv_gt_pose exactly; dataset __getitem__ may normalize poses.\n            return sb_pose(d[f][-1])\n        return np.linalg.inv(get(ta,tf))@get(sa,sf)\n    def dense(self,aid,sid):\n        key=(aid,sid)\n        if key in self.dense_memo: self.dense_memo.move_to_end(key); return self.dense_memo[key]\n        path=self.output/\'dense_cache\'/f\'agent{aid}_submap{sid:06d}.npz\'\n        t=time.perf_counter(); hit=path.exists()\n        if hit:\n            with np.load(path) as z:\n                cloud=sb_o3d.geometry.PointCloud()\n                cloud.points=sb_o3d.utility.Vector3dVector(z[\'points\'])\n                cloud.normals=sb_o3d.utility.Vector3dVector(z[\'normals\'])\n                if \'colors\' in z: cloud.colors=sb_o3d.utility.Vector3dVector(z[\'colors\'])\n        else:\n            with SBMonitor() as monitor:\n                sub=self.submaps[aid][sid]\n                rgb,depth=sb_repo_utils.get_rgbd(sub)\n                cloud=sb_repo_utils.rgbd2ptcloud(rgb,depth,sub[\'intrinsics\'],np.eye(4))\n                cloud.estimate_normals(); sb_sync()\n            path.parent.mkdir(parents=True,exist_ok=True)\n            arrays={\'points\':np.asarray(cloud.points),\'normals\':np.asarray(cloud.normals)}\n            if cloud.has_colors(): arrays[\'colors\']=np.asarray(cloud.colors)\n            tmp=path.with_suffix(\'.tmp.npz\'); np.savez_compressed(tmp,**arrays); tmp.replace(path)\n            sb_write_json(path.with_suffix(\'.json\'),dict(agent=aid,submap=sid,points=len(cloud.points),\n                build_time_s=time.perf_counter()-t,**monitor.metrics))\n        info=json.loads(path.with_suffix(\'.json\').read_text()) if path.with_suffix(\'.json\').exists() else {}\n        info={**info,\'cache_hit\':hit,\'access_time_s\':time.perf_counter()-t}\n        self.dense_rows[key]={\'scene\':self.scene,**info,\'agent\':aid,\'submap\':sid}\n        pd.DataFrame(self.dense_rows.values()).to_csv(self.output/\'dense_runtime.csv\',index=False)\n        self.dense_memo[key]=(cloud,info)\n        if len(self.dense_memo)>2: self.dense_memo.popitem(last=False)\n        if len(cloud.points)<3 or not np.isfinite(np.asarray(cloud.points)).all() or not np.isfinite(np.asarray(cloud.normals)).all(): raise RuntimeError(\'Invalid dense cloud\')\n        return cloud,info\n\ndef sb_namespace(ctx):\n    cfg=ctx.cfg; opts=cfg[\'submap\'].get(\'registration_options\',{})\n    ns=dict(np=np,pd=pd,torch=sb_torch,F=sb_F,o3d=sb_o3d,time=time,math=math,json=json,\n        hashlib=hashlib,Path=Path,Image=sb_Image,repo_utils=sb_repo_utils,\n        load_3dgs=sb_repo_utils.load_3dgs,_rigidity_compatibility_mask=sb_repo_utils._rigidity_compatibility_mask,\n        _ransac_from_correspondences=sb_repo_utils._ransac_from_correspondences,\n        mv_config=cfg,mv_datasets=ctx.datasets,mv_submap_paths=ctx.paths,mv_submaps=ctx.submaps,\n        B_DEVICE=f\'cuda:{BENCH_GPU_DEVICE}\' if sb_torch.cuda.is_available() else \'cpu\',\n        MV_DEVICE=f\'cuda:{BENCH_GPU_DEVICE}\' if sb_torch.cuda.is_available() else \'cpu\',\n        MV_MIN_SIMILARITY=float(opts.get(\'dino_min_similarity\',-1)),MV_MIN_MARGIN=0.0,\n        MV_COMPATIBILITY_THRESHOLD_M=float(opts.get(\'dino_compatibility_threshold_m\',0.03)),\n        MV_MIN_COMPATIBILITY_SUPPORT=int(opts.get(\'dino_min_compatibility_support\',16)),\n        MV_RANSAC_DISTANCE_THRESHOLD_M=0.05,A_SEEDS=list(BENCH_SEEDS),\n        A_MIN_SIMILARITY=float(opts.get(\'dino_min_similarity\',-1)),A_MIN_MARGIN=0.0,\n        A_COMPATIBILITY_THRESHOLD_M=float(opts.get(\'dino_compatibility_threshold_m\',0.03)),\n        A_MIN_COMPATIBILITY_SUPPORT=int(opts.get(\'dino_min_compatibility_support\',16)),\n        A_RANSAC_DISTANCE_THRESHOLD_M=0.05,A_MIN_CORRESPONDENCES=int(opts.get(\'min_correspondences\',8)),\n        MV_DEPTH_TOLERANCE_M=float(opts.get(\'gaussian_depth_tolerance_m\',0.05)),\n        MV_MIN_OPACITY=float(opts.get(\'gaussian_min_opacity\',0.05)),MV_MIN_OBSERVATIONS=2,\n        MV_VOXEL_SIZE_M=0.03,MV_REBUILD_CACHE=False,MV_MODEL_NAME=cfg[\'loop_detection\'].get(\'weights_path\',\'facebook/dinov2-small\'),\n        MV_CACHE_ROOT=ctx.output/\'descriptor_cache\',MV_CACHE_VERSION=\'shared-master-\'+_SB_CELL_VERSION,\n        MV_MAX_LANDMARKS=max([BENCH_PRIMARY_K]+(BENCH_K_VALUES if BENCH_RUN_ABLATIONS else [])))\n    exec(compile(_SB_HELPERS,\'frozen_recent_notebook_helpers\',\'exec\'),ns)\n    ns[\'mv_find_submap_index\']=ctx.index\n    return ns\n\ndef sb_cache_compatible(cache,ctx,aid,sid):\n    need=[\'xyz\',\'gaussian_index\',\'descriptor_multi_view\',\'descriptor_single_best_view\',\'stability\',\n          \'observation_count\',\'depth_residual_mean\',\'opacity\',\'score\']\n    if any(k not in cache for k in need): return False\n    if cache.get(\'version\') not in {\'notebook-office0-ablation-v4\',\'heldout-frozen-mv-v331-v1\',ctx.ns[\'MV_CACHE_VERSION\']}: return False\n    n=len(cache[\'xyz\'])\n    if any(len(cache[k])!=n for k in need): return False\n    if n==0 or np.any(np.asarray(cache[\'observation_count\'])<2): return False\n    for key in [\'xyz\',\'descriptor_multi_view\',\'descriptor_single_best_view\']:\n        arr=np.asarray(cache[key])\n        if arr.ndim!=2 or not np.isfinite(arr).all(): return False\n    if cache[\'descriptor_multi_view\'].shape!=cache[\'descriptor_single_best_view\'].shape: return False\n    if int(cache.get(\'agent_id\',aid))!=aid or int(cache.get(\'submap_index\',sid))!=sid: return False\n    # Existing caches may have no provenance. Compare each selected center to this checkpoint.\n    sub=ctx.submaps[aid][sid]\n    if \'keyframe_ids\' not in cache or not np.array_equal(np.asarray(cache[\'keyframe_ids\']),np.asarray(sub[\'keyframe_ids\'])): return False\n    model,xyz,opacity=ctx.ns[\'mv_restore_gaussians\'](sub)\n    idx=cache[\'gaussian_index\'].long().to(xyz.device)\n    valid=bool(len(idx) and int(idx.max())<len(xyz) and int(idx.min())>=0)\n    if valid:\n        cur=xyz[idx].detach().cpu().numpy()\n        old=np.asarray(cache[\'xyz\'],dtype=np.float64)\n        # float16 center storage: compare against the same precision, not a loose metric gate.\n        valid=np.array_equal(cur.astype(old.dtype).astype(np.float16),old.astype(np.float16))\n        valid=valid and np.array_equal(opacity[idx].detach().cpu().numpy().astype(np.float16),np.asarray(cache[\'opacity\']).astype(np.float16))\n    del model,xyz,opacity,idx; gc.collect()\n    return valid\n\ndef sb_prepare_caches(ctx):\n    ns=ctx.ns; target=ns[\'MV_MAX_LANDMARKS\']; rows=[]\n    for aid,sid in sorted(ctx.required):\n        dest=ns[\'MV_CACHE_ROOT\']/f\'agent_{aid}\'/f\'{sid:06d}.pt\'\n        candidates=[dest]\n        for d in sorted(ctx.root.glob(\'notebook*cache*\')):\n            candidates.extend([d/f\'agent_{aid}\'/f\'{sid:06d}.pt\',d/f\'agent_{aid}\'/f\'{sid:03d}.pt\'])\n        chosen=None\n        for p in candidates:\n            if not p.exists(): continue\n            cache=sb_torch.load(p,map_location=\'cpu\',weights_only=False)\n            if not sb_cache_compatible(cache,ctx,aid,sid): continue\n            enough=len(cache[\'xyz\'])>=target or int(cache.get(\'num_gaussians_multiview\',10**12))<=len(cache[\'xyz\'])\n            if enough:\n                chosen=(p,cache); break\n        measured={}; built=False; changed=False\n        if chosen:\n            p,cache=chosen; hit=True\n            # Reused times are historical; allocator peaks are not invented.\n            measured={\'cache_build_time_s_measured\':None,\'cache_build_time_s_historical\':cache.get(\'build_time_s\')}\n        else:\n            if not BENCH_BUILD_MISSING_CACHE: raise RuntimeError(f\'Missing validated K={target} cache for {aid}/{sid}; enable BENCH_BUILD_MISSING_CACHE\')\n            if \'mv_feature_extractor\' not in ns:\n                from src.entities.loop_detection.feature_extractors import get_patch_feature_extractor\n                t=time.perf_counter()\n                ns[\'mv_feature_extractor\']=get_patch_feature_extractor(ctx.cfg[\'loop_detection\'],\n                    {\'registration_weights_path\':ns[\'MV_MODEL_NAME\'],\'registration_device\':ns[\'MV_DEVICE\']},existing_extractor=None)\n                ns[\'MV_EMBED_DIM\']=int(ns[\'mv_feature_extractor\'].embed_size)\n                sb_sync(); ctx.model_init_s=time.perf_counter()-t\n            calls=defaultdict(float); originals={}\n            for name,category in [(\'mv_extract_feature_map\',\'dino_feature_s\'),(\'mv_bilinear_sample\',\'bilinear_sample_s\')]:\n                original=ns[name]\n                originals[name]=original\n                def wrapped(*a,_fn=original,_cat=category,**kw):\n                    sb_sync(); t=time.perf_counter(); ret=_fn(*a,**kw); sb_sync()\n                    calls[_cat]+=time.perf_counter()-t; return ret\n                ns[name]=wrapped\n            ns[\'MV_REBUILD_CACHE\']=True\n            try:\n                with SBMonitor() as monitor:\n                    sb_sync(); t=time.perf_counter()\n                    with contextlib.redirect_stdout(io.StringIO()): cache=ns[\'mv_build_cache\'](aid,sid)\n                    sb_sync(); elapsed=time.perf_counter()-t\n            finally:\n                ns[\'MV_REBUILD_CACHE\']=False\n                ns.update(originals)\n            measured={**monitor.metrics,**calls,\'cache_build_time_s_measured\':elapsed,\n                      \'cache_build_time_s_historical\':cache.get(\'build_time_s\')}\n            cache[\'sb_build_profile\']=measured\n            p=dest; hit=False; built=True; changed=True\n        if BENCH_RUN_ABLATIONS and not {\'scale_sorted\',\'rotation_axes_sorted\',\'rotation_axis_valid\'}.issubset(cache):\n            # An unsupported shape API must not block the main single/multi experiment.\n            try:\n                cache={**cache,**sb_shape_from_checkpoint(ctx.submaps[aid][sid],cache[\'gaussian_index\'])}\n                cache.pop(\'shape_metadata_error\',None)\n            except Exception as e:\n                cache=dict(cache); cache[\'shape_metadata_error\']=f\'{type(e).__name__}: {e}\'\n            changed=True\n        # Save validated private cache; never modify the notebook\'s original cache.\n        dest.parent.mkdir(parents=True,exist_ok=True)\n        if changed or p!=dest or not dest.exists():\n            tmp=dest.with_suffix(\'.tmp.pt\'); sb_torch.save(cache,tmp); tmp.replace(dest)\n        ctx.caches[(aid,sid)]=cache\n        prof=cache.get(\'sb_build_profile\',{}) if hit else measured\n        rows.append(dict(scene=ctx.scene,agent=aid,submap=sid,landmarks=len(cache[\'xyz\']),\n             eligible=cache.get(\'num_gaussians_multiview\'),views=cache.get(\'num_views_used\'),\n             cache_hit=hit,cache_built_this_run=built,cache_sha256=sb_file_hash(dest),cache_content_sha256=sb_cache_content_hash(cache),\n             source_cache_path=str(p),source_cache_version=cache.get(\'version\'),\n             gpu_profile_is_historical=hit,\n             current_access_measured=not hit,construction_measurement=\'shared_builder_stores_both_variants\',\n             **{**prof,**measured}))\n    ctx.cache_rows=rows\n    pd.DataFrame(rows).to_csv(ctx.output/\'descriptor_runtime.csv\',index=False)\n    return sb_digest([r[\'cache_content_sha256\'] for r in rows])\n\ndef sb_profile_namespace(ns):\n    totals=defaultdict(float); active=defaultdict(int)\n    groups={\n      \'matching_s\':[\'a_match_descriptors\',\'v2_build_adaptive_bidirectional\',\'v32_build_hard_candidates\'],\n      \'geometry_s\':[\'a_geometric_compatibility\',\'mv_topk_rigidity_support\'],\n      \'ransac_s\':[\'a_ransac\',\'v2_prosac_ransac\',\'v3_prosac_ransac\'],\n      \'local_refinement_s\':[\'v32_robust_local_raw_candidates\',\'v32_select_local_candidate\',\'v33_irls_full\',\n          \'v33_irls_translation\',\'v33_irls_rotation\',\'v331_validation_eval\',\'v331_candidate_acceptable\']}\n    for category,names in groups.items():\n        for name in names:\n            original=ns[name]\n            def wrapped(*a,_fn=original,_cat=category,**kw):\n                outer=active[_cat]==0\n                if outer: sb_sync(); t=time.perf_counter()\n                active[_cat]+=1\n                try: return _fn(*a,**kw)\n                finally:\n                    active[_cat]-=1\n                    if outer: sb_sync(); totals[_cat]+=time.perf_counter()-t\n            ns[name]=wrapped\n    return totals\n\ndef sb_fpfh(ctx,aid,sid):\n    key=(aid,sid)\n    if key in ctx.fpfh_memo:\n        ctx.fpfh_memo.move_to_end(key)\n        down,feat,info=ctx.fpfh_memo[key]\n        return down,feat,{**info,\'cache_hit\':True}\n    cloud,_=ctx.dense(aid,sid); path=ctx.output/\'fpfh_cache\'/f\'a{aid}_s{sid:06d}.npz\'\n    if path.exists():\n        with np.load(path) as z:\n            down=sb_o3d.geometry.PointCloud(); down.points=sb_o3d.utility.Vector3dVector(z[\'points\'])\n            down.normals=sb_o3d.utility.Vector3dVector(z[\'normals\'])\n            feat=sb_o3d.pipelines.registration.Feature(); feat.data=z[\'features\']\n        info=json.loads(path.with_suffix(\'.json\').read_text())\n        info={**info,\'cache_hit\':True}\n    else:\n        with SBMonitor() as monitor:\n            t=time.perf_counter(); down=cloud.voxel_down_sample(BENCH_FPFH_VOXEL_M)\n            down.estimate_normals(sb_o3d.geometry.KDTreeSearchParamHybrid(radius=2*BENCH_FPFH_VOXEL_M,max_nn=30))\n            feat=sb_o3d.pipelines.registration.compute_fpfh_feature(down,\n                sb_o3d.geometry.KDTreeSearchParamHybrid(radius=5*BENCH_FPFH_VOXEL_M,max_nn=100))\n            elapsed=time.perf_counter()-t\n        info={\'feature_build_time_s\':elapsed,\'points\':len(down.points),\'cache_hit\':False,**monitor.metrics}\n        path.parent.mkdir(parents=True,exist_ok=True)\n        np.savez_compressed(path,points=np.asarray(down.points),normals=np.asarray(down.normals),features=np.asarray(feat.data))\n        sb_write_json(path.with_suffix(\'.json\'),info)\n    ctx.fpfh_rows[key]={\'scene\':ctx.scene,\'agent\':aid,\'submap\':sid,**info}\n    pd.DataFrame(ctx.fpfh_rows.values()).to_csv(ctx.output/\'fpfh_runtime.csv\',index=False)\n    ctx.fpfh_memo[key]=(down,feat,info)\n    if len(ctx.fpfh_memo)>2: ctx.fpfh_memo.popitem(last=False)\n    return down,feat,info\n\ndef sb_fpfh_register(ctx,info,seed):\n    sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=info\n    s,fs,is_=sb_fpfh(ctx,sa,si); t,ft,it_=sb_fpfh(ctx,ta,ti)\n    sb_o3d.utility.random.seed(int(seed)); reg=sb_o3d.pipelines.registration\n    dist=1.5*BENCH_FPFH_VOXEL_M\n    t0=time.perf_counter()\n    res=reg.registration_ransac_based_on_feature_matching(s,t,fs,ft,True,dist,\n        reg.TransformationEstimationPointToPoint(False),4,\n        [reg.CorrespondenceCheckerBasedOnEdgeLength(0.9),reg.CorrespondenceCheckerBasedOnDistance(dist)],\n        reg.RANSACConvergenceCriteria(BENCH_FPFH_MAX_ITERS,BENCH_FPFH_CONFIDENCE))\n    elapsed=time.perf_counter()-t0\n    T0=sb_pose(res.transformation)\n    good=len(res.correspondence_set)>=3\n    return dict(T_est=sb_start_to_loop(T0,cs0,ct0,cs,ct) if good else None,\n        n_ransac_inliers=len(res.correspondence_set),fpfh_ransac_fitness=float(res.fitness),\n        fpfh_ransac_rmse_cm=100*float(res.inlier_rmse),matcher_runtime_s=elapsed,\n        fpfh_source_points=len(s.points),fpfh_target_points=len(t.points),\n        fpfh_feature_build_s_historical=is_[\'feature_build_time_s\']+it_[\'feature_build_time_s\'],\n        fpfh_feature_build_s_new=sum(x[\'feature_build_time_s\'] for x in [is_,it_] if not x[\'cache_hit\']))\n\ndef sb_audit_pairs(sc,tc,cs,ct,T,gt):\n    src=np.array(sc[\'descriptor_multi_view\'],dtype=np.float64,copy=True); tgt=np.array(tc[\'descriptor_multi_view\'],dtype=np.float64,copy=True)\n    src/=np.maximum(np.linalg.norm(src,axis=1,keepdims=True),1e-12)\n    tgt/=np.maximum(np.linalg.norm(tgt,axis=1,keepdims=True),1e-12)\n    sim=src@tgt.T; j=sim.argmax(1); i=np.arange(len(src)); keep=sim.argmax(0)[j]==i; i=i[keep];j=j[keep]\n    P=np.asarray(sc[\'xyz\'],dtype=np.float64); Q=np.asarray(tc[\'xyz\'],dtype=np.float64)\n    P=(P-cs[:3,3])@cs[:3,:3]\n    Q=(Q-ct[:3,3])@ct[:3,:3]\n    out={\'audit_mutual_pairs\':len(i),\'audit_mean_similarity\':float(sim[i,j].mean()) if len(i) else None}\n    for name,pose in [(\'estimated\',T),(\'gt\',gt)]:\n        if pose is None or not len(i): continue\n        r=sb_residual(pose,P[i],Q[j]); out.update({f\'audit_{name}_paired_ratio_5cm\':float((r<=0.05).mean()),\n            f\'audit_{name}_paired_ratio_3cm\':float((r<=0.03).mean()),f\'audit_{name}_paired_rmse_cm\':100*float(np.sqrt(np.mean(r*r)))})\n    return out\n\ndef sb_success(row,prefix):\n    if not row.get(\'gt_available\',False):\n        for label in [\'5deg_10cm\',\'1deg_2cm\',\'05deg_1cm\']: row[f\'{prefix}_success_{label}\']=None\n        return\n    r,t=row.get(prefix+\'_RRE_deg\'),row.get(prefix+\'_RTE_cm\')\n    for label,rr,tt in [(\'5deg_10cm\',5,10),(\'1deg_2cm\',1,2),(\'05deg_1cm\',0.5,1)]:\n        row[f\'{prefix}_success_{label}\']=bool(r is not None and t is not None and r<rr and t<tt)\n\ndef sb_attempt(ctx,spec,loop_id,loop,seed,run_hash):\n    f=ctx.frame_info(loop); sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=f\n    row={\'scene\':ctx.scene,\'scope\':BENCH_SCOPE,\'candidate_id\':loop_id,\'seed\':seed,**spec,\n         \'source_agent\':sa,\'source_frame\':sf,\'target_agent\':ta,\'target_frame\':tf,\'source_submap\':si,\'target_submap\':ti,\n         \'baseline_admitted\':sb_loop_key(loop) in ctx.admitted,\'run_hash\':run_hash,\n         \'status\':\'algorithm_failed\',\'coarse_valid\':False,\'icp_evaluated\':seed in BENCH_ICP_SEEDS,\n         \'final_valid\':False,\'filter_pass\':False,\'gt_available\':False,\'T_coarse\':None,\'T_final\':None}\n    sc=tc=None; totals={}; T=None\n    try:\n        if spec[\'method\']==\'FPFH_reference\':\n            # Feature/cloud preparation is one-time and excluded from query timer.\n            _,_,sp=sb_fpfh(ctx,sa,si); _,_,tp=sb_fpfh(ctx,ta,ti)\n            row[\'fpfh_preparation_new_s\']=sum(x[\'feature_build_time_s\'] for x in [sp,tp] if not x[\'cache_hit\'])\n        else:\n            ns=sb_namespace(ctx); ns[\'A_MIN_MARGIN\']=spec[\'margin\']\n            sc=sb_slice_cache(ctx.caches[(sa,si)],spec[\'k\'],spec[\'view\'])\n            tc=sb_slice_cache(ctx.caches[(ta,ti)],spec[\'k\'],spec[\'view\'])\n            row.update(n_source_landmarks=len(sc[\'xyz\']),n_target_landmarks=len(tc[\'xyz\']))\n            if spec[\'geometry\']==\'distance_scale_relative_axis\':\n                if any(\'shape_metadata_error\' in c for c in [sc,tc]):\n                    raise RuntimeError(\'Shape ablation unavailable: \'+str([c.get(\'shape_metadata_error\') for c in [sc,tc]]))\n                def shape_support(p,P,Q,threshold_m=0.03): return sb_shape_support(p,P,Q,sc,tc,threshold_m)\n                ns[\'mv_topk_rigidity_support\']=shape_support\n            totals=sb_profile_namespace(ns)\n        with SBMonitor() as monitor:\n            sb_sync(); start=time.perf_counter()\n            if spec[\'method\']==\'FPFH_reference\': out=sb_fpfh_register(ctx,f,seed)\n            else:\n                name={\'A_mnn\':\'a_registration_once\',\'B_v32\':\'v32_registration_once\',\'B_v331\':\'v331_registration_once\'}[spec[\'method\']]\n                out=ns[name](sc,tc,cs,ct,seed)\n            sb_sync(); elapsed=time.perf_counter()-start\n        reserved=set(row)|{\'matcher_runtime_s\',\'T_est\'}\n        row.update({k:sb_jsonable(v) for k,v in out.items() if k not in reserved and (v is None or isinstance(v,(str,bool,int,float,np.generic)))})\n        row[\'implementation_label\']=out.get(\'method\',spec[\'method\'])\n        for key,value in out.items():\n            if isinstance(value,np.ndarray) and value.shape==(4,4) and key!=\'T_est\': row[key]=sb_jsonable(value)\n        row.update(matcher_runtime_s=elapsed,**dict(totals),**{\'coarse_\'+k:v for k,v in monitor.metrics.items()})\n        if spec[\'method\']==\'FPFH_reference\': row[\'fpfh_matching_ransac_s\']=out[\'matcher_runtime_s\']\n        raw=out.get(\'T_est\'); T=None if raw is None else sb_pose(raw)\n        row[\'T_coarse\']=None if T is None else T.tolist(); row[\'coarse_valid\']=T is not None\n        if T is not None:\n            row[\'status\']=\'done\'\n            if seed in BENCH_ICP_SEEDS:\n                source,sinfo=ctx.dense(sa,si); target,tinfo=ctx.dense(ta,ti)\n                T0=sb_loop_to_start(T,cs0,ct0,cs,ct)\n                if not np.allclose(sb_start_to_loop(T0,cs0,ct0,cs,ct),T,atol=1e-8): raise AssertionError(\'ICP frame conversion round-trip failed\')\n                with SBMonitor() as im:\n                    res=ctx.ns[\'exact_production_icp\'](source,target,T0)\n                final=sb_pose(sb_start_to_loop(res[\'T_final\'],cs0,ct0,cs,ct))\n                row.update(T_final=final.tolist(),final_valid=True,icp_runtime_s=res[\'runtime_s\'],\n                    initial_fitness_5mm=res[\'initial_fitness_5mm\'],final_fitness_5mm=res[\'final_fitness_5mm\'],\n                    initial_rmse_cm=100*res[\'initial_rmse_5mm_m\'],final_rmse_cm=100*res[\'final_rmse_5mm_m\'],\n                    dense_source_points=len(source.points),dense_target_points=len(target.points),\n                    dense_build_s_historical=sinfo.get(\'build_time_s\',0)+tinfo.get(\'build_time_s\',0),\n                    **{\'icp_\'+k:v for k,v in im.metrics.items()})\n                thresholds=ctx.cfg[\'loop_detection\']\n                row[\'filter_pass\']=bool(res[\'final_fitness_5mm\']>float(thresholds[\'fitness_threshold\'])\n                       and np.isfinite(res[\'final_fitness_5mm\']) and np.isfinite(res[\'final_rmse_5mm_m\'])\n                       and res[\'final_rmse_5mm_m\']<float(thresholds[\'inlier_rmse_threshold\']))\n                row[\'online_runtime_s\']=elapsed+res[\'runtime_s\']\n    except Exception as e:\n        row[\'status\']=\'error\'; row[\'error\']=f\'{type(e).__name__}: {e}\'\n    # Ground truth is queried AFTER every decision, including failed ICP attempts.\n    # A valid coarse estimate remains evaluable if only ICP raises an exception.\n    try: gt=ctx.gt(loop); row[\'gt_available\']=True\n    except Exception as e: gt=None; row[\'gt_error\']=str(e)\n    if T is not None and gt is not None: row[\'coarse_RRE_deg\'],row[\'coarse_RTE_cm\']=sb_error(T,gt)\n    if row[\'final_valid\'] and gt is not None: row[\'final_RRE_deg\'],row[\'final_RTE_cm\']=sb_error(np.asarray(row[\'T_final\']),gt)\n    if sc is not None:\n        try: row.update(sb_audit_pairs(sc,tc,cs,ct,T,gt))\n        except Exception as e: row[\'audit_error\']=str(e)\n    sb_success(row,\'coarse\'); sb_success(row,\'final\')\n    return sb_jsonable(row)\n\ndef sb_bootstrap(values,seed=1701):\n    v=np.asarray(values,dtype=float);v=v[np.isfinite(v)]\n    if len(v)<2: return [None,None]\n    rng=np.random.default_rng(seed)\n    means=np.mean(rng.choice(v,size=(4000,len(v)),replace=True),axis=1)\n    return [float(x) for x in np.quantile(means,[0.025,0.975])]\n\ndef sb_summaries(df):\n    rows=[]; keys=[\'scene\',\'id\',\'method\',\'view\',\'k\',\'margin\',\'geometry\']\n    for key,d in df.groupby(keys,dropna=False):\n        base=dict(zip(keys,key)); base.update(attempts=len(d),unique_loops=d.candidate_id.nunique(),\n            coarse_valid=int(d.coarse_valid.fillna(False).sum()),errors=int((d.status==\'error\').sum()),\n            gt_evaluable=int(d.gt_available.fillna(False).sum()),\n            icp_attempts=int(d.icp_evaluated.fillna(False).sum()),\n            final_valid=int(d.final_valid.fillna(False).sum()),filter_pass=int(d.filter_pass.fillna(False).sum()))\n        for stage in [\'coarse\',\'final\']:\n            dd=d if stage==\'coarse\' else d[d.icp_evaluated.fillna(False)]\n            evaluable=dd[dd.gt_available.fillna(False)]\n            for label in [\'5deg_10cm\',\'1deg_2cm\',\'05deg_1cm\']:\n                col=f\'{stage}_success_{label}\'\n                base[col+\'_percent\']=100*evaluable[col].eq(True).mean() if len(evaluable) else np.nan\n            for metric in [\'RRE_deg\',\'RTE_cm\']:\n                col=f\'{stage}_{metric}\'\n                v=pd.to_numeric(dd[col],errors=\'coerce\').dropna() if col in dd else pd.Series(dtype=float)\n                for stat,fn in [(\'mean\',lambda x:x.mean()),(\'median\',lambda x:x.median()),(\'max\',lambda x:x.max()),(\'p90\',lambda x:x.quantile(.9))]:\n                    base[col+\'_\'+stat]=float(fn(v)) if len(v) else np.nan\n                loop_means=dd.assign(**{col:pd.to_numeric(dd[col],errors=\'coerce\')}).groupby(\'candidate_id\')[col].mean().dropna() if col in dd else pd.Series(dtype=float)\n                base[col+\'_std_loop_mean\']=float(loop_means.std()) if len(loop_means)>1 else np.nan\n        for col in [\'matching_s\',\'geometry_s\',\'ransac_s\',\'local_refinement_s\',\'matcher_runtime_s\',\'icp_runtime_s\',\'online_runtime_s\',\n                \'audit_estimated_paired_ratio_5cm\',\'audit_gt_paired_ratio_5cm\',\'selected_inlier_ratio_5cm\',\'selected_inlier_ratio_3cm\',\n                \'final_fitness_5mm\',\'final_rmse_cm\',\'selected_pairs\',\'n_ransac_inliers\',\'fallback_triggered\',\'fallback_selected\',\n                \'v331_used_holdout\',\'v331_rotation_allowed\',\'v331_validation_gain\',\'v331_candidate_count\',\'v331_accepted_count\',\n                \'fpfh_preparation_new_s\',\'fpfh_feature_build_s_historical\']:\n            if col in d: base[col+\'_mean\']=pd.to_numeric(d[col],errors=\'coerce\').mean()\n        for col in [c for c in d if \'gpu_peak_\' in c]: base[col+\'_max\']=pd.to_numeric(d[col],errors=\'coerce\').max()\n        if \'n_source_landmarks\' in d:\n            for col in [\'n_source_landmarks\',\'n_target_landmarks\']:\n                v=pd.to_numeric(d[col],errors=\'coerce\');base[col+\'_min\']=v.min();base[col+\'_mean\']=v.mean()\n            base[\'below_requested_K_attempts\']=int(((d.n_source_landmarks<base[\'k\'])|(d.n_target_landmarks<base[\'k\'])).sum())\n        rows.append(base)\n    return pd.DataFrame(rows)\n\ndef sb_pairwise(df):\n    pair_keys=[\'scene\',\'candidate_id\',\'seed\',\'method\',\'k\',\'margin\',\'geometry\']\n    s=df[df.view==\'single_best_view\']; m=df[df.view==\'multi_view\']\n    paired=s.merge(m,on=pair_keys,suffixes=(\'_single\',\'_multi\'),validate=\'one_to_one\')\n    report=[]\n    for (scene,method,k,margin,geometry),d in paired.groupby([\'scene\',\'method\',\'k\',\'margin\',\'geometry\']):\n        for stage in [\'coarse\',\'final\']:\n            eligible=d if stage==\'coarse\' else d[d.icp_evaluated_single&d.icp_evaluated_multi]\n            valid=eligible[eligible[f\'{stage}_valid_single\']&eligible[f\'{stage}_valid_multi\']&eligible.gt_available_single&eligible.gt_available_multi]\n            result=dict(scene=scene,method=method,k=k,margin=margin,geometry=geometry,stage=stage,\n                pairs=len(eligible),common_valid_pairs=len(valid),unique_loops=eligible.candidate_id.nunique())\n            for metric in [\'RRE_deg\',\'RTE_cm\']:\n                a=f\'{stage}_{metric}_single\';b=f\'{stage}_{metric}_multi\'\n                delta=pd.to_numeric(valid[b],errors=\'coerce\')-pd.to_numeric(valid[a],errors=\'coerce\')\n                paired.loc[valid.index,f\'{stage}_{metric}_delta_multi_minus_single\']=delta\n                grouped=delta.groupby(valid.candidate_id).mean().dropna()\n                lo,hi=sb_bootstrap(grouped)\n                result.update({metric+\'_delta_mean\':delta.mean(),metric+\'_delta_loop_mean\':grouped.mean(),metric+\'_win_pairs\':int((delta<0).sum()),\n                    metric+\'_delta_loop_ci_low\':lo,metric+\'_delta_loop_ci_high\':hi})\n            cc=f\'{stage}_success_5deg_10cm\'\n            evaluable=eligible[eligible.gt_available_single&eligible.gt_available_multi]\n            sv=evaluable[cc+\'_single\'].eq(True);mv=evaluable[cc+\'_multi\'].eq(True)\n            result.update(gt_evaluable_pairs=len(evaluable),single_only_success=int((sv&~mv).sum()),multi_only_success=int((mv&~sv).sum()),\n                both_success=int((sv&mv).sum()),both_failed=int((~sv&~mv).sum()))\n            report.append(result)\n    if not report:\n        return paired,pd.DataFrame(columns=[\'scene\',\'method\',\'k\',\'margin\',\'geometry\',\'stage\',\'pairs\',\'common_valid_pairs\'])\n    return paired,pd.DataFrame(report)\n\ndef sb_tex_table(folder,filename,frame,columns,caption,label):\n    def esc(x):\n        x={\'B_v331\':\'V3.3.1\',\'B_v32\':\'V3.2\',\'A_mnn\':\'MNN\',\'FPFH_reference\':\'FPFH\',\n           \'single_best_view\':\'Single\',\'multi_view\':\'Multi\',\'geometry\':\'Geom.\',\n           \'distance\':\'Dist.\',\'distance_scale_relative_axis\':\'Dist.+scale+axes\'}.get(str(x),x)\n        return str(x).replace(\'\\\\\',r\'\\textbackslash{}\').replace(\'_\',r\'\\_\').replace(\'%\',r\'\\%\').replace(\'&\',r\'\\&\').replace(\'#\',r\'\\#\')\n    lines=[r\'\\begin{table*}[t]\',r\'\\centering\',r\'\\caption{\'+caption+\'}\',\n        r\'\\resizebox{\\textwidth}{!}{%\',\n        r\'\\begin{tabular}{\'+\' \'.join(\'l\' for _ in columns)+\'}\',r\'\\hline\',\n        \' & \'.join(heading for _,heading in columns)+r\' \\\\\',r\'\\hline\']\n    for _,row in frame.iterrows():\n        values=[]\n        for field,_ in columns:\n            value=row.get(field)\n            if value is None or (not isinstance(value,(list,dict)) and pd.isna(value)): values.append(\'--\')\n            elif isinstance(value,(int,np.integer)): values.append(str(value))\n            elif isinstance(value,(float,np.floating)): values.append(f\'{float(value):.3f}\')\n            else: values.append(esc(value))\n        lines.append(\' & \'.join(values)+r\' \\\\\')\n    lines.extend([r\'\\hline\',r\'\\end{tabular}\',r\'}\',r\'\\label{\'+label+\'}\',r\'\\end{table*}\'])\n    (Path(folder)/filename).write_text(\'\\n\'.join(lines))\n\ndef sb_report(folder,df,coverage,cache_df):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    # Missing numeric fields are NaN; failures remain explicit rows.\n    for c in [\'coarse_RRE_deg\',\'coarse_RTE_cm\',\'final_RRE_deg\',\'final_RTE_cm\']:\n        if c not in df: df[c]=np.nan\n        df[c]=pd.to_numeric(df[c],errors=\'coerce\')\n    df.to_csv(folder/\'per_run.csv\',index=False)\n    summary=sb_summaries(df); summary.to_csv(folder/\'summary.csv\',index=False)\n    per_seed=pd.concat([sb_summaries(d).assign(seed=int(seed)) for seed,d in df.groupby(\'seed\')],ignore_index=True)\n    per_seed.to_csv(folder/\'per_seed_summary.csv\',index=False)\n    paired,ps=sb_pairwise(df); paired.to_csv(folder/\'paired_single_multi.csv\',index=False)\n    ps.to_csv(folder/\'paired_summary.csv\',index=False)\n    df.groupby([\'scene\',\'id\',\'candidate_id\'],dropna=False).agg(\n        runs=(\'seed\',\'size\'),coarse_RRE_deg=(\'coarse_RRE_deg\',\'mean\'),coarse_RTE_cm=(\'coarse_RTE_cm\',\'mean\'),\n        final_RRE_deg=(\'final_RRE_deg\',\'mean\'),final_RTE_cm=(\'final_RTE_cm\',\'mean\')).reset_index().to_csv(folder/\'per_loop.csv\',index=False)\n    df[[\'scene\',\'candidate_id\',\'source_agent\',\'source_frame\',\'target_agent\',\'target_frame\',\'source_submap\',\'target_submap\',\'baseline_admitted\']].drop_duplicates().to_csv(folder/\'loop_manifest.csv\',index=False)\n    pd.DataFrame(coverage).to_csv(folder/\'dataset_coverage.csv\',index=False)\n    cache_df.to_csv(folder/\'descriptor_runtime_all.csv\',index=False)\n    hard=df[df.get(\'fallback_triggered\',pd.Series(False,index=df.index)).fillna(False).astype(bool)]\n    hard.to_csv(folder/\'hard_cases.csv\',index=False)\n    # Conditional paper subset: same baseline admission keys for every method.\n    conditional=df[df.baseline_admitted.fillna(False)]\n    if len(conditional): sb_summaries(conditional).to_csv(folder/\'paper_common_summary.csv\',index=False)\n    primary=summary[(summary.method==\'B_v331\')&(summary.k==BENCH_PRIMARY_K)&\n                    (summary.geometry==\'distance\')&(summary.margin==0)]\n    columns=[\'scene\',\'view\',\'unique_loops\',\'attempts\',\'coarse_valid\',\'coarse_success_5deg_10cm_percent\',\n        \'coarse_RRE_deg_mean\',\'coarse_RTE_cm_mean\',\'final_RRE_deg_mean\',\'final_RTE_cm_mean\',\n        \'final_success_5deg_10cm_percent\',\'filter_pass\',\'matcher_runtime_s_mean\',\'icp_runtime_s_mean\',\'final_fitness_5mm_mean\',\'final_rmse_cm_mean\']\n    primary[[c for c in columns if c in primary]].to_csv(folder/\'paper_single_multi.csv\',index=False)\n    text=[\'KẾT QUẢ BENCHMARK SINGLE/MULTI — \'+_SB_CELL_VERSION,\n        \'Sai số tịnh tiến/RMSE: cm. RRE: độ. Thời gian: giây. GPU: MiB.\',\n        \'Mean/median sai số tính trên pose hợp lệ có GT; success rate giữ các cặp thất bại trong mẫu số. Error rows được báo riêng.\',\n        \'Các seed RANSAC dùng chung một lần chạy SLAM. Bootstrap lấy mẫu theo loop đã gộp seed.\',\n        \'Paired ratio theo pose ước lượng và theo GT được xuất riêng; không đồng nhất hai loại chỉ số.\',\n        \'Dense ICP: point-to-plane 5 mm, tối đa 500 vòng lặp; giữ nguyên helper notebook gần đây.\',\n        \'paper_common_summary đánh giá có điều kiện trên tập baseline đã nhận; không phải detection recall.\',\n        \'Ablation hình học mới dùng scale và quan hệ giữa các hệ trục Gaussian, bỏ các trục có scale gần nhau; không so sánh quaternion thô giữa hai map.\',\n        \'PSNR/SSIM/LPIPS/ATE giữ lại từ SLAM gốc chỉ là thông tin nền; chưa chạy lại PGO/render cho từng descriptor.\',\n        \'Device GPU peak là số đo lấy mẫu và bao gồm tiến trình khác; peak PyTorch allocator được ghi riêng.\',\'\']\n    for _,r in ps[(ps.method==\'B_v331\')&(ps.k==BENCH_PRIMARY_K)&(ps.geometry==\'distance\')].iterrows():\n        text.append(f"{r.scene} / {\'trước ICP\' if r.stage==\'coarse\' else \'sau ICP\'}: cả hai có pose hợp lệ {int(r.common_valid_pairs)}/{int(r.pairs)}; "\n            f"delta Multi-Single RRE={r.RRE_deg_delta_mean:.6f} độ, RTE={r.RTE_cm_delta_mean:.6f} cm; "\n            f"chỉ multi thành công={int(r.multi_only_success)}, chỉ single thành công={int(r.single_only_success)}.")\n        text.append(f"CI 95% của delta trung bình theo loop: RRE=[{r.RRE_deg_delta_loop_ci_low}, {r.RRE_deg_delta_loop_ci_high}], "\n                    f"RTE=[{r.RTE_cm_delta_loop_ci_low}, {r.RTE_cm_delta_loop_ci_high}]. Delta âm có lợi cho multi.")\n        if r.common_valid_pairs==r.pairs and r.RRE_deg_delta_mean<0 and r.RTE_cm_delta_mean<0 and r.multi_only_success>=r.single_only_success:\n            text.append(\'Multi có mean RRE và RTE thấp hơn tại scene/giai đoạn này; cần xem CI và các scene khác trước khi khái quát.\')\n        else: text.append(\'Chưa chứng minh multi thắng đồng đều trên scene/giai đoạn này; xem cả sai số, failure và success rate.\')\n    for (scene,view),d in df[(df.method==\'B_v331\')&(df.k==BENCH_PRIMARY_K)&(df.geometry==\'distance\')].groupby([\'scene\',\'view\']):\n        d=d[d.coarse_valid&d.final_valid&d.gt_available]\n        if not len(d): continue\n        change=d.final_RTE_cm-d.coarse_RTE_cm\n        text.append(f\'{scene}/{view}: trên {len(d)} trial có cả coarse/final, ICP làm RTE tốt hơn {(change<0).sum()} trial, \'\n                    f\'tệ hơn {(change>0).sum()} trial; delta mean Final-Coarse={change.mean():.6f} cm.\')\n    for scene,d in paired[(paired.method==\'B_v331\')&(paired.k==BENCH_PRIMARY_K)&(paired.geometry==\'distance\')].groupby(\'scene\'):\n        same=d[d.coarse_valid_single&d.coarse_valid_multi&d.final_valid_single&d.final_valid_multi&d.gt_available_single&d.gt_available_multi]\n        if len(same):\n            c=(same.coarse_RTE_cm_multi-same.coarse_RTE_cm_single).mean()\n            f=(same.final_RTE_cm_multi-same.final_RTE_cm_single).mean()\n            text.append(f\'{scene}: trên cùng {len(same)} paired trial có đủ cả 4 pose, độ lớn chênh lệch RTE giữa multi/single \'\n                +(\'thu hẹp\' if abs(f)<abs(c) else \'không thu hẹp\')+f\' sau ICP ({c:.6f} -> {f:.6f} cm).\')\n    missing=[r for r in coverage if r.get(\'status\')!=\'complete\']\n    if missing: text.append(\'CHƯA ĐỦ TOÀN BỘ DATASET/ĐÁNH GIÁ: \'+json.dumps(sb_jsonable(missing),ensure_ascii=False))\n    if \'below_requested_K_attempts\' in summary:\n        short=summary[summary.below_requested_K_attempts.fillna(0)>0]\n        if len(short): text.append(\'Một số submap có ít landmark đủ điều kiện hơn K yêu cầu. Xem n_source_landmarks/n_target_landmarks và below_requested_K_attempts; không diễn giải chúng là đã dùng đủ K.\')\n    (folder/\'assessment_vi.txt\').write_text(\'\\n\'.join(text))\n    # Complete machine-generated LaTeX: measured values only; no TODO filled by invented data.\n    lines=[r\'\\begin{table*}[t]\',r\'\\centering\',r\'\\caption{Controlled single-best-view and multi-view registration using identical landmarks and frozen V3.3.1. Errors are means over valid estimates; success includes failed candidates. \'+str(len(BENCH_SEEDS))+\' seeds share one SLAM run.}\',\n        r\'\\begin{tabular}{l l r r r r r r r}\',r\'\\hline\',r\'Scene & View & Loops & Trials & Coarse RRE & Coarse RTE (cm) & Final RRE & Final RTE (cm) & Final success (\\%) \\\\\',r\'\\hline\']\n    def fmt(x): return \'--\' if pd.isna(x) else f\'{float(x):.3f}\'\n    for _,r in primary.iterrows():\n        vals=[str(r.scene).replace(\'_\',r\'\\_\'),\'Single\' if r.view==\'single_best_view\' else \'Multi\',str(int(r.unique_loops)),str(int(r.attempts)),\n            fmt(r.coarse_RRE_deg_mean),fmt(r.coarse_RTE_cm_mean),fmt(r.final_RRE_deg_mean),fmt(r.final_RTE_cm_mean),fmt(r.final_success_5deg_10cm_percent)]\n        lines.append(\' & \'.join(vals)+r\' \\\\\')\n    lines.extend([r\'\\hline\',r\'\\end{tabular}\',r\'\\label{tab:single-multi-current}\',r\'\\end{table*}\'])\n    (folder/\'paper_single_multi.tex\').write_text(\'\\n\'.join(lines))\n    # All five ablation groups retain their actual method and settings.\n    summary.to_csv(folder/\'paper_all_ablations.csv\',index=False)\n    # Separate reviewable tables for each requested ablation and method history.\n    cuts={\n      \'K_ablation\':summary[(summary.method==\'B_v331\')&(summary.geometry==\'distance\')],\n      \'margin_ablation\':summary[(summary.method==\'A_mnn\')&(summary.view==\'multi_view\')&(summary.k==BENCH_PRIMARY_K)],\n      \'geometry_ablation\':summary[(summary.method==\'B_v331\')&(summary.view==\'multi_view\')&(summary.k==BENCH_PRIMARY_K)],\n      \'coarse_icp_fpfh\':summary[((summary.method==\'B_v331\')&(summary.k==BENCH_PRIMARY_K)&(summary.geometry==\'distance\'))|(summary.method==\'FPFH_reference\')],\n      \'method_evolution\':summary[(summary.k==BENCH_PRIMARY_K)&(summary.geometry==\'distance\')&(summary.margin==0)],\n    }\n    ablation_columns=[(\'scene\',\'Scene\'),(\'method\',\'Method\'),(\'view\',\'View\'),(\'k\',\'K\'),(\'margin\',\'Margin\'),\n        (\'geometry\',\'Geometry\'),(\'coarse_RRE_deg_mean\',\'Coarse RRE\'),(\'coarse_RTE_cm_mean\',\'Coarse RTE (cm)\'),\n        (\'final_RRE_deg_mean\',\'Final RRE\'),(\'final_RTE_cm_mean\',\'Final RTE (cm)\'),\n        (\'final_success_5deg_10cm_percent\',r\'Final success (\\%)\')]\n    for name,cut in cuts.items():\n        cut.to_csv(folder/(name+\'.csv\'),index=False)\n        sb_tex_table(folder,name+\'.tex\',cut,ablation_columns,\n            \'Measured \'+name.replace(\'_\',\' \')+\' on fixed candidate loops. Error means condition on valid estimates; success retains failures. Shape filtering is experimental.\',\n            \'tab:\'+name.replace(\'_\',\'-\'))\n    runtime_columns=[c for c in summary if c in [\'scene\',\'method\',\'view\',\'k\',\'margin\',\'geometry\',\'attempts\',\'coarse_valid\',\'final_valid\']\n        or c.endswith(\'_s_mean\') or \'gpu_peak_\' in c]\n    runtime=summary[runtime_columns]\n    runtime.to_csv(folder/\'runtime_per_pair.csv\',index=False)\n    sb_tex_table(folder,\'runtime_per_pair.tex\',runtime,\n        [(\'scene\',\'Scene\'),(\'method\',\'Method\'),(\'view\',\'View\'),(\'k\',\'K\'),\n         (\'matcher_runtime_s_mean\',\'Coarse (s)\'),(\'matching_s_mean\',\'Matching (s)\'),(\'ransac_s_mean\',\'RANSAC (s)\'),\n         (\'icp_runtime_s_mean\',\'ICP (s)\'),(\'coarse_gpu_peak_torch_allocated_mb_max\',\'Torch peak (MiB)\'),\n         (\'coarse_gpu_peak_device_used_mb_max\',\'Device peak (MiB)\')],\n        \'Per-pair runtime excludes one-time model, descriptor and dense-cloud construction. Coarse includes matching, geometry, RANSAC and local refinement. Device peak is sampled and includes other processes.\',\n        \'tab:runtime-pair\')\n    descriptor=[]\n    if len(cache_df):\n        for scene,d in cache_df.groupby(\'scene\'):\n            item={\'scene\':scene,\'submaps\':len(d),\'fresh_builds\':int(d.cache_built_this_run.fillna(False).sum()),\n                  \'construction\':\'shared builder constructs both descriptor variants\'}\n            for c in [\'landmarks\',\'views\',\'cache_build_time_s_measured\',\'cache_build_time_s_historical\',\'dino_feature_s\',\'bilinear_sample_s\']:\n                if c in d: item[c+\'_mean\']=pd.to_numeric(d[c],errors=\'coerce\').mean()\n            for c in [c for c in d if c.startswith(\'gpu_peak_\') and c.endswith(\'_mb\')]:\n                item[c+\'_max\']=pd.to_numeric(d[c],errors=\'coerce\').max()\n            descriptor.append(item)\n    dc=pd.DataFrame(descriptor);dc.to_csv(folder/\'runtime_descriptor_shared.csv\',index=False)\n    sb_tex_table(folder,\'runtime_descriptor_shared.tex\',dc,\n        [(\'scene\',\'Scene\'),(\'submaps\',\'Submaps\'),(\'fresh_builds\',\'Fresh builds\'),\n         (\'cache_build_time_s_measured_mean\',\'Fresh mean (s)\'),(\'cache_build_time_s_historical_mean\',\'Stored mean (s)\'),\n         (\'gpu_peak_torch_allocated_mb_max\',\'Torch peak (MiB)\'),(\'gpu_peak_device_used_mb_max\',\'Device peak (MiB)\')],\n        \'The shared descriptor builder stores single-best-view and multi-view descriptors for identical Gaussians. Historical cache times are distinct from fresh measurements; no individual-variant construction speedup is inferred.\',\n        \'tab:runtime-descriptor\')\n    counts=df[[\'scene\',\'candidate_id\']].drop_duplicates().groupby(\'scene\').size().to_dict()\n    completion_text=(f\'Observed unique candidate loops by scene: {counts}; total={sum(counts.values())}; \'\n        f\'paper reference={BENCH_EXPECTED_TOTAL_LOOPS}. \'+\n        (\'Reference count matches.\' if sum(counts.values())==BENCH_EXPECTED_TOTAL_LOOPS else \'Reference count differs; do not label this as the same 21-loop experiment.\'))\n    with (folder/\'assessment_vi.txt\').open(\'a\') as out: out.write(\'\\n\'+completion_text+\'\\n\')\n    sb_write_json(folder/\'paper_scope_check.json\',{\'candidate_counts\':counts,\'total_unique_candidates\':sum(counts.values()),\n        \'expected_paper_loops\':BENCH_EXPECTED_TOTAL_LOOPS,\'matches_paper_loop_count\':sum(counts.values())==BENCH_EXPECTED_TOTAL_LOOPS})\n    (folder/\'paper_tables.tex\').write_text(\'\\n\'.join(r\'\\input{\'+name+\'.tex}\' for name in\n        [\'paper_single_multi\',\'K_ablation\',\'margin_ablation\',\'geometry_ablation\',\'coarse_icp_fpfh\',\'method_evolution\',\'runtime_per_pair\',\'runtime_descriptor_shared\']))\n    try:\n        import matplotlib.pyplot as plt\n        scenes=list(primary.scene.unique());fig,ax=plt.subplots(1,2,figsize=(11,4))\n        for a,metric,label in [(ax[0],\'coarse_RTE_cm_mean\',\'Coarse RTE (cm)\'),(ax[1],\'final_RTE_cm_mean\',\'Final RTE (cm)\')]:\n            x=np.arange(len(scenes))\n            for offset,view in [(-.18,\'single_best_view\'),(.18,\'multi_view\')]:\n                vals=[float(primary[(primary.scene==s)&(primary.view==view)][metric].iloc[0])\n                      if len(primary[(primary.scene==s)&(primary.view==view)]) else np.nan for s in scenes]\n                a.bar(x+offset,vals,width=.36,label=view)\n            a.set_xticks(x,scenes);a.set_ylabel(label);a.legend()\n        fig.tight_layout();fig.savefig(folder/\'single_multi_rte.png\',dpi=220);plt.close(fig)\n    except Exception as e: sb_write_json(folder/\'plot_status.json\',{\'error\':str(e)})\n    return primary,summary,ps\n\ndef sb_pair_figure(ctx,row,reference_rows):\n    if not BENCH_EXPORT_PAIR_FIGURES or row[\'seed\']!=BENCH_SEEDS[0] or row[\'method\']!=\'B_v331\' or row[\'k\']!=BENCH_PRIMARY_K or row[\'geometry\']!=\'distance\': return\n    if not row.get(\'T_coarse\'): return\n    import matplotlib.pyplot as plt\n    l=ctx.loops[int(row[\'candidate_id\'])];sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=ctx.frame_info(l)\n    s,_=ctx.dense(sa,si);t,_=ctx.dense(ta,ti)\n    P=np.asarray(s.points);Q=np.asarray(t.points)\n    # Deterministic uniform sampling, identical across methods; display has no role in inference.\n    P=P[np.linspace(0,len(P)-1,min(12000,len(P)),dtype=int)];Q=Q[np.linspace(0,len(Q)-1,min(12000,len(Q)),dtype=int)]\n    transforms=[np.eye(4),sb_loop_to_start(np.asarray(row[\'T_coarse\']),cs0,ct0,cs,ct),\n        None if not row.get(\'T_final\') else sb_loop_to_start(np.asarray(row[\'T_final\']),cs0,ct0,cs,ct)]\n    common_transforms=[np.eye(4)]\n    for other in reference_rows:\n        if other[\'candidate_id\']!=row[\'candidate_id\'] or other[\'seed\']!=row[\'seed\'] or other[\'method\']!=\'B_v331\' or other[\'k\']!=BENCH_PRIMARY_K or other[\'geometry\']!=\'distance\': continue\n        for field in [\'T_coarse\',\'T_final\']:\n            if other.get(field): common_transforms.append(sb_loop_to_start(np.asarray(other[field]),cs0,ct0,cs,ct))\n    fig=plt.figure(figsize=(15,4));combined=np.vstack([Q]+[P@T[:3,:3].T+T[:3,3] for T in common_transforms])\n    low,high=combined.min(0),combined.max(0);center=(low+high)/2;radius=max((high-low).max()/2,0.1)\n    for i,(name,T) in enumerate(zip([\'Unaligned\',\'Coarse\',\'ICP\'],transforms)):\n        ax=fig.add_subplot(1,3,i+1,projection=\'3d\');ax.scatter(*Q.T,s=.15,c=\'#dd8023\',alpha=.6)\n        if T is not None:\n            R=P@T[:3,:3].T+T[:3,3];ax.scatter(*R.T,s=.15,c=\'#2372b3\',alpha=.6)\n        ax.set_title(name);ax.set_xlim(center[0]-radius,center[0]+radius);ax.set_ylim(center[1]-radius,center[1]+radius);ax.set_zlim(center[2]-radius,center[2]+radius)\n        ax.set_box_aspect([1,1,1]);ax.set_axis_off();ax.view_init(22,-55)\n    fig.suptitle(f"{ctx.scene} candidate {row[\'candidate_id\']} {row[\'view\']}");fig.tight_layout()\n    d=ctx.output/\'figures\';d.mkdir(exist_ok=True)\n    fig.savefig(d/f"loop{row[\'candidate_id\']:03d}_{row[\'view\']}_coarse_icp.png",dpi=220);plt.close(fig)\n\ndef sb_resolve_runs():\n    explicit={s:Path(p) for s,p in BENCH_RUNS.items()}; found=defaultdict(list)\n    current=globals().get(\'OUTPUT_ROOT\')\n    candidates=[]\n    if current: candidates.append(Path(current))\n    for root in BENCH_SEARCH_ROOTS:\n        if root.exists(): candidates.extend(p.parent for p in root.rglob(\'detected_loops.pkl\'))\n    import re\n    for p in sorted(set(candidates)):\n        if not (p/\'config.yaml\').exists(): continue\n        try:\n            cfg=sb_load_config(str(p/\'config.yaml\'))\n            text=(str(cfg[\'data\'].get(\'input_path\',\'\'))+\' \'+str(p)).lower().replace(\'-\',\'_\')\n            hits=re.findall(r\'(?<![a-z])(office|apart|apartment)_?(\\d+)\',text)\n            names={(\'apart\' if a==\'apartment\' else a)+\'_\'+b for a,b in hits}\n            for scene in BENCH_SCENES:\n                if scene in names: found[scene].append(p)\n        except Exception: continue\n    runs={}; coverage=[]\n    for scene in BENCH_SCENES:\n        if scene in explicit: runs[scene]=explicit[scene]\n        elif current and Path(current) in found[scene]: runs[scene]=Path(current)\n        elif len(found[scene])==1: runs[scene]=found[scene][0]\n        else: coverage.append({\'scene\':scene,\'status\':\'missing\' if not found[scene] else \'ambiguous\',\n                    \'candidates\':[str(p) for p in found[scene]],\'action\':\'Set BENCH_RUNS with the exact finished SLAM output folder\'})\n    return runs,coverage\n\ndef sb_run():\n    global sb_torch,sb_o3d,sb_F,sb_Image,sb_load_config,sb_get_dataset,sb_repo_utils\n    if str(BENCH_REPO) not in sys.path: sys.path.insert(0,str(BENCH_REPO))\n    import torch as sb_torch\n    import open3d as sb_o3d\n    import torch.nn.functional as sb_F\n    from PIL import Image as sb_Image\n    from src.utils.io_utils import load_config as sb_load_config\n    from src.entities.datasets import get_dataset as sb_get_dataset\n    from src.utils import utils as sb_repo_utils\n    if not sb_torch.cuda.is_available(): raise RuntimeError(\'Run in the recent Kaggle CUDA environment; descriptor/render-depth construction needs CUDA\')\n    sb_torch.cuda.set_device(BENCH_GPU_DEVICE)\n    if set(BENCH_ICP_SEEDS)-set(BENCH_SEEDS): raise ValueError(\'ICP seeds must be a subset of coarse seeds\')\n    specs=sb_specs()\n    try: head=subprocess.check_output([\'git\',\'-C\',str(BENCH_REPO),\'rev-parse\',\'HEAD\'],text=True).strip()\n    except Exception: head=\'unknown\'\n    protocol={\'version\':_SB_CELL_VERSION,\'repo_head\':head,\'helpers\':_SB_PROVENANCE,\n        \'driver_sha256\':_SB_DRIVER_SHA256,\'specs\':specs,\'seeds\':BENCH_SEEDS,\'icp_seeds\':BENCH_ICP_SEEDS,\'scope\':BENCH_SCOPE,\n        \'expected_paper_total_loops\':BENCH_EXPECTED_TOTAL_LOOPS,\n        \'repo_python_sha256\':sb_digest([(str(p.relative_to(BENCH_REPO)),sb_file_hash(p)) for p in sorted((BENCH_REPO/\'src\').rglob(\'*.py\'))]),\n        \'shape\':{\'scale_log_threshold\':BENCH_SCALE_LOG_THRESHOLD,\'relative_axis_cos_threshold\':BENCH_AXIS_COS_THRESHOLD,\n                 \'anisotropy_min\':BENCH_AXIS_ANISOTROPY_MIN,\'quaternion_order\':BENCH_QUATERNION_ORDER,\'experimental\':True},\n        \'fpfh\':{\'voxel_m\':BENCH_FPFH_VOXEL_M,\'ransac_max_iters\':BENCH_FPFH_MAX_ITERS,\'confidence\':BENCH_FPFH_CONFIDENCE},\n        \'torch\':sb_torch.__version__,\'open3d\':sb_o3d.__version__,\'cuda\':sb_torch.version.cuda,\n        \'gpu_model\':sb_torch.cuda.get_device_name(BENCH_GPU_DEVICE),\n        \'gpu_poll_seconds\':BENCH_GPU_POLL_SECONDS,\'frame_convention\':\'source loop camera -> target loop camera; explicit conversion for first-submap-frame dense ICP\'}\n    protocol_hash=sb_digest(protocol); root=BENCH_OUTPUT/protocol_hash[:16];root.mkdir(parents=True,exist_ok=True)\n    sb_write_json(root/\'protocol.json\',protocol)\n    runs,coverage=sb_resolve_runs();all_rows=[];all_cache=[];expected_by_scene={};active_outputs=set()\n    for scene,path in runs.items():\n        print(f\'[{scene}] Load fixed candidates and finalized submaps: {path}\',flush=True)\n        try:\n            ctx=SBContext(scene,path,root/scene)\n            # Namespace/config and input identity are frozen before any variant runs.\n            ctx.output=root/scene/ctx.input_hash[:16];ctx.output.mkdir(parents=True,exist_ok=True)\n            ctx.ns=sb_namespace(ctx);cache_hash=sb_prepare_caches(ctx)\n            run_hash=sb_digest([protocol_hash,ctx.input_hash,cache_hash])\n            expected_by_scene[scene]=len(ctx.loops)\n            sb_write_json(ctx.output/\'run_meta.json\',{\'scene\':scene,\'root\':str(path),\'protocol_hash\':protocol_hash,\n                \'run_hash\':run_hash,\'input_fingerprint\':ctx.input_fingerprint,\'cache_hash\':cache_hash,\n                \'fixed_loop_keys\':[sb_loop_key(l) for l in ctx.loops],\n                \'model_init_s\':getattr(ctx,\'model_init_s\',None),\'candidate_count\':len(ctx.loops),\n                \'expected_rows\':len(ctx.loops)*len(BENCH_SEEDS)*len(specs)})\n            # Preserve system metrics without claiming they changed under pairwise ablation.\n            system={}\n            for name in [\'coarse_render_metrics.json\',\'fine_render_metrics.json\',\'novel_render_metrics.json\',\n                         \'walltime_stats.json\',\'registration_summary.json\',\'ate.json\',\'loop_pipeline_summary.json\']:\n                p=ctx.root/name\n                if p.exists():\n                    try: system[name]=json.loads(p.read_text())\n                    except Exception: system[name]={\'source_file\':str(p),\'parse_failed\':True}\n            sb_write_json(ctx.output/\'original_system_metrics_context.json\',system)\n            local=[];checkpoints=ctx.output/\'checkpoints\';checkpoints.mkdir(exist_ok=True)\n            for lid,loop in enumerate(ctx.loops):\n                for seed in BENCH_SEEDS:\n                    # Alternate pair order by seed/loop, independent of GT, to limit order bias.\n                    ordered=list(specs)\n                    if (lid+seed)%2: ordered=list(reversed(ordered))\n                    for spec in ordered:\n                        token=sb_digest([run_hash,sb_loop_key(loop),seed,spec])\n                        p=checkpoints/(token+\'.json\');row=None\n                        if p.exists():\n                            old=json.loads(p.read_text())\n                            if old.get(\'run_hash\')==run_hash and (old.get(\'status\')!=\'error\' or not BENCH_RETRY_ERRORS): row=old\n                        if row is None:\n                            row=sb_attempt(ctx,spec,lid,loop,seed,run_hash);sb_write_json(p,row)\n                            if BENCH_VERBOSE:\n                                print(f"[{scene}] loop={lid} seed={seed} {spec[\'id\']}: {row[\'status\']}; "\n                                    f"coarse={row.get(\'coarse_RRE_deg\')}/{row.get(\'coarse_RTE_cm\')} final={row.get(\'final_RRE_deg\')}/{row.get(\'final_RTE_cm\')}",flush=True)\n                        local.append(row)\n                    for figure_row in [r for r in local if r[\'candidate_id\']==lid and r[\'seed\']==seed]:\n                        try: sb_pair_figure(ctx,figure_row,local)\n                        except Exception as e: sb_write_json(ctx.output/\'figure_error.json\',{\'error\':str(e)})\n                    # Persist CSV after each loop/seed as well as each atomic checkpoint.\n                    pd.DataFrame(local).to_csv(ctx.output/\'per_run_partial.csv\',index=False)\n                    print(f\'[{scene}] loop={lid} seed={seed}: {len(local)}/{len(ctx.loops)*len(BENCH_SEEDS)*len(specs)} saved rows\',flush=True)\n            local_df=pd.DataFrame(local);cd=pd.DataFrame(ctx.cache_rows)\n            status=\'complete\' if not (local_df.status==\'error\').any() else \'complete_with_errors\'\n            if not local_df.gt_available.fillna(False).all(): status=\'complete_without_gt\'\n            coverage.append({\'scene\':scene,\'status\':status,\'run_hash\':run_hash,\'candidates\':len(ctx.loops),\n                \'expected_rows\':len(ctx.loops)*len(BENCH_SEEDS)*len(specs),\'rows\':len(local),\n                \'error_rows\':int((local_df.status==\'error\').sum())})\n            all_rows.extend(local);all_cache.extend(ctx.cache_rows)\n            active_outputs.add(ctx.output)\n            sb_report(ctx.output/\'report\',local_df,[coverage[-1]],cd)\n            del ctx;gc.collect();sb_torch.cuda.empty_cache()\n        except Exception as e:\n            coverage.append({\'scene\':scene,\'status\':\'setup_error\',\'error\':f\'{type(e).__name__}: {e}\'})\n            print(f\'[{scene}] SETUP ERROR: {e}\',flush=True)\n        sb_write_json(root/\'dataset_coverage.json\',coverage)\n    # Import only identical protocol results, detect duplicate/conflicting runs.\n    for base in BENCH_IMPORT_RESULTS:\n        for p in Path(base).rglob(\'run_meta.json\'):\n            meta=json.loads(p.read_text())\n            if meta.get(\'protocol_hash\')!=protocol_hash: continue\n            if meta.get(\'scene\') not in BENCH_SCENES: continue\n            expected_by_scene[meta[\'scene\']]=int(meta[\'candidate_count\'])\n            dest=p.parent if p.parent.is_relative_to(root) else root/\'imports\'/meta[\'scene\']/meta[\'run_hash\'][:16]\n            active_outputs.add(dest)\n            if dest!=p.parent:\n                sb_write_json(dest/\'run_meta.json\',meta)\n                for name in [\'descriptor_runtime.csv\',\'dense_runtime.csv\',\'fpfh_runtime.csv\',\'original_system_metrics_context.json\']:\n                    if (p.parent/name).exists(): (dest/name).write_bytes((p.parent/name).read_bytes())\n            for c in (p.parent/\'checkpoints\').glob(\'*.json\'):\n                row=json.loads(c.read_text())\n                if row.get(\'run_hash\')==meta[\'run_hash\']:\n                    all_rows.append(row)\n                    if dest!=p.parent: sb_write_json(dest/\'checkpoints\'/c.name,row)\n            if (p.parent/\'descriptor_runtime.csv\').exists():\n                all_cache.extend(pd.read_csv(p.parent/\'descriptor_runtime.csv\').to_dict(\'records\'))\n    if not all_rows:\n        raise RuntimeError(\'No runnable datasets. See \'+str(root/\'dataset_coverage.json\'))\n    df=pd.DataFrame(all_rows)\n    # One run per scene is required; no cherry-picking across repeated SLAM runs.\n    if (df.groupby(\'scene\').run_hash.nunique()>1).any(): raise RuntimeError(\'Multiple distinct SLAM runs for one scene; select BENCH_RUNS/imports explicitly\')\n    # Prefer an executed attempt over a technical exception; otherwise current\n    # local rows precede imports. Never select rows by success or pose error.\n    df=df.assign(_technical_error=(df.status==\'error\').astype(int)).sort_values(\'_technical_error\',kind=\'stable\')\n    df=df.drop_duplicates([\'run_hash\',\'id\',\'candidate_id\',\'seed\'],keep=\'first\').drop(columns=\'_technical_error\')\n    # Imported finished scenes can complete coverage; verify the entire fixed matrix.\n    for scene,d in df.groupby(\'scene\'):\n        nloops=expected_by_scene[scene];expected=nloops*len(specs)*len(BENCH_SEEDS)\n        all_keys={(int(lid),int(seed),s[\'id\']) for lid in range(nloops) for seed in BENCH_SEEDS for s in specs}\n        observed=set(zip(d.candidate_id.astype(int),d.seed.astype(int),d.id))\n        complete=all_keys==observed\n        status=\'partial\' if not complete else (\'complete_with_errors\' if (d.status==\'error\').any() else (\'complete_without_gt\' if not d.gt_available.fillna(False).all() else \'complete\'))\n        coverage=[c for c in coverage if c.get(\'scene\')!=scene]\n        coverage.append({\'scene\':scene,\'status\':status,\'run_hash\':str(d.run_hash.iloc[0]),\'candidates\':int(nloops),\n            \'expected_rows\':int(expected),\'rows\':len(d),\'error_rows\':int((d.status==\'error\').sum())})\n    cache_df=pd.DataFrame(all_cache)\n    if len(cache_df): cache_df=cache_df.drop_duplicates([\'scene\',\'agent\',\'submap\',\'cache_content_sha256\'],keep=\'first\')\n    primary,summary,paired=sb_report(root/\'all_datasets\',df,coverage,cache_df)\n    sb_write_json(root/\'dataset_coverage.json\',coverage)\n    sb_write_json(root/\'completion.json\',{\'protocol_hash\':protocol_hash,\'scenes\':coverage,\'rows\':len(df),\n        \'complete_requested_scenes\':all(any(x.get(\'scene\')==s and x.get(\'status\')==\'complete\' for x in coverage) for s in BENCH_SCENES)})\n    export=BENCH_OUTPUT/(\'single_multi_v331_\'+protocol_hash[:16]+\'_results.zip\')\n    with zipfile.ZipFile(export,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob(\'*\')):\n            current=p.parent==root or p.is_relative_to(root/\'all_datasets\') or any(p.is_relative_to(d) for d in active_outputs)\n            if p.is_file() and current and not any(x in p.parts for x in [\'dense_cache\',\'descriptor_cache\',\'fpfh_cache\']): z.write(p,p.relative_to(root))\n    try:\n        from IPython.display import display\n        display(primary); display(paired[(paired.method==\'B_v331\')&(paired.k==BENCH_PRIMARY_K)])\n    except Exception: print(primary.to_string(index=False))\n    print(\'RESULTS:\',root/\'all_datasets\');print(\'EXPORT ZIP:\',export)\n    print(\'Dataset coverage:\',coverage)\n    return {\'root\':root,\'zip\':export,\'per_run\':df,\'summary\':summary,\'paired\':paired,\'primary\':primary}\n'
_SB_CELL_VERSION = 'sv-mv-v331-complete-2-py311'

def sb_jsonable(x):
    if isinstance(x, dict): return {str(k): sb_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return [sb_jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return sb_jsonable(x.tolist())
    if isinstance(x, np.generic): return sb_jsonable(x.item())
    if isinstance(x, float) and not np.isfinite(x): return None
    return x

def sb_write_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(sb_jsonable(value),ensure_ascii=False,indent=2,allow_nan=False))
    tmp.replace(path)

def sb_digest(value):
    return hashlib.sha256(json.dumps(sb_jsonable(value),sort_keys=True).encode()).hexdigest()

def sb_file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def sb_cache_content_hash(cache):
    # Tensor content, not torch.save archive bytes; serialization IDs can change.
    h=hashlib.sha256()
    keys=['xyz','gaussian_index','descriptor_multi_view','descriptor_single_best_view','stability',
          'observation_count','depth_residual_mean','opacity','score','scale_sorted','rotation_axes_sorted','rotation_axis_valid']
    for key in keys:
        if key not in cache: continue
        value=cache[key]
        if hasattr(value,'detach'): value=value.detach().cpu().numpy()
        value=np.ascontiguousarray(value)
        h.update(key.encode()); h.update(str(value.dtype).encode()); h.update(str(value.shape).encode())
        h.update(value.tobytes())
    return h.hexdigest()

def sb_pose(x):
    if hasattr(x,'detach'): x=x.detach().cpu().numpy()
    a=np.asarray(x,dtype=np.float64)
    if a.shape!=(4,4) or not np.isfinite(a).all(): raise ValueError('Invalid SE(3) matrix')
    if not np.allclose(a[3],[0,0,0,1],atol=1e-5): raise ValueError('Invalid homogeneous row')
    if not np.allclose(a[:3,:3].T@a[:3,:3],np.eye(3),atol=1e-3) or np.linalg.det(a[:3,:3])<0:
        raise ValueError('Invalid rigid rotation')
    return a

def sb_error(T, gt):
    T,gt=sb_pose(T),sb_pose(gt)
    angle=np.degrees(np.arccos(np.clip((np.trace(T[:3,:3].T@gt[:3,:3])-1)/2,-1,1)))
    return float(angle),float(100*np.linalg.norm(T[:3,3]-gt[:3,3]))

def sb_loop_key(loop):
    return tuple(int(getattr(loop,k)) for k in ['source_agent_id','source_frame_id','target_agent_id','target_frame_id'])

def sb_residual(T,P,Q):
    return np.linalg.norm(np.asarray(P)@T[:3,:3].T+T[:3,3]-np.asarray(Q),axis=1)

def sb_loop_to_start(T,cs0,ct0,csl,ctl):
    return np.linalg.inv(ct0)@ctl@T@np.linalg.inv(csl)@cs0

def sb_start_to_loop(T,cs0,ct0,csl,ctl):
    return np.linalg.inv(ctl)@ct0@T@np.linalg.inv(cs0)@csl

def sb_slice_cache(cache,k,view):
    n=min(int(k),len(cache['xyz'])); out={}
    for name,val in cache.items():
        if (hasattr(val,'shape') and len(val.shape)>0 and len(val)==len(cache['xyz'])
                and name not in {'keyframe_ids'}): out[name]=val[:n]
        else: out[name]=val
    key={'single_best_view':'descriptor_single_best_view','multi_view':'descriptor_multi_view'}[view]
    out['descriptor_multi_view']=out[key]  # private alias, all frozen paths see the chosen variant
    out['benchmark_variant']=view
    return out

def sb_quaternion_rotation(q,order='wxyz'):
    q=np.asarray(q,dtype=np.float64)
    if order=='xyzw': q=q[:,[3,0,1,2]]
    elif order!='wxyz': raise ValueError(order)
    q=q/np.maximum(np.linalg.norm(q,axis=1,keepdims=True),1e-12)
    w,x,y,z=q.T
    R=np.stack([1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y),
                2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x),
                2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)],axis=1).reshape(-1,3,3)
    return R

def sb_quaternion_axes(q,scales,order='wxyz'):
    R=sb_quaternion_rotation(q,order)
    axes=R[np.arange(len(R)),:,np.argmax(scales,axis=1)]
    return axes/np.maximum(np.linalg.norm(axes,axis=1,keepdims=True),1e-12)

def sb_shape_from_checkpoint(sub,indices):
    # Exact raw-field convention already used by the recent Gaussian-glyph export.
    def arr(v): return v.detach().cpu().numpy() if hasattr(v,'detach') else np.asarray(v)
    params=sub['gaussian_model_params']; idx=arr(indices).astype(np.int64)
    scales=np.exp(arr(params['scaling'])[idx].astype(np.float64))
    quat=arr(params['rotation'])[idx].astype(np.float64)
    if scales.shape!=(len(idx),3) or quat.shape!=(len(idx),4) or not np.isfinite(scales).all() or not np.isfinite(quat).all() or not np.all(scales>0) or np.any(np.linalg.norm(quat,axis=1)<1e-12):
        raise ValueError('Invalid Gaussian scaling/rotation fields')
    sorted_scale=np.sort(scales,axis=1)
    R=sb_quaternion_rotation(quat,BENCH_QUATERNION_ORDER)
    order=np.argsort(-scales,axis=1)
    axes=np.take_along_axis(R,order[:,None,:],axis=2).astype(np.float32)
    desc_scale=np.take_along_axis(scales,order,axis=1)
    gap01=desc_scale[:,0]/desc_scale[:,1]>=BENCH_AXIS_ANISOTROPY_MIN
    gap12=desc_scale[:,1]/desc_scale[:,2]>=BENCH_AXIS_ANISOTROPY_MIN
    return {'scale_sorted':sorted_scale,'rotation_axes_sorted':axes,
        'rotation_axis_valid':np.column_stack([gap01,gap01&gap12,gap12]),
        'principal_axis':axes[:,:,0],'axis_valid':gap01}

def sb_shape_support(pairs,P,Q,sc,tc,threshold):
    pairs=np.asarray(pairs,dtype=np.int64)
    if len(pairs)==0: return np.empty(0,dtype=np.int64)
    i,j=pairs.T; pp=np.asarray(P)[i]; qq=np.asarray(Q)[j]
    ss=np.asarray(sc['scale_sorted'])[i]; ts=np.asarray(tc['scale_sorted'])[j]
    scale_ok=np.max(np.abs(np.log(np.maximum(ss,1e-12)/np.maximum(ts,1e-12))),axis=1)<=BENCH_SCALE_LOG_THRESHOLD
    sa=np.asarray(sc['rotation_axes_sorted'])[i];ta=np.asarray(tc['rotation_axes_sorted'])[j]
    usable=np.asarray(sc['rotation_axis_valid'])[i]&np.asarray(tc['rotation_axis_valid'])[j]
    support=np.zeros(len(pairs),dtype=np.int64)
    # Stream all pair relationships. abs(R_i^T R_j) is invariant under a common
    # rigid rotation and independent Gaussian-axis sign flips; skip ambiguous axes.
    for start in range(0,len(pairs),256):
        stop=min(start+256,len(pairs));sl=slice(start,stop)
        compatible=np.abs(np.linalg.norm(pp[sl,None]-pp[None,:],axis=2)-
                          np.linalg.norm(qq[sl,None]-qq[None,:],axis=2))<=threshold
        compatible &= scale_ok[sl,None]&scale_ok[None,:]
        for a in range(3):
            for b in range(3):
                use=usable[sl,a,None]&usable[None,:,b]
                ds=np.abs(sa[sl,:,a]@sa[:,:,b].T);dt=np.abs(ta[sl,:,a]@ta[:,:,b].T)
                compatible &= (~use)|(np.abs(ds-dt)<=BENCH_AXIS_COS_THRESHOLD)
        compatible &= (i[sl,None]!=i[None,:])&(j[sl,None]!=j[None,:])
        support[sl]=compatible.sum(axis=1)
    return support

def sb_specs():
    specs=[]
    for method in ['A_mnn','B_v32','B_v331']:
        for view in ['single_best_view','multi_view']:
            specs.append(dict(method=method,view=view,k=BENCH_PRIMARY_K,margin=0.0,geometry='distance'))
    if BENCH_RUN_ABLATIONS:
        for k in BENCH_K_VALUES:
            if k==BENCH_PRIMARY_K: continue
            for view in ['single_best_view','multi_view']:
                specs.append(dict(method='B_v331',view=view,k=k,margin=0.0,geometry='distance'))
        specs.append(dict(method='A_mnn',view='multi_view',k=BENCH_PRIMARY_K,
                          margin=BENCH_MARGIN_POSITIVE,geometry='distance'))
        specs.append(dict(method='B_v331',view='multi_view',k=BENCH_PRIMARY_K,
                          margin=0.0,geometry='distance_scale_relative_axis'))
    if BENCH_RUN_FPFH:
        specs.append(dict(method='FPFH_reference',view='geometry',k=0,margin=0.0,geometry='fpfh'))
    for s in specs:
        s['id']=f"{s['method']}_{s['view']}_K{s['k']}_m{s['margin']:g}_{s['geometry']}"
    return specs

class SBMonitor:
    """PyTorch allocator peak, plus sampled device memory from nvidia-smi.
    Device memory includes other processes; it is not mislabeled as process memory.
    """
    def __enter__(self):
        self.samples=[]; self.stop=threading.Event(); self.torch_base=None
        if sb_torch.cuda.is_available():
            sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)
            self.torch_base=sb_torch.cuda.memory_allocated(BENCH_GPU_DEVICE)/2**20
            sb_torch.cuda.reset_peak_memory_stats(BENCH_GPU_DEVICE)
            self.thread=threading.Thread(target=self.poll,daemon=True); self.thread.start()
        return self
    def poll(self):
        while not self.stop.is_set():
            try:
                p=subprocess.run(['nvidia-smi','-i',str(BENCH_GPU_DEVICE),'--query-gpu=memory.used',
                    '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=2)
                if p.returncode==0: self.samples.append(float(p.stdout.strip().splitlines()[0]))
            except Exception: pass
            self.stop.wait(BENCH_GPU_POLL_SECONDS)
    def __exit__(self,*exc):
        self.stop.set()
        if hasattr(self,'thread'): self.thread.join(timeout=3)
        if self.torch_base is not None:
            sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)
            peak=sb_torch.cuda.max_memory_allocated(BENCH_GPU_DEVICE)/2**20
        else: peak=None
        self.metrics={'gpu_peak_torch_allocated_mb':peak,
            'gpu_peak_torch_incremental_mb':None if peak is None else max(0,peak-self.torch_base),
            'gpu_peak_device_used_mb':max(self.samples) if self.samples else None,
            'gpu_device_samples':len(self.samples),'gpu_peak_device_is_sampled':True}

def sb_sync():
    if sb_torch.cuda.is_available(): sb_torch.cuda.synchronize(BENCH_GPU_DEVICE)

class SBSubmaps:
    def __init__(self,paths): self.paths=paths; self.memo=OrderedDict()
    def __getitem__(self,i):
        i=int(i)
        if i not in self.memo:
            self.memo[i]=sb_torch.load(self.paths[i],map_location='cpu',weights_only=False)
            if len(self.memo)>2: self.memo.popitem(last=False)
        self.memo.move_to_end(i); return self.memo[i]
    def __len__(self): return len(self.paths)

class SBContext:
    def __init__(self,scene,root,output):
        self.scene=scene; self.root=Path(root); self.output=Path(output); self.output.mkdir(parents=True,exist_ok=True)
        self.cfg=sb_load_config(str(self.root/'config.yaml'))
        self.data_root=Path(BENCH_DATA_ROOT_OVERRIDES.get(scene,self.cfg['data']['input_path']))
        if not self.data_root.exists(): raise FileNotFoundError(f'{scene}: RGB-D not found: {self.data_root}; set BENCH_DATA_ROOT_OVERRIDES')
        self.paths={}; self.submaps={}; self.headers={}; self.datasets={}; self.dense_memo=OrderedDict(); self.fpfh_memo=OrderedDict()
        self.dense_rows={}; self.fpfh_rows={}
        dirs=sorted(p for p in self.data_root.iterdir() if p.is_dir() and not p.name.startswith('.'))
        for aid in map(int,self.cfg['data']['agent_ids']):
            exact=[p for p in dirs if p.name.lower() in {f'agent_{aid}',f'agent{aid}'}]
            path=exact[0] if len(exact)==1 else (dirs[aid] if 0<=aid<len(dirs) else None)
            if path is None: raise RuntimeError(f'Cannot resolve agent {aid} input folder')
            dc={**self.cfg['data'],'input_path':str(path),**self.cfg['cam'],**self.cfg['submap']}
            self.datasets[aid]=sb_get_dataset(self.cfg['dataset_name'])(dc)
            paths=sorted((self.root/f'agent_{aid}'/'submaps').glob('*.ckpt'))
            if not paths: raise FileNotFoundError(f'{scene}: missing finalized submaps for agent {aid}')
            self.paths[aid]=paths; self.submaps[aid]=SBSubmaps(paths); self.headers[aid]=[]
            for p in paths:
                sub=sb_torch.load(p,map_location='cpu',weights_only=False)
                self.headers[aid].append({k:sub[k] for k in ['submap_start_frame_id','submap_end_frame_id','submap_c2ws','keyframe_ids']})
                del sub
        candidate=self.root/'detected_loops.pkl'
        if not candidate.exists(): raise FileNotFoundError('Requires pre-registration detected_loops.pkl; loops.pkl is not a substitute')
        with candidate.open('rb') as f: all_loops=pickle.load(f)
        self.loops=[l for l in all_loops if int(l.source_agent_id)!=int(l.target_agent_id)]
        keys=[sb_loop_key(l) for l in self.loops]
        if len(set(keys))!=len(keys): raise RuntimeError('Duplicate candidate keys; resolve duplicates before paired evaluation')
        self.admitted=set()
        if (self.root/'filtered_loops.pkl').exists():
            with (self.root/'filtered_loops.pkl').open('rb') as f:
                self.admitted={sb_loop_key(l) for l in pickle.load(f) if int(l.source_agent_id)!=int(l.target_agent_id)}
        if BENCH_SCOPE=='paper_common': self.loops=[l for l in self.loops if sb_loop_key(l) in self.admitted]
        elif BENCH_SCOPE!='detected': raise ValueError(BENCH_SCOPE)
        if not self.loops: raise RuntimeError('No inter-agent candidates in selected fixed scope')
        self.required=set()
        for l in self.loops:
            sa,sf,ta,tf=sb_loop_key(l); self.required.update([(sa,self.index(sa,sf)),(ta,self.index(ta,tf))])
        fingerprints=[]
        for aid,sid in sorted(self.required):
            p=self.paths[aid][sid]; st=p.stat()
            fingerprints.append({'agent':aid,'submap':sid,'name':p.name,'bytes':st.st_size,'mtime_ns':st.st_mtime_ns,
                 'sha256':sb_file_hash(p) if BENCH_STRICT_INPUT_HASH else None})
        self.input_fingerprint={'config_sha256':sb_file_hash(self.root/'config.yaml'),
            'candidates_sha256':sb_file_hash(candidate),'submaps':fingerprints,'data_root':str(self.data_root),
            'baseline_admission_sha256':sb_file_hash(self.root/'filtered_loops.pkl') if (self.root/'filtered_loops.pkl').exists() else None,
            'loop_keys':[sb_loop_key(l) for l in self.loops]}
        # RGB-D paths/size/mtime invalidate a cached descriptor if mounted inputs change.
        self.input_fingerprint['dataset_inventory']=sb_digest([
            (str(p.relative_to(self.data_root)),p.stat().st_size,p.stat().st_mtime_ns)
            for p in sorted(self.data_root.rglob('*')) if p.is_file()])
        self.input_hash=sb_digest(self.input_fingerprint)
        self.caches={}; self.cache_rows=[]; self.ns=None
    def index(self,aid,frame):
        for i,h in enumerate(self.headers[int(aid)]):
            if int(h['submap_start_frame_id'])<=int(frame)<=int(h['submap_end_frame_id']): return i
        raise KeyError((aid,frame))
    def c2w(self,aid,sid,frame):
        h=self.headers[int(aid)][int(sid)]; off=int(frame)-int(h['submap_start_frame_id'])
        if off<0 or off>=len(h['submap_c2ws']): raise IndexError((aid,sid,frame))
        return sb_pose(h['submap_c2ws'][off])
    def frame_info(self,loop):
        sa,sf,ta,tf=sb_loop_key(loop); si,ti=self.index(sa,sf),self.index(ta,tf)
        cs=self.c2w(sa,si,sf); ct=self.c2w(ta,ti,tf)
        cs0=self.c2w(sa,si,int(self.headers[sa][si]['submap_start_frame_id']))
        ct0=self.c2w(ta,ti,int(self.headers[ta][ti]['submap_start_frame_id']))
        return sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0
    def gt(self,loop):
        sa,sf,ta,tf=sb_loop_key(loop)
        def get(a,f):
            d=self.datasets[a]
            # Match recent mv_gt_pose exactly; dataset __getitem__ may normalize poses.
            return sb_pose(d[f][-1])
        return np.linalg.inv(get(ta,tf))@get(sa,sf)
    def dense(self,aid,sid):
        key=(aid,sid)
        if key in self.dense_memo: self.dense_memo.move_to_end(key); return self.dense_memo[key]
        path=self.output/'dense_cache'/f'agent{aid}_submap{sid:06d}.npz'
        t=time.perf_counter(); hit=path.exists()
        if hit:
            with np.load(path) as z:
                cloud=sb_o3d.geometry.PointCloud()
                cloud.points=sb_o3d.utility.Vector3dVector(z['points'])
                cloud.normals=sb_o3d.utility.Vector3dVector(z['normals'])
                if 'colors' in z: cloud.colors=sb_o3d.utility.Vector3dVector(z['colors'])
        else:
            with SBMonitor() as monitor:
                sub=self.submaps[aid][sid]
                rgb,depth=sb_repo_utils.get_rgbd(sub)
                cloud=sb_repo_utils.rgbd2ptcloud(rgb,depth,sub['intrinsics'],np.eye(4))
                cloud.estimate_normals(); sb_sync()
            path.parent.mkdir(parents=True,exist_ok=True)
            arrays={'points':np.asarray(cloud.points),'normals':np.asarray(cloud.normals)}
            if cloud.has_colors(): arrays['colors']=np.asarray(cloud.colors)
            tmp=path.with_suffix('.tmp.npz'); np.savez_compressed(tmp,**arrays); tmp.replace(path)
            sb_write_json(path.with_suffix('.json'),dict(agent=aid,submap=sid,points=len(cloud.points),
                build_time_s=time.perf_counter()-t,**monitor.metrics))
        info=json.loads(path.with_suffix('.json').read_text()) if path.with_suffix('.json').exists() else {}
        info={**info,'cache_hit':hit,'access_time_s':time.perf_counter()-t}
        self.dense_rows[key]={'scene':self.scene,**info,'agent':aid,'submap':sid}
        pd.DataFrame(self.dense_rows.values()).to_csv(self.output/'dense_runtime.csv',index=False)
        self.dense_memo[key]=(cloud,info)
        if len(self.dense_memo)>2: self.dense_memo.popitem(last=False)
        if len(cloud.points)<3 or not np.isfinite(np.asarray(cloud.points)).all() or not np.isfinite(np.asarray(cloud.normals)).all(): raise RuntimeError('Invalid dense cloud')
        return cloud,info

def sb_namespace(ctx):
    cfg=ctx.cfg; opts=cfg['submap'].get('registration_options',{})
    ns=dict(np=np,pd=pd,torch=sb_torch,F=sb_F,o3d=sb_o3d,time=time,math=math,json=json,
        hashlib=hashlib,Path=Path,Image=sb_Image,repo_utils=sb_repo_utils,
        load_3dgs=sb_repo_utils.load_3dgs,_rigidity_compatibility_mask=sb_repo_utils._rigidity_compatibility_mask,
        _ransac_from_correspondences=sb_repo_utils._ransac_from_correspondences,
        mv_config=cfg,mv_datasets=ctx.datasets,mv_submap_paths=ctx.paths,mv_submaps=ctx.submaps,
        B_DEVICE=f'cuda:{BENCH_GPU_DEVICE}' if sb_torch.cuda.is_available() else 'cpu',
        MV_DEVICE=f'cuda:{BENCH_GPU_DEVICE}' if sb_torch.cuda.is_available() else 'cpu',
        MV_MIN_SIMILARITY=float(opts.get('dino_min_similarity',-1)),MV_MIN_MARGIN=0.0,
        MV_COMPATIBILITY_THRESHOLD_M=float(opts.get('dino_compatibility_threshold_m',0.03)),
        MV_MIN_COMPATIBILITY_SUPPORT=int(opts.get('dino_min_compatibility_support',16)),
        MV_RANSAC_DISTANCE_THRESHOLD_M=0.05,A_SEEDS=list(BENCH_SEEDS),
        A_MIN_SIMILARITY=float(opts.get('dino_min_similarity',-1)),A_MIN_MARGIN=0.0,
        A_COMPATIBILITY_THRESHOLD_M=float(opts.get('dino_compatibility_threshold_m',0.03)),
        A_MIN_COMPATIBILITY_SUPPORT=int(opts.get('dino_min_compatibility_support',16)),
        A_RANSAC_DISTANCE_THRESHOLD_M=0.05,A_MIN_CORRESPONDENCES=int(opts.get('min_correspondences',8)),
        MV_DEPTH_TOLERANCE_M=float(opts.get('gaussian_depth_tolerance_m',0.05)),
        MV_MIN_OPACITY=float(opts.get('gaussian_min_opacity',0.05)),MV_MIN_OBSERVATIONS=2,
        MV_VOXEL_SIZE_M=0.03,MV_REBUILD_CACHE=False,MV_MODEL_NAME=cfg['loop_detection'].get('weights_path','facebook/dinov2-small'),
        MV_CACHE_ROOT=ctx.output/'descriptor_cache',MV_CACHE_VERSION='shared-master-'+_SB_CELL_VERSION,
        MV_MAX_LANDMARKS=max([BENCH_PRIMARY_K]+(BENCH_K_VALUES if BENCH_RUN_ABLATIONS else [])))
    exec(compile(_SB_HELPERS,'frozen_recent_notebook_helpers','exec'),ns)
    ns['mv_find_submap_index']=ctx.index
    return ns

def sb_cache_compatible(cache,ctx,aid,sid):
    need=['xyz','gaussian_index','descriptor_multi_view','descriptor_single_best_view','stability',
          'observation_count','depth_residual_mean','opacity','score']
    if any(k not in cache for k in need): return False
    if cache.get('version') not in {'notebook-office0-ablation-v4','heldout-frozen-mv-v331-v1',ctx.ns['MV_CACHE_VERSION']}: return False
    n=len(cache['xyz'])
    if any(len(cache[k])!=n for k in need): return False
    if n==0 or np.any(np.asarray(cache['observation_count'])<2): return False
    for key in ['xyz','descriptor_multi_view','descriptor_single_best_view']:
        arr=np.asarray(cache[key])
        if arr.ndim!=2 or not np.isfinite(arr).all(): return False
    if cache['descriptor_multi_view'].shape!=cache['descriptor_single_best_view'].shape: return False
    if int(cache.get('agent_id',aid))!=aid or int(cache.get('submap_index',sid))!=sid: return False
    # Existing caches may have no provenance. Compare each selected center to this checkpoint.
    sub=ctx.submaps[aid][sid]
    if 'keyframe_ids' not in cache or not np.array_equal(np.asarray(cache['keyframe_ids']),np.asarray(sub['keyframe_ids'])): return False
    model,xyz,opacity=ctx.ns['mv_restore_gaussians'](sub)
    idx=cache['gaussian_index'].long().to(xyz.device)
    valid=bool(len(idx) and int(idx.max())<len(xyz) and int(idx.min())>=0)
    if valid:
        cur=xyz[idx].detach().cpu().numpy()
        old=np.asarray(cache['xyz'],dtype=np.float64)
        # float16 center storage: compare against the same precision, not a loose metric gate.
        valid=np.array_equal(cur.astype(old.dtype).astype(np.float16),old.astype(np.float16))
        valid=valid and np.array_equal(opacity[idx].detach().cpu().numpy().astype(np.float16),np.asarray(cache['opacity']).astype(np.float16))
    del model,xyz,opacity,idx; gc.collect()
    return valid

def sb_prepare_caches(ctx):
    ns=ctx.ns; target=ns['MV_MAX_LANDMARKS']; rows=[]
    for aid,sid in sorted(ctx.required):
        dest=ns['MV_CACHE_ROOT']/f'agent_{aid}'/f'{sid:06d}.pt'
        candidates=[dest]
        for d in sorted(ctx.root.glob('notebook*cache*')):
            candidates.extend([d/f'agent_{aid}'/f'{sid:06d}.pt',d/f'agent_{aid}'/f'{sid:03d}.pt'])
        chosen=None
        for p in candidates:
            if not p.exists(): continue
            cache=sb_torch.load(p,map_location='cpu',weights_only=False)
            if not sb_cache_compatible(cache,ctx,aid,sid): continue
            enough=len(cache['xyz'])>=target or int(cache.get('num_gaussians_multiview',10**12))<=len(cache['xyz'])
            if enough:
                chosen=(p,cache); break
        measured={}; built=False; changed=False
        if chosen:
            p,cache=chosen; hit=True
            # Reused times are historical; allocator peaks are not invented.
            measured={'cache_build_time_s_measured':None,'cache_build_time_s_historical':cache.get('build_time_s')}
        else:
            if not BENCH_BUILD_MISSING_CACHE: raise RuntimeError(f'Missing validated K={target} cache for {aid}/{sid}; enable BENCH_BUILD_MISSING_CACHE')
            if 'mv_feature_extractor' not in ns:
                from src.entities.loop_detection.feature_extractors import get_patch_feature_extractor
                t=time.perf_counter()
                ns['mv_feature_extractor']=get_patch_feature_extractor(ctx.cfg['loop_detection'],
                    {'registration_weights_path':ns['MV_MODEL_NAME'],'registration_device':ns['MV_DEVICE']},existing_extractor=None)
                ns['MV_EMBED_DIM']=int(ns['mv_feature_extractor'].embed_size)
                sb_sync(); ctx.model_init_s=time.perf_counter()-t
            calls=defaultdict(float); originals={}
            for name,category in [('mv_extract_feature_map','dino_feature_s'),('mv_bilinear_sample','bilinear_sample_s')]:
                original=ns[name]
                originals[name]=original
                def wrapped(*a,_fn=original,_cat=category,**kw):
                    sb_sync(); t=time.perf_counter(); ret=_fn(*a,**kw); sb_sync()
                    calls[_cat]+=time.perf_counter()-t; return ret
                ns[name]=wrapped
            ns['MV_REBUILD_CACHE']=True
            try:
                with SBMonitor() as monitor:
                    sb_sync(); t=time.perf_counter()
                    with contextlib.redirect_stdout(io.StringIO()): cache=ns['mv_build_cache'](aid,sid)
                    sb_sync(); elapsed=time.perf_counter()-t
            finally:
                ns['MV_REBUILD_CACHE']=False
                ns.update(originals)
            measured={**monitor.metrics,**calls,'cache_build_time_s_measured':elapsed,
                      'cache_build_time_s_historical':cache.get('build_time_s')}
            cache['sb_build_profile']=measured
            p=dest; hit=False; built=True; changed=True
        if BENCH_RUN_ABLATIONS and not {'scale_sorted','rotation_axes_sorted','rotation_axis_valid'}.issubset(cache):
            # An unsupported shape API must not block the main single/multi experiment.
            try:
                cache={**cache,**sb_shape_from_checkpoint(ctx.submaps[aid][sid],cache['gaussian_index'])}
                cache.pop('shape_metadata_error',None)
            except Exception as e:
                cache=dict(cache); cache['shape_metadata_error']=f'{type(e).__name__}: {e}'
            changed=True
        # Save validated private cache; never modify the notebook's original cache.
        dest.parent.mkdir(parents=True,exist_ok=True)
        if changed or p!=dest or not dest.exists():
            tmp=dest.with_suffix('.tmp.pt'); sb_torch.save(cache,tmp); tmp.replace(dest)
        ctx.caches[(aid,sid)]=cache
        prof=cache.get('sb_build_profile',{}) if hit else measured
        rows.append(dict(scene=ctx.scene,agent=aid,submap=sid,landmarks=len(cache['xyz']),
             eligible=cache.get('num_gaussians_multiview'),views=cache.get('num_views_used'),
             cache_hit=hit,cache_built_this_run=built,cache_sha256=sb_file_hash(dest),cache_content_sha256=sb_cache_content_hash(cache),
             source_cache_path=str(p),source_cache_version=cache.get('version'),
             gpu_profile_is_historical=hit,
             current_access_measured=not hit,construction_measurement='shared_builder_stores_both_variants',
             **{**prof,**measured}))
    ctx.cache_rows=rows
    pd.DataFrame(rows).to_csv(ctx.output/'descriptor_runtime.csv',index=False)
    return sb_digest([r['cache_content_sha256'] for r in rows])

def sb_profile_namespace(ns):
    totals=defaultdict(float); active=defaultdict(int)
    groups={
      'matching_s':['a_match_descriptors','v2_build_adaptive_bidirectional','v32_build_hard_candidates'],
      'geometry_s':['a_geometric_compatibility','mv_topk_rigidity_support'],
      'ransac_s':['a_ransac','v2_prosac_ransac','v3_prosac_ransac'],
      'local_refinement_s':['v32_robust_local_raw_candidates','v32_select_local_candidate','v33_irls_full',
          'v33_irls_translation','v33_irls_rotation','v331_validation_eval','v331_candidate_acceptable']}
    for category,names in groups.items():
        for name in names:
            original=ns[name]
            def wrapped(*a,_fn=original,_cat=category,**kw):
                outer=active[_cat]==0
                if outer: sb_sync(); t=time.perf_counter()
                active[_cat]+=1
                try: return _fn(*a,**kw)
                finally:
                    active[_cat]-=1
                    if outer: sb_sync(); totals[_cat]+=time.perf_counter()-t
            ns[name]=wrapped
    return totals

def sb_fpfh(ctx,aid,sid):
    key=(aid,sid)
    if key in ctx.fpfh_memo:
        ctx.fpfh_memo.move_to_end(key)
        down,feat,info=ctx.fpfh_memo[key]
        return down,feat,{**info,'cache_hit':True}
    cloud,_=ctx.dense(aid,sid); path=ctx.output/'fpfh_cache'/f'a{aid}_s{sid:06d}.npz'
    if path.exists():
        with np.load(path) as z:
            down=sb_o3d.geometry.PointCloud(); down.points=sb_o3d.utility.Vector3dVector(z['points'])
            down.normals=sb_o3d.utility.Vector3dVector(z['normals'])
            feat=sb_o3d.pipelines.registration.Feature(); feat.data=z['features']
        info=json.loads(path.with_suffix('.json').read_text())
        info={**info,'cache_hit':True}
    else:
        with SBMonitor() as monitor:
            t=time.perf_counter(); down=cloud.voxel_down_sample(BENCH_FPFH_VOXEL_M)
            down.estimate_normals(sb_o3d.geometry.KDTreeSearchParamHybrid(radius=2*BENCH_FPFH_VOXEL_M,max_nn=30))
            feat=sb_o3d.pipelines.registration.compute_fpfh_feature(down,
                sb_o3d.geometry.KDTreeSearchParamHybrid(radius=5*BENCH_FPFH_VOXEL_M,max_nn=100))
            elapsed=time.perf_counter()-t
        info={'feature_build_time_s':elapsed,'points':len(down.points),'cache_hit':False,**monitor.metrics}
        path.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(path,points=np.asarray(down.points),normals=np.asarray(down.normals),features=np.asarray(feat.data))
        sb_write_json(path.with_suffix('.json'),info)
    ctx.fpfh_rows[key]={'scene':ctx.scene,'agent':aid,'submap':sid,**info}
    pd.DataFrame(ctx.fpfh_rows.values()).to_csv(ctx.output/'fpfh_runtime.csv',index=False)
    ctx.fpfh_memo[key]=(down,feat,info)
    if len(ctx.fpfh_memo)>2: ctx.fpfh_memo.popitem(last=False)
    return down,feat,info

def sb_fpfh_register(ctx,info,seed):
    sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=info
    s,fs,is_=sb_fpfh(ctx,sa,si); t,ft,it_=sb_fpfh(ctx,ta,ti)
    sb_o3d.utility.random.seed(int(seed)); reg=sb_o3d.pipelines.registration
    dist=1.5*BENCH_FPFH_VOXEL_M
    t0=time.perf_counter()
    res=reg.registration_ransac_based_on_feature_matching(s,t,fs,ft,True,dist,
        reg.TransformationEstimationPointToPoint(False),4,
        [reg.CorrespondenceCheckerBasedOnEdgeLength(0.9),reg.CorrespondenceCheckerBasedOnDistance(dist)],
        reg.RANSACConvergenceCriteria(BENCH_FPFH_MAX_ITERS,BENCH_FPFH_CONFIDENCE))
    elapsed=time.perf_counter()-t0
    T0=sb_pose(res.transformation)
    good=len(res.correspondence_set)>=3
    return dict(T_est=sb_start_to_loop(T0,cs0,ct0,cs,ct) if good else None,
        n_ransac_inliers=len(res.correspondence_set),fpfh_ransac_fitness=float(res.fitness),
        fpfh_ransac_rmse_cm=100*float(res.inlier_rmse),matcher_runtime_s=elapsed,
        fpfh_source_points=len(s.points),fpfh_target_points=len(t.points),
        fpfh_feature_build_s_historical=is_['feature_build_time_s']+it_['feature_build_time_s'],
        fpfh_feature_build_s_new=sum(x['feature_build_time_s'] for x in [is_,it_] if not x['cache_hit']))

def sb_audit_pairs(sc,tc,cs,ct,T,gt):
    src=np.array(sc['descriptor_multi_view'],dtype=np.float64,copy=True); tgt=np.array(tc['descriptor_multi_view'],dtype=np.float64,copy=True)
    src/=np.maximum(np.linalg.norm(src,axis=1,keepdims=True),1e-12)
    tgt/=np.maximum(np.linalg.norm(tgt,axis=1,keepdims=True),1e-12)
    sim=src@tgt.T; j=sim.argmax(1); i=np.arange(len(src)); keep=sim.argmax(0)[j]==i; i=i[keep];j=j[keep]
    P=np.asarray(sc['xyz'],dtype=np.float64); Q=np.asarray(tc['xyz'],dtype=np.float64)
    P=(P-cs[:3,3])@cs[:3,:3]
    Q=(Q-ct[:3,3])@ct[:3,:3]
    out={'audit_mutual_pairs':len(i),'audit_mean_similarity':float(sim[i,j].mean()) if len(i) else None}
    for name,pose in [('estimated',T),('gt',gt)]:
        if pose is None or not len(i): continue
        r=sb_residual(pose,P[i],Q[j]); out.update({f'audit_{name}_paired_ratio_5cm':float((r<=0.05).mean()),
            f'audit_{name}_paired_ratio_3cm':float((r<=0.03).mean()),f'audit_{name}_paired_rmse_cm':100*float(np.sqrt(np.mean(r*r)))})
    return out

def sb_success(row,prefix):
    if not row.get('gt_available',False):
        for label in ['5deg_10cm','1deg_2cm','05deg_1cm']: row[f'{prefix}_success_{label}']=None
        return
    r,t=row.get(prefix+'_RRE_deg'),row.get(prefix+'_RTE_cm')
    for label,rr,tt in [('5deg_10cm',5,10),('1deg_2cm',1,2),('05deg_1cm',0.5,1)]:
        row[f'{prefix}_success_{label}']=bool(r is not None and t is not None and r<rr and t<tt)

def sb_attempt(ctx,spec,loop_id,loop,seed,run_hash):
    f=ctx.frame_info(loop); sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=f
    row={'scene':ctx.scene,'scope':BENCH_SCOPE,'candidate_id':loop_id,'seed':seed,**spec,
         'source_agent':sa,'source_frame':sf,'target_agent':ta,'target_frame':tf,'source_submap':si,'target_submap':ti,
         'baseline_admitted':sb_loop_key(loop) in ctx.admitted,'run_hash':run_hash,
         'status':'algorithm_failed','coarse_valid':False,'icp_evaluated':seed in BENCH_ICP_SEEDS,
         'final_valid':False,'filter_pass':False,'gt_available':False,'T_coarse':None,'T_final':None}
    sc=tc=None; totals={}; T=None
    try:
        if spec['method']=='FPFH_reference':
            # Feature/cloud preparation is one-time and excluded from query timer.
            _,_,sp=sb_fpfh(ctx,sa,si); _,_,tp=sb_fpfh(ctx,ta,ti)
            row['fpfh_preparation_new_s']=sum(x['feature_build_time_s'] for x in [sp,tp] if not x['cache_hit'])
        else:
            ns=sb_namespace(ctx); ns['A_MIN_MARGIN']=spec['margin']
            sc=sb_slice_cache(ctx.caches[(sa,si)],spec['k'],spec['view'])
            tc=sb_slice_cache(ctx.caches[(ta,ti)],spec['k'],spec['view'])
            row.update(n_source_landmarks=len(sc['xyz']),n_target_landmarks=len(tc['xyz']))
            if spec['geometry']=='distance_scale_relative_axis':
                if any('shape_metadata_error' in c for c in [sc,tc]):
                    raise RuntimeError('Shape ablation unavailable: '+str([c.get('shape_metadata_error') for c in [sc,tc]]))
                def shape_support(p,P,Q,threshold_m=0.03): return sb_shape_support(p,P,Q,sc,tc,threshold_m)
                ns['mv_topk_rigidity_support']=shape_support
            totals=sb_profile_namespace(ns)
        with SBMonitor() as monitor:
            sb_sync(); start=time.perf_counter()
            if spec['method']=='FPFH_reference': out=sb_fpfh_register(ctx,f,seed)
            else:
                name={'A_mnn':'a_registration_once','B_v32':'v32_registration_once','B_v331':'v331_registration_once'}[spec['method']]
                out=ns[name](sc,tc,cs,ct,seed)
            sb_sync(); elapsed=time.perf_counter()-start
        reserved=set(row)|{'matcher_runtime_s','T_est'}
        row.update({k:sb_jsonable(v) for k,v in out.items() if k not in reserved and (v is None or isinstance(v,(str,bool,int,float,np.generic)))})
        row['implementation_label']=out.get('method',spec['method'])
        for key,value in out.items():
            if isinstance(value,np.ndarray) and value.shape==(4,4) and key!='T_est': row[key]=sb_jsonable(value)
        row.update(matcher_runtime_s=elapsed,**dict(totals),**{'coarse_'+k:v for k,v in monitor.metrics.items()})
        if spec['method']=='FPFH_reference': row['fpfh_matching_ransac_s']=out['matcher_runtime_s']
        raw=out.get('T_est'); T=None if raw is None else sb_pose(raw)
        row['T_coarse']=None if T is None else T.tolist(); row['coarse_valid']=T is not None
        if T is not None:
            row['status']='done'
            if seed in BENCH_ICP_SEEDS:
                source,sinfo=ctx.dense(sa,si); target,tinfo=ctx.dense(ta,ti)
                T0=sb_loop_to_start(T,cs0,ct0,cs,ct)
                if not np.allclose(sb_start_to_loop(T0,cs0,ct0,cs,ct),T,atol=1e-8): raise AssertionError('ICP frame conversion round-trip failed')
                with SBMonitor() as im:
                    res=ctx.ns['exact_production_icp'](source,target,T0)
                final=sb_pose(sb_start_to_loop(res['T_final'],cs0,ct0,cs,ct))
                row.update(T_final=final.tolist(),final_valid=True,icp_runtime_s=res['runtime_s'],
                    initial_fitness_5mm=res['initial_fitness_5mm'],final_fitness_5mm=res['final_fitness_5mm'],
                    initial_rmse_cm=100*res['initial_rmse_5mm_m'],final_rmse_cm=100*res['final_rmse_5mm_m'],
                    dense_source_points=len(source.points),dense_target_points=len(target.points),
                    dense_build_s_historical=sinfo.get('build_time_s',0)+tinfo.get('build_time_s',0),
                    **{'icp_'+k:v for k,v in im.metrics.items()})
                thresholds=ctx.cfg['loop_detection']
                row['filter_pass']=bool(res['final_fitness_5mm']>float(thresholds['fitness_threshold'])
                       and np.isfinite(res['final_fitness_5mm']) and np.isfinite(res['final_rmse_5mm_m'])
                       and res['final_rmse_5mm_m']<float(thresholds['inlier_rmse_threshold']))
                row['online_runtime_s']=elapsed+res['runtime_s']
    except Exception as e:
        row['status']='error'; row['error']=f'{type(e).__name__}: {e}'
    # Ground truth is queried AFTER every decision, including failed ICP attempts.
    # A valid coarse estimate remains evaluable if only ICP raises an exception.
    try: gt=ctx.gt(loop); row['gt_available']=True
    except Exception as e: gt=None; row['gt_error']=str(e)
    if T is not None and gt is not None: row['coarse_RRE_deg'],row['coarse_RTE_cm']=sb_error(T,gt)
    if row['final_valid'] and gt is not None: row['final_RRE_deg'],row['final_RTE_cm']=sb_error(np.asarray(row['T_final']),gt)
    if sc is not None:
        try: row.update(sb_audit_pairs(sc,tc,cs,ct,T,gt))
        except Exception as e: row['audit_error']=str(e)
    sb_success(row,'coarse'); sb_success(row,'final')
    return sb_jsonable(row)

def sb_bootstrap(values,seed=1701):
    v=np.asarray(values,dtype=float);v=v[np.isfinite(v)]
    if len(v)<2: return [None,None]
    rng=np.random.default_rng(seed)
    means=np.mean(rng.choice(v,size=(4000,len(v)),replace=True),axis=1)
    return [float(x) for x in np.quantile(means,[0.025,0.975])]

def sb_summaries(df):
    rows=[]; keys=['scene','id','method','view','k','margin','geometry']
    for key,d in df.groupby(keys,dropna=False):
        base=dict(zip(keys,key)); base.update(attempts=len(d),unique_loops=d.candidate_id.nunique(),
            coarse_valid=int(d.coarse_valid.fillna(False).sum()),errors=int((d.status=='error').sum()),
            gt_evaluable=int(d.gt_available.fillna(False).sum()),
            icp_attempts=int(d.icp_evaluated.fillna(False).sum()),
            final_valid=int(d.final_valid.fillna(False).sum()),filter_pass=int(d.filter_pass.fillna(False).sum()))
        for stage in ['coarse','final']:
            dd=d if stage=='coarse' else d[d.icp_evaluated.fillna(False)]
            evaluable=dd[dd.gt_available.fillna(False)]
            for label in ['5deg_10cm','1deg_2cm','05deg_1cm']:
                col=f'{stage}_success_{label}'
                base[col+'_percent']=100*evaluable[col].eq(True).mean() if len(evaluable) else np.nan
            for metric in ['RRE_deg','RTE_cm']:
                col=f'{stage}_{metric}'
                v=pd.to_numeric(dd[col],errors='coerce').dropna() if col in dd else pd.Series(dtype=float)
                for stat,fn in [('mean',lambda x:x.mean()),('median',lambda x:x.median()),('max',lambda x:x.max()),('p90',lambda x:x.quantile(.9))]:
                    base[col+'_'+stat]=float(fn(v)) if len(v) else np.nan
                loop_means=dd.assign(**{col:pd.to_numeric(dd[col],errors='coerce')}).groupby('candidate_id')[col].mean().dropna() if col in dd else pd.Series(dtype=float)
                base[col+'_std_loop_mean']=float(loop_means.std()) if len(loop_means)>1 else np.nan
        for col in ['matching_s','geometry_s','ransac_s','local_refinement_s','matcher_runtime_s','icp_runtime_s','online_runtime_s',
                'audit_estimated_paired_ratio_5cm','audit_gt_paired_ratio_5cm','selected_inlier_ratio_5cm','selected_inlier_ratio_3cm',
                'final_fitness_5mm','final_rmse_cm','selected_pairs','n_ransac_inliers','fallback_triggered','fallback_selected',
                'v331_used_holdout','v331_rotation_allowed','v331_validation_gain','v331_candidate_count','v331_accepted_count',
                'fpfh_preparation_new_s','fpfh_feature_build_s_historical']:
            if col in d: base[col+'_mean']=pd.to_numeric(d[col],errors='coerce').mean()
        for col in [c for c in d if 'gpu_peak_' in c]: base[col+'_max']=pd.to_numeric(d[col],errors='coerce').max()
        if 'n_source_landmarks' in d:
            for col in ['n_source_landmarks','n_target_landmarks']:
                v=pd.to_numeric(d[col],errors='coerce');base[col+'_min']=v.min();base[col+'_mean']=v.mean()
            base['below_requested_K_attempts']=int(((d.n_source_landmarks<base['k'])|(d.n_target_landmarks<base['k'])).sum())
        rows.append(base)
    return pd.DataFrame(rows)

def sb_pairwise(df):
    pair_keys=['scene','candidate_id','seed','method','k','margin','geometry']
    s=df[df.view=='single_best_view']; m=df[df.view=='multi_view']
    paired=s.merge(m,on=pair_keys,suffixes=('_single','_multi'),validate='one_to_one')
    report=[]
    for (scene,method,k,margin,geometry),d in paired.groupby(['scene','method','k','margin','geometry']):
        for stage in ['coarse','final']:
            eligible=d if stage=='coarse' else d[d.icp_evaluated_single&d.icp_evaluated_multi]
            valid=eligible[eligible[f'{stage}_valid_single']&eligible[f'{stage}_valid_multi']&eligible.gt_available_single&eligible.gt_available_multi]
            result=dict(scene=scene,method=method,k=k,margin=margin,geometry=geometry,stage=stage,
                pairs=len(eligible),common_valid_pairs=len(valid),unique_loops=eligible.candidate_id.nunique())
            for metric in ['RRE_deg','RTE_cm']:
                a=f'{stage}_{metric}_single';b=f'{stage}_{metric}_multi'
                delta=pd.to_numeric(valid[b],errors='coerce')-pd.to_numeric(valid[a],errors='coerce')
                paired.loc[valid.index,f'{stage}_{metric}_delta_multi_minus_single']=delta
                grouped=delta.groupby(valid.candidate_id).mean().dropna()
                lo,hi=sb_bootstrap(grouped)
                result.update({metric+'_delta_mean':delta.mean(),metric+'_delta_loop_mean':grouped.mean(),metric+'_win_pairs':int((delta<0).sum()),
                    metric+'_delta_loop_ci_low':lo,metric+'_delta_loop_ci_high':hi})
            cc=f'{stage}_success_5deg_10cm'
            evaluable=eligible[eligible.gt_available_single&eligible.gt_available_multi]
            sv=evaluable[cc+'_single'].eq(True);mv=evaluable[cc+'_multi'].eq(True)
            result.update(gt_evaluable_pairs=len(evaluable),single_only_success=int((sv&~mv).sum()),multi_only_success=int((mv&~sv).sum()),
                both_success=int((sv&mv).sum()),both_failed=int((~sv&~mv).sum()))
            report.append(result)
    if not report:
        return paired,pd.DataFrame(columns=['scene','method','k','margin','geometry','stage','pairs','common_valid_pairs'])
    return paired,pd.DataFrame(report)

def sb_tex_table(folder,filename,frame,columns,caption,label):
    def esc(x):
        x={'B_v331':'V3.3.1','B_v32':'V3.2','A_mnn':'MNN','FPFH_reference':'FPFH',
           'single_best_view':'Single','multi_view':'Multi','geometry':'Geom.',
           'distance':'Dist.','distance_scale_relative_axis':'Dist.+scale+axes'}.get(str(x),x)
        return str(x).replace('\\',r'\textbackslash{}').replace('_',r'\_').replace('%',r'\%').replace('&',r'\&').replace('#',r'\#')
    lines=[r'\begin{table*}[t]',r'\centering',r'\caption{'+caption+'}',
        r'\resizebox{\textwidth}{!}{%',
        r'\begin{tabular}{'+' '.join('l' for _ in columns)+'}',r'\hline',
        ' & '.join(heading for _,heading in columns)+r' \\',r'\hline']
    for _,row in frame.iterrows():
        values=[]
        for field,_ in columns:
            value=row.get(field)
            if value is None or (not isinstance(value,(list,dict)) and pd.isna(value)): values.append('--')
            elif isinstance(value,(int,np.integer)): values.append(str(value))
            elif isinstance(value,(float,np.floating)): values.append(f'{float(value):.3f}')
            else: values.append(esc(value))
        lines.append(' & '.join(values)+r' \\')
    lines.extend([r'\hline',r'\end{tabular}',r'}',r'\label{'+label+'}',r'\end{table*}'])
    (Path(folder)/filename).write_text('\n'.join(lines))

def sb_report(folder,df,coverage,cache_df):
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    # Missing numeric fields are NaN; failures remain explicit rows.
    for c in ['coarse_RRE_deg','coarse_RTE_cm','final_RRE_deg','final_RTE_cm']:
        if c not in df: df[c]=np.nan
        df[c]=pd.to_numeric(df[c],errors='coerce')
    df.to_csv(folder/'per_run.csv',index=False)
    summary=sb_summaries(df); summary.to_csv(folder/'summary.csv',index=False)
    per_seed=pd.concat([sb_summaries(d).assign(seed=int(seed)) for seed,d in df.groupby('seed')],ignore_index=True)
    per_seed.to_csv(folder/'per_seed_summary.csv',index=False)
    paired,ps=sb_pairwise(df); paired.to_csv(folder/'paired_single_multi.csv',index=False)
    ps.to_csv(folder/'paired_summary.csv',index=False)
    df.groupby(['scene','id','candidate_id'],dropna=False).agg(
        runs=('seed','size'),coarse_RRE_deg=('coarse_RRE_deg','mean'),coarse_RTE_cm=('coarse_RTE_cm','mean'),
        final_RRE_deg=('final_RRE_deg','mean'),final_RTE_cm=('final_RTE_cm','mean')).reset_index().to_csv(folder/'per_loop.csv',index=False)
    df[['scene','candidate_id','source_agent','source_frame','target_agent','target_frame','source_submap','target_submap','baseline_admitted']].drop_duplicates().to_csv(folder/'loop_manifest.csv',index=False)
    pd.DataFrame(coverage).to_csv(folder/'dataset_coverage.csv',index=False)
    cache_df.to_csv(folder/'descriptor_runtime_all.csv',index=False)
    hard=df[df.get('fallback_triggered',pd.Series(False,index=df.index)).fillna(False).astype(bool)]
    hard.to_csv(folder/'hard_cases.csv',index=False)
    # Conditional paper subset: same baseline admission keys for every method.
    conditional=df[df.baseline_admitted.fillna(False)]
    if len(conditional): sb_summaries(conditional).to_csv(folder/'paper_common_summary.csv',index=False)
    primary=summary[(summary.method=='B_v331')&(summary.k==BENCH_PRIMARY_K)&
                    (summary.geometry=='distance')&(summary.margin==0)]
    columns=['scene','view','unique_loops','attempts','coarse_valid','coarse_success_5deg_10cm_percent',
        'coarse_RRE_deg_mean','coarse_RTE_cm_mean','final_RRE_deg_mean','final_RTE_cm_mean',
        'final_success_5deg_10cm_percent','filter_pass','matcher_runtime_s_mean','icp_runtime_s_mean','final_fitness_5mm_mean','final_rmse_cm_mean']
    primary[[c for c in columns if c in primary]].to_csv(folder/'paper_single_multi.csv',index=False)
    text=['KẾT QUẢ BENCHMARK SINGLE/MULTI — '+_SB_CELL_VERSION,
        'Sai số tịnh tiến/RMSE: cm. RRE: độ. Thời gian: giây. GPU: MiB.',
        'Mean/median sai số tính trên pose hợp lệ có GT; success rate giữ các cặp thất bại trong mẫu số. Error rows được báo riêng.',
        'Các seed RANSAC dùng chung một lần chạy SLAM. Bootstrap lấy mẫu theo loop đã gộp seed.',
        'Paired ratio theo pose ước lượng và theo GT được xuất riêng; không đồng nhất hai loại chỉ số.',
        'Dense ICP: point-to-plane 5 mm, tối đa 500 vòng lặp; giữ nguyên helper notebook gần đây.',
        'paper_common_summary đánh giá có điều kiện trên tập baseline đã nhận; không phải detection recall.',
        'Ablation hình học mới dùng scale và quan hệ giữa các hệ trục Gaussian, bỏ các trục có scale gần nhau; không so sánh quaternion thô giữa hai map.',
        'PSNR/SSIM/LPIPS/ATE giữ lại từ SLAM gốc chỉ là thông tin nền; chưa chạy lại PGO/render cho từng descriptor.',
        'Device GPU peak là số đo lấy mẫu và bao gồm tiến trình khác; peak PyTorch allocator được ghi riêng.','']
    for _,r in ps[(ps.method=='B_v331')&(ps.k==BENCH_PRIMARY_K)&(ps.geometry=='distance')].iterrows():
        text.append(f"{r.scene} / {'trước ICP' if r.stage=='coarse' else 'sau ICP'}: cả hai có pose hợp lệ {int(r.common_valid_pairs)}/{int(r.pairs)}; "
            f"delta Multi-Single RRE={r.RRE_deg_delta_mean:.6f} độ, RTE={r.RTE_cm_delta_mean:.6f} cm; "
            f"chỉ multi thành công={int(r.multi_only_success)}, chỉ single thành công={int(r.single_only_success)}.")
        text.append(f"CI 95% của delta trung bình theo loop: RRE=[{r.RRE_deg_delta_loop_ci_low}, {r.RRE_deg_delta_loop_ci_high}], "
                    f"RTE=[{r.RTE_cm_delta_loop_ci_low}, {r.RTE_cm_delta_loop_ci_high}]. Delta âm có lợi cho multi.")
        if r.common_valid_pairs==r.pairs and r.RRE_deg_delta_mean<0 and r.RTE_cm_delta_mean<0 and r.multi_only_success>=r.single_only_success:
            text.append('Multi có mean RRE và RTE thấp hơn tại scene/giai đoạn này; cần xem CI và các scene khác trước khi khái quát.')
        else: text.append('Chưa chứng minh multi thắng đồng đều trên scene/giai đoạn này; xem cả sai số, failure và success rate.')
    for (scene,view),d in df[(df.method=='B_v331')&(df.k==BENCH_PRIMARY_K)&(df.geometry=='distance')].groupby(['scene','view']):
        d=d[d.coarse_valid&d.final_valid&d.gt_available]
        if not len(d): continue
        change=d.final_RTE_cm-d.coarse_RTE_cm
        text.append(f'{scene}/{view}: trên {len(d)} trial có cả coarse/final, ICP làm RTE tốt hơn {(change<0).sum()} trial, '
                    f'tệ hơn {(change>0).sum()} trial; delta mean Final-Coarse={change.mean():.6f} cm.')
    for scene,d in paired[(paired.method=='B_v331')&(paired.k==BENCH_PRIMARY_K)&(paired.geometry=='distance')].groupby('scene'):
        same=d[d.coarse_valid_single&d.coarse_valid_multi&d.final_valid_single&d.final_valid_multi&d.gt_available_single&d.gt_available_multi]
        if len(same):
            c=(same.coarse_RTE_cm_multi-same.coarse_RTE_cm_single).mean()
            f=(same.final_RTE_cm_multi-same.final_RTE_cm_single).mean()
            text.append(f'{scene}: trên cùng {len(same)} paired trial có đủ cả 4 pose, độ lớn chênh lệch RTE giữa multi/single '
                +('thu hẹp' if abs(f)<abs(c) else 'không thu hẹp')+f' sau ICP ({c:.6f} -> {f:.6f} cm).')
    missing=[r for r in coverage if r.get('status')!='complete']
    if missing: text.append('CHƯA ĐỦ TOÀN BỘ DATASET/ĐÁNH GIÁ: '+json.dumps(sb_jsonable(missing),ensure_ascii=False))
    if 'below_requested_K_attempts' in summary:
        short=summary[summary.below_requested_K_attempts.fillna(0)>0]
        if len(short): text.append('Một số submap có ít landmark đủ điều kiện hơn K yêu cầu. Xem n_source_landmarks/n_target_landmarks và below_requested_K_attempts; không diễn giải chúng là đã dùng đủ K.')
    (folder/'assessment_vi.txt').write_text('\n'.join(text))
    # Complete machine-generated LaTeX: measured values only; no TODO filled by invented data.
    lines=[r'\begin{table*}[t]',r'\centering',r'\caption{Controlled single-best-view and multi-view registration using identical landmarks and frozen V3.3.1. Errors are means over valid estimates; success includes failed candidates. '+str(len(BENCH_SEEDS))+' seeds share one SLAM run.}',
        r'\begin{tabular}{l l r r r r r r r}',r'\hline',r'Scene & View & Loops & Trials & Coarse RRE & Coarse RTE (cm) & Final RRE & Final RTE (cm) & Final success (\%) \\',r'\hline']
    def fmt(x): return '--' if pd.isna(x) else f'{float(x):.3f}'
    for _,r in primary.iterrows():
        vals=[str(r.scene).replace('_',r'\_'),'Single' if r.view=='single_best_view' else 'Multi',str(int(r.unique_loops)),str(int(r.attempts)),
            fmt(r.coarse_RRE_deg_mean),fmt(r.coarse_RTE_cm_mean),fmt(r.final_RRE_deg_mean),fmt(r.final_RTE_cm_mean),fmt(r.final_success_5deg_10cm_percent)]
        lines.append(' & '.join(vals)+r' \\')
    lines.extend([r'\hline',r'\end{tabular}',r'\label{tab:single-multi-current}',r'\end{table*}'])
    (folder/'paper_single_multi.tex').write_text('\n'.join(lines))
    # All five ablation groups retain their actual method and settings.
    summary.to_csv(folder/'paper_all_ablations.csv',index=False)
    # Separate reviewable tables for each requested ablation and method history.
    cuts={
      'K_ablation':summary[(summary.method=='B_v331')&(summary.geometry=='distance')],
      'margin_ablation':summary[(summary.method=='A_mnn')&(summary.view=='multi_view')&(summary.k==BENCH_PRIMARY_K)],
      'geometry_ablation':summary[(summary.method=='B_v331')&(summary.view=='multi_view')&(summary.k==BENCH_PRIMARY_K)],
      'coarse_icp_fpfh':summary[((summary.method=='B_v331')&(summary.k==BENCH_PRIMARY_K)&(summary.geometry=='distance'))|(summary.method=='FPFH_reference')],
      'method_evolution':summary[(summary.k==BENCH_PRIMARY_K)&(summary.geometry=='distance')&(summary.margin==0)],
    }
    ablation_columns=[('scene','Scene'),('method','Method'),('view','View'),('k','K'),('margin','Margin'),
        ('geometry','Geometry'),('coarse_RRE_deg_mean','Coarse RRE'),('coarse_RTE_cm_mean','Coarse RTE (cm)'),
        ('final_RRE_deg_mean','Final RRE'),('final_RTE_cm_mean','Final RTE (cm)'),
        ('final_success_5deg_10cm_percent',r'Final success (\%)')]
    for name,cut in cuts.items():
        cut.to_csv(folder/(name+'.csv'),index=False)
        sb_tex_table(folder,name+'.tex',cut,ablation_columns,
            'Measured '+name.replace('_',' ')+' on fixed candidate loops. Error means condition on valid estimates; success retains failures. Shape filtering is experimental.',
            'tab:'+name.replace('_','-'))
    runtime_columns=[c for c in summary if c in ['scene','method','view','k','margin','geometry','attempts','coarse_valid','final_valid']
        or c.endswith('_s_mean') or 'gpu_peak_' in c]
    runtime=summary[runtime_columns]
    runtime.to_csv(folder/'runtime_per_pair.csv',index=False)
    sb_tex_table(folder,'runtime_per_pair.tex',runtime,
        [('scene','Scene'),('method','Method'),('view','View'),('k','K'),
         ('matcher_runtime_s_mean','Coarse (s)'),('matching_s_mean','Matching (s)'),('ransac_s_mean','RANSAC (s)'),
         ('icp_runtime_s_mean','ICP (s)'),('coarse_gpu_peak_torch_allocated_mb_max','Torch peak (MiB)'),
         ('coarse_gpu_peak_device_used_mb_max','Device peak (MiB)')],
        'Per-pair runtime excludes one-time model, descriptor and dense-cloud construction. Coarse includes matching, geometry, RANSAC and local refinement. Device peak is sampled and includes other processes.',
        'tab:runtime-pair')
    descriptor=[]
    if len(cache_df):
        for scene,d in cache_df.groupby('scene'):
            item={'scene':scene,'submaps':len(d),'fresh_builds':int(d.cache_built_this_run.fillna(False).sum()),
                  'construction':'shared builder constructs both descriptor variants'}
            for c in ['landmarks','views','cache_build_time_s_measured','cache_build_time_s_historical','dino_feature_s','bilinear_sample_s']:
                if c in d: item[c+'_mean']=pd.to_numeric(d[c],errors='coerce').mean()
            for c in [c for c in d if c.startswith('gpu_peak_') and c.endswith('_mb')]:
                item[c+'_max']=pd.to_numeric(d[c],errors='coerce').max()
            descriptor.append(item)
    dc=pd.DataFrame(descriptor);dc.to_csv(folder/'runtime_descriptor_shared.csv',index=False)
    sb_tex_table(folder,'runtime_descriptor_shared.tex',dc,
        [('scene','Scene'),('submaps','Submaps'),('fresh_builds','Fresh builds'),
         ('cache_build_time_s_measured_mean','Fresh mean (s)'),('cache_build_time_s_historical_mean','Stored mean (s)'),
         ('gpu_peak_torch_allocated_mb_max','Torch peak (MiB)'),('gpu_peak_device_used_mb_max','Device peak (MiB)')],
        'The shared descriptor builder stores single-best-view and multi-view descriptors for identical Gaussians. Historical cache times are distinct from fresh measurements; no individual-variant construction speedup is inferred.',
        'tab:runtime-descriptor')
    counts=df[['scene','candidate_id']].drop_duplicates().groupby('scene').size().to_dict()
    completion_text=(f'Observed unique candidate loops by scene: {counts}; total={sum(counts.values())}; '
        f'paper reference={BENCH_EXPECTED_TOTAL_LOOPS}. '+
        ('Reference count matches.' if sum(counts.values())==BENCH_EXPECTED_TOTAL_LOOPS else 'Reference count differs; do not label this as the same 21-loop experiment.'))
    with (folder/'assessment_vi.txt').open('a') as out: out.write('\n'+completion_text+'\n')
    sb_write_json(folder/'paper_scope_check.json',{'candidate_counts':counts,'total_unique_candidates':sum(counts.values()),
        'expected_paper_loops':BENCH_EXPECTED_TOTAL_LOOPS,'matches_paper_loop_count':sum(counts.values())==BENCH_EXPECTED_TOTAL_LOOPS})
    (folder/'paper_tables.tex').write_text('\n'.join(r'\input{'+name+'.tex}' for name in
        ['paper_single_multi','K_ablation','margin_ablation','geometry_ablation','coarse_icp_fpfh','method_evolution','runtime_per_pair','runtime_descriptor_shared']))
    try:
        import matplotlib.pyplot as plt
        scenes=list(primary.scene.unique());fig,ax=plt.subplots(1,2,figsize=(11,4))
        for a,metric,label in [(ax[0],'coarse_RTE_cm_mean','Coarse RTE (cm)'),(ax[1],'final_RTE_cm_mean','Final RTE (cm)')]:
            x=np.arange(len(scenes))
            for offset,view in [(-.18,'single_best_view'),(.18,'multi_view')]:
                vals=[float(primary[(primary.scene==s)&(primary.view==view)][metric].iloc[0])
                      if len(primary[(primary.scene==s)&(primary.view==view)]) else np.nan for s in scenes]
                a.bar(x+offset,vals,width=.36,label=view)
            a.set_xticks(x,scenes);a.set_ylabel(label);a.legend()
        fig.tight_layout();fig.savefig(folder/'single_multi_rte.png',dpi=220);plt.close(fig)
    except Exception as e: sb_write_json(folder/'plot_status.json',{'error':str(e)})
    return primary,summary,ps

def sb_pair_figure(ctx,row,reference_rows):
    if not BENCH_EXPORT_PAIR_FIGURES or row['seed']!=BENCH_SEEDS[0] or row['method']!='B_v331' or row['k']!=BENCH_PRIMARY_K or row['geometry']!='distance': return
    if not row.get('T_coarse'): return
    import matplotlib.pyplot as plt
    l=ctx.loops[int(row['candidate_id'])];sa,sf,ta,tf,si,ti,cs,ct,cs0,ct0=ctx.frame_info(l)
    s,_=ctx.dense(sa,si);t,_=ctx.dense(ta,ti)
    P=np.asarray(s.points);Q=np.asarray(t.points)
    # Deterministic uniform sampling, identical across methods; display has no role in inference.
    P=P[np.linspace(0,len(P)-1,min(12000,len(P)),dtype=int)];Q=Q[np.linspace(0,len(Q)-1,min(12000,len(Q)),dtype=int)]
    transforms=[np.eye(4),sb_loop_to_start(np.asarray(row['T_coarse']),cs0,ct0,cs,ct),
        None if not row.get('T_final') else sb_loop_to_start(np.asarray(row['T_final']),cs0,ct0,cs,ct)]
    common_transforms=[np.eye(4)]
    for other in reference_rows:
        if other['candidate_id']!=row['candidate_id'] or other['seed']!=row['seed'] or other['method']!='B_v331' or other['k']!=BENCH_PRIMARY_K or other['geometry']!='distance': continue
        for field in ['T_coarse','T_final']:
            if other.get(field): common_transforms.append(sb_loop_to_start(np.asarray(other[field]),cs0,ct0,cs,ct))
    fig=plt.figure(figsize=(15,4));combined=np.vstack([Q]+[P@T[:3,:3].T+T[:3,3] for T in common_transforms])
    low,high=combined.min(0),combined.max(0);center=(low+high)/2;radius=max((high-low).max()/2,0.1)
    for i,(name,T) in enumerate(zip(['Unaligned','Coarse','ICP'],transforms)):
        ax=fig.add_subplot(1,3,i+1,projection='3d');ax.scatter(*Q.T,s=.15,c='#dd8023',alpha=.6)
        if T is not None:
            R=P@T[:3,:3].T+T[:3,3];ax.scatter(*R.T,s=.15,c='#2372b3',alpha=.6)
        ax.set_title(name);ax.set_xlim(center[0]-radius,center[0]+radius);ax.set_ylim(center[1]-radius,center[1]+radius);ax.set_zlim(center[2]-radius,center[2]+radius)
        ax.set_box_aspect([1,1,1]);ax.set_axis_off();ax.view_init(22,-55)
    fig.suptitle(f"{ctx.scene} candidate {row['candidate_id']} {row['view']}");fig.tight_layout()
    d=ctx.output/'figures';d.mkdir(exist_ok=True)
    fig.savefig(d/f"loop{row['candidate_id']:03d}_{row['view']}_coarse_icp.png",dpi=220);plt.close(fig)

def sb_resolve_runs():
    explicit={s:Path(p) for s,p in BENCH_RUNS.items()}; found=defaultdict(list)
    current=globals().get('OUTPUT_ROOT')
    candidates=[]
    if current: candidates.append(Path(current))
    for root in BENCH_SEARCH_ROOTS:
        if root.exists(): candidates.extend(p.parent for p in root.rglob('detected_loops.pkl'))
    import re
    for p in sorted(set(candidates)):
        if not (p/'config.yaml').exists(): continue
        try:
            cfg=sb_load_config(str(p/'config.yaml'))
            text=(str(cfg['data'].get('input_path',''))+' '+str(p)).lower().replace('-','_')
            hits=re.findall(r'(?<![a-z])(office|apart|apartment)_?(\d+)',text)
            names={('apart' if a=='apartment' else a)+'_'+b for a,b in hits}
            for scene in BENCH_SCENES:
                if scene in names: found[scene].append(p)
        except Exception: continue
    runs={}; coverage=[]
    for scene in BENCH_SCENES:
        if scene in explicit: runs[scene]=explicit[scene]
        elif current and Path(current) in found[scene]: runs[scene]=Path(current)
        elif len(found[scene])==1: runs[scene]=found[scene][0]
        else: coverage.append({'scene':scene,'status':'missing' if not found[scene] else 'ambiguous',
                    'candidates':[str(p) for p in found[scene]],'action':'Set BENCH_RUNS with the exact finished SLAM output folder'})
    return runs,coverage

def sb_run():
    global sb_torch,sb_o3d,sb_F,sb_Image,sb_load_config,sb_get_dataset,sb_repo_utils
    if str(BENCH_REPO) not in sys.path: sys.path.insert(0,str(BENCH_REPO))
    import torch as sb_torch
    import open3d as sb_o3d
    import torch.nn.functional as sb_F
    from PIL import Image as sb_Image
    from src.utils.io_utils import load_config as sb_load_config
    from src.entities.datasets import get_dataset as sb_get_dataset
    from src.utils import utils as sb_repo_utils
    if not sb_torch.cuda.is_available(): raise RuntimeError('Run in the recent Kaggle CUDA environment; descriptor/render-depth construction needs CUDA')
    sb_torch.cuda.set_device(BENCH_GPU_DEVICE)
    if set(BENCH_ICP_SEEDS)-set(BENCH_SEEDS): raise ValueError('ICP seeds must be a subset of coarse seeds')
    specs=sb_specs()
    try: head=subprocess.check_output(['git','-C',str(BENCH_REPO),'rev-parse','HEAD'],text=True).strip()
    except Exception: head='unknown'
    protocol={'version':_SB_CELL_VERSION,'repo_head':head,'helpers':_SB_PROVENANCE,
        'driver_sha256':_SB_DRIVER_SHA256,'specs':specs,'seeds':BENCH_SEEDS,'icp_seeds':BENCH_ICP_SEEDS,'scope':BENCH_SCOPE,
        'expected_paper_total_loops':BENCH_EXPECTED_TOTAL_LOOPS,
        'repo_python_sha256':sb_digest([(str(p.relative_to(BENCH_REPO)),sb_file_hash(p)) for p in sorted((BENCH_REPO/'src').rglob('*.py'))]),
        'shape':{'scale_log_threshold':BENCH_SCALE_LOG_THRESHOLD,'relative_axis_cos_threshold':BENCH_AXIS_COS_THRESHOLD,
                 'anisotropy_min':BENCH_AXIS_ANISOTROPY_MIN,'quaternion_order':BENCH_QUATERNION_ORDER,'experimental':True},
        'fpfh':{'voxel_m':BENCH_FPFH_VOXEL_M,'ransac_max_iters':BENCH_FPFH_MAX_ITERS,'confidence':BENCH_FPFH_CONFIDENCE},
        'torch':sb_torch.__version__,'open3d':sb_o3d.__version__,'cuda':sb_torch.version.cuda,
        'gpu_model':sb_torch.cuda.get_device_name(BENCH_GPU_DEVICE),
        'gpu_poll_seconds':BENCH_GPU_POLL_SECONDS,'frame_convention':'source loop camera -> target loop camera; explicit conversion for first-submap-frame dense ICP'}
    protocol_hash=sb_digest(protocol); root=BENCH_OUTPUT/protocol_hash[:16];root.mkdir(parents=True,exist_ok=True)
    sb_write_json(root/'protocol.json',protocol)
    runs,coverage=sb_resolve_runs();all_rows=[];all_cache=[];expected_by_scene={};active_outputs=set()
    for scene,path in runs.items():
        print(f'[{scene}] Load fixed candidates and finalized submaps: {path}',flush=True)
        try:
            ctx=SBContext(scene,path,root/scene)
            # Namespace/config and input identity are frozen before any variant runs.
            ctx.output=root/scene/ctx.input_hash[:16];ctx.output.mkdir(parents=True,exist_ok=True)
            ctx.ns=sb_namespace(ctx);cache_hash=sb_prepare_caches(ctx)
            run_hash=sb_digest([protocol_hash,ctx.input_hash,cache_hash])
            expected_by_scene[scene]=len(ctx.loops)
            sb_write_json(ctx.output/'run_meta.json',{'scene':scene,'root':str(path),'protocol_hash':protocol_hash,
                'run_hash':run_hash,'input_fingerprint':ctx.input_fingerprint,'cache_hash':cache_hash,
                'fixed_loop_keys':[sb_loop_key(l) for l in ctx.loops],
                'model_init_s':getattr(ctx,'model_init_s',None),'candidate_count':len(ctx.loops),
                'expected_rows':len(ctx.loops)*len(BENCH_SEEDS)*len(specs)})
            # Preserve system metrics without claiming they changed under pairwise ablation.
            system={}
            for name in ['coarse_render_metrics.json','fine_render_metrics.json','novel_render_metrics.json',
                         'walltime_stats.json','registration_summary.json','ate.json','loop_pipeline_summary.json']:
                p=ctx.root/name
                if p.exists():
                    try: system[name]=json.loads(p.read_text())
                    except Exception: system[name]={'source_file':str(p),'parse_failed':True}
            sb_write_json(ctx.output/'original_system_metrics_context.json',system)
            local=[];checkpoints=ctx.output/'checkpoints';checkpoints.mkdir(exist_ok=True)
            for lid,loop in enumerate(ctx.loops):
                for seed in BENCH_SEEDS:
                    # Alternate pair order by seed/loop, independent of GT, to limit order bias.
                    ordered=list(specs)
                    if (lid+seed)%2: ordered=list(reversed(ordered))
                    for spec in ordered:
                        token=sb_digest([run_hash,sb_loop_key(loop),seed,spec])
                        p=checkpoints/(token+'.json');row=None
                        if p.exists():
                            old=json.loads(p.read_text())
                            if old.get('run_hash')==run_hash and (old.get('status')!='error' or not BENCH_RETRY_ERRORS): row=old
                        if row is None:
                            row=sb_attempt(ctx,spec,lid,loop,seed,run_hash);sb_write_json(p,row)
                            if BENCH_VERBOSE:
                                print(f"[{scene}] loop={lid} seed={seed} {spec['id']}: {row['status']}; "
                                    f"coarse={row.get('coarse_RRE_deg')}/{row.get('coarse_RTE_cm')} final={row.get('final_RRE_deg')}/{row.get('final_RTE_cm')}",flush=True)
                        local.append(row)
                    for figure_row in [r for r in local if r['candidate_id']==lid and r['seed']==seed]:
                        try: sb_pair_figure(ctx,figure_row,local)
                        except Exception as e: sb_write_json(ctx.output/'figure_error.json',{'error':str(e)})
                    # Persist CSV after each loop/seed as well as each atomic checkpoint.
                    pd.DataFrame(local).to_csv(ctx.output/'per_run_partial.csv',index=False)
                    print(f'[{scene}] loop={lid} seed={seed}: {len(local)}/{len(ctx.loops)*len(BENCH_SEEDS)*len(specs)} saved rows',flush=True)
            local_df=pd.DataFrame(local);cd=pd.DataFrame(ctx.cache_rows)
            status='complete' if not (local_df.status=='error').any() else 'complete_with_errors'
            if not local_df.gt_available.fillna(False).all(): status='complete_without_gt'
            coverage.append({'scene':scene,'status':status,'run_hash':run_hash,'candidates':len(ctx.loops),
                'expected_rows':len(ctx.loops)*len(BENCH_SEEDS)*len(specs),'rows':len(local),
                'error_rows':int((local_df.status=='error').sum())})
            all_rows.extend(local);all_cache.extend(ctx.cache_rows)
            active_outputs.add(ctx.output)
            sb_report(ctx.output/'report',local_df,[coverage[-1]],cd)
            del ctx;gc.collect();sb_torch.cuda.empty_cache()
        except Exception as e:
            coverage.append({'scene':scene,'status':'setup_error','error':f'{type(e).__name__}: {e}'})
            print(f'[{scene}] SETUP ERROR: {e}',flush=True)
        sb_write_json(root/'dataset_coverage.json',coverage)
    # Import only identical protocol results, detect duplicate/conflicting runs.
    for base in BENCH_IMPORT_RESULTS:
        for p in Path(base).rglob('run_meta.json'):
            meta=json.loads(p.read_text())
            if meta.get('protocol_hash')!=protocol_hash: continue
            if meta.get('scene') not in BENCH_SCENES: continue
            expected_by_scene[meta['scene']]=int(meta['candidate_count'])
            dest=p.parent if p.parent.is_relative_to(root) else root/'imports'/meta['scene']/meta['run_hash'][:16]
            active_outputs.add(dest)
            if dest!=p.parent:
                sb_write_json(dest/'run_meta.json',meta)
                for name in ['descriptor_runtime.csv','dense_runtime.csv','fpfh_runtime.csv','original_system_metrics_context.json']:
                    if (p.parent/name).exists(): (dest/name).write_bytes((p.parent/name).read_bytes())
            for c in (p.parent/'checkpoints').glob('*.json'):
                row=json.loads(c.read_text())
                if row.get('run_hash')==meta['run_hash']:
                    all_rows.append(row)
                    if dest!=p.parent: sb_write_json(dest/'checkpoints'/c.name,row)
            if (p.parent/'descriptor_runtime.csv').exists():
                all_cache.extend(pd.read_csv(p.parent/'descriptor_runtime.csv').to_dict('records'))
    if not all_rows:
        raise RuntimeError('No runnable datasets. See '+str(root/'dataset_coverage.json'))
    df=pd.DataFrame(all_rows)
    # One run per scene is required; no cherry-picking across repeated SLAM runs.
    if (df.groupby('scene').run_hash.nunique()>1).any(): raise RuntimeError('Multiple distinct SLAM runs for one scene; select BENCH_RUNS/imports explicitly')
    # Prefer an executed attempt over a technical exception; otherwise current
    # local rows precede imports. Never select rows by success or pose error.
    df=df.assign(_technical_error=(df.status=='error').astype(int)).sort_values('_technical_error',kind='stable')
    df=df.drop_duplicates(['run_hash','id','candidate_id','seed'],keep='first').drop(columns='_technical_error')
    # Imported finished scenes can complete coverage; verify the entire fixed matrix.
    for scene,d in df.groupby('scene'):
        nloops=expected_by_scene[scene];expected=nloops*len(specs)*len(BENCH_SEEDS)
        all_keys={(int(lid),int(seed),s['id']) for lid in range(nloops) for seed in BENCH_SEEDS for s in specs}
        observed=set(zip(d.candidate_id.astype(int),d.seed.astype(int),d.id))
        complete=all_keys==observed
        status='partial' if not complete else ('complete_with_errors' if (d.status=='error').any() else ('complete_without_gt' if not d.gt_available.fillna(False).all() else 'complete'))
        coverage=[c for c in coverage if c.get('scene')!=scene]
        coverage.append({'scene':scene,'status':status,'run_hash':str(d.run_hash.iloc[0]),'candidates':int(nloops),
            'expected_rows':int(expected),'rows':len(d),'error_rows':int((d.status=='error').sum())})
    cache_df=pd.DataFrame(all_cache)
    if len(cache_df): cache_df=cache_df.drop_duplicates(['scene','agent','submap','cache_content_sha256'],keep='first')
    primary,summary,paired=sb_report(root/'all_datasets',df,coverage,cache_df)
    sb_write_json(root/'dataset_coverage.json',coverage)
    sb_write_json(root/'completion.json',{'protocol_hash':protocol_hash,'scenes':coverage,'rows':len(df),
        'complete_requested_scenes':all(any(x.get('scene')==s and x.get('status')=='complete' for x in coverage) for s in BENCH_SCENES)})
    export=BENCH_OUTPUT/('single_multi_v331_'+protocol_hash[:16]+'_results.zip')
    with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(root.rglob('*')):
            current=p.parent==root or p.is_relative_to(root/'all_datasets') or any(p.is_relative_to(d) for d in active_outputs)
            if p.is_file() and current and not any(x in p.parts for x in ['dense_cache','descriptor_cache','fpfh_cache']): z.write(p,p.relative_to(root))
    try:
        from IPython.display import display
        display(primary); display(paired[(paired.method=='B_v331')&(paired.k==BENCH_PRIMARY_K)])
    except Exception: print(primary.to_string(index=False))
    print('RESULTS:',root/'all_datasets');print('EXPORT ZIP:',export)
    print('Dataset coverage:',coverage)
    return {'root':root,'zip':export,'per_run':df,'summary':summary,'paired':paired,'primary':primary}

def sb_launch():
    target=Path(BENCH_PYTHON)
    if not target.exists(): raise FileNotFoundError('Benchmark Python not found: '+str(target))
    if target.absolute()==Path(sys.executable).absolute(): return sb_run()
    # The latest paper-assets notebook builds extensions in PY311, not in its kernel.
    # Pass actual edited options to that interpreter, stream progress, retain full log.
    BENCH_OUTPUT.mkdir(parents=True,exist_ok=True)
    worker=BENCH_OUTPUT/'run_single_multi_v331.py';params=BENCH_OUTPUT/'benchmark_options.json'
    configs={k:sb_jsonable(v) for k,v in globals().items() if k.startswith('BENCH_')}
    if globals().get('OUTPUT_ROOT') is not None: configs['OUTPUT_ROOT']=str(globals()['OUTPUT_ROOT'])
    sb_write_json(params,configs)
    tail="""\nif __name__ == '__main__':
    options=json.loads(Path(sys.argv[1]).read_text())
    globals().update(options)
    BENCH_REPO=Path(BENCH_REPO); BENCH_OUTPUT=Path(BENCH_OUTPUT)
    BENCH_SEARCH_ROOTS=[Path(p) for p in BENCH_SEARCH_ROOTS]
    result=sb_run()
    sb_write_json(BENCH_OUTPUT/'last_result.json',{'root':str(result['root']),'zip':str(result['zip'])})
"""
    worker.write_text(_SB_WORKER_SOURCE+tail)
    log=BENCH_OUTPUT/'benchmark_console.log';env=os.environ.copy()
    env.setdefault('CUDA_HOME','/usr/local/cuda');env.setdefault('PYTHONUNBUFFERED','1')
    print('Using recent notebook interpreter:',target,flush=True)
    with log.open('a') as out:
        with subprocess.Popen([str(target),str(worker),str(params)],cwd=BENCH_REPO,env=env,
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
            try:
                for line in proc.stdout: out.write(line);out.flush();print(line,end='',flush=True)
                code=proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=15)
                except subprocess.TimeoutExpired: proc.kill();proc.wait()
                raise
    if code: raise RuntimeError(f'Benchmark worker exited {code}; see {log}')
    result=json.loads((BENCH_OUTPUT/'last_result.json').read_text());folder=Path(result['root'])/'all_datasets'
    result.update(primary=pd.read_csv(folder/'paper_single_multi.csv'),summary=pd.read_csv(folder/'summary.csv'),
                  paired=pd.read_csv(folder/'paired_summary.csv'),per_run=pd.read_csv(folder/'per_run.csv'))
    return result

if BENCH_EXECUTE:
    SINGLE_MULTI_BENCHMARK = sb_launch()


## 12. Xem ảnh pipeline

In [ ]:
from IPython.display import display,Image as DisplayImage
for scene,path in BENCH_RUNS.items():
    print('Scene:',scene)
    images=sorted((Path(path)/'pipeline_assets_v2'/'images').glob('*review*.png'))
    if not images: images=sorted((Path(path)/'pipeline_assets'/'images').glob('*contact_sheet*.png'))
    for image in images[:1]:display(DisplayImage(filename=str(image),width=1400))


## 13. Đóng gói metric và frame

In [ ]:
import zipfile
ZIP_PATH=Path('/kaggle/working/office0_single_multi_metrics_frames.zip')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as archive:
    for path in FULL_SLAM_OUTPUT_BASE.rglob('*'):
        if not path.is_file() or any('_partial_' in part for part in path.parts):continue
        if path.suffix in {'.ply','.ckpt','.pt'}:continue  # bản đồ/submap giữ riêng, không mất khỏi output
        archive.write(path,Path('full_slam')/path.relative_to(FULL_SLAM_OUTPUT_BASE))
    if 'SINGLE_MULTI_BENCHMARK' in globals():
        result_zip=Path(SINGLE_MULTI_BENCHMARK['zip'])
        archive.write(result_zip,Path('single_multi')/result_zip.name)
    for path in TOOLS_DIR.iterdir():
        if path.is_file():archive.write(path,Path('scripts_and_setup')/path.name)
from IPython.display import FileLink
print('ZIP metrics/frame:',ZIP_PATH)
display(FileLink(str(ZIP_PATH)))
print('Giữ riêng PLY/CKPT và RGB-D để tiếp tục:',FULL_SLAM_OUTPUT_BASE,FULL_SLAM_STAGING_ROOT)


## Optional: giảm Gaussian sau full SLAM

Phần này chỉ chạy backend trên submap và pose đã hoàn tất; không chạy lại tracking, loop retrieval, registration hoặc PGO. Mặc định tắt để bảo toàn baseline. Bật `RUN_OPTIONAL_GS_COMPACTION=True` và luôn giữ output mới tách khỏi kết quả gốc. Profile mặc định: voxel 5 mm, tối đa 2 triệu Gaussian.

Với Magic-SLAM, kết quả compact phải được báo dưới tên **Magic-SLAM + compaction**, không thay thế số liệu baseline gốc.

In [ ]:
# Optional post-PGO Gaussian compaction. It is disabled by default.
from pathlib import Path
import json, os, shutil, subprocess, sys

RUN_OPTIONAL_GS_COMPACTION = False
COMPACTION_METHOD_LABEL = "single_multi"
COMPACTION_SOURCE_OUTPUT = None  # Set an exact completed output path if auto-detection is ambiguous.
COMPACTION_TARGET_MAX_GAUSSIANS = 2_000_000
COMPACTION_VOXEL_SIZE_M = 0.005
COMPACTION_MIN_OPACITY = 0.05
COMPACTION_REFINE_ITERATIONS = 3000
COMPACTION_EXPORT_FRAMES = True
COMPACTION_ZIP_FINAL_MAP = True

# Final triangle-mesh export. This runs after full SLAM even when compaction is disabled.
EXPORT_FINAL_MESH = True
MESH_SOURCE_OUTPUT = None  # Set an exact completed output path if auto-detection is ambiguous.
MESH_VOXEL_SIZE_M = 0.010
MESH_MAX_INPUT_POINTS = 250_000
MESH_POISSON_DEPTH = 9
MESH_DENSITY_QUANTILE = 0.01
MESH_PREVIEW_TRIANGLES = 150_000
MESH_ZIP_INCLUDE_GAUSSIAN_MAP = True

COMPACT_EXPECTED_REPO_COMMIT = "80a05c955d67f7be666d1c96ab26244843e7e7c8"
COMPACT_REPO = Path("/kaggle/working/3DGS-SLAM-TURTLEBOT4-gscompact")
COMPACT_TOOLS_DIR = Path("/kaggle/working/global_compaction_setup")
COMPACT_TOOLS_DIR.mkdir(parents=True, exist_ok=True)
PY311 = Path(globals().get("PY311", "/kaggle/working/gs_slam_py311/bin/python"))


def _is_frontend_output(path):
    path = Path(path)
    if not (path / "config.yaml").is_file():
        return False
    try:
        cfg = __import__("yaml").safe_load((path / "config.yaml").read_text())
        agent_ids = [int(x) for x in cfg["data"]["agent_ids"]]
    except Exception:
        return False
    for aid in agent_ids:
        if not (path / f"agent_{aid}" / "optimized_kf_c2w.ckpt").is_file():
            return False
        if not list((path / f"agent_{aid}" / "submaps").glob("*.ckpt")):
            return False
    return True


def _source_candidates():
    candidates = []
    for name in ["OUTPUT_ROOT", "BASELINE_OUTPUT", "PATCHED_OUTPUT"]:
        value = globals().get(name)
        if value and _is_frontend_output(Path(value)):
            candidates.append(Path(value).resolve())
    for name in ["OUTPUT_BASE", "FULL_SLAM_OUTPUT_BASE"]:
        value = globals().get(name)
        if value:
            base = Path(value)
            if _is_frontend_output(base):
                candidates.append(base.resolve())
            if base.exists():
                candidates.extend(
                    p.parent.resolve() for p in base.rglob("config.yaml")
                    if _is_frontend_output(p.parent)
                )
    unique = []
    for path in candidates:
        if path not in unique:
            unique.append(path)
    return unique


if RUN_OPTIONAL_GS_COMPACTION:
    if not PY311.is_file():
        raise FileNotFoundError(PY311)
    if COMPACTION_SOURCE_OUTPUT is None:
        detected = _source_candidates()
        if len(detected) != 1:
            raise RuntimeError(
                "Set COMPACTION_SOURCE_OUTPUT to one exact completed SLAM output. "
                f"Detected candidates: {[str(x) for x in detected]}"
            )
        COMPACTION_SOURCE_OUTPUT = detected[0]
    else:
        COMPACTION_SOURCE_OUTPUT = Path(COMPACTION_SOURCE_OUTPUT).resolve()
    if not _is_frontend_output(COMPACTION_SOURCE_OUTPUT):
        raise RuntimeError(
            "Source must contain config.yaml, finalized submaps and optimized poses: "
            + str(COMPACTION_SOURCE_OUTPUT)
        )

    import yaml
    source_cfg = yaml.safe_load((COMPACTION_SOURCE_OUTPUT / "config.yaml").read_text())
    COMPACTION_SCENE = str(source_cfg["data"].get("scene_name", "scene"))
    COMPACTION_SEED = int(source_cfg.get("seed", 0))
    COMPACTION_OUTPUT = Path(
        f"/kaggle/working/gslam_outputs/{COMPACTION_METHOD_LABEL}_global_compaction_2m/"
        f"{COMPACTION_SCENE}/seed_{COMPACTION_SEED}"
    )
    COMPACTION_CONFIG = COMPACT_TOOLS_DIR / (
        f"{COMPACTION_METHOD_LABEL}_{COMPACTION_SCENE}_2m.yaml"
    )
    profile = {
        "enabled": True,
        "min_opacity": COMPACTION_MIN_OPACITY,
        "voxel_size_m": COMPACTION_VOXEL_SIZE_M,
        "max_gaussians": COMPACTION_TARGET_MAX_GAUSSIANS,
        "refine_iterations": COMPACTION_REFINE_ITERATIONS,
        "refine_prune_min_opacity": 0.01,
        "refine_prune_interval": 100,
        "refine_prune_start_fraction": 0.2,
        "refine_prune_end_fraction": 0.8,
    }
    source_cfg["global_compaction"] = profile
    source_cfg["data"]["output_path"] = str(COMPACTION_OUTPUT)
    COMPACTION_CONFIG.write_text(yaml.safe_dump(source_cfg, sort_keys=False))
    print("Source:", COMPACTION_SOURCE_OUTPUT)
    print("Output:", COMPACTION_OUTPUT)
    print("Profile:", json.dumps(profile, indent=2))
else:
    print("Optional Gaussian compaction is disabled. Set RUN_OPTIONAL_GS_COMPACTION=True to run it.")


### Chuẩn bị checkout và áp dụng patch backend

In [ ]:
if RUN_OPTIONAL_GS_COMPACTION:
    repo_url = "https://github.com/DucVu2610/3DGS-SLAM-TURTLEBOT4.git"
    if not (COMPACT_REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--no-checkout", repo_url, str(COMPACT_REPO)], check=True)
        subprocess.run([
            "git", "-C", str(COMPACT_REPO), "checkout", "--detach",
            COMPACT_EXPECTED_REPO_COMMIT,
        ], check=True)
    actual = subprocess.check_output(
        ["git", "-C", str(COMPACT_REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != COMPACT_EXPECTED_REPO_COMMIT:
        raise RuntimeError(
            f"Compaction checkout is {actual}, expected {COMPACT_EXPECTED_REPO_COMMIT}. "
            f"Use a clean directory: {COMPACT_REPO}"
        )
    print("Compaction checkout:", actual)
else:
    print("Skipped compaction checkout.")


In [ ]:
if RUN_OPTIONAL_GS_COMPACTION:
    # Apply the reviewed post-PGO compaction patch idempotently.
    PATCH_TEXT = 'diff --git a/configs/ReplicaMultiagent/office_0.yaml b/configs/ReplicaMultiagent/office_0.yaml\nindex 5e3f0a7..62665b7 100644\n--- a/configs/ReplicaMultiagent/office_0.yaml\n+++ b/configs/ReplicaMultiagent/office_0.yaml\n@@ -4,4 +4,17 @@ data:\n   frame_limit: 1500\n   scene_name: office_0\n   input_path: ReplicaMultiagent/Office-0/\n-  output_path: output/CP-SLAM_dataset/Office-0/\n\\ No newline at end of file\n+  output_path: output/CP-SLAM_dataset/Office-0/\n+\n+# Applied only after pose-graph optimization. Front-end mapping, loop\n+# detection, Gaussian landmark registration and PGO remain unchanged.\n+global_compaction:\n+  enabled: true\n+  min_opacity: 0.05\n+  voxel_size_m: 0.003\n+  max_gaussians: 4500000\n+  refine_iterations: 3000\n+  refine_prune_min_opacity: 0.01\n+  refine_prune_interval: 100\n+  refine_prune_start_fraction: 0.2\n+  refine_prune_end_fraction: 0.8\ndiff --git a/src/entities/magic_slam.py b/src/entities/magic_slam.py\nindex 85827f2..ad610e1 100644\n--- a/src/entities/magic_slam.py\n+++ b/src/entities/magic_slam.py\n@@ -223,11 +223,14 @@ class MAGiCSLAM(object):\n             agents_kf_ids[agent_id] = agents_kf_ids[agent_id].astype(int)\n         opt_args = self.agents[0].mapper.opt\n \n+        compaction_options = self.config.get("global_compaction", {})\n         total_merge_time = 0\n         start = time.time()\n         merged_map = merge_submaps(\n-            self.agents_submaps, agents_kf_ids, agents_opt_kf_c2ws, opt_args)\n-        total_merge_time += time.time() - start\n+            self.agents_submaps, agents_kf_ids, agents_opt_kf_c2ws, opt_args,\n+            compaction_options=compaction_options)\n+        merge_only_time = time.time() - start\n+        total_merge_time += merge_only_time\n \n         merged_map.save_ply(str(self.output_path / "merged_coarse.ply"))\n \n@@ -244,9 +247,13 @@ class MAGiCSLAM(object):\n                           directory=self.output_path)\n \n         start = time.time()\n+        refine_iterations = int(compaction_options.get("refine_iterations", 3000))\n         refined_map = refine_map(\n-            merged_map, agents_datasets, agents_kf_ids, agents_opt_kf_c2ws, iterations=3000)\n-        total_merge_time += time.time() - start\n+            merged_map, agents_datasets, agents_kf_ids, agents_opt_kf_c2ws,\n+            iterations=refine_iterations,\n+            compaction_options=compaction_options)\n+        refine_only_time = time.time() - start\n+        total_merge_time += refine_only_time\n \n         print(f"Total merge time: {total_merge_time}")\n \n@@ -256,11 +263,20 @@ class MAGiCSLAM(object):\n \n         save_dict_to_json({\n             "registration_time": registration_time,\n+            "merge_only_time_s": merge_only_time,\n+            "refine_only_time_s": refine_only_time,\n             "total_merge_time": total_merge_time,\n             },\n             "walltime_stats.json",\n             directory=self.output_path)\n \n+        save_dict_to_json({\n+            "merge": getattr(merged_map, "compaction_stats", {}),\n+            "refinement": getattr(refined_map, "refinement_stats", {}),\n+            },\n+            "gaussian_compaction_stats.json",\n+            directory=self.output_path)\n+\n         refined_map.save_ply(str(self.output_path / "merged_refined.ply"))\n         print("Eval fine merged map")\n         fine_psnr, fine_lpips, fine_ssim, fine_dl1 = eval_agents_rendering(\ndiff --git a/src/entities/magic_slam_checkpoint.py b/src/entities/magic_slam_checkpoint.py\nindex b51a528..f4718e6 100644\n--- a/src/entities/magic_slam_checkpoint.py\n+++ b/src/entities/magic_slam_checkpoint.py\n@@ -215,11 +215,14 @@ class MAGiCSLAM(object):\n             agents_kf_ids[agent_id] = agents_kf_ids[agent_id].astype(int)\n         opt_args = self.agents[0].mapper.opt\n \n+        compaction_options = self.config.get("global_compaction", {})\n         total_merge_time = 0\n         start = time.time()\n         merged_map = merge_submaps(\n-            self.agents_submaps, agents_kf_ids, agents_opt_kf_c2ws, opt_args)\n-        total_merge_time += time.time() - start\n+            self.agents_submaps, agents_kf_ids, agents_opt_kf_c2ws, opt_args,\n+            compaction_options=compaction_options)\n+        merge_only_time = time.time() - start\n+        total_merge_time += merge_only_time\n \n         merged_map.save_ply(str(self.output_path / "merged_coarse.ply"))\n \n@@ -236,9 +239,13 @@ class MAGiCSLAM(object):\n                           directory=self.output_path)\n \n         start = time.time()\n+        refine_iterations = int(compaction_options.get("refine_iterations", 3000))\n         refined_map = refine_map(\n-            merged_map, agents_datasets, agents_kf_ids, agents_opt_kf_c2ws, iterations=3000)\n-        total_merge_time += time.time() - start\n+            merged_map, agents_datasets, agents_kf_ids, agents_opt_kf_c2ws,\n+            iterations=refine_iterations,\n+            compaction_options=compaction_options)\n+        refine_only_time = time.time() - start\n+        total_merge_time += refine_only_time\n \n         print(f"Total merge time: {total_merge_time}")\n \n@@ -248,11 +255,20 @@ class MAGiCSLAM(object):\n \n         save_dict_to_json({\n             "registration_time": registration_time,\n+            "merge_only_time_s": merge_only_time,\n+            "refine_only_time_s": refine_only_time,\n             "total_merge_time": total_merge_time,\n             },\n             "walltime_stats.json",\n             directory=self.output_path)\n \n+        save_dict_to_json({\n+            "merge": getattr(merged_map, "compaction_stats", {}),\n+            "refinement": getattr(refined_map, "refinement_stats", {}),\n+            },\n+            "gaussian_compaction_stats.json",\n+            directory=self.output_path)\n+\n         refined_map.save_ply(str(self.output_path / "merged_refined.ply"))\n         print("Eval fine merged map")\n         fine_psnr, fine_lpips, fine_ssim, fine_dl1 = eval_agents_rendering(\ndiff --git a/src/utils/magic_slam_utils.py b/src/utils/magic_slam_utils.py\nindex a15a008..9857746 100644\n--- a/src/utils/magic_slam_utils.py\n+++ b/src/utils/magic_slam_utils.py\n@@ -89,7 +89,8 @@ class Registration(object):\n         self.coarse_transformation = None\n \n \n-def refine_map(gaussian_model, agents_datasets: dict, agents_keyframe_ids: dict, agents_c2ws: dict, iterations=3000):\n+def refine_map(gaussian_model, agents_datasets: dict, agents_keyframe_ids: dict,\n+               agents_c2ws: dict, iterations=3000, compaction_options=None):\n     """ Refine a Gaussian model using keyframes from agents\' datasets: Section 3.4\n     Args:\n         gaussian_model: The Gaussian model.\n@@ -100,7 +101,18 @@ def refine_map(gaussian_model, agents_datasets: dict, agents_keyframe_ids: dict,\n     Returns:\n         gaussian_model: The refined Gaussian model.\n     """\n-    print("Refining map")\n+    options = compaction_options or {}\n+    prune_interval = max(1, int(options.get("refine_prune_interval", 100)))\n+    prune_min_opacity = float(options.get("refine_prune_min_opacity", 0.005))\n+    prune_start = float(options.get("refine_prune_start_fraction", 0.2))\n+    prune_end = float(options.get("refine_prune_end_fraction", 0.8))\n+    before_count = int(gaussian_model.get_xyz().shape[0])\n+    pruned_total = 0\n+    refine_start = time.perf_counter()\n+    print(\n+        "Refining map | Gaussians:", before_count,\n+        "| iterations:", iterations,\n+        "| prune interval:", prune_interval)\n     for iteration in tqdm(range(iterations)):\n         agent_id = random.choice(list(agents_datasets.keys()))\n         sample_id = random.randint(0, len(agents_keyframe_ids[agent_id]) - 1)\n@@ -124,17 +136,134 @@ def refine_map(gaussian_model, agents_datasets: dict, agents_keyframe_ids: dict,\n             gaussian_model.max_radii2D[visibility_filter] = torch.max(\n                 gaussian_model.max_radii2D[visibility_filter], radii[visibility_filter])\n \n-            if iteration > iterations * 0.2 and iteration < iterations * 0.8:\n-                prune_mask = (gaussian_model.get_opacity() < 0.005).squeeze()\n-                gaussian_model.prune_points(prune_mask)\n+            in_prune_window = (\n+                iteration > iterations * prune_start\n+                and iteration < iterations * prune_end)\n+            if in_prune_window and iteration % prune_interval == 0:\n+                prune_mask = (\n+                    gaussian_model.get_opacity() < prune_min_opacity).squeeze()\n+                num_pruned = int(prune_mask.sum().item())\n+                if num_pruned:\n+                    gaussian_model.prune_points(prune_mask)\n+                    pruned_total += num_pruned\n \n             gaussian_model.optimizer.step()\n             gaussian_model.optimizer.zero_grad(set_to_none=True)\n             # gaussian_model.update_learning_rate(iteration)\n+    gaussian_model.refinement_stats = {\n+        "iterations": int(iterations),\n+        "prune_interval": int(prune_interval),\n+        "prune_min_opacity": float(prune_min_opacity),\n+        "gaussians_before_refine": int(before_count),\n+        "gaussians_pruned_during_refine": int(pruned_total),\n+        "gaussians_after_refine": int(gaussian_model.get_xyz().shape[0]),\n+        "refine_wall_s": float(time.perf_counter() - refine_start),\n+    }\n+    return gaussian_model\n+\n+\n+def compact_gaussian_model(gaussian_model, options=None):\n+    """Remove low-value and spatially duplicated Gaussians after submap merge.\n+\n+    The operation keeps the highest-opacity Gaussian in every voxel, so all\n+    retained SH, scale, rotation and opacity parameters remain valid model\n+    parameters rather than averaged approximations.\n+    """\n+    options = options or {}\n+    enabled = bool(options.get("enabled", False))\n+    before = int(gaussian_model.get_xyz().shape[0])\n+    stats = {\n+        "enabled": enabled,\n+        "gaussians_before_compaction": before,\n+        "min_opacity": float(options.get("min_opacity", 0.0)),\n+        "voxel_size_m": float(options.get("voxel_size_m", 0.0)),\n+        "max_gaussians": options.get("max_gaussians"),\n+    }\n+    if not enabled or before == 0:\n+        stats["gaussians_after_compaction"] = before\n+        stats["removed_gaussians"] = 0\n+        stats["removed_percent"] = 0.0\n+        stats["compaction_wall_s"] = 0.0\n+        gaussian_model.compaction_stats = stats\n+        return gaussian_model\n+\n+    start = time.perf_counter()\n+    xyz = gaussian_model.get_xyz().detach()\n+    opacity = gaussian_model.get_opacity().detach().reshape(-1)\n+    keep = torch.isfinite(xyz).all(dim=1) & torch.isfinite(opacity)\n+    keep &= opacity >= stats["min_opacity"]\n+    stats["gaussians_after_opacity_filter"] = int(keep.sum().item())\n+\n+    voxel_size = stats["voxel_size_m"]\n+    if voxel_size > 0 and bool(keep.any()):\n+        candidate_idx = torch.where(keep)[0]\n+        quantized = torch.floor(xyz[candidate_idx] / voxel_size).to(torch.int64)\n+        quantized -= quantized.amin(dim=0, keepdim=True)\n+        dims = quantized.amax(dim=0) + 1\n+        max_key = int((dims[0] * dims[1] * dims[2]).item())\n+        if max_key >= torch.iinfo(torch.int64).max:\n+            raise OverflowError("Voxel grid is too large for collision-free int64 keys.")\n+        keys = quantized[:, 0] + dims[0] * (\n+            quantized[:, 1] + dims[1] * quantized[:, 2])\n+        _, inverse = torch.unique(keys, return_inverse=True)\n+        cell_count = int(inverse.max().item()) + 1\n+        candidate_opacity = opacity[candidate_idx]\n+        best_opacity = torch.full(\n+            (cell_count,), -torch.inf,\n+            dtype=candidate_opacity.dtype, device=candidate_opacity.device)\n+        best_opacity.scatter_reduce_(\n+            0, inverse, candidate_opacity, reduce="amax", include_self=True)\n+        is_best = candidate_opacity >= best_opacity[inverse]\n+        sentinel = torch.full_like(candidate_idx, before)\n+        best_candidate = torch.where(is_best, candidate_idx, sentinel)\n+        winner = torch.full(\n+            (cell_count,), before, dtype=torch.long, device=xyz.device)\n+        winner.scatter_reduce_(\n+            0, inverse, best_candidate, reduce="amin", include_self=True)\n+        winner = winner[winner < before]\n+        keep.zero_()\n+        keep[winner] = True\n+        stats["occupied_voxels"] = int(winner.numel())\n+        del candidate_idx, quantized, keys, inverse\n+        del candidate_opacity, best_opacity, best_candidate, winner\n+\n+    max_gaussians = options.get("max_gaussians")\n+    if max_gaussians is not None:\n+        max_gaussians = int(max_gaussians)\n+        kept_count = int(keep.sum().item())\n+        if max_gaussians > 0 and kept_count > max_gaussians:\n+            kept_idx = torch.where(keep)[0]\n+            selected = torch.topk(\n+                opacity[kept_idx], k=max_gaussians,\n+                largest=True, sorted=False).indices\n+            selected_idx = kept_idx[selected]\n+            keep.zero_()\n+            keep[selected_idx] = True\n+            stats["global_opacity_cap_applied"] = True\n+        else:\n+            stats["global_opacity_cap_applied"] = False\n+\n+    prune_mask = ~keep\n+    removed = int(prune_mask.sum().item())\n+    if removed:\n+        gaussian_model.prune_points(prune_mask)\n+    after = int(gaussian_model.get_xyz().shape[0])\n+    stats.update({\n+        "gaussians_after_compaction": after,\n+        "removed_gaussians": removed,\n+        "removed_percent": 100.0 * removed / max(before, 1),\n+        "compaction_wall_s": float(time.perf_counter() - start),\n+    })\n+    gaussian_model.compaction_stats = stats\n+    print(\n+        "Global Gaussian compaction:", before, "->", after,\n+        f"({stats[\'removed_percent\']:.1f}% removed)")\n     return gaussian_model\n \n \n-def merge_submaps(agents_submaps: dict, agents_kf_ids: dict, agents_opt_kf_c2ws: dict, opt_args):\n+def merge_submaps(agents_submaps: dict, agents_kf_ids: dict,\n+                  agents_opt_kf_c2ws: dict, opt_args,\n+                  compaction_options=None):\n     """ Merge submaps from agents in a coarse manner: Section 3.4\n     Args:\n         agents_submaps: A dictionary of agent submaps.\n@@ -179,7 +308,7 @@ def merge_submaps(agents_submaps: dict, agents_kf_ids: dict, agents_opt_kf_c2ws:\n                 scaling,\n                 opt_rotations)\n \n-    return merged_map\n+    return compact_gaussian_model(merged_map, compaction_options)\n \n \n def register_submaps(agents_submaps: dict, registration: Registration):\n'
    PATCH_PATH = COMPACT_TOOLS_DIR / 'global_compaction.patch'
    PATCH_PATH.write_text(PATCH_TEXT)

    # The dedicated checkout may contain the source patch from an earlier pass,
    # while office_0.yaml may have been rewritten later by the experiment cell.
    # Restore only the four patch targets from the pinned commit, then reapply.
    # Datasets, checkpoints and SLAM outputs are not touched.
    PATCH_TARGETS = [
        'configs/ReplicaMultiagent/office_0.yaml',
        'src/entities/magic_slam.py',
        'src/entities/magic_slam_checkpoint.py',
        'src/utils/magic_slam_utils.py',
    ]
    for relative in PATCH_TARGETS:
        if not (COMPACT_REPO / relative).is_file():
            raise FileNotFoundError(COMPACT_REPO / relative)

    subprocess.run([
        'git', '-C', str(COMPACT_REPO), 'restore',
        '--source', COMPACT_EXPECTED_REPO_COMMIT, '--worktree', '--', *PATCH_TARGETS
    ], check=True)

    forward = subprocess.run(
        ['git', '-C', str(COMPACT_REPO), 'apply', '--check', str(PATCH_PATH)],
        capture_output=True, text=True)
    if forward.returncode:
        raise RuntimeError(
            'Patch does not match the clean pinned files:\n' + forward.stderr[-5000:])
    subprocess.run(
        ['git', '-C', str(COMPACT_REPO), 'apply', str(PATCH_PATH)], check=True)
    print('Restored pinned targets and applied the compaction patch.')

    subprocess.run([sys.executable, '-m', 'py_compile',
        str(COMPACT_REPO/'src/utils/magic_slam_utils.py'),
        str(COMPACT_REPO/'src/entities/magic_slam.py'),
        str(COMPACT_REPO/'src/entities/magic_slam_checkpoint.py')], check=True)
    subprocess.run(['git','-C',str(COMPACT_REPO),'diff','--check'], check=True)
    print('Patch syntax and whitespace checks: OK')
    print(subprocess.check_output(['git','-C',str(COMPACT_REPO),'diff','--stat'], text=True))
else:
    print('Optional Gaussian compaction is disabled.')


### Chạy backend-only compaction

In [ ]:
if RUN_OPTIONAL_GS_COMPACTION:
    import yaml

    runner = COMPACT_TOOLS_DIR / "run_global_compaction_backend.py"
    runner.write_text(r"""
import argparse, json, random, shutil, sys, time
from argparse import ArgumentParser
from pathlib import Path
import numpy as np
import torch
import yaml


def read_ply_count(path):
    count = None
    with Path(path).open("rb") as handle:
        if handle.readline().strip() != b"ply":
            raise ValueError(path)
        for _ in range(1000):
            line = handle.readline().decode("ascii").strip()
            if line.startswith("element vertex "):
                count = int(line.split()[-1])
            if line == "end_header":
                if count is None:
                    raise ValueError("PLY has no vertex count: " + str(path))
                return count
    raise ValueError("Invalid PLY header: " + str(path))


parser = argparse.ArgumentParser()
parser.add_argument("--repo", required=True)
parser.add_argument("--source", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--config", required=True)
args = parser.parse_args()

repo = Path(args.repo)
source = Path(args.source)
output = Path(args.output)
config_path = Path(args.config)
sys.path.insert(0, str(repo))

from src.entities.arguments import OptimizationParams
from src.entities.datasets import get_dataset
from src.utils.io_utils import load_config, save_dict_to_json
from src.utils.magic_slam_utils import merge_submaps, refine_map
from src.utils.mapping_eval import eval_agents_rendering

if output.exists() and any(output.iterdir()):
    marker = output / "RUN_COMPLETE.json"
    if marker.is_file() and json.loads(marker.read_text()).get("complete"):
        print("Reuse completed compaction:", output)
        raise SystemExit(0)
    raise RuntimeError("Nonempty incomplete output; select a new output: " + str(output))

output.mkdir(parents=True, exist_ok=True)
cfg = load_config(str(config_path))
cfg.pop("inherit_from", None)
cfg["data"]["output_path"] = str(output)
(output / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

seed = int(cfg.get("seed", 0))
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.set_device(0)

agents_submaps = {}
agents_kf_ids = {}
agents_opt = {}
datasets = {}
input_root = Path(cfg["data"]["input_path"])
agent_ids = [int(x) for x in cfg["data"]["agent_ids"]]

for aid in agent_ids:
    files = sorted((source / f"agent_{aid}" / "submaps").glob("*.ckpt"))
    if not files:
        raise FileNotFoundError(source / f"agent_{aid}" / "submaps")
    agents_submaps[aid] = [
        torch.load(path, map_location="cpu", weights_only=False) for path in files
    ]
    agents_kf_ids[aid] = np.concatenate([
        np.asarray(submap["keyframe_ids"], dtype=int)
        for submap in agents_submaps[aid]
    ])
    pose_path = source / f"agent_{aid}" / "optimized_kf_c2w.ckpt"
    optimized = torch.load(pose_path, map_location="cpu", weights_only=False)
    agents_opt[aid] = (
        optimized.detach().cpu().numpy()
        if torch.is_tensor(optimized) else np.asarray(optimized)
    )
    if len(agents_opt[aid]) != len(agents_kf_ids[aid]):
        raise RuntimeError(
            f"Agent {aid}: optimized poses {len(agents_opt[aid])} != "
            f"keyframes {len(agents_kf_ids[aid])}"
        )
    dataset_cfg = {
        **cfg["data"],
        "input_path": str(input_root / f"agent_{aid}"),
        **cfg["cam"],
        **cfg["submap"],
    }
    datasets[aid] = get_dataset(cfg["dataset_name"])(dataset_cfg)

for name in [
    "detected_loops.pkl", "loops.pkl", "filtered_loops.pkl",
    "registration_metrics.csv", "registration_summary.json",
    "loop_pipeline_summary.json",
]:
    src = source / name
    if src.is_file():
        shutil.copy2(src, output / name)

for aid in agent_ids:
    destination = output / f"agent_{aid}"
    destination.mkdir(parents=True, exist_ok=True)
    for name in ["kf_ids.ckpt", "estimated_kf_c2w.ckpt", "optimized_kf_c2w.ckpt"]:
        src = source / f"agent_{aid}" / name
        if src.is_file():
            shutil.copy2(src, destination / name)

options = cfg.get("global_compaction", {})
optimization = OptimizationParams(ArgumentParser(description="global compaction"))
backend_start = time.perf_counter()

merge_start = time.perf_counter()
merged = merge_submaps(
    agents_submaps, agents_kf_ids, agents_opt, optimization,
    compaction_options=options,
)
merge_s = time.perf_counter() - merge_start
merged.save_ply(str(output / "merged_coarse.ply"))
coarse = eval_agents_rendering(merged, datasets, agents_kf_ids, agents_opt)
save_dict_to_json(
    dict(zip(["psnr", "lpips", "ssim", "depth_l1"], coarse)),
    "coarse_render_metrics.json", directory=output,
)

refine_start = time.perf_counter()
refined = refine_map(
    merged, datasets, agents_kf_ids, agents_opt,
    iterations=int(options.get("refine_iterations", 3000)),
    compaction_options=options,
)
refine_s = time.perf_counter() - refine_start
refined.save_ply(str(output / "merged_refined.ply"))
fine = eval_agents_rendering(refined, datasets, agents_kf_ids, agents_opt)
save_dict_to_json(
    dict(zip(["psnr", "lpips", "ssim", "depth_l1"], fine)),
    "fine_render_metrics.json", directory=output,
)

baseline_wall = json.loads((source / "walltime_stats.json").read_text())
wall = {
    "registration_time_reused_from_baseline_s": baseline_wall.get("registration_time"),
    "merge_only_time_s": merge_s,
    "refine_only_time_s": refine_s,
    "total_merge_time": merge_s + refine_s,
    "backend_only_wall_s": time.perf_counter() - backend_start,
}
save_dict_to_json(wall, "walltime_stats.json", directory=output)
save_dict_to_json(
    {
        "merge": getattr(merged, "compaction_stats", {}),
        "refinement": getattr(refined, "refinement_stats", {}),
    },
    "gaussian_compaction_stats.json", directory=output,
)

complete = {
    "complete": True,
    "scene": str(cfg["data"].get("scene_name", "scene")),
    "seed": seed,
    "backend_only_frozen_frontend": True,
    "baseline_output": str(source),
    "gaussian_counts": {
        "coarse": read_ply_count(output / "merged_coarse.ply"),
        "refined": read_ply_count(output / "merged_refined.ply"),
    },
    "backend_wall_s": time.perf_counter() - backend_start,
}
save_dict_to_json(complete, "RUN_COMPLETE.json", directory=output)
print(json.dumps(complete, indent=2))
""", encoding="utf-8")

    subprocess.run([
        str(PY311), str(runner),
        "--repo", str(COMPACT_REPO),
        "--source", str(COMPACTION_SOURCE_OUTPUT),
        "--output", str(COMPACTION_OUTPUT),
        "--config", str(COMPACTION_CONFIG),
    ], cwd=COMPACT_REPO, check=True)
    print("Compacted output:", COMPACTION_OUTPUT)
else:
    print("Skipped backend compaction.")


### So sánh định lượng trước và sau

In [ ]:
if RUN_OPTIONAL_GS_COMPACTION:
    import pandas as pd
    from IPython.display import display

    def _ply_count(path):
        count = None
        with Path(path).open("rb") as handle:
            if handle.readline().strip() != b"ply":
                raise ValueError(path)
            for _ in range(1000):
                line = handle.readline().decode("ascii").strip()
                if line.startswith("element vertex "):
                    count = int(line.split()[-1])
                if line == "end_header":
                    return count
        raise ValueError(path)

    def _summary(label, root):
        root = Path(root)
        fine = json.loads((root / "fine_render_metrics.json").read_text())
        wall = json.loads((root / "walltime_stats.json").read_text())
        run_path = root / "RUN_COMPLETE.json"
        run = json.loads(run_path.read_text()) if run_path.is_file() else {}
        count = run.get("gaussian_counts", {}).get("refined")
        if count is None:
            count = _ply_count(root / "merged_refined.ply")
        return {
            "Version": label,
            "Gaussians": int(count),
            "Map (MiB)": (root / "merged_refined.ply").stat().st_size / 2**20,
            "Merge+refine (s)": float(wall["total_merge_time"]),
            "PSNR (dB)": float(fine["psnr"]),
            "SSIM": float(fine["ssim"]),
            "LPIPS": float(fine["lpips"]),
            "Depth L1 (cm)": 100.0 * float(fine["depth_l1"]),
        }

    baseline_row = _summary("Original", COMPACTION_SOURCE_OUTPUT)
    compact_row = _summary("Compaction 2M", COMPACTION_OUTPUT)
    comparison = pd.DataFrame([baseline_row, compact_row])
    display(comparison.style.hide(axis="index"))

    changes = {
        "gaussian_reduction_percent": 100.0 * (
            1.0 - compact_row["Gaussians"] / baseline_row["Gaussians"]
        ),
        "map_reduction_percent": 100.0 * (
            1.0 - compact_row["Map (MiB)"] / baseline_row["Map (MiB)"]
        ),
        "merge_refine_reduction_percent": 100.0 * (
            1.0 - compact_row["Merge+refine (s)"] / baseline_row["Merge+refine (s)"]
        ),
        "psnr_change_db": compact_row["PSNR (dB)"] - baseline_row["PSNR (dB)"],
        "ssim_change": compact_row["SSIM"] - baseline_row["SSIM"],
        "lpips_change": compact_row["LPIPS"] - baseline_row["LPIPS"],
        "depth_l1_change_cm": compact_row["Depth L1 (cm)"] - baseline_row["Depth L1 (cm)"],
    }
    assessment = {
        "baseline": baseline_row,
        "compacted": compact_row,
        "changes": changes,
        "acceptance": {
            "psnr_drop_le_0p3_db": bool(changes["psnr_change_db"] >= -0.3),
            "ssim_drop_le_0p002": bool(changes["ssim_change"] >= -0.002),
            "lpips_increase_le_0p005": bool(changes["lpips_change"] <= 0.005),
            "depth_increase_le_0p1_cm": bool(changes["depth_l1_change_cm"] <= 0.1),
        },
    }
    assessment["accept"] = bool(all(assessment["acceptance"].values()))
    (COMPACTION_OUTPUT / "compaction_comparison.json").write_text(
        json.dumps(assessment, indent=2)
    )
    comparison.to_csv(COMPACTION_OUTPUT / "compaction_comparison.csv", index=False)
    print("Decision:", "ACCEPT" if assessment["accept"] else "KEEP ORIGINAL")
    print(json.dumps(changes, indent=2))
else:
    print("Skipped compaction comparison.")


### Xuất frame GT | Original | Compacted

In [ ]:
if RUN_OPTIONAL_GS_COMPACTION and COMPACTION_EXPORT_FRAMES:
    render_script = COMPACT_TOOLS_DIR / "export_compaction_frames.py"
    render_script.write_text(r"""
import gc, json, sys
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw

repo, baseline, compacted, output = map(Path, sys.argv[1:5])
sys.path.insert(0, str(repo))
from src.entities.datasets import get_dataset
from src.entities.gaussian_model import GaussianModel
from src.utils.io_utils import load_config
from src.utils.utils import get_render_settings, render_gaussian_model

output.mkdir(parents=True, exist_ok=True)
for name in ["gt", "original", "compacted", "comparison"]:
    (output / name).mkdir(exist_ok=True)

cfg = load_config(str(compacted / "config.yaml"))
root = Path(cfg["data"]["input_path"])
datasets = {}

def dataset(aid):
    if aid not in datasets:
        dc = {
            **cfg["data"], "input_path": str(root / f"agent_{aid}"),
            **cfg["cam"], **cfg["submap"],
        }
        datasets[aid] = get_dataset(cfg["dataset_name"])(dc)
    return datasets[aid]

def u8(value):
    array = value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)
    if array.ndim == 3 and array.shape[0] in [1, 3, 4]:
        array = np.moveaxis(array, 0, -1)
    if np.issubdtype(array.dtype, np.floating) and np.nanmax(array) <= 1.5:
        array = array * 255.0
    return np.clip(np.nan_to_num(array), 0, 255).astype(np.uint8)[..., :3]

records = []
for aid in map(int, cfg["data"]["agent_ids"]):
    ds = dataset(aid)
    for frame in sorted(set([len(ds) // 3, 2 * len(ds) // 3])):
        name = f"agent{aid}_frame{frame:04d}"
        _, rgb, _, pose = ds[frame]
        pose = pose.detach().cpu().numpy() if torch.is_tensor(pose) else np.asarray(pose)
        Image.fromarray(u8(rgb)).save(output / "gt" / f"{name}_gt.png")
        records.append((aid, frame, name, np.linalg.inv(pose)))

def render(label, ply, folder):
    model = GaussianModel(0)
    model.load_ply(str(ply))
    with torch.no_grad():
        for aid, frame, name, world_to_camera in records:
            ds = dataset(aid)
            settings = get_render_settings(ds.width, ds.height, ds.intrinsics, world_to_camera)
            image = u8(render_gaussian_model(model, settings)["color"])
            Image.fromarray(image).save(output / folder / f"{name}_{folder}.png")
    del model
    gc.collect()
    torch.cuda.empty_cache()

render("Original", baseline / "merged_refined.ply", "original")
render("Compacted", compacted / "merged_refined.ply", "compacted")

panels = []
for _, _, name, _ in records:
    images = [
        Image.open(output / "gt" / f"{name}_gt.png").convert("RGB"),
        Image.open(output / "original" / f"{name}_original.png").convert("RGB"),
        Image.open(output / "compacted" / f"{name}_compacted.png").convert("RGB"),
    ]
    header = 40
    panel = Image.new("RGB", (sum(x.width for x in images), max(x.height for x in images) + header), "white")
    draw = ImageDraw.Draw(panel)
    x = 0
    for title, image in zip(["GT", "Original", "Compacted"], images):
        panel.paste(image, (x, header))
        draw.text((x + 10, 12), title, fill="black")
        x += image.width
    panel_path = output / "comparison" / f"{name}_comparison.png"
    panel.save(panel_path)
    panels.append(panel_path)

print(json.dumps({"frames": len(records), "output": str(output)}, indent=2))
""", encoding="utf-8")

    COMPACTION_FRAME_DIR = COMPACTION_OUTPUT / "paper_frames"
    subprocess.run([
        str(PY311), str(render_script), str(COMPACT_REPO),
        str(COMPACTION_SOURCE_OUTPUT), str(COMPACTION_OUTPUT),
        str(COMPACTION_FRAME_DIR),
    ], cwd=COMPACT_REPO, check=True)

    from IPython.display import Image as DisplayImage, display
    panels = sorted((COMPACTION_FRAME_DIR / "comparison").glob("*.png"))
    if panels:
        display(DisplayImage(filename=str(panels[0]), width=1500))
    print("All paper frames:", COMPACTION_FRAME_DIR)
elif RUN_OPTIONAL_GS_COMPACTION:
    print("Frame export disabled.")
else:
    print("Skipped compaction frame export.")


### Đóng gói metric, frame và map cuối

In [ ]:
if RUN_OPTIONAL_GS_COMPACTION:
    import zipfile
    from IPython.display import FileLink, display

    metrics_zip = Path(
        f"/kaggle/working/{COMPACTION_METHOD_LABEL}_{COMPACTION_SCENE}_compaction_2m_metrics_frames.zip"
    )
    with zipfile.ZipFile(metrics_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
        for path in COMPACTION_OUTPUT.rglob("*"):
            if path.is_file() and path.suffix not in {".ply", ".ckpt", ".pt"}:
                archive.write(path, path.relative_to(COMPACTION_OUTPUT))
        for path in COMPACT_TOOLS_DIR.iterdir():
            if path.is_file():
                archive.write(path, Path("scripts_and_setup") / path.name)
    display(FileLink(str(metrics_zip)))

    if COMPACTION_ZIP_FINAL_MAP:
        map_zip = Path(
            f"/kaggle/working/{COMPACTION_METHOD_LABEL}_{COMPACTION_SCENE}_compaction_2m_map.zip"
        )
        with zipfile.ZipFile(map_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
            for name in [
                "merged_refined.ply", "config.yaml", "RUN_COMPLETE.json",
                "fine_render_metrics.json", "walltime_stats.json",
                "gaussian_compaction_stats.json", "compaction_comparison.json",
            ]:
                path = COMPACTION_OUTPUT / name
                if path.is_file():
                    archive.write(path, path.name)
        display(FileLink(str(map_zip)))

    print("Compacted map:", COMPACTION_OUTPUT / "merged_refined.ply")
else:
    print("Skipped compaction packaging.")


## Xuất mesh tam giác và ZIP kết quả cuối

Cell này chạy sau full SLAM, kể cả khi không bật compaction. Mesh màu được dựng từ các tâm Gaussian bằng Poisson reconstruction; file Gaussian gốc vẫn được giữ nguyên. Nếu auto-detection thấy nhiều output, đặt `MESH_SOURCE_OUTPUT` thành đúng thư mục cần xuất.

In [ ]:
if EXPORT_FINAL_MESH:
    import time

    if not PY311.is_file():
        raise FileNotFoundError(PY311)

    if MESH_SOURCE_OUTPUT is not None:
        FINAL_MESH_SOURCE = Path(MESH_SOURCE_OUTPUT).resolve()
    elif (
        RUN_OPTIONAL_GS_COMPACTION
        and "COMPACTION_OUTPUT" in globals()
        and (Path(COMPACTION_OUTPUT) / "merged_refined.ply").is_file()
    ):
        FINAL_MESH_SOURCE = Path(COMPACTION_OUTPUT).resolve()
    else:
        detected = [
            p for p in _source_candidates()
            if (p / "merged_refined.ply").is_file()
        ]
        if len(detected) != 1:
            raise RuntimeError(
                "Set MESH_SOURCE_OUTPUT to one exact completed output containing "
                "merged_refined.ply. Detected candidates: "
                + str([str(x) for x in detected])
            )
        FINAL_MESH_SOURCE = detected[0]

    FINAL_GAUSSIAN_PLY = FINAL_MESH_SOURCE / "merged_refined.ply"
    FINAL_MESH_PLY = FINAL_MESH_SOURCE / "merged_refined_mesh.ply"
    FINAL_MESH_PREVIEW_PLY = FINAL_MESH_SOURCE / "merged_refined_mesh_preview.ply"
    FINAL_MESH_STATS = FINAL_MESH_SOURCE / "mesh_stats.json"
    if not FINAL_GAUSSIAN_PLY.is_file():
        raise FileNotFoundError(FINAL_GAUSSIAN_PLY)

    mesh_script = COMPACT_TOOLS_DIR / "export_final_gaussian_mesh.py"
    mesh_script.write_text(r"""
import argparse, json, time
from pathlib import Path
import numpy as np
import open3d as o3d
from plyfile import PlyData


def gaussian_points(path, min_opacity=0.05):
    vertex = PlyData.read(str(path))["vertex"].data
    names = set(vertex.dtype.names or [])
    required = {"x", "y", "z"}
    if not required.issubset(names):
        raise ValueError("PLY has no xyz Gaussian fields: " + str(path))
    xyz = np.column_stack([vertex["x"], vertex["y"], vertex["z"]]).astype(np.float64)
    valid = np.isfinite(xyz).all(axis=1)
    if "opacity" in names:
        raw = np.asarray(vertex["opacity"], dtype=np.float64)
        opacity = 1.0 / (1.0 + np.exp(-np.clip(raw, -60.0, 60.0)))
        valid &= np.isfinite(opacity) & (opacity >= float(min_opacity))
    xyz = xyz[valid]
    if {"f_dc_0", "f_dc_1", "f_dc_2"}.issubset(names):
        dc = np.column_stack([
            vertex["f_dc_0"], vertex["f_dc_1"], vertex["f_dc_2"]
        ]).astype(np.float64)[valid]
        rgb = np.clip(0.5 + 0.28209479177387814 * dc, 0.0, 1.0)
    elif {"red", "green", "blue"}.issubset(names):
        rgb = np.column_stack([
            vertex["red"], vertex["green"], vertex["blue"]
        ]).astype(np.float64)[valid] / 255.0
    else:
        rgb = np.full((len(xyz), 3), 0.7, dtype=np.float64)
    return xyz, rgb, int(len(vertex))


parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
parser.add_argument("--mesh", required=True)
parser.add_argument("--preview", required=True)
parser.add_argument("--stats", required=True)
parser.add_argument("--voxel", type=float, default=0.01)
parser.add_argument("--max-points", type=int, default=250000)
parser.add_argument("--depth", type=int, default=9)
parser.add_argument("--density-quantile", type=float, default=0.01)
parser.add_argument("--preview-triangles", type=int, default=150000)
args = parser.parse_args()

start = time.perf_counter()
xyz, rgb, gaussian_count = gaussian_points(args.input)
if len(xyz) < 1000:
    raise RuntimeError(f"Only {len(xyz)} valid Gaussian centers; cannot build a mesh")

cloud = o3d.geometry.PointCloud()
cloud.points = o3d.utility.Vector3dVector(xyz)
cloud.colors = o3d.utility.Vector3dVector(rgb)
cloud = cloud.voxel_down_sample(float(args.voxel))
if len(cloud.points) > int(args.max_points):
    ids = np.linspace(0, len(cloud.points) - 1, int(args.max_points)).round().astype(np.int64)
    points = np.asarray(cloud.points)[ids]
    colors = np.asarray(cloud.colors)[ids]
    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(points)
    cloud.colors = o3d.utility.Vector3dVector(colors)

radius = max(4.0 * float(args.voxel), 0.03)
cloud.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=40)
)
cloud.normalize_normals()
try:
    cloud.orient_normals_consistent_tangent_plane(30)
except RuntimeError as error:
    print("Normal orientation fallback:", repr(error))
    cloud.orient_normals_towards_camera_location(np.asarray(cloud.get_center()))

mesh, density = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    cloud, depth=int(args.depth), scale=1.08, linear_fit=False
)
density = np.asarray(density)
if len(density) and float(args.density_quantile) > 0:
    threshold = float(np.quantile(density, float(args.density_quantile)))
    mesh.remove_vertices_by_mask(density < threshold)

bbox = cloud.get_axis_aligned_bounding_box()
padding = max(2.0 * float(args.voxel), 0.02)
bbox = o3d.geometry.AxisAlignedBoundingBox(
    bbox.get_min_bound() - padding, bbox.get_max_bound() + padding
)
mesh = mesh.crop(bbox)
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.remove_degenerate_triangles()
mesh.remove_non_manifold_edges()
mesh.remove_unreferenced_vertices()
mesh.compute_vertex_normals()
if len(mesh.vertices) == 0 or len(mesh.triangles) == 0:
    raise RuntimeError("Poisson reconstruction returned an empty triangle mesh")

mesh_path = Path(args.mesh)
mesh_path.parent.mkdir(parents=True, exist_ok=True)
if not o3d.io.write_triangle_mesh(str(mesh_path), mesh, write_ascii=False):
    raise RuntimeError("Cannot write mesh: " + str(mesh_path))

target = min(int(args.preview_triangles), len(mesh.triangles))
preview = mesh.simplify_quadric_decimation(target) if target < len(mesh.triangles) else mesh
preview.remove_unreferenced_vertices()
preview.compute_vertex_normals()
if not o3d.io.write_triangle_mesh(str(args.preview), preview, write_ascii=False):
    raise RuntimeError("Cannot write preview mesh: " + str(args.preview))

stats = {
    "source_gaussians": gaussian_count,
    "eligible_gaussians": int(len(xyz)),
    "reconstruction_points": int(len(cloud.points)),
    "mesh_vertices": int(len(mesh.vertices)),
    "mesh_triangles": int(len(mesh.triangles)),
    "preview_vertices": int(len(preview.vertices)),
    "preview_triangles": int(len(preview.triangles)),
    "mesh_mib": mesh_path.stat().st_size / 2**20,
    "preview_mib": Path(args.preview).stat().st_size / 2**20,
    "voxel_size_m": float(args.voxel),
    "poisson_depth": int(args.depth),
    "density_quantile": float(args.density_quantile),
    "runtime_s": time.perf_counter() - start,
    "method": "colored Gaussian centers -> voxel sampling -> normal estimation -> Poisson",
}
Path(args.stats).write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))
""", encoding="utf-8")

    subprocess.run([
        str(PY311), str(mesh_script),
        "--input", str(FINAL_GAUSSIAN_PLY),
        "--mesh", str(FINAL_MESH_PLY),
        "--preview", str(FINAL_MESH_PREVIEW_PLY),
        "--stats", str(FINAL_MESH_STATS),
        "--voxel", str(MESH_VOXEL_SIZE_M),
        "--max-points", str(MESH_MAX_INPUT_POINTS),
        "--depth", str(MESH_POISSON_DEPTH),
        "--density-quantile", str(MESH_DENSITY_QUANTILE),
        "--preview-triangles", str(MESH_PREVIEW_TRIANGLES),
    ], check=True)

    print("Triangle mesh:", FINAL_MESH_PLY)
    print("Preview mesh:", FINAL_MESH_PREVIEW_PLY)
else:
    print("Final mesh export is disabled.")


### Đóng gói map Gaussian, mesh, metric và frame

In [ ]:
if EXPORT_FINAL_MESH:
    import zipfile
    from IPython.display import FileLink, display

    source_cfg = {}
    config_path = FINAL_MESH_SOURCE / "config.yaml"
    if config_path.is_file():
        import yaml
        source_cfg = yaml.safe_load(config_path.read_text()) or {}
    scene = str(source_cfg.get("data", {}).get("scene_name", FINAL_MESH_SOURCE.parent.name))
    FINAL_RESULT_ZIP = Path(
        f"/kaggle/working/{COMPACTION_METHOD_LABEL}_{scene}_final_with_mesh.zip"
    )

    allowed_ply = {
        "merged_refined_mesh.ply",
        "merged_refined_mesh_preview.ply",
    }
    if MESH_ZIP_INCLUDE_GAUSSIAN_MAP:
        allowed_ply.add("merged_refined.ply")

    with zipfile.ZipFile(
        FINAL_RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
    ) as archive:
        for path in FINAL_MESH_SOURCE.rglob("*"):
            if not path.is_file() or any("_partial_" in part for part in path.parts):
                continue
            if path.suffix in {".ckpt", ".pt", ".zip"}:
                continue
            if path.suffix == ".ply" and path.name not in allowed_ply:
                continue
            archive.write(path, Path("final_result") / path.relative_to(FINAL_MESH_SOURCE))
        for path in COMPACT_TOOLS_DIR.iterdir():
            if path.is_file():
                archive.write(path, Path("scripts_and_setup") / path.name)

    print("Final ZIP includes triangle mesh:", FINAL_RESULT_ZIP)
    print("Gaussian PLY included:", bool(MESH_ZIP_INCLUDE_GAUSSIAN_MAP))
    display(FileLink(str(FINAL_RESULT_ZIP)))
else:
    print("Skipped final mesh packaging.")


## Export colored TSDF mesh (after full SLAM)

This optional final cell fuses the RGB-D frames with the **optimized SLAM poses**. It exports a colored watertight-style TSDF surface for visualization; it does not replace the Gaussian map used for rendering or evaluation. Run it after the full-SLAM cell has completed. Increase `TSDF_FRAME_STRIDE` or `TSDF_VOXEL_M` if memory is limited.


In [ ]:
# tsdf-export-v1 — run after the full-SLAM output is complete
import json, os, subprocess, sys, textwrap, zipfile
from pathlib import Path
from IPython.display import FileLink, display

# User-adjustable settings. 1 cm is appropriate for paper figures; 1.5–2 cm is faster.
RUN_TSDF_EXPORT = globals().get("RUN_TSDF_EXPORT", True)
TSDF_VOXEL_M = float(globals().get("TSDF_VOXEL_M", 0.01))
TSDF_TRUNC_M = float(globals().get("TSDF_TRUNC_M", 0.04))
TSDF_DEPTH_TRUNC_M = float(globals().get("TSDF_DEPTH_TRUNC_M", 5.0))
TSDF_FRAME_STRIDE = int(globals().get("TSDF_FRAME_STRIDE", 2))
TSDF_MIN_COMPONENT_TRIANGLES = int(globals().get("TSDF_MIN_COMPONENT_TRIANGLES", 1000))
TSDF_FORCE_REBUILD = bool(globals().get("TSDF_FORCE_REBUILD", False))

TSDF_SCENE = "office_0"
TSDF_METHOD = "multi_view"
TSDF_REPO = Path(globals().get("TSDF_REPO", "/kaggle/working/3DGS-SLAM-TURTLEBOT4"))
TSDF_SOURCE_OUTPUT = Path(globals().get("TSDF_SOURCE_OUTPUT", "/kaggle/working/gslam_outputs/office0_full_slam/office_0/seed_0"))
TSDF_PYTHON = Path(globals().get("PY311", "/kaggle/working/gs_slam_py311/bin/python"))
TSDF_EXPORT_DIR = TSDF_SOURCE_OUTPUT / "tsdf_mesh"
TSDF_ZIP = Path("/kaggle/working") / f"{TSDF_METHOD}_{TSDF_SCENE}_tsdf_mesh.zip"

if TSDF_FRAME_STRIDE < 1 or min(TSDF_VOXEL_M, TSDF_TRUNC_M, TSDF_DEPTH_TRUNC_M) <= 0:
    raise ValueError("TSDF voxel, truncation, depth limit, and frame stride must be positive.")
if not RUN_TSDF_EXPORT:
    print("TSDF export disabled. Set RUN_TSDF_EXPORT=True and rerun this cell.")
else:
    required = [TSDF_SOURCE_OUTPUT / "config.yaml", TSDF_SOURCE_OUTPUT / "merged_refined.ply"]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(
            "Full-SLAM output is incomplete. Run the full-SLAM cells first. Missing: "
            + ", ".join(missing)
        )
    if not TSDF_PYTHON.is_file():
        raise FileNotFoundError(f"Python 3.11 environment not found: {TSDF_PYTHON}")

    worker = r"""
import argparse, csv, json, re, sys, time
from pathlib import Path

import numpy as np
import open3d as o3d
import torch


def array(value, dtype=None):
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def natural_key(path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", path.name)]


def rgb_u8(value):
    image = array(value)
    while image.ndim > 3 and 1 in image.shape:
        image = np.squeeze(image)
    if image.ndim != 3:
        raise ValueError(f"Unexpected RGB shape: {image.shape}")
    if image.shape[0] in (1, 3, 4) and image.shape[-1] not in (1, 3, 4):
        image = np.moveaxis(image, 0, -1)
    if image.shape[-1] == 1:
        image = np.repeat(image, 3, axis=-1)
    image = image[..., :3]
    if np.issubdtype(image.dtype, np.floating) and np.nanmax(image) <= 1.5:
        image = image * 255.0
    return np.ascontiguousarray(np.clip(np.nan_to_num(image), 0, 255).astype(np.uint8))


def depth_f32(value):
    depth = array(value, np.float32)
    while depth.ndim > 2 and 1 in depth.shape:
        depth = np.squeeze(depth)
    if depth.ndim == 3:
        depth = depth[0] if depth.shape[0] <= 4 else depth[..., 0]
    if depth.ndim != 2:
        raise ValueError(f"Unexpected depth shape: {depth.shape}")
    depth = np.nan_to_num(depth, nan=0.0, posinf=0.0, neginf=0.0)
    depth[depth <= 0] = 0
    return np.ascontiguousarray(depth)


def cfg_number(cam, names, default=None):
    for name in names:
        if name in cam and cam[name] is not None:
            return float(cam[name])
    if default is not None:
        return float(default)
    raise KeyError(f"Missing camera value; tried {names}")


def intrinsics(dataset, cam, width, height):
    def get(names, default=None):
        for name in names:
            if hasattr(dataset, name):
                value = getattr(dataset, name)
                if value is not None:
                    return float(value)
        return cfg_number(cam, names, default)

    fx, fy = get(["fx"]), get(["fy"])
    cx, cy = get(["cx"], width / 2), get(["cy"], height / 2)
    nominal_w = get(["W", "width"], width)
    nominal_h = get(["H", "height"], height)
    sx, sy = width / nominal_w, height / nominal_h
    return o3d.camera.PinholeCameraIntrinsic(
        width, height, fx * sx, fy * sy, cx * sx, cy * sy
    )


def clean_mesh(mesh, min_triangles):
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.remove_degenerate_triangles()
    mesh.remove_unreferenced_vertices()
    mesh.remove_non_manifold_edges()
    if len(mesh.triangles) and min_triangles > 0:
        labels, counts, _ = mesh.cluster_connected_triangles()
        labels, counts = np.asarray(labels), np.asarray(counts)
        remove = counts[labels] < min_triangles
        mesh.remove_triangles_by_mask(remove)
        mesh.remove_unreferenced_vertices()
    mesh.compute_vertex_normals()
    return mesh


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("params")
    args = parser.parse_args()
    p = json.loads(Path(args.params).read_text())
    repo, source, output = Path(p["repo"]), Path(p["source"]), Path(p["output"])
    output.mkdir(parents=True, exist_ok=True)
    mesh_path = output / f'{p["scene"]}_tsdf_mesh.ply'
    preview_path = output / f'{p["scene"]}_tsdf_mesh_preview.ply'
    stats_path = output / "tsdf_stats.json"
    records_path = output / "tsdf_integrated_frames.csv"

    signature = {
        key: p[key] for key in (
            "scene", "method", "voxel_m", "trunc_m", "depth_trunc_m",
            "frame_stride", "min_component_triangles"
        )
    }
    if not p["force"] and mesh_path.is_file() and stats_path.is_file():
        old = json.loads(stats_path.read_text())
        if old.get("signature") == signature:
            print("Reuse existing TSDF mesh:", mesh_path)
            return

    sys.path.insert(0, str(repo))
    from src.entities.datasets import get_dataset
    from src.utils.io_utils import load_config

    cfg = load_config(str(source / "config.yaml"))
    input_root = Path(cfg["data"]["input_path"])
    if not input_root.is_absolute():
        candidates = [repo / input_root, repo / "data" / input_root]
        input_root = next((x for x in candidates if x.exists()), candidates[0])
    agent_ids = [int(x) for x in cfg["data"]["agent_ids"]]
    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=float(p["voxel_m"]),
        sdf_trunc=float(p["trunc_m"]),
        color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8,
    )
    records, seen = [], set()
    start_time = time.monotonic()

    for aid in agent_ids:
        agent_dir = source / f"agent_{aid}"
        submap_files = sorted((agent_dir / "submaps").glob("*.ckpt"), key=natural_key)
        pose_file = agent_dir / "optimized_kf_c2w.ckpt"
        if not submap_files or not pose_file.is_file():
            raise FileNotFoundError(
                f"Agent {aid}: missing submaps or optimized_kf_c2w.ckpt in {agent_dir}"
            )
        submaps = [torch.load(x, map_location="cpu", weights_only=False) for x in submap_files]
        optimized = array(torch.load(pose_file, map_location="cpu", weights_only=False), np.float64)
        keyframe_count = sum(len(array(sub["keyframe_ids"]).reshape(-1)) for sub in submaps)
        if len(optimized) != keyframe_count:
            raise RuntimeError(
                f"Agent {aid}: {len(optimized)} optimized poses != {keyframe_count} saved keyframes"
            )

        dataset_cfg = {
            **cfg["data"], "input_path": str(input_root / f"agent_{aid}"),
            **cfg["cam"], **cfg["submap"],
        }
        dataset = get_dataset(cfg["dataset_name"])(dataset_cfg)
        keyframe_offset = 0
        for submap_index, submap in enumerate(submaps):
            local_poses = array(submap["submap_c2ws"], np.float64)
            keyframes = array(submap["keyframe_ids"]).reshape(-1)
            if local_poses.ndim != 3 or local_poses.shape[1:] != (4, 4):
                raise ValueError(f"Agent {aid}, submap {submap_index}: invalid local pose array")
            correction = optimized[keyframe_offset] @ np.linalg.inv(local_poses[0])
            keyframe_offset += len(keyframes)
            frame_start = int(submap["submap_start_frame_id"])
            offsets = list(range(0, len(local_poses), int(p["frame_stride"])))
            if offsets and offsets[-1] != len(local_poses) - 1:
                offsets.append(len(local_poses) - 1)
            for local_index in offsets:
                frame_id = frame_start + local_index
                identity = (aid, frame_id)
                if identity in seen or frame_id < 0 or frame_id >= len(dataset):
                    continue
                seen.add(identity)
                _, rgb, depth, _ = dataset[frame_id]
                color_np, depth_np = rgb_u8(rgb), depth_f32(depth)
                if depth_np.shape != color_np.shape[:2]:
                    raise ValueError(
                        f"Agent {aid}, frame {frame_id}: RGB {color_np.shape} and depth {depth_np.shape} differ"
                    )
                c2w = correction @ local_poses[local_index]
                if not np.isfinite(c2w).all() or not np.isclose(np.linalg.det(c2w[:3, :3]), 1, atol=0.05):
                    raise ValueError(f"Agent {aid}, frame {frame_id}: invalid optimized rigid pose")
                intrinsic = intrinsics(dataset, cfg["cam"], color_np.shape[1], color_np.shape[0])
                rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
                    o3d.geometry.Image(color_np), o3d.geometry.Image(depth_np),
                    depth_scale=1.0, depth_trunc=float(p["depth_trunc_m"]),
                    convert_rgb_to_intensity=False,
                )
                volume.integrate(rgbd, intrinsic, np.linalg.inv(c2w))
                valid_depth = int(np.count_nonzero((depth_np > 0) & (depth_np <= p["depth_trunc_m"])))
                records.append({
                    "agent": aid, "submap": submap_index, "frame": frame_id,
                    "valid_depth_pixels": valid_depth,
                })
                if len(records) % 100 == 0:
                    print(f"Integrated {len(records)} RGB-D frames")

    mesh = clean_mesh(volume.extract_triangle_mesh(), int(p["min_component_triangles"]))
    if not len(mesh.vertices) or not len(mesh.triangles):
        raise RuntimeError("TSDF produced an empty mesh; check depth units and camera poses.")
    if not o3d.io.write_triangle_mesh(str(mesh_path), mesh, write_ascii=False, compressed=True):
        raise RuntimeError(f"Could not write {mesh_path}")

    target_triangles = min(300000, len(mesh.triangles))
    preview = mesh
    if len(mesh.triangles) > target_triangles:
        preview = mesh.simplify_quadric_decimation(target_number_of_triangles=target_triangles)
        preview.compute_vertex_normals()
    if not o3d.io.write_triangle_mesh(str(preview_path), preview, write_ascii=False, compressed=True):
        raise RuntimeError(f"Could not write {preview_path}")

    with records_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["agent", "submap", "frame", "valid_depth_pixels"])
        writer.writeheader()
        writer.writerows(records)
    bbox = mesh.get_axis_aligned_bounding_box()
    stats = {
        "signature": signature,
        "source_output": str(source),
        "input_root": str(input_root),
        "agents": agent_ids,
        "integrated_frames": len(records),
        "vertices": len(mesh.vertices),
        "triangles": len(mesh.triangles),
        "preview_vertices": len(preview.vertices),
        "preview_triangles": len(preview.triangles),
        "bounds_min_m": np.asarray(bbox.min_bound).tolist(),
        "bounds_max_m": np.asarray(bbox.max_bound).tolist(),
        "runtime_s": time.monotonic() - start_time,
        "mesh_bytes": mesh_path.stat().st_size,
    }
    stats_path.write_text(json.dumps(stats, indent=2))
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()
"""

    TSDF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tsdf_params = {
        "scene": TSDF_SCENE,
        "method": TSDF_METHOD,
        "repo": str(TSDF_REPO),
        "source": str(TSDF_SOURCE_OUTPUT),
        "output": str(TSDF_EXPORT_DIR),
        "voxel_m": TSDF_VOXEL_M,
        "trunc_m": TSDF_TRUNC_M,
        "depth_trunc_m": TSDF_DEPTH_TRUNC_M,
        "frame_stride": TSDF_FRAME_STRIDE,
        "min_component_triangles": TSDF_MIN_COMPONENT_TRIANGLES,
        "force": TSDF_FORCE_REBUILD,
    }
    tsdf_script = TSDF_EXPORT_DIR / "export_tsdf_mesh.py"
    tsdf_params_path = TSDF_EXPORT_DIR / "tsdf_params.json"
    tsdf_script.write_text(worker)
    tsdf_params_path.write_text(json.dumps(tsdf_params, indent=2))
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", WANDB_MODE="disabled")
    subprocess.run(
        [str(TSDF_PYTHON), str(tsdf_script), str(tsdf_params_path)],
        cwd=TSDF_REPO, env=env, check=True,
    )

    with zipfile.ZipFile(TSDF_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
        for path in sorted(TSDF_EXPORT_DIR.iterdir()):
            if path.is_file():
                archive.write(path, Path("tsdf_mesh") / path.name)
    print("TSDF output:", TSDF_EXPORT_DIR)
    print("Download:", TSDF_ZIP)
    display(FileLink(str(TSDF_ZIP)))
    display(FileLink(str(TSDF_EXPORT_DIR / f"{TSDF_SCENE}_tsdf_mesh.ply")))
